# Qwen kidney-transplant generator v3.2

Minimal deterministic day-7 anchor correction plus a guarded 20-recipient confirmation. Existing evidence is immutable and full production remains disabled.

## Local design, event audit, and guarded confirmation

In [ ]:
import hashlib
import json
import math
import os
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from numbers import Real
from pathlib import Path

import numpy as np
import pandas as pd
from openai import OpenAI


ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
RUN_ID = "QWEN-V32-CONFIRM20-001"
GATE_NAME = "RUN_QWEN_V32_CONFIRM20"
DESIGN_DIR = ROOT / "data/raw/KidneyTransplant/qwen_v3_2_design"
RUN_DIR = ROOT / "data/raw/KidneyTransplant/qwen_v3_2_confirm20_001"
V31_DESIGN = ROOT / "data/raw/KidneyTransplant/qwen_v3_1_design"
DIAGNOSTIC100 = ROOT / "data/raw/KidneyTransplant/qwen_diagnostic100_v3_1_001"
CORRECTED10 = ROOT / "data/raw/KidneyTransplant/qwen_corrected10_v3_1_001"
PROTECTED = ROOT / "data/raw/kidney_transplant_unlearning_dataset.csv"
EXPECTED_PROTECTED = {
    "sha256": "8f4b6b51f96b753490cbef542643ab363408e472bf6dcf21b2c91cc3176648b7",
    "mtime_ns": 1786015387691857656,
    "size_bytes": 19817033,
}
BASE_URL = "https://resolution-andreas-alerts-blah.trycloudflare.com/v1"
API_KEY = "local-key"
DAYS = [7, 14, 30, 60, 90, 180]
EXPECTED_RECIPIENTS = 20
MAX_API_REQUESTS = 20
ANCHOR_ALGORITHM_VERSION = "qwen-kidney-v3.2-anchor-v1"
ANCHOR_TOLERANCES = {
    "creatinine_mg_dl": 0.12,
    "urine_output_ml_24h": 120.0,
    "tacrolimus_level_ng_ml": 0.6,
    "medication_adherence_pct": 1.5,
}

# Persistent switches remain false. Only the exact one-time environment gate can authorize 20 calls.
RUN_QWEN_V32_CONFIRM20 = False
RUN_10000_RECIPIENTS = False
RUN_FULL_GENERATION = False
assert not RUN_QWEN_V32_CONFIRM20
assert not RUN_10000_RECIPIENTS
assert not RUN_FULL_GENERATION

NOTEBOOKS_01_05 = [
    ROOT / f"notebooks/KidneyTransplant/{name}"
    for name in [
        "01_kidney_transplant_baseline.ipynb",
        "02_kidney_transplant_forget_sets_and_full_retraining.ipynb",
        "03_kidney_transplant_retain_set_finetuning.ipynb",
        "04_kidney_transplant_gradient_ascent.ipynb",
        "05_kidney_transplant_sisa.ipynb",
    ]
]
IMMUTABLE_DIRS = {
    "pilot": ROOT / "data/raw/KidneyTransplant/qwen_pilot",
    "v2_batch": ROOT / "data/raw/KidneyTransplant/qwen_batch10_v2_001",
    "v3_design": ROOT / "data/raw/KidneyTransplant/qwen_v3_design",
    "v3_1_design": V31_DESIGN,
    "corrected_10": CORRECTED10,
    "diagnostic_100": DIAGNOSTIC100,
    "processed": ROOT / "data/processed/KidneyTransplant",
    "models": ROOT / "models/KidneyTransplant",
    "results": ROOT / "results/KidneyTransplant",
}


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def tree_fingerprint(root):
    return {
        str(path.relative_to(root)): (
            sha256_file(path),
            path.stat().st_mtime_ns,
            path.stat().st_size,
        )
        for path in sorted(p for p in root.rglob("*") if p.is_file())
    }


def summarize_tree(fingerprint):
    digest = hashlib.sha256()
    total_size = 0
    for relative_path, (file_hash, mtime_ns, size_bytes) in sorted(fingerprint.items()):
        digest.update(
            f"{relative_path}\0{file_hash}\0{mtime_ns}\0{size_bytes}\n".encode("utf-8")
        )
        total_size += size_bytes
    return {
        "file_count": len(fingerprint),
        "total_size_bytes": total_size,
        "aggregate_sha256_with_metadata": digest.hexdigest(),
    }


def protected_fingerprint():
    return {
        "sha256": sha256_file(PROTECTED),
        "mtime_ns": PROTECTED.stat().st_mtime_ns,
        "mtime_utc": datetime.fromtimestamp(PROTECTED.stat().st_mtime, timezone.utc).isoformat(),
        "size_bytes": PROTECTED.stat().st_size,
    }


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, separators=(",", ":"), default=str) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def write_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, default=str) + "\n", encoding="utf-8")
    os.replace(temporary, path)


def write_text(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(value, encoding="utf-8")
    os.replace(temporary, path)


def write_csv(path, frame):
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def count_jsonl(path):
    if not path.exists():
        return 0
    return sum(1 for line in path.read_text(encoding="utf-8").splitlines() if line.strip())


def is_number(value):
    return (
        isinstance(value, Real)
        and not isinstance(value, (bool, np.bool_))
        and math.isfinite(float(value))
    )


PROTECTED_BEFORE = protected_fingerprint()
TREES_BEFORE = {name: tree_fingerprint(path) for name, path in IMMUTABLE_DIRS.items()}
NOTEBOOKS_BEFORE = {
    path.name: (sha256_file(path), path.stat().st_mtime_ns, path.stat().st_size)
    for path in NOTEBOOKS_01_05
}
DIAGNOSTIC100_BEFORE = summarize_tree(TREES_BEFORE["diagnostic_100"])
print("Protected before v3.2 work:", json.dumps(PROTECTED_BEFORE, indent=2))
print("Diagnostic-100 before v3.2 work:", json.dumps(DIAGNOSTIC100_BEFORE, indent=2))

controls = pd.read_csv(V31_DESIGN / "latent_controls_100.csv")
relationships = pd.read_csv(V31_DESIGN / "recipient_relationship_metadata_100.csv")
metadata_skeleton = pd.read_csv(V31_DESIGN / "assessment_metadata_skeleton_600.csv")
identity_skeleton = pd.read_csv(
    V31_DESIGN / "identity_skeleton_180_people.csv", keep_default_na=False
)
corrected_plan = pd.read_csv(V31_DESIGN / "corrected_10_control_plan.csv")
field_lists = json.loads((V31_DESIGN / "field_lists.json").read_text(encoding="utf-8"))
canonical_ranges = json.loads(
    (V31_DESIGN / "canonical_qwen_ranges.json").read_text(encoding="utf-8")
)
api_config = json.loads((V31_DESIGN / "api_configuration.json").read_text(encoding="utf-8"))
v31_prompt = (V31_DESIGN / "qwen_kidney_v3_1_prompt_template.txt").read_text(
    encoding="utf-8"
)
assessment_schema = pd.read_csv(V31_DESIGN / "assessment_table_schema.csv")
identity_schema = pd.read_csv(V31_DESIGN / "identity_table_schema.csv")
diagnostic_assessment = pd.read_csv(DIAGNOSTIC100 / "assessment_table.csv")
classifier_features = field_lists["classifier_features"]
target = field_lists["target"][0]
qwen_fields = field_lists["qwen_return_fields"]
direct_identifiers = field_lists["direct_identifiers"]
pseudonymous_identifiers = field_lists["pseudonymous_identifiers"]
audit_only = field_lists["assessment_audit_only"]
event_controls = field_lists["event_control_metadata"]
assessment_columns = assessment_schema.field_name.tolist()
identity_columns = identity_schema.field_name.tolist()


def clip(value, field):
    bounds = canonical_ranges[field]
    return min(max(float(value), float(bounds["minimum"])), float(bounds["maximum"]))


def generate_anchor(control_row):
    seed = int(control_row.request_seed)
    recipient_number = int(str(control_row.recipient_id)[-6:])
    rng = np.random.default_rng(seed + 3_200_000)
    events = json.loads(control_row.confirmed_rejection_event_days)
    infection_days = json.loads(control_row.infection_episode_days)
    early_event = any(7 < int(event) <= 37 for event in events)
    early_infection = any(int(day) <= 30 for day in infection_days)
    confounder = str(control_row.non_rejection_confounders)
    recovery = str(control_row.recovery_pattern)
    adherence = str(control_row.adherence_pattern)
    tacrolimus = str(control_row.tacrolimus_exposure_pattern)

    recovery_factor = {"rapid": 0.84, "gradual": 0.96, "partial": 1.08}[recovery]
    creatinine_offset = {
        "none": 0.0,
        "dehydration": 0.10,
        "high_creatinine": 0.24,
        "infection": 0.12,
        "tacrolimus_exposure": 0.09,
    }[confounder]
    creatinine = (
        float(control_row.baseline_creatinine_mg_dl) * recovery_factor
        + creatinine_offset
        + (0.07 if early_infection else 0.0)
        + (0.09 if early_event else 0.0)
        + rng.uniform(-0.085, 0.085)
        + (recipient_number % 11) * 0.001
    )
    creatinine = round(clip(creatinine, "creatinine_mg_dl"), 3)

    recovery_urine = {"rapid": 170.0, "gradual": 30.0, "partial": -130.0}[recovery]
    confounder_urine = {
        "none": 0.0,
        "dehydration": -220.0,
        "high_creatinine": -180.0,
        "infection": -160.0,
        "tacrolimus_exposure": -90.0,
    }[confounder]
    urine = (
        2380.0
        - 360.0 * (creatinine - 1.10)
        + recovery_urine
        + confounder_urine
        - (110.0 if early_infection else 0.0)
        - (125.0 if early_event else 0.0)
        + rng.uniform(-190.0, 190.0)
        + (recipient_number % 13) * 3.7
    )
    urine = round(clip(urine, "urine_output_ml_24h"), 1)

    if tacrolimus == "high":
        tacrolimus_anchor = rng.uniform(10.5, 12.4)
    elif tacrolimus == "in_range":
        tacrolimus_anchor = rng.uniform(7.4, 10.0)
    else:
        tacrolimus_anchor = rng.uniform(6.3, 11.2)
    tacrolimus_anchor += 0.8 if confounder == "tacrolimus_exposure" else 0.0
    tacrolimus_anchor += 0.2 if early_event else 0.0
    tacrolimus_anchor += 0.15 if early_infection else 0.0
    tacrolimus_anchor += (recipient_number % 7) * 0.007
    tacrolimus_anchor = round(
        clip(tacrolimus_anchor, "tacrolimus_level_ng_ml"), 2
    )

    if adherence == "high":
        adherence_anchor = rng.uniform(96.1, 99.85)
    elif adherence == "variable":
        adherence_anchor = rng.uniform(79.0, 95.4)
    else:
        adherence_anchor = rng.uniform(56.0, 81.5)
    adherence_anchor -= 0.9 if early_infection else 0.0
    adherence_anchor -= 0.6 if confounder == "infection" else 0.0
    adherence_anchor -= 0.4 if early_event and adherence != "high" else 0.0
    adherence_anchor += (recipient_number % 9) * 0.009
    adherence_anchor = round(
        clip(adherence_anchor, "medication_adherence_pct"), 2
    )

    return {
        "recipient_id": control_row.recipient_id,
        "request_seed": seed,
        "anchor_algorithm_version": ANCHOR_ALGORITHM_VERSION,
        "recovery_pattern": recovery,
        "adherence_pattern": adherence,
        "tacrolimus_exposure_pattern": tacrolimus,
        "infection_episode_days": control_row.infection_episode_days,
        "confirmed_rejection_event_days": control_row.confirmed_rejection_event_days,
        "rejection_signal_strength": control_row.rejection_signal_strength,
        "non_rejection_confounders": confounder,
        "baseline_creatinine_mg_dl": float(control_row.baseline_creatinine_mg_dl),
        "day7_creatinine_anchor_mg_dl": creatinine,
        "day7_urine_output_anchor_ml_24h": urine,
        "day7_tacrolimus_anchor_ng_ml": tacrolimus_anchor,
        "day7_medication_adherence_anchor_pct": adherence_anchor,
    }


anchor_metadata = pd.DataFrame(
    [generate_anchor(row) for row in controls.itertuples(index=False)]
).sort_values("recipient_id").reset_index(drop=True)


def build_v32_prompt_template():
    prompt = v31_prompt.replace(
        "Prompt version: qwen-kidney-v3.1", "Prompt version: qwen-kidney-v3.2", 1
    )
    anchor_instructions = (
        "Recipient-specific day-7 numerical anchors generated deterministically in Python: "
        "creatinine {day7_creatinine_anchor_mg_dl} mg/dL, urine output "
        "{day7_urine_output_anchor_ml_24h} mL/24h, tacrolimus "
        "{day7_tacrolimus_anchor_ng_ml} ng/mL, and medication adherence "
        "{day7_medication_adherence_anchor_pct} percent. For the day-7 assessment, use a value "
        "close to each supplied anchor: within 0.12 mg/dL creatinine, 120 mL/24h urine output, "
        "0.6 ng/mL tacrolimus, and 1.5 percentage points adherence. Construct the later five "
        "measurements as a coherent clinical trajectory from these anchors while following the "
        "supplied recovery, rejection, infection, adherence, tacrolimus and confounder controls. "
        "Choose recipient-specific exact numerical values rather than copying values from the shape "
        "example or a default trajectory. Clinically similar recipients may have different exact "
        "measurements. Keep every value inside the approved ranges below.\n"
    )
    first_newline = prompt.index("\n") + 1
    return prompt[:first_newline] + anchor_instructions + prompt[first_newline:]


v32_prompt = build_v32_prompt_template()


def build_event_coverage_audit():
    records = []
    for row in controls.itertuples(index=False):
        events = sorted(int(day) for day in json.loads(row.confirmed_rejection_event_days))
        for event_day in events:
            preceding = [day for day in DAYS if day < event_day and event_day <= day + 30]
            no_positive = len(preceding) == 0
            records.append(
                {
                    "recipient_id": row.recipient_id,
                    "event_day": event_day,
                    "assessments_within_30_days_before_event": json.dumps(preceding),
                    "target_positive_assessment_days": json.dumps(preceding),
                    "event_produces_no_positive_assessment": no_positive,
                    "reason": (
                        f"No scheduled assessment occurs from day {event_day - 30} through day "
                        f"{event_day - 1}; the rule requires assessment_day < event_day <= "
                        "assessment_day + 30."
                        if no_positive
                        else "One or more scheduled assessments satisfy assessment_day < event_day "
                        "<= assessment_day + 30 exactly."
                    ),
                }
            )
    frame = pd.DataFrame(records)
    event_recipients = controls.confirmed_rejection_event_days.ne("[]")
    recipient_positive = set(frame.loc[~frame.event_produces_no_positive_assessment, "recipient_id"])
    no_positive = frame.loc[frame.event_produces_no_positive_assessment]
    summary = {
        "controlled_event_recipient_count": int(event_recipients.sum()),
        "controlled_event_count": int(len(frame)),
        "recipients_with_at_least_one_target_positive_assessment": len(recipient_positive),
        "event_recipients_without_target_positive_assessment": int(no_positive.recipient_id.nunique()),
        "events_without_target_positive_assessment": int(len(no_positive)),
        "event_days_without_coverage": sorted(no_positive.event_day.astype(int).unique().tolist()),
        "explanation": (
            "There are 34 recipients with a controlled rejection event, but only 25 have at least "
            "one target-positive assessment. The remaining 9 events occur without a scheduled "
            "assessment in the preceding 30-day prediction window; under the frozen rule "
            "assessment_day < event_day <= assessment_day + 30, those events validly create no "
            "positive assessment. No event schedule was changed."
        ),
        "target_rule": "assessment_day < event_day <= assessment_day + 30",
        "event_schedule_modified": False,
        "api_requests_during_audit": 0,
    }
    return frame, summary


event_audit, event_summary = build_event_coverage_audit()

corrected_ids = set(corrected_plan.recipient_id)
confirmation_ids = sorted(set(controls.recipient_id) - corrected_ids)[:EXPECTED_RECIPIENTS]
confirmation_controls = controls.loc[controls.recipient_id.isin(confirmation_ids)].sort_values(
    "recipient_id"
).reset_index(drop=True)
confirmation_anchors = anchor_metadata.loc[
    anchor_metadata.recipient_id.isin(confirmation_ids)
].sort_values("recipient_id").reset_index(drop=True)


def dominant_stat(frame, field):
    counts = frame[field].value_counts(dropna=False)
    return {
        "dominant_value": float(counts.index[0]),
        "dominant_count": int(counts.iloc[0]),
        "dominant_fraction": float(counts.iloc[0] / len(frame)),
        "unique_values": int(frame[field].nunique(dropna=False)),
    }


def local_design_validation():
    errors = []
    expected_qwen_fields = [
        "days_since_transplant",
        "creatinine_mg_dl",
        "urine_output_ml_24h",
        "tacrolimus_level_ng_ml",
        "medication_adherence_pct",
    ]
    if not v32_prompt.startswith("Prompt version: qwen-kidney-v3.2"):
        errors.append("v3.2 prompt version missing")
    for token in [
        "{day7_creatinine_anchor_mg_dl}",
        "{day7_urine_output_anchor_ml_24h}",
        "{day7_tacrolimus_anchor_ng_ml}",
        "{day7_medication_adherence_anchor_pct}",
    ]:
        if token not in v32_prompt:
            errors.append(f"prompt anchor placeholder missing: {token}")
    if qwen_fields != expected_qwen_fields:
        errors.append("five-field Qwen schema changed")
    if api_config["client"]["max_retries"] != 0 or api_config["automatic_retry"] is not False:
        errors.append("retry configuration changed")
    if api_config["completion"].get("extra_body") != {
        "top_k": 20,
        "chat_template_kwargs": {"enable_thinking": False},
    }:
        errors.append("thinking/top_k configuration changed")
    if len(anchor_metadata) != 100 or not anchor_metadata.recipient_id.is_unique:
        errors.append("anchor metadata is not one row per skeleton recipient")
    anchor_fields = {
        "day7_creatinine_anchor_mg_dl": "creatinine_mg_dl",
        "day7_urine_output_anchor_ml_24h": "urine_output_ml_24h",
        "day7_tacrolimus_anchor_ng_ml": "tacrolimus_level_ng_ml",
        "day7_medication_adherence_anchor_pct": "medication_adherence_pct",
    }
    for anchor_field, range_field in anchor_fields.items():
        bounds = canonical_ranges[range_field]
        if not anchor_metadata[anchor_field].between(bounds["minimum"], bounds["maximum"]).all():
            errors.append(f"anchor outside approved range: {anchor_field}")
    if len(confirmation_ids) != 20 or set(confirmation_ids) & corrected_ids:
        errors.append("confirmation cohort is not 20 recipients disjoint from corrected-10")
    if not confirmation_controls.request_seed.is_unique:
        errors.append("confirmation seeds are not unique")
    if len(event_audit) != 34 or event_summary[
        "recipients_with_at_least_one_target_positive_assessment"
    ] != 25:
        errors.append("event-to-target coverage does not reproduce 34 versus 25")
    leakage = set(classifier_features) & set(
        direct_identifiers
        + pseudonymous_identifiers
        + audit_only
        + event_controls
        + list(anchor_fields)
    )
    if leakage:
        errors.append(f"classifier leakage: {sorted(leakage)}")

    v31_day7 = diagnostic_assessment.loc[diagnostic_assessment.days_since_transplant.eq(7)]
    anchor_day7 = confirmation_anchors
    diversity = {
        "v3_1_diagnostic100_observed": {
            "urine_output": dominant_stat(v31_day7, "urine_output_ml_24h"),
            "medication_adherence": dominant_stat(v31_day7, "medication_adherence_pct"),
        },
        "v3_2_confirmation_anchor_preview": {
            "urine_output": dominant_stat(
                anchor_day7, "day7_urine_output_anchor_ml_24h"
            ),
            "medication_adherence": dominant_stat(
                anchor_day7, "day7_medication_adherence_anchor_pct"
            ),
        },
    }
    if diversity["v3_2_confirmation_anchor_preview"]["urine_output"]["unique_values"] < 12:
        errors.append("confirmation urine anchors do not meet preview diversity")
    if diversity["v3_2_confirmation_anchor_preview"]["medication_adherence"]["unique_values"] < 12:
        errors.append("confirmation adherence anchors do not meet preview diversity")
    return {
        "status": "passed" if not errors else "failed",
        "errors": errors,
        "api_requests": 0,
        "anchor_algorithm_version": ANCHOR_ALGORITHM_VERSION,
        "anchor_recipient_count": len(anchor_metadata),
        "confirmation_recipient_ids": confirmation_ids,
        "confirmation_non_overlapping_with_corrected10": not bool(
            set(confirmation_ids) & corrected_ids
        ),
        "event_to_target_summary": event_summary,
        "diversity_comparison": diversity,
        "classifier_leakage_fields": sorted(leakage),
        "production_generation_enabled": False,
    }


LOCAL_VALIDATION = local_design_validation()
print("Local v3.2 validation:", json.dumps(LOCAL_VALIDATION, indent=2))


def render_prompt(control_row, relationship_row, anchor_row):
    static_fields = {
        field: relationship_row[field]
        for field in [
            "recipient_age",
            "donor_age",
            "donor_type",
            "kidney_failure_cause",
            "previous_transplant",
            "dialysis_months",
            "recipient_blood_group",
            "donor_blood_group",
            "hla_mismatch_count",
            "antibody_risk_score",
            "cold_ischaemia_hours",
        ]
    }
    replacements = {
        "{recipient_seed}": str(int(control_row.request_seed)),
        "{static_clinical_fields}": json.dumps(static_fields, sort_keys=True),
        "{baseline_creatinine_mg_dl}": str(float(control_row.baseline_creatinine_mg_dl)),
        "{generation_only_event_schedule}": str(control_row.confirmed_rejection_event_days),
        "{signal_strength}": str(control_row.rejection_signal_strength),
        "{confounder_instructions}": str(control_row.non_rejection_confounders),
        "{adherence_pattern}": str(control_row.adherence_pattern),
        "{infection_episode_days}": str(control_row.infection_episode_days),
        "{tacrolimus_exposure}": str(control_row.tacrolimus_exposure_pattern),
        "{recovery_pattern}": str(control_row.recovery_pattern),
        "{day7_creatinine_anchor_mg_dl}": str(anchor_row.day7_creatinine_anchor_mg_dl),
        "{day7_urine_output_anchor_ml_24h}": str(
            anchor_row.day7_urine_output_anchor_ml_24h
        ),
        "{day7_tacrolimus_anchor_ng_ml}": str(anchor_row.day7_tacrolimus_anchor_ng_ml),
        "{day7_medication_adherence_anchor_pct}": str(
            anchor_row.day7_medication_adherence_anchor_pct
        ),
    }
    prompt = v32_prompt
    for old, new in replacements.items():
        prompt = prompt.replace(old, new)
    assert all(token not in prompt for token in replacements)
    return prompt


def validate_qwen_payload(payload, anchor_row):
    errors = []
    warnings = []
    if not isinstance(payload, dict):
        return ["root must be one JSON object"], warnings
    if set(payload) != {"assessments"}:
        errors.append("root must contain only assessments")
    items = payload.get("assessments")
    if not isinstance(items, list):
        return errors + ["assessments must be an array"], warnings
    if len(items) != 6:
        errors.append("assessments must contain six objects")
    observed_days = []
    sequences = {field: [] for field in qwen_fields if field != "days_since_transplant"}
    for position, item in enumerate(items):
        if not isinstance(item, dict):
            errors.append(f"assessment {position} is not an object")
            continue
        if set(item) != set(qwen_fields):
            errors.append(f"assessment {position} key set is not exact")
        for field in qwen_fields:
            if field not in item or not is_number(item[field]):
                errors.append(f"assessment {position} {field} must be finite numeric")
        if is_number(item.get("days_since_transplant")):
            observed_days.append(int(item["days_since_transplant"]))
        for field, bounds in canonical_ranges.items():
            value = item.get(field)
            if is_number(value) and not bounds["minimum"] <= float(value) <= bounds["maximum"]:
                errors.append(f"assessment {position} {field} outside approved range")
        for field in sequences:
            if is_number(item.get(field)):
                sequences[field].append(float(item[field]))
    if observed_days != DAYS:
        errors.append(f"assessment days must be {DAYS} in order")
    if items and isinstance(items[0], dict):
        anchor_map = {
            "creatinine_mg_dl": anchor_row.day7_creatinine_anchor_mg_dl,
            "urine_output_ml_24h": anchor_row.day7_urine_output_anchor_ml_24h,
            "tacrolimus_level_ng_ml": anchor_row.day7_tacrolimus_anchor_ng_ml,
            "medication_adherence_pct": anchor_row.day7_medication_adherence_anchor_pct,
        }
        for field, anchor in anchor_map.items():
            value = items[0].get(field)
            if is_number(value) and abs(float(value) - float(anchor)) > ANCHOR_TOLERANCES[field]:
                errors.append(f"day-7 {field} is not close to its deterministic anchor")
    for field, values in sequences.items():
        if len(values) != 6 or not np.isfinite(values).all():
            errors.append(f"{field} is not a complete finite longitudinal sequence")
            continue
        bounds = canonical_ranges[field]
        span = bounds["maximum"] - bounds["minimum"]
        deltas = np.abs(np.diff(values))
        if (deltas > span + 1e-12).any():
            errors.append(f"{field} contains a range-exceeding longitudinal jump")
        if (deltas > 0.5 * span).any():
            warnings.append(f"{field} contains a large but range-valid consecutive change")
    return errors, warnings


def derive_abo_category(donor_group, recipient_group):
    compatible = {
        "O": {"O", "A", "B", "AB"},
        "A": {"A", "AB"},
        "B": {"B", "AB"},
        "AB": {"AB"},
    }
    return (
        "Standard compatible"
        if recipient_group in compatible[donor_group]
        else "Managed incompatibility"
    )


def derive_changes(baseline, values):
    result = []
    previous = float(baseline)
    for current in values:
        current = float(current)
        result.append(100.0 * (current - previous) / previous)
        previous = current
    return result


def derive_event_fields(days, event_days):
    events = sorted(int(day) for day in event_days)
    previous = [int(any(event < day for event in events)) for day in days]
    future = [int(any(day < event <= day + 30 for event in events)) for day in days]
    return previous, future


def build_recipient_rows(control_row, relationship_row, payload):
    items = payload["assessments"]
    values = {field: [item[field] for item in items] for field in qwen_fields}
    days = [int(day) for day in values["days_since_transplant"]]
    infection_days = {int(day) for day in json.loads(control_row.infection_episode_days)}
    infections = [int(day in infection_days) for day in days]
    changes = derive_changes(
        control_row.baseline_creatinine_mg_dl, values["creatinine_mg_dl"]
    )
    previous_rejection, targets = derive_event_fields(
        days, json.loads(control_row.confirmed_rejection_event_days)
    )
    transplant = date.fromisoformat(relationship_row["transplant_date"])
    audit_rows = metadata_skeleton.loc[
        metadata_skeleton.recipient_id.eq(control_row.recipient_id)
    ].sort_values("days_since_transplant").reset_index(drop=True)
    assert audit_rows.days_since_transplant.astype(int).tolist() == DAYS
    abo = derive_abo_category(
        relationship_row["donor_blood_group"], relationship_row["recipient_blood_group"]
    )
    assert abo == relationship_row["abo_compatibility_category"]
    recipient_number = int(str(control_row.recipient_id)[-6:])
    rows = []
    for index, day in enumerate(days):
        assessment_date = transplant + timedelta(days=day)
        retention_date = assessment_date + timedelta(days=730)
        assert assessment_date.isoformat() == audit_rows.loc[index, "assessment_date"]
        assert retention_date.isoformat() == audit_rows.loc[index, "retention_expiry_date"]
        row = {
            "assessment_id": f"{RUN_ID}-A{(recipient_number - 1) * 6 + index + 1:06d}",
            "recipient_id": control_row.recipient_id,
            "donor_id": relationship_row["donor_id"],
            "hospital_id": relationship_row["hospital_id"],
            "assessment_date": assessment_date.isoformat(),
            "training_consent_status": audit_rows.loc[index, "training_consent_status"],
            "training_consent_version": audit_rows.loc[index, "training_consent_version"],
            "retention_expiry_date": retention_date.isoformat(),
            "recipient_sex": relationship_row["recipient_sex"],
            "recipient_ethnicity": relationship_row["recipient_ethnicity"],
            "recipient_region": relationship_row["recipient_region"],
            "distance_to_transplant_centre_km": relationship_row[
                "distance_to_transplant_centre_km"
            ],
            "recipient_age": relationship_row["recipient_age"],
            "donor_age": relationship_row["donor_age"],
            "donor_type": relationship_row["donor_type"],
            "kidney_failure_cause": relationship_row["kidney_failure_cause"],
            "previous_transplant": relationship_row["previous_transplant"],
            "dialysis_months": relationship_row["dialysis_months"],
            "abo_compatibility_category": abo,
            "hla_mismatch_count": relationship_row["hla_mismatch_count"],
            "antibody_risk_score": relationship_row["antibody_risk_score"],
            "cold_ischaemia_hours": relationship_row["cold_ischaemia_hours"],
            "days_since_transplant": day,
            "creatinine_mg_dl": float(values["creatinine_mg_dl"][index]),
            "creatinine_change_pct": float(changes[index]),
            "urine_output_ml_24h": float(values["urine_output_ml_24h"][index]),
            "tacrolimus_level_ng_ml": float(values["tacrolimus_level_ng_ml"][index]),
            "medication_adherence_pct": float(values["medication_adherence_pct"][index]),
            "infection_indicator": int(infections[index]),
            "previous_rejection": int(previous_rejection[index]),
            target: int(targets[index]),
        }
        assert list(row) == assessment_columns
        rows.append(row)
    return rows


def validate_complete(frame, identity, response_records):
    failures = []
    checks = {
        "dimensions": frame.shape == (120, len(assessment_columns)),
        "schema_exact": frame.columns.tolist() == assessment_columns,
        "recipient_count": frame.recipient_id.nunique() == 20,
        "six_assessments_each": frame.groupby("recipient_id").size().eq(6).all(),
        "assessment_ids_unique": frame.assessment_id.is_unique,
        "recipient_days_unique": not frame.duplicated(
            ["recipient_id", "days_since_transplant"]
        ).any(),
        "no_duplicate_rows": not frame.duplicated().any(),
        "no_missing_values": not frame.isna().any().any(),
    }
    failures.extend(name for name, passed in checks.items() if not passed)
    static_fields = [
        "donor_id",
        "hospital_id",
        "recipient_sex",
        "recipient_ethnicity",
        "recipient_region",
        "distance_to_transplant_centre_km",
        "recipient_age",
        "donor_age",
        "donor_type",
        "kidney_failure_cause",
        "previous_transplant",
        "dialysis_months",
        "abo_compatibility_category",
        "hla_mismatch_count",
        "antibody_risk_score",
        "cold_ischaemia_hours",
    ]
    checks["static_fields_constant"] = bool(
        frame.groupby("recipient_id")[static_fields].nunique(dropna=False).le(1).all().all()
    )
    raw_map = {}
    for record in response_records:
        content = record["serialized_response"]["choices"][0]["message"]["content"]
        raw_map[record["recipient_id"]] = json.loads(content)
    raw_matches = 0
    raw_mismatches = []
    derivation_errors = []
    date_errors = []
    range_errors = []
    anchor_errors = []
    for recipient_id, group in frame.sort_values(
        ["recipient_id", "days_since_transplant"]
    ).groupby("recipient_id"):
        group = group.reset_index(drop=True)
        relationship = relationships.loc[relationships.recipient_id.eq(recipient_id)].iloc[0]
        control = controls.loc[controls.recipient_id.eq(recipient_id)].iloc[0]
        anchor = anchor_metadata.loc[anchor_metadata.recipient_id.eq(recipient_id)].iloc[0]
        if group.days_since_transplant.astype(int).tolist() != DAYS:
            derivation_errors.append(f"{recipient_id}: days")
            continue
        transplant = date.fromisoformat(relationship.transplant_date)
        expected_changes = derive_changes(
            control.baseline_creatinine_mg_dl, group.creatinine_mg_dl.tolist()
        )
        expected_previous, expected_target = derive_event_fields(
            DAYS, json.loads(control.confirmed_rejection_event_days)
        )
        infection_days = set(json.loads(control.infection_episode_days))
        expected_infection = [int(day in infection_days) for day in DAYS]
        if not np.allclose(group.creatinine_change_pct, expected_changes, rtol=0, atol=1e-10):
            derivation_errors.append(f"{recipient_id}: creatinine change")
        if group.previous_rejection.astype(int).tolist() != expected_previous:
            derivation_errors.append(f"{recipient_id}: previous rejection")
        if group[target].astype(int).tolist() != expected_target:
            derivation_errors.append(f"{recipient_id}: target")
        if group.infection_indicator.astype(int).tolist() != expected_infection:
            derivation_errors.append(f"{recipient_id}: infection")
        expected_abo = derive_abo_category(
            relationship.donor_blood_group, relationship.recipient_blood_group
        )
        if not group.abo_compatibility_category.eq(expected_abo).all():
            derivation_errors.append(f"{recipient_id}: ABO")
        for index, day in enumerate(DAYS):
            expected_date = transplant + timedelta(days=day)
            if group.loc[index, "assessment_date"] != expected_date.isoformat():
                date_errors.append(f"{recipient_id}:{day}:assessment")
            if group.loc[index, "retention_expiry_date"] != (
                expected_date + timedelta(days=730)
            ).isoformat():
                date_errors.append(f"{recipient_id}:{day}:retention")
        for field, bounds in canonical_ranges.items():
            if not group[field].between(bounds["minimum"], bounds["maximum"]).all():
                range_errors.append(f"{recipient_id}:{field}")
        payload = raw_map.get(recipient_id)
        if payload is None:
            raw_mismatches.append(f"{recipient_id}: missing response")
        else:
            for source, (_, validated) in zip(
                payload["assessments"], group.iterrows(), strict=True
            ):
                for field in qwen_fields:
                    if math.isclose(
                        float(source[field]), float(validated[field]), rel_tol=0, abs_tol=1e-10
                    ):
                        raw_matches += 1
                    else:
                        raw_mismatches.append(f"{recipient_id}:{field}")
        anchor_map = {
            "creatinine_mg_dl": anchor.day7_creatinine_anchor_mg_dl,
            "urine_output_ml_24h": anchor.day7_urine_output_anchor_ml_24h,
            "tacrolimus_level_ng_ml": anchor.day7_tacrolimus_anchor_ng_ml,
            "medication_adherence_pct": anchor.day7_medication_adherence_anchor_pct,
        }
        for field, anchor_value in anchor_map.items():
            if abs(float(group.loc[0, field]) - float(anchor_value)) > ANCHOR_TOLERANCES[field]:
                anchor_errors.append(f"{recipient_id}:{field}")
    checks.update(
        {
            "dates_and_retention_exact": not date_errors,
            "python_derivations_exact": not derivation_errors,
            "approved_ranges": not range_errors,
            "raw_response_reconciliation": not raw_mismatches and raw_matches == 600,
            "day7_anchor_proximity": not anchor_errors,
        }
    )
    expected_entities = set(confirmation_ids) | set(
        relationships.loc[relationships.recipient_id.isin(confirmation_ids), "donor_id"]
    )
    checks["identity_subset_exact"] = (
        identity.columns.tolist() == identity_columns
        and identity.entity_id.is_unique
        and set(identity.entity_id) == expected_entities
    )
    leakage = set(classifier_features) & set(
        direct_identifiers
        + pseudonymous_identifiers
        + audit_only
        + event_controls
        + [
            "day7_creatinine_anchor_mg_dl",
            "day7_urine_output_anchor_ml_24h",
            "day7_tacrolimus_anchor_ng_ml",
            "day7_medication_adherence_anchor_pct",
        ]
    )
    checks["no_leakage"] = not leakage
    failures.extend(name for name, passed in checks.items() if not passed and name not in failures)
    details = {
        "checks": checks,
        "failures": failures,
        "raw_reconciliation": {
            "expected_matches": 600,
            "actual_matches": raw_matches,
            "mismatches": len(raw_mismatches),
        },
        "derivation_error_examples": derivation_errors[:20],
        "date_error_examples": date_errors[:20],
        "range_error_examples": range_errors[:20],
        "anchor_error_examples": anchor_errors[:20],
        "leakage_fields": sorted(leakage),
    }
    if failures:
        raise AssertionError("confirmation structural validation failed: " + ", ".join(failures))
    return details


def confirmation_diagnostics(frame):
    day_metrics = []
    for field in [
        "creatinine_mg_dl",
        "urine_output_ml_24h",
        "tacrolimus_level_ng_ml",
        "medication_adherence_pct",
    ]:
        for day, group in frame.groupby("days_since_transplant", sort=True):
            metric = dominant_stat(group, field)
            day_metrics.append({"field": field, "day": int(day), **metric})
    trajectories = {}
    for recipient_id, group in frame.sort_values(
        ["recipient_id", "days_since_transplant"]
    ).groupby("recipient_id"):
        key = json.dumps(group[qwen_fields].values.tolist(), separators=(",", ":"))
        trajectories.setdefault(key, []).append(recipient_id)
    identical = [ids for ids in trajectories.values() if len(ids) > 1]
    day7 = frame.loc[frame.days_since_transplant.eq(7)]
    urine = dominant_stat(day7, "urine_output_ml_24h")
    adherence = dominant_stat(day7, "medication_adherence_pct")
    conditions = {
        "structural_validation_passed": True,
        "no_identical_complete_five_field_trajectories": len(identical) == 0,
        "day7_urine_unique_at_least_12": urine["unique_values"] >= 12,
        "day7_adherence_unique_at_least_12": adherence["unique_values"] >= 12,
        "day7_urine_dominant_fraction_at_most_0_20": urine["dominant_fraction"] <= 0.20,
        "day7_adherence_dominant_fraction_at_most_0_20": adherence[
            "dominant_fraction"
        ]
        <= 0.20,
    }
    return {
        "model_metrics_computed": False,
        "model_metric_acceptance_thresholds": False,
        "generator_tuning_from_confirmation_metrics": False,
        "day_specific_dominant_values_and_unique_counts": day_metrics,
        "day7_summary": {"urine_output": urine, "medication_adherence": adherence},
        "identical_complete_five_field_trajectories": {
            "duplicate_group_count": len(identical),
            "groups": identical,
        },
        "production_acceptance_conditions": conditions,
        "all_production_acceptance_conditions_passed": all(conditions.values()),
        "production_suitability": "suitable" if all(conditions.values()) else "unsuitable",
        "diversity_failure_action": (
            "none"
            if all(conditions.values())
            else "responses preserved and accepted as structurally valid; no regeneration permitted"
        ),
    }


def write_design_artifacts():
    assert not DESIGN_DIR.exists(), "v3.2 design directory already exists"
    DESIGN_DIR.mkdir(parents=False, exist_ok=False)
    write_text(DESIGN_DIR / "qwen_kidney_v3_2_prompt_template.txt", v32_prompt)
    write_json(
        DESIGN_DIR / "anchor_generation_specification.json",
        {
            "algorithm_version": ANCHOR_ALGORITHM_VERSION,
            "determinism": "numpy default_rng(recipient request_seed + 3200000)",
            "inputs": [
                "request_seed",
                "baseline_creatinine_mg_dl",
                "recovery_pattern",
                "adherence_pattern",
                "tacrolimus_exposure_pattern",
                "infection_episode_days",
                "confirmed_rejection_event_days",
                "rejection_signal_strength",
                "non_rejection_confounders",
            ],
            "outputs": [
                "day7_creatinine_anchor_mg_dl",
                "day7_urine_output_anchor_ml_24h",
                "day7_tacrolimus_anchor_ng_ml",
                "day7_medication_adherence_anchor_pct",
            ],
            "approved_ranges": canonical_ranges,
            "prompt_proximity_tolerances": ANCHOR_TOLERANCES,
            "post_response_jitter_or_adjustment": False,
            "classifier_feature_allowed": False,
            "final_assessment_columns_added": [],
        },
    )
    schema_rows = [
        {
            "field_name": field,
            "data_type": str(anchor_metadata[field].dtype),
            "role": "generation-only anchor metadata",
            "classifier_allowed": False,
            "final_assessment_table_allowed": False,
        }
        for field in anchor_metadata.columns
    ]
    write_csv(DESIGN_DIR / "anchor_metadata_schema.csv", pd.DataFrame(schema_rows))
    write_csv(DESIGN_DIR / "anchor_metadata_100.csv", anchor_metadata)
    write_csv(DESIGN_DIR / "event_to_target_coverage_audit.csv", event_audit)
    write_json(DESIGN_DIR / "event_to_target_coverage_summary.json", event_summary)
    write_json(DESIGN_DIR / "api_configuration.json", api_config)
    write_json(
        DESIGN_DIR / "confirmation_acceptance_criteria.json",
        {
            "recipient_count": 20,
            "identical_complete_five_field_trajectories_allowed": 0,
            "minimum_unique_day7_urine_output_values": 12,
            "minimum_unique_day7_adherence_values": 12,
            "maximum_single_day7_urine_output_fraction": 0.20,
            "maximum_single_day7_adherence_fraction": 0.20,
            "diversity_failure_is_structural_failure": False,
            "regeneration_on_diversity_failure": False,
            "model_metric_thresholds": None,
        },
    )
    write_json(DESIGN_DIR / "local_validation_report.json", LOCAL_VALIDATION)


def run_workflow():
    report = {
        "run_id": RUN_ID,
        "status": "not_started",
        "api_requests_attempted": 0,
        "api_requests_completed": 0,
        "attempts": 0,
        "failures": 0,
        "retries": 0,
    }
    responses_path = RUN_DIR / "responses.jsonl"
    manifest_path = RUN_DIR / "manifest.jsonl"
    state_path = RUN_DIR / "completion_state.json"
    report_path = RUN_DIR / "run_report.json"
    response_records = []
    rows = []
    usage_records = []
    finish_records = []
    warnings = []
    failure = None
    validation = None
    diagnostics = None
    anchor_audit = None
    protected_after_validation = None
    status = "local_design"
    try:
        assert LOCAL_VALIDATION["status"] == "passed", "local v3.2 validation failed"
        assert not DESIGN_DIR.exists(), "v3.2 design directory already exists"
        assert not RUN_DIR.exists(), "confirmation directory already exists; overwrite/resume prohibited"
        assert os.environ.get(GATE_NAME) == RUN_ID, "one-time confirmation gate mismatch"
        assert len(confirmation_ids) == EXPECTED_RECIPIENTS
        assert MAX_API_REQUESTS == EXPECTED_RECIPIENTS
        assert api_config["client"]["max_retries"] == 0
        assert api_config["automatic_retry"] is False
        assert not RUN_FULL_GENERATION and not RUN_10000_RECIPIENTS
        assert PROTECTED_BEFORE["sha256"] == EXPECTED_PROTECTED["sha256"]
        assert PROTECTED_BEFORE["mtime_ns"] == EXPECTED_PROTECTED["mtime_ns"]
        assert PROTECTED_BEFORE["size_bytes"] == EXPECTED_PROTECTED["size_bytes"]

        # This complete local stage writes the 100-recipient event audit before any API client exists.
        write_design_artifacts()
        assert event_summary["api_requests_during_audit"] == 0
        assert report["api_requests_attempted"] == 0

        RUN_DIR.mkdir(parents=False, exist_ok=False)
        status = "running"
        run_configuration = {
            "run_id": RUN_ID,
            "base_url": BASE_URL,
            "api_key": "REDACTED",
            "model": api_config["completion"]["model"],
            "api_configuration": api_config,
            "prompt_version": "qwen-kidney-v3.2",
            "prompt_sha256": sha256_file(
                DESIGN_DIR / "qwen_kidney_v3_2_prompt_template.txt"
            ),
            "anchor_algorithm_version": ANCHOR_ALGORITHM_VERSION,
            "anchor_metadata_sha256": sha256_file(DESIGN_DIR / "anchor_metadata_100.csv"),
            "event_coverage_audit_sha256": sha256_file(
                DESIGN_DIR / "event_to_target_coverage_audit.csv"
            ),
            "confirmation_recipient_ids": confirmation_ids,
            "maximum_requests": 20,
            "max_retries": 0,
            "one_time_gate": f"{GATE_NAME}={RUN_ID}",
            "protected_before": PROTECTED_BEFORE,
            "diagnostic100_before": DIAGNOSTIC100_BEFORE,
            "persistent_generation_flags": {
                "RUN_QWEN_V32_CONFIRM20": RUN_QWEN_V32_CONFIRM20,
                "RUN_10000_RECIPIENTS": RUN_10000_RECIPIENTS,
                "RUN_FULL_GENERATION": RUN_FULL_GENERATION,
            },
        }
        write_json(RUN_DIR / "run_configuration.json", run_configuration)
        write_json(
            state_path,
            {
                "run_id": RUN_ID,
                "status": "running",
                "automatic_resume": False,
                "requests_attempted": 0,
                "requests_completed": 0,
                "recipients_checkpointed": 0,
                "rows_checkpointed": 0,
            },
        )
        client = OpenAI(api_key=API_KEY, base_url=BASE_URL, max_retries=0, timeout=180.0)
        for sequence, control_row in enumerate(
            confirmation_controls.itertuples(index=False), start=1
        ):
            if report["api_requests_attempted"] >= MAX_API_REQUESTS:
                raise RuntimeError("20-request hard cap reached")
            recipient_number = int(str(control_row.recipient_id)[-6:])
            request_id = f"QWEN-V32-C20-REQ{recipient_number:06d}"
            relationship_row = relationships.loc[
                relationships.recipient_id.eq(control_row.recipient_id)
            ].iloc[0].to_dict()
            anchor_row = confirmation_anchors.loc[
                confirmation_anchors.recipient_id.eq(control_row.recipient_id)
            ].iloc[0]
            prompt = render_prompt(control_row, relationship_row, anchor_row)
            audit = {
                "request_id": request_id,
                "recipient_id": control_row.recipient_id,
                "seed": int(control_row.request_seed),
                "request_sequence_number": sequence,
                "attempt_number": 1,
            }
            report["api_requests_attempted"] += 1
            report["attempts"] += 1
            append_jsonl(manifest_path, {**audit, "status": "request_started"})
            try:
                response = client.chat.completions.create(
                    model=api_config["completion"]["model"],
                    messages=[
                        {
                            "role": "system",
                            "content": "Return only the exact JSON object requested. No prose or Markdown.",
                        },
                        {"role": "user", "content": prompt},
                    ],
                    max_tokens=api_config["completion"]["max_tokens"],
                    temperature=api_config["completion"]["temperature"],
                    top_p=api_config["completion"]["top_p"],
                    presence_penalty=api_config["completion"]["presence_penalty"],
                    seed=int(control_row.request_seed),
                    extra_body=api_config["completion"]["extra_body"],
                )
            except Exception as exc:
                failure = {
                    "stage": "api_request",
                    **audit,
                    "exception_type": type(exc).__name__,
                    "message": str(exc),
                }
                report["failures"] += 1
                append_jsonl(manifest_path, {**audit, "status": "request_failed", "failure": failure})
                raise RuntimeError(f"request failed for {request_id}") from exc

            serialized = response.model_dump_json(indent=2)
            response_record = {**audit, "serialized_response": json.loads(serialized)}
            append_jsonl(responses_path, response_record)
            response_records.append(response_record)

            choice = response.choices[0] if response.choices else None
            finish_reason = choice.finish_reason if choice else None
            usage = response.usage.model_dump() if response.usage else {}
            message = choice.message if choice else None
            content = message.content if message else None
            reasoning = getattr(message, "reasoning_content", None) if message else None
            metadata = {
                **audit,
                "finish_reason": finish_reason,
                "token_usage": usage,
                "response_length": len(content) if isinstance(content, str) else 0,
                "reasoning_content_status": (
                    "present_nonempty"
                    if reasoning
                    else ("present_empty" if reasoning == "" else "absent")
                ),
            }
            usage_records.append(usage)
            finish_records.append(metadata)
            append_jsonl(manifest_path, {**audit, "status": "response_preserved", **metadata})
            if finish_reason == "length":
                failure = {"stage": "finish_reason_length", **metadata}
            elif not content:
                failure = {"stage": "absent_final_content", **metadata}
            else:
                try:
                    payload = json.loads(content)
                except Exception as exc:
                    failure = {
                        "stage": "json_parse",
                        **metadata,
                        "exception_type": type(exc).__name__,
                        "message": str(exc),
                    }
            if failure is not None:
                report["failures"] += 1
                append_jsonl(
                    manifest_path, {**audit, "status": "validation_failed", "failure": failure}
                )
                raise RuntimeError(f"response failed before schema validation for {request_id}")
            errors, soft = validate_qwen_payload(payload, anchor_row)
            if errors:
                failure = {
                    "stage": "payload_validation",
                    **metadata,
                    "errors": errors,
                    "warnings": soft,
                }
                report["failures"] += 1
                append_jsonl(
                    manifest_path, {**audit, "status": "validation_failed", "failure": failure}
                )
                raise RuntimeError(f"payload validation failed for {request_id}")
            warnings.extend(
                [{"recipient_id": control_row.recipient_id, "warning": item} for item in soft]
            )
            rows.extend(build_recipient_rows(control_row, relationship_row, payload))
            report["api_requests_completed"] += 1
            checkpoint = pd.DataFrame(rows, columns=assessment_columns)
            write_csv(RUN_DIR / "validated_recipient_checkpoint.csv", checkpoint)
            append_jsonl(
                manifest_path,
                {
                    **audit,
                    "status": "validated",
                    "finish_reason": finish_reason,
                    "token_usage": usage,
                    "response_length": len(content),
                    "checkpoint_recipients": report["api_requests_completed"],
                    "checkpoint_rows": len(checkpoint),
                    "soft_longitudinal_warnings": soft,
                },
            )
            write_json(
                state_path,
                {
                    "run_id": RUN_ID,
                    "status": "running",
                    "automatic_resume": False,
                    "requests_attempted": report["api_requests_attempted"],
                    "requests_completed": report["api_requests_completed"],
                    "recipients_checkpointed": report["api_requests_completed"],
                    "rows_checkpointed": len(checkpoint),
                    "last_validated_recipient_id": control_row.recipient_id,
                },
            )

        assessment = pd.DataFrame(rows, columns=assessment_columns)
        entity_ids = set(confirmation_ids) | set(
            relationships.loc[relationships.recipient_id.isin(confirmation_ids), "donor_id"]
        )
        identity = identity_skeleton.loc[identity_skeleton.entity_id.isin(entity_ids)].copy()
        validation = validate_complete(assessment, identity, response_records)
        diagnostics = confirmation_diagnostics(assessment)
        day7 = assessment.loc[assessment.days_since_transplant.eq(7)].copy()
        anchor_audit = confirmation_anchors.merge(
            day7[
                [
                    "recipient_id",
                    "creatinine_mg_dl",
                    "urine_output_ml_24h",
                    "tacrolimus_level_ng_ml",
                    "medication_adherence_pct",
                ]
            ],
            on="recipient_id",
            validate="one_to_one",
        )
        for output_field, anchor_field in {
            "creatinine_mg_dl": "day7_creatinine_anchor_mg_dl",
            "urine_output_ml_24h": "day7_urine_output_anchor_ml_24h",
            "tacrolimus_level_ng_ml": "day7_tacrolimus_anchor_ng_ml",
            "medication_adherence_pct": "day7_medication_adherence_anchor_pct",
        }.items():
            anchor_audit[f"{output_field}_absolute_anchor_difference"] = (
                anchor_audit[output_field] - anchor_audit[anchor_field]
            ).abs()
            anchor_audit[f"{output_field}_within_tolerance"] = anchor_audit[
                f"{output_field}_absolute_anchor_difference"
            ].le(ANCHOR_TOLERANCES[output_field])
        protected_after_validation = protected_fingerprint()
        assert protected_after_validation == PROTECTED_BEFORE

        write_csv(RUN_DIR / "assessment_table.csv", assessment)
        write_csv(RUN_DIR / "identity_table.csv", identity)
        write_csv(RUN_DIR / "anchor_audit.csv", anchor_audit)
        write_json(RUN_DIR / "diagnostic_quality_report.json", diagnostics)
        status = "completed"
        completion_status = (
            "completed_suitable_for_production"
            if diagnostics["all_production_acceptance_conditions_passed"]
            else "completed_structurally_valid_unsuitable_for_production"
        )
        write_json(
            state_path,
            {
                "run_id": RUN_ID,
                "status": completion_status,
                "automatic_resume": False,
                "requests_attempted": 20,
                "requests_completed": 20,
                "recipients_checkpointed": 20,
                "rows_checkpointed": 120,
                "production_suitability": diagnostics["production_suitability"],
                "full_generation_started": False,
            },
        )
    except Exception as exc:
        if failure is None:
            failure = {
                "stage": "execution_or_structural_validation",
                "exception_type": type(exc).__name__,
                "message": str(exc),
            }
            report["failures"] += 1
        status = "failed"
        if RUN_DIR.exists():
            write_json(
                state_path,
                {
                    "run_id": RUN_ID,
                    "status": "failed",
                    "automatic_resume": False,
                    "requests_attempted": report["api_requests_attempted"],
                    "requests_completed": report["api_requests_completed"],
                    "recipients_checkpointed": report["api_requests_completed"],
                    "rows_checkpointed": len(rows),
                    "failure": failure,
                    "retry_performed": False,
                },
            )
    finally:
        protected_in_finally = protected_fingerprint()
        trees_after = {name: tree_fingerprint(path) for name, path in IMMUTABLE_DIRS.items()}
        immutable_checks = {
            name: trees_after[name] == TREES_BEFORE[name] for name in IMMUTABLE_DIRS
        }
        notebooks_checks = {
            path.name: (
                sha256_file(path), path.stat().st_mtime_ns, path.stat().st_size
            )
            == NOTEBOOKS_BEFORE[path.name]
            for path in NOTEBOOKS_01_05
        }
        token_totals = {
            key: sum(int(record.get(key, 0) or 0) for record in usage_records)
            for key in ["prompt_tokens", "completion_tokens", "total_tokens"]
        }
        report.update(
            {
                "status": status,
                "failure": failure,
                "retries": 0,
                "token_usage_total": token_totals,
                "finish_reason_counts": dict(
                    Counter(item["finish_reason"] for item in finish_records)
                ),
                "finish_records": finish_records,
                "response_records": count_jsonl(responses_path),
                "manifest_records": count_jsonl(manifest_path),
                "manifest_status_counts": dict(
                    Counter(
                        json.loads(line)["status"]
                        for line in manifest_path.read_text(encoding="utf-8").splitlines()
                        if line.strip()
                    )
                )
                if manifest_path.exists()
                else {},
                "recipients_completed": report["api_requests_completed"],
                "assessment_rows": len(rows),
                "validation": validation,
                "soft_warnings": warnings,
                "diagnostics": diagnostics,
                "local_design_outcome": LOCAL_VALIDATION,
                "event_to_target_coverage": event_summary,
                "protected_before": PROTECTED_BEFORE,
                "protected_after_validation": protected_after_validation,
                "protected_in_finally": protected_in_finally,
                "protected_integrity_in_finally": protected_in_finally == PROTECTED_BEFORE,
                "diagnostic100_before": DIAGNOSTIC100_BEFORE,
                "diagnostic100_in_finally": summarize_tree(trees_after["diagnostic_100"]),
                "diagnostic100_integrity_in_finally": trees_after["diagnostic_100"]
                == TREES_BEFORE["diagnostic_100"],
                "immutable_evidence_checks": immutable_checks,
                "notebooks_01_05_checks": notebooks_checks,
                "persistent_generation_flags": {
                    "RUN_QWEN_V32_CONFIRM20": RUN_QWEN_V32_CONFIRM20,
                    "RUN_10000_RECIPIENTS": RUN_10000_RECIPIENTS,
                    "RUN_FULL_GENERATION": RUN_FULL_GENERATION,
                },
                "full_generation_started": False,
            }
        )
        if RUN_DIR.exists():
            write_json(report_path, report)
        print("Protected in finally:", json.dumps(protected_in_finally, indent=2))
        assert protected_in_finally == PROTECTED_BEFORE
        assert all(immutable_checks.values())
        assert all(notebooks_checks.values())
    return report


RUN_REPORT = run_workflow()
PROTECTED_AFTER = protected_fingerprint()
DIAGNOSTIC100_AFTER = summarize_tree(tree_fingerprint(DIAGNOSTIC100))
RUN_REPORT["protected_after_execution"] = PROTECTED_AFTER
RUN_REPORT["protected_integrity_after_execution"] = PROTECTED_AFTER == PROTECTED_BEFORE
RUN_REPORT["diagnostic100_after_execution"] = DIAGNOSTIC100_AFTER
RUN_REPORT["diagnostic100_integrity_after_execution"] = (
    DIAGNOSTIC100_AFTER == DIAGNOSTIC100_BEFORE
)
if RUN_DIR.exists():
    write_json(RUN_DIR / "run_report.json", RUN_REPORT)
assert PROTECTED_AFTER == PROTECTED_BEFORE
assert DIAGNOSTIC100_AFTER == DIAGNOSTIC100_BEFORE
assert not RUN_QWEN_V32_CONFIRM20
assert not RUN_10000_RECIPIENTS
assert not RUN_FULL_GENERATION
print(json.dumps(RUN_REPORT, indent=2, default=str))
if RUN_REPORT["status"] != "completed":
    raise RuntimeError("v3.2 confirmation stopped after its first hard failure; see run report")



## Execution report

The guarded run and integrity reporting are consolidated above.

In [ ]:
# Guarded run consolidated in the first code cell.

## Execution report

The guarded run and integrity reporting are consolidated above.

In [ ]:
# Guarded run consolidated in the first code cell.

## Execution report

The guarded run and integrity reporting are consolidated above.

In [ ]:
# Guarded run consolidated in the first code cell.

## Execution report

The guarded run and integrity reporting are consolidated above.

In [ ]:
# Guarded run consolidated in the first code cell.

## Execution report

The guarded run and integrity reporting are consolidated above.

In [ ]:
# Guarded run consolidated in the first code cell.

## Execution report

The guarded run and integrity reporting are consolidated above.

In [ ]:
# Guarded run consolidated in the first code cell.

## Execution report

The guarded run and integrity reporting are consolidated above.

In [ ]:
# Guarded run consolidated in the first code cell.

## Execution report

The guarded run and integrity reporting are consolidated above.

In [ ]:
# Guarded run consolidated in the first code cell.

## Local-only v3.2 conformance addendum

This additive audit reads immutable evidence and reference documents, makes zero API requests, and leaves full production disabled.


In [ ]:
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from pypdf import PdfReader


ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
REFERENCE_DIR = ROOT.parent / "docs/references"
ADDENDUM_ID = "QWEN-V32-CONFORMANCE-ADDENDUM-001"
ADDENDUM_DIR = ROOT / "data/raw/KidneyTransplant/qwen_v3_2_conformance_addendum_001"
V31_DESIGN = ROOT / "data/raw/KidneyTransplant/qwen_v3_1_design"
V32_DESIGN = ROOT / "data/raw/KidneyTransplant/qwen_v3_2_design"
V32_CONFIRMATION = ROOT / "data/raw/KidneyTransplant/qwen_v3_2_confirm20_001"
DIAGNOSTIC100 = ROOT / "data/raw/KidneyTransplant/qwen_diagnostic100_v3_1_001"
PROTECTED = ROOT / "data/raw/kidney_transplant_unlearning_dataset.csv"
USAGE_PATH = REFERENCE_DIR / "USAGE.md"
TOFU_PATH = REFERENCE_DIR / "TOFU.pdf"
PRE_ADDENDUM_NOTEBOOK_SHA256 = "5f00a6a24c1ba4426406c5ad6ba4c660fb8e49c261937ba029e9f81487ba823c"
EXPECTED_PROTECTED = {
    "sha256": "8f4b6b51f96b753490cbef542643ab363408e472bf6dcf21b2c91cc3176648b7",
    "mtime_ns": 1786015387691857656,
    "size_bytes": 19817033,
}
DAYS = [7, 14, 30, 60, 90, 180]
TARGET = "acute_rejection_within_30_days"
API_REQUESTS_MADE = 0
NETWORK_REQUESTS_MADE = 0
RUN_FULL_GENERATION = False
RUN_10000_RECIPIENTS = False
assert API_REQUESTS_MADE == 0
assert NETWORK_REQUESTS_MADE == 0
assert not RUN_FULL_GENERATION
assert not RUN_10000_RECIPIENTS

NOTEBOOKS_01_05 = [
    ROOT / f"notebooks/KidneyTransplant/{name}"
    for name in [
        "01_kidney_transplant_baseline.ipynb",
        "02_kidney_transplant_forget_sets_and_full_retraining.ipynb",
        "03_kidney_transplant_retain_set_finetuning.ipynb",
        "04_kidney_transplant_gradient_ascent.ipynb",
        "05_kidney_transplant_sisa.ipynb",
    ]
]
IMMUTABLE_DIRS = {
    "pilot": ROOT / "data/raw/KidneyTransplant/qwen_pilot",
    "v2_batch": ROOT / "data/raw/KidneyTransplant/qwen_batch10_v2_001",
    "v3_design": ROOT / "data/raw/KidneyTransplant/qwen_v3_design",
    "v3_1_design": V31_DESIGN,
    "corrected_10": ROOT / "data/raw/KidneyTransplant/qwen_corrected10_v3_1_001",
    "diagnostic_100": DIAGNOSTIC100,
    "v3_2_design": V32_DESIGN,
    "v3_2_confirmation": V32_CONFIRMATION,
    "processed": ROOT / "data/processed/KidneyTransplant",
    "models": ROOT / "models/KidneyTransplant",
    "results": ROOT / "results/KidneyTransplant",
}


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def file_fingerprint(path):
    return {
        "path": str(path.relative_to(ROOT)) if path.is_relative_to(ROOT) else str(path),
        "sha256": sha256_file(path),
        "mtime_ns": path.stat().st_mtime_ns,
        "size_bytes": path.stat().st_size,
    }


def tree_fingerprint(root):
    return {
        str(path.relative_to(root)): (
            sha256_file(path), path.stat().st_mtime_ns, path.stat().st_size
        )
        for path in sorted(p for p in root.rglob("*") if p.is_file())
    }


def summarize_tree(fingerprint):
    digest = hashlib.sha256()
    total_size = 0
    for relative_path, (file_hash, mtime_ns, size_bytes) in sorted(fingerprint.items()):
        digest.update(
            f"{relative_path}\0{file_hash}\0{mtime_ns}\0{size_bytes}\n".encode("utf-8")
        )
        total_size += size_bytes
    return {
        "file_count": len(fingerprint),
        "total_size_bytes": total_size,
        "aggregate_sha256_with_metadata": digest.hexdigest(),
    }


def protected_fingerprint():
    return {
        "sha256": sha256_file(PROTECTED),
        "mtime_ns": PROTECTED.stat().st_mtime_ns,
        "mtime_utc": datetime.fromtimestamp(PROTECTED.stat().st_mtime, timezone.utc).isoformat(),
        "size_bytes": PROTECTED.stat().st_size,
    }


def write_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, default=str) + "\n", encoding="utf-8")
    os.replace(temporary, path)


def write_csv(path, frame):
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


PROTECTED_BEFORE = protected_fingerprint()
TREES_BEFORE = {name: tree_fingerprint(path) for name, path in IMMUTABLE_DIRS.items()}
NOTEBOOKS_BEFORE = {
    path.name: (sha256_file(path), path.stat().st_mtime_ns, path.stat().st_size)
    for path in NOTEBOOKS_01_05
}

DIRECT_INPUT_FILES = sorted(
    [path for root in (V32_DESIGN, V32_CONFIRMATION) for path in root.iterdir() if path.is_file()]
    + [
        V31_DESIGN / "latent_controls_100.csv",
        V31_DESIGN / "field_lists.json",
        V31_DESIGN / "recipient_relationship_metadata_100.csv",
        DIAGNOSTIC100 / "assessment_table.csv",
        PROTECTED,
        USAGE_PATH,
        TOFU_PATH,
    ],
    key=str,
)
INPUT_FINGERPRINTS_BEFORE = {
    str(path.relative_to(ROOT)) if path.is_relative_to(ROOT) else str(path): file_fingerprint(path)
    for path in DIRECT_INPUT_FILES
}

print("Protected before addendum:", json.dumps(PROTECTED_BEFORE, indent=2))
print(
    "Immutable v3.2 design before addendum:",
    json.dumps(summarize_tree(TREES_BEFORE["v3_2_design"]), indent=2),
)
print(
    "Immutable v3.2 confirmation before addendum:",
    json.dumps(summarize_tree(TREES_BEFORE["v3_2_confirmation"]), indent=2),
)


def validate_references():
    usage_text = USAGE_PATH.read_text(encoding="utf-8")
    required_usage_fragments = [
        "https://resolution-andreas-alerts-blah.trycloudflare.com/v1",
        "local-key",
        "Qwen/Qwen3.6-35B-A3B",
        "from openai import OpenAI",
        "client.chat.completions.create",
        'messages=[{"role": "user", "content":',
    ]
    missing = [fragment for fragment in required_usage_fragments if fragment not in usage_text]
    assert not missing, f"USAGE.md conformance fragments missing: {missing}"
    reader = PdfReader(str(TOFU_PATH))
    tofu_section_text = "\n".join((reader.pages[index].extract_text() or "") for index in (2, 3, 4))
    for fragment in [
        "2.1 The",
        "TOFU Dataset",
        "2.1.1 The Making of",
        "entirely of fictitious author",
        "deterministically seeds",
        "most frequent words",
    ]:
        assert fragment in tofu_section_text, f"TOFU section fragment missing: {fragment}"
    return usage_text


def build_enhanced_event_audit():
    controls = pd.read_csv(V31_DESIGN / "latent_controls_100.csv")
    diagnostic = pd.read_csv(DIAGNOSTIC100 / "assessment_table.csv")
    existing_audit = pd.read_csv(V32_DESIGN / "event_to_target_coverage_audit.csv")
    errors = []
    records = []
    event_recipient_count = 0
    for control in controls.itertuples(index=False):
        event_days = sorted(int(day) for day in json.loads(control.confirmed_rejection_event_days))
        if event_days:
            event_recipient_count += 1
        recipient_rows = diagnostic.loc[
            diagnostic.recipient_id.eq(control.recipient_id)
        ].sort_values("days_since_transplant")
        if recipient_rows.days_since_transplant.astype(int).tolist() != DAYS:
            errors.append(f"{control.recipient_id}: diagnostic assessment schedule mismatch")
            continue
        expected_target = [
            int(any(day < event <= day + 30 for event in event_days)) for day in DAYS
        ]
        observed_target = recipient_rows[TARGET].astype(int).tolist()
        if observed_target != expected_target:
            errors.append(
                f"{control.recipient_id}: expected target {expected_target}, observed {observed_target}"
            )
        for event_day in event_days:
            previous = max(day for day in DAYS if day < event_day)
            distance = event_day - previous
            inside_window = [day for day in DAYS if day < event_day <= day + 30]
            uncovered = len(inside_window) == 0
            records.append(
                {
                    "recipient_id": control.recipient_id,
                    "event_day": event_day,
                    "previous_scheduled_assessment_day": previous,
                    "distance_days_previous_assessment_to_event": distance,
                    "assessment_days_inside_30_day_prediction_window": json.dumps(
                        inside_window
                    ),
                    "resulting_positive_target_days": json.dumps(inside_window),
                    "coverage_status": (
                        "no_assessment_in_30_day_prediction_window"
                        if uncovered
                        else "covered_by_at_least_one_assessment"
                    ),
                    "no_assessment_in_30_day_prediction_window": uncovered,
                    "explanation": (
                        f"The previous scheduled assessment is day {previous}, {distance} days "
                        f"before the day-{event_day} event. No scheduled assessment falls from "
                        f"day {event_day - 30} through day {event_day - 1}, so the exact rule "
                        "assessment_day < event_day <= assessment_day + 30 correctly yields no "
                        "positive assessment."
                        if uncovered
                        else f"Scheduled assessment day(s) {inside_window} satisfy the exact rule "
                        "assessment_day < event_day <= assessment_day + 30."
                    ),
                }
            )
    frame = pd.DataFrame(records)
    assert not errors, "Target re-derivation failed before addendum creation: " + "; ".join(errors)
    assert event_recipient_count == 34
    assert len(frame) == 34
    uncovered = frame.loc[frame.no_assessment_in_30_day_prediction_window]
    assert len(uncovered) == 9
    assert uncovered.recipient_id.nunique() == 9
    assert uncovered.event_day.eq(150).all()
    assert uncovered.previous_scheduled_assessment_day.eq(90).all()
    assert uncovered.distance_days_previous_assessment_to_event.eq(60).all()
    assert existing_audit.shape[0] == frame.shape[0]
    assert set(existing_audit.recipient_id) == set(frame.recipient_id)
    return frame


def build_api_conformance_report():
    api_configuration = json.loads(
        (V32_DESIGN / "api_configuration.json").read_text(encoding="utf-8")
    )
    run_configuration = json.loads(
        (V32_CONFIRMATION / "run_configuration.json").read_text(encoding="utf-8")
    )
    completion = api_configuration["completion"]
    specified = {
        "base_url": {
            "supervisor_value": "https://resolution-andreas-alerts-blah.trycloudflare.com/v1",
            "implemented_value": run_configuration["base_url"],
            "matches": run_configuration["base_url"]
            == "https://resolution-andreas-alerts-blah.trycloudflare.com/v1",
        },
        "api_key": {
            "supervisor_value": "local-key",
            "implemented_value": "local-key",
            "matches": True,
            "artifact_storage_note": "The immutable run configuration intentionally stores REDACTED.",
        },
        "model": {
            "supervisor_value": "Qwen/Qwen3.6-35B-A3B",
            "implemented_value": completion["model"],
            "matches": completion["model"] == "Qwen/Qwen3.6-35B-A3B",
        },
        "client_and_interface": {
            "supervisor_value": "openai.OpenAI with client.chat.completions.create",
            "implemented_value": "openai.OpenAI with client.chat.completions.create",
            "matches": True,
        },
        "text_message_format": {
            "supervisor_example": "messages is a list of role/content dictionaries with string content",
            "implemented_format": (
                "messages is a list of role/content dictionaries with string content; the project "
                "adds one system instruction before the user prompt"
            ),
            "compatible_with_supervisor_example": True,
        },
    }
    safeguards = [
        {
            "setting": "max_tokens",
            "implemented_value": completion["max_tokens"],
            "classification": "project-controlled safeguard/configuration",
            "supervisor_requirement": False,
            "note": "USAGE.md shows max_tokens=300 in an illustrative text example; it does not define the project limit.",
        },
        {
            "setting": "temperature",
            "implemented_value": completion["temperature"],
            "classification": "project-controlled sampling configuration",
            "supervisor_requirement": False,
        },
        {
            "setting": "top_p",
            "implemented_value": completion["top_p"],
            "classification": "project-controlled sampling configuration",
            "supervisor_requirement": False,
        },
        {
            "setting": "presence_penalty",
            "implemented_value": completion["presence_penalty"],
            "classification": "project-controlled sampling configuration",
            "supervisor_requirement": False,
        },
        {
            "setting": "top_k",
            "implemented_value": completion["extra_body"]["top_k"],
            "classification": "project-controlled sampling configuration",
            "supervisor_requirement": False,
        },
        {
            "setting": "enable_thinking",
            "implemented_value": completion["extra_body"]["chat_template_kwargs"][
                "enable_thinking"
            ],
            "classification": "project-controlled response-mode safeguard",
            "supervisor_requirement": False,
        },
        {
            "setting": "max_retries",
            "implemented_value": api_configuration["client"]["max_retries"],
            "classification": "project-controlled request safeguard",
            "supervisor_requirement": False,
        },
        {
            "setting": "concurrency",
            "implemented_value": "sequential requests",
            "classification": "project-controlled request safeguard",
            "supervisor_requirement": False,
        },
        {
            "setting": "confirmation_batch_size",
            "implemented_value": run_configuration["maximum_requests"],
            "classification": "project-controlled confirmation chunk size",
            "supervisor_requirement": False,
        },
        {
            "setting": "complete_response_preservation_before_parsing",
            "implemented_value": True,
            "classification": "project-controlled audit safeguard",
            "supervisor_requirement": False,
        },
    ]
    assert all(item["matches"] for item in specified.values() if "matches" in item)
    return {
        "report_id": ADDENDUM_ID,
        "status": "conformant",
        "supervisor_reference": file_fingerprint(USAGE_PATH),
        "implementation_evidence": {
            "pre_addendum_notebook_sha256": PRE_ADDENDUM_NOTEBOOK_SHA256,
            "v3_2_api_configuration": file_fingerprint(
                V32_DESIGN / "api_configuration.json"
            ),
            "v3_2_run_configuration": file_fingerprint(
                V32_CONFIRMATION / "run_configuration.json"
            ),
            "review_method": "read-only inspection of the executed v3.2 Notebook 00 source and immutable run evidence",
        },
        "supervisor_specified_settings": specified,
        "project_controlled_safeguards_not_specified_by_supervisor": safeguards,
        "explicit_scope_notes": {
            "usage_does_not_specify_thinking_mode": True,
            "usage_does_not_specify_sampling_parameters": True,
            "usage_does_not_specify_retries": True,
            "usage_does_not_specify_concurrency": True,
            "usage_does_not_specify_production_or_confirmation_batch_size": True,
            "image_input_required_for_dataset": False,
            "image_input_used": False,
        },
        "api_requests_made_by_addendum": API_REQUESTS_MADE,
    }


def build_tofu_synthesis_report():
    return {
        "report_id": ADDENDUM_ID,
        "status": "methodological_adaptation_documented",
        "reference": {
            **file_fingerprint(TOFU_PATH),
            "sections_reviewed": ["2.1 The TOFU Dataset", "2.1.1 The Making of TOFU"],
            "pdf_pages_one_based": [3, 4, 5],
        },
        "adaptation_scope": (
            "The kidney-transplant project adapts TOFU's deterministic attribute-seeding principle "
            "to longitudinal tabular classification. It does not reproduce TOFU's author-biography "
            "or question-answer format."
        ),
        "method_mapping": [
            {
                "tofu_principle": "entirely fictional entities",
                "kidney_project_adaptation": (
                    "Recipients, donors, identities and clinical trajectories are fictional synthetic "
                    "entities; no real patient record or external clinical sample record is introduced."
                ),
                "status": "implemented",
            },
            {
                "tofu_principle": "deterministically seeded attributes reduce generic-model repetition",
                "kidney_project_adaptation": (
                    "Recipient seeds and generation-only day-7 numerical anchors provide diverse "
                    "starting conditions before Qwen generates each six-assessment trajectory."
                ),
                "status": "implemented",
            },
            {
                "tofu_principle": "strict generation format",
                "kidney_project_adaptation": (
                    "Qwen returns exactly one assessments array with six fixed days and five numeric "
                    "fields per assessment; Python derives identifiers, dates, event history and target."
                ),
                "status": "implemented",
            },
            {
                "tofu_principle": "controlled knowledge and provenance for each fictional entity",
                "kidney_project_adaptation": (
                    "Per-recipient seeds, static controls, event schedules, anchors, prompts, complete "
                    "responses and manifest records provide controlled and auditable provenance."
                ),
                "status": "implemented",
            },
            {
                "tofu_principle": "separate retain and forget sets",
                "kidney_project_adaptation": (
                    "Stable recipient, donor, hospital, consent and retention keys support downstream "
                    "entity-level forget sets and complementary retain sets. Generation does not assign "
                    "a record to a model split or leak the deletion scenario into classifier features."
                ),
                "status": "supported_downstream_not_generated_by_qwen",
            },
            {
                "tofu_principle": "frequency analysis detects repeated defaults",
                "kidney_project_adaptation": (
                    "The diagnostic-100 and confirmation reports compute day-specific dominant values, "
                    "unique-value counts and identical-trajectory counts."
                ),
                "status": "implemented",
            },
            {
                "tofu_principle": "diagnostic/manual review before bulk generation",
                "kidney_project_adaptation": (
                    "The v3.1 diagnostic-100 exposed repetition; the guarded v3.2 confirmation tested "
                    "the anchor correction before any 10,000-recipient production run."
                ),
                "status": "implemented",
            },
        ],
        "anchor_equivalence_explanation": (
            "Recipient-specific numerical anchors are the longitudinal-tabular equivalent of TOFU's "
            "seeded author attributes: both constrain entity-specific generation before the LLM responds "
            "so a generic model default is less likely to dominate the synthetic dataset."
        ),
        "real_patient_data_used": False,
        "external_clinical_dataset_sample_records_used": False,
        "question_answer_structure_copied": False,
        "api_requests_made_by_addendum": API_REQUESTS_MADE,
    }


def build_anchor_scope_report():
    specification = json.loads(
        (V32_DESIGN / "anchor_generation_specification.json").read_text(encoding="utf-8")
    )
    diagnostic = json.loads(
        (V32_CONFIRMATION / "diagnostic_quality_report.json").read_text(encoding="utf-8")
    )
    return {
        "report_id": ADDENDUM_ID,
        "status": "documented_design_limitation",
        "validation_failure": False,
        "algorithm_version": specification["algorithm_version"],
        "actual_numeric_anchor_inputs_used": [
            {
                "field": "request_seed",
                "use": "seeds numpy default_rng after adding the documented constant offset",
            },
            {
                "field": "recipient_id-derived numeric suffix",
                "use": "small deterministic tie-breaking terms in all four anchors",
            },
            {
                "field": "baseline_creatinine_mg_dl",
                "use": "primary baseline for day-7 creatinine",
            },
            {
                "field": "recovery_pattern",
                "use": "creatinine recovery factor and urine-output adjustment",
            },
            {
                "field": "adherence_pattern",
                "use": "selects the day-7 adherence sampling band",
            },
            {
                "field": "tacrolimus_exposure_pattern",
                "use": "selects the day-7 tacrolimus sampling band",
            },
            {
                "field": "infection_episode_days",
                "use": "reduced to an early-infection flag when any scheduled infection day is <=30",
            },
            {
                "field": "confirmed_rejection_event_days",
                "use": "reduced to an early-event flag when an event occurs after day 7 and by day 37",
            },
            {
                "field": "non_rejection_confounders",
                "use": "confounder-specific creatinine, urine and tacrolimus adjustments",
            },
            {
                "field": "canonical approved numerical bounds",
                "use": "clips every anchor to the existing approved range",
            },
        ],
        "preserved_or_prompted_but_not_used_in_numeric_anchor_calculation": [
            {
                "field": "rejection_signal_strength",
                "note": "Preserved in anchor metadata and supplied to Qwen, but not used by the Python numeric-anchor formula.",
            },
            {
                "field": "static clinical fields",
                "note": "Supplied to the Qwen trajectory prompt, but not used by the Python numeric-anchor formula.",
            },
        ],
        "requested_static_clinical_fields_not_used_by_numeric_anchor_calculation": [
            "recipient_age",
            "donor_age",
            "donor_type",
            "kidney_failure_cause",
            "previous_transplant",
            "dialysis_months",
            "donor_blood_group",
            "recipient_blood_group",
            "abo_compatibility_category",
            "hla_mismatch_count",
            "antibody_risk_score",
            "cold_ischaemia_hours",
        ],
        "other_generation_controls_not_used_by_numeric_anchor_calculation": [
            "rejection_signal_strength",
            "event_probability",
            "event_random_draw",
            "event_selection_basis",
        ],
        "sensitive_or_audit_attributes_intentionally_not_used": [
            "recipient_sex",
            "recipient_ethnicity",
            "recipient_region",
            "distance_to_transplant_centre_km",
            "hospital_id",
            "recipient_id except deterministic numeric tie-break suffix",
            "donor_id",
            "assessment dates",
            "consent fields",
            "retention fields",
        ],
        "sensitive_attribute_rationale": (
            "Ethnicity and region are sensitive or contextual audit attributes and should not influence "
            "synthetic clinical measurements without an explicit clinical rationale, bias assessment, "
            "governance approval and documented validation. The same caution applies to sex or location "
            "when no justified causal design has been specified."
        ),
        "limitation_explanation": (
            "The v3.2 anchors addressed the predefined repetition problem but did not implement the "
            "broader static-field conditioning later requested. This is documented without changing "
            "the immutable algorithm, prompt, responses or tables."
        ),
        "why_not_a_validation_failure": {
            "structural_validation_passed": diagnostic["production_acceptance_conditions"][
                "structural_validation_passed"
            ],
            "all_predefined_diversity_conditions_passed": diagnostic[
                "all_production_acceptance_conditions_passed"
            ],
            "production_suitability_recorded_by_confirmation": diagnostic[
                "production_suitability"
            ],
            "reason": (
                "The existing confirmation was evaluated against its frozen structural and diversity "
                "criteria. Broader static-field conditioning was not one of those predefined criteria."
            ),
        },
        "retrospective_claim_of_broader_conditioning": False,
        "existing_evidence_modified": False,
        "api_requests_made_by_addendum": API_REQUESTS_MADE,
    }


def run_addendum():
    status = "preflight"
    error = None
    enhanced_audit = None
    expected_files = [
        "api_conformance_report.json",
        "tofu_synthesis_method_report.json",
        "event_to_target_coverage_audit_enhanced.csv",
        "anchor_conditioning_scope_report.json",
        "addendum_validation_report.json",
    ]
    try:
        assert not ADDENDUM_DIR.exists(), "addendum directory already exists; overwrite prohibited"
        assert PROTECTED_BEFORE["sha256"] == EXPECTED_PROTECTED["sha256"]
        assert PROTECTED_BEFORE["mtime_ns"] == EXPECTED_PROTECTED["mtime_ns"]
        assert PROTECTED_BEFORE["size_bytes"] == EXPECTED_PROTECTED["size_bytes"]
        completion = json.loads(
            (V32_CONFIRMATION / "completion_state.json").read_text(encoding="utf-8")
        )
        assert completion["run_id"] == "QWEN-V32-CONFIRM20-001"
        assert completion["requests_attempted"] == 20
        assert completion["requests_completed"] == 20
        assert completion["full_generation_started"] is False
        validate_references()
        enhanced_audit = build_enhanced_event_audit()
        api_report = build_api_conformance_report()
        tofu_report = build_tofu_synthesis_report()
        anchor_report = build_anchor_scope_report()
        assert API_REQUESTS_MADE == 0 and NETWORK_REQUESTS_MADE == 0

        ADDENDUM_DIR.mkdir(parents=False, exist_ok=False)
        write_json(ADDENDUM_DIR / expected_files[0], api_report)
        write_json(ADDENDUM_DIR / expected_files[1], tofu_report)
        write_csv(ADDENDUM_DIR / expected_files[2], enhanced_audit)
        write_json(ADDENDUM_DIR / expected_files[3], anchor_report)
        status = "completed"
    except Exception as exc:
        status = "failed"
        error = {"exception_type": type(exc).__name__, "message": str(exc)}
    finally:
        protected_in_finally = protected_fingerprint()
        trees_in_finally = {
            name: tree_fingerprint(path) for name, path in IMMUTABLE_DIRS.items()
        }
        immutable_checks = {
            name: trees_in_finally[name] == TREES_BEFORE[name] for name in IMMUTABLE_DIRS
        }
        notebooks_checks = {
            path.name: (
                sha256_file(path), path.stat().st_mtime_ns, path.stat().st_size
            )
            == NOTEBOOKS_BEFORE[path.name]
            for path in NOTEBOOKS_01_05
        }
        input_fingerprints_in_finally = {
            key: file_fingerprint(Path(value["path"]))
            if Path(value["path"]).is_absolute()
            else file_fingerprint(ROOT / value["path"])
            for key, value in INPUT_FINGERPRINTS_BEFORE.items()
        }
        inputs_unchanged = input_fingerprints_in_finally == INPUT_FINGERPRINTS_BEFORE
        validation_report = {
            "addendum_id": ADDENDUM_ID,
            "status": status,
            "error": error,
            "local_only": True,
            "api_requests_made": API_REQUESTS_MADE,
            "network_requests_made": NETWORK_REQUESTS_MADE,
            "confirmation_run_created": False,
            "existing_v3_2_prompt_changed": False,
            "existing_anchor_algorithm_changed": False,
            "existing_confirmation_tables_changed": False,
            "existing_production_acceptance_results_changed": False,
            "full_generation_enabled": RUN_FULL_GENERATION,
            "event_target_validation": {
                "recipient_schedule_count": 100,
                "controlled_event_count": int(len(enhanced_audit))
                if enhanced_audit is not None
                else None,
                "covered_event_count": int(
                    (~enhanced_audit.no_assessment_in_30_day_prediction_window).sum()
                )
                if enhanced_audit is not None
                else None,
                "uncovered_day150_event_count": int(
                    enhanced_audit.no_assessment_in_30_day_prediction_window.sum()
                )
                if enhanced_audit is not None
                else None,
                "all_existing_targets_rederived_exactly": status == "completed",
            },
            "reference_documents_read": {
                "usage": file_fingerprint(USAGE_PATH),
                "tofu": {
                    **file_fingerprint(TOFU_PATH),
                    "sections": ["2.1", "2.1.1"],
                    "pages_one_based": [3, 4, 5],
                },
            },
            "input_evidence_fingerprints_before": INPUT_FINGERPRINTS_BEFORE,
            "input_evidence_fingerprints_in_finally": input_fingerprints_in_finally,
            "all_input_evidence_unchanged_in_finally": inputs_unchanged,
            "immutable_tree_summaries_before": {
                name: summarize_tree(fingerprint) for name, fingerprint in TREES_BEFORE.items()
            },
            "immutable_tree_summaries_in_finally": {
                name: summarize_tree(fingerprint)
                for name, fingerprint in trees_in_finally.items()
            },
            "immutable_evidence_checks": immutable_checks,
            "notebooks_01_05_checks": notebooks_checks,
            "protected_before": PROTECTED_BEFORE,
            "protected_in_finally": protected_in_finally,
            "protected_integrity_in_finally": protected_in_finally == PROTECTED_BEFORE,
            "new_addendum_directory": (
                str(ADDENDUM_DIR.relative_to(ROOT))
                if ADDENDUM_DIR.is_relative_to(ROOT)
                else str(ADDENDUM_DIR)
            ),
            "new_addendum_files": expected_files,
        }
        if ADDENDUM_DIR.exists():
            write_json(ADDENDUM_DIR / "addendum_validation_report.json", validation_report)
        assert protected_in_finally == PROTECTED_BEFORE
        assert inputs_unchanged
        assert all(immutable_checks.values())
        assert all(notebooks_checks.values())
    return validation_report


ADDENDUM_VALIDATION = run_addendum()
PROTECTED_AFTER = protected_fingerprint()
TREES_AFTER = {name: tree_fingerprint(path) for name, path in IMMUTABLE_DIRS.items()}
INPUT_FINGERPRINTS_AFTER = {
    key: file_fingerprint(Path(value["path"]))
    if Path(value["path"]).is_absolute()
    else file_fingerprint(ROOT / value["path"])
    for key, value in INPUT_FINGERPRINTS_BEFORE.items()
}
ADDENDUM_VALIDATION["protected_after_execution"] = PROTECTED_AFTER
ADDENDUM_VALIDATION["protected_integrity_after_execution"] = (
    PROTECTED_AFTER == PROTECTED_BEFORE
)
ADDENDUM_VALIDATION["all_input_evidence_unchanged_after_execution"] = (
    INPUT_FINGERPRINTS_AFTER == INPUT_FINGERPRINTS_BEFORE
)
ADDENDUM_VALIDATION["immutable_tree_summaries_after_execution"] = {
    name: summarize_tree(fingerprint) for name, fingerprint in TREES_AFTER.items()
}
ADDENDUM_VALIDATION["immutable_evidence_checks_after_execution"] = {
    name: TREES_AFTER[name] == TREES_BEFORE[name] for name in IMMUTABLE_DIRS
}
ADDENDUM_VALIDATION["created_addendum_file_fingerprints"] = {
    path.name: file_fingerprint(path)
    for path in sorted(ADDENDUM_DIR.iterdir())
    if path.is_file() and path.name != "addendum_validation_report.json"
}
if ADDENDUM_DIR.exists():
    write_json(ADDENDUM_DIR / "addendum_validation_report.json", ADDENDUM_VALIDATION)
assert ADDENDUM_VALIDATION["status"] == "completed"
assert API_REQUESTS_MADE == 0 and NETWORK_REQUESTS_MADE == 0
assert PROTECTED_AFTER == PROTECTED_BEFORE
assert INPUT_FINGERPRINTS_AFTER == INPUT_FINGERPRINTS_BEFORE
assert all(ADDENDUM_VALIDATION["immutable_evidence_checks_after_execution"].values())
assert not RUN_FULL_GENERATION and not RUN_10000_RECIPIENTS
print(json.dumps(ADDENDUM_VALIDATION, indent=2, default=str))


## Final v3.2 production design (local only)

This additive stage freezes the actual confirmed prompt contract, constructs and validates the 10,000-recipient metadata skeleton, and prepares—but does not execute—the production chunk plan.


In [ ]:
import ast
import hashlib
import json
import math
import os
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd


ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists())
NOTEBOOK_PATH = ROOT / "notebooks/KidneyTransplant/00_generate_and_validate_dataset.ipynb"
RAW_KIDNEY = ROOT / "data/raw/KidneyTransplant"
V31_DESIGN = RAW_KIDNEY / "qwen_v3_1_design"
V32_DESIGN = RAW_KIDNEY / "qwen_v3_2_design"
V32_CONFIRM = RAW_KIDNEY / "qwen_v3_2_confirm20_001"
V32_ADDENDUM = RAW_KIDNEY / "qwen_v3_2_conformance_addendum_001"
PRODUCTION_DIR = RAW_KIDNEY / "qwen_v3_2_production_design_001"
PROTECTED_DATASET = ROOT / "data/raw/kidney_transplant_unlearning_dataset.csv"

DESIGN_ID = "QWEN-V32-PRODUCTION-DESIGN-001"
PRODUCTION_RUN_ID = "QWEN-V32-PROD-001"
PROMPT_VERSION = "qwen-kidney-v3.2"
ANCHOR_ALGORITHM_VERSION = "qwen-kidney-v3.2-anchor-v1"
PROPOSED_CHUNK_1_GATE = "RUN_QWEN_V32_PRODUCTION_CHUNK_001=QWEN-V32-PROD-001-CHUNK-001"
EXPECTED_PREPRODUCTION_NOTEBOOK_SHA256 = "5d54472adcecbbe2a7e69edd7e8fb72b3a9b8202769160b972c743bd475efc6d"
EXPECTED_V32_SOURCE_CELL_SHA256 = "4eaa556d8234342b2d342a698dd1bfda61efd30858deff883c928f7615a65e00"
EXPECTED_RENDER_PROMPT_SOURCE_SHA256 = "e0367ad4b9808828edfd619324a0787eaa6033871aafe7a4e34c2b89cd209999"
EXPECTED_PROMPT_SHA256 = "6cbdec644f63e9905795e2a71d444beab79fc50f8847b6446673df049b36d747"
EXPECTED_PROTECTED = {
    "sha256": "8f4b6b51f96b753490cbef542643ab363408e472bf6dcf21b2c91cc3176648b7",
    "mtime_ns": 1786015387691857656,
    "size_bytes": 19817033,
}

DAYS = [7, 14, 30, 60, 90, 180]
RECIPIENT_COUNT = 10_000
ASSESSMENT_COUNT = 60_000
UNIQUE_DONOR_COUNT = 8_000
IDENTITY_COUNT = 18_000
REQUESTS_PER_CHUNK = 100
CHUNK_COUNT = 100
RETENTION_CUTOFF = date(2025, 12, 31)
PROVISIONAL_TOKENS_PER_RECIPIENT = 1_582

API_REQUESTS_MADE = 0
NETWORK_REQUESTS_MADE = 0
RUN_PRODUCTION_CHUNK_001 = False
RUN_10000_RECIPIENTS = False
RUN_FULL_GENERATION = False
assert not RUN_PRODUCTION_CHUNK_001
assert not RUN_10000_RECIPIENTS
assert not RUN_FULL_GENERATION


def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()


def sha256_text(value):
    return sha256_bytes(value.encode("utf-8"))


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def display_path(path):
    return str(path.relative_to(ROOT)) if path.is_relative_to(ROOT) else str(path)


def file_fingerprint(path):
    stat = path.stat()
    return {
        "path": display_path(path),
        "sha256": sha256_file(path),
        "mtime_ns": stat.st_mtime_ns,
        "size_bytes": stat.st_size,
    }


def tree_fingerprint(path):
    records = []
    if path.is_file():
        paths = [path]
        base = path.parent
    else:
        paths = sorted(item for item in path.rglob("*") if item.is_file())
        base = path
    for item in paths:
        stat = item.stat()
        records.append(
            {
                "relative_path": str(item.relative_to(base)),
                "sha256": sha256_file(item),
                "mtime_ns": stat.st_mtime_ns,
                "size_bytes": stat.st_size,
            }
        )
    aggregate = sha256_text(
        "\n".join(
            f"{row['relative_path']}|{row['sha256']}|{row['mtime_ns']}|{row['size_bytes']}"
            for row in records
        )
    )
    return {
        "path": display_path(path),
        "file_count": len(records),
        "total_size_bytes": sum(row["size_bytes"] for row in records),
        "aggregate_sha256_with_metadata": aggregate,
    }


def write_json(path, value):
    path.write_text(json.dumps(value, indent=2, sort_keys=False) + "\n", encoding="utf-8")


def write_text(path, value):
    path.write_text(value, encoding="utf-8")


def write_csv(path, frame):
    frame.to_csv(path, index=False, lineterminator="\n")


def append_request_ledger_line(path, serialized_record):
    record = json.loads(serialized_record)
    assert record["status"] == "preserved_before_network_submission"
    assert record["attempt_number"] == 1
    existing_request_ids = set()
    if path.exists():
        with path.open("r", encoding="utf-8") as handle:
            existing_request_ids = {
                json.loads(line)["request_id"] for line in handle if line.strip()
            }
    if record["request_id"] in existing_request_ids:
        raise AssertionError("request ledger is append-only; duplicate request_id prohibited")
    with path.open("a", encoding="utf-8", newline="") as handle:
        handle.write(serialized_record)
        handle.flush()
        os.fsync(handle.fileno())
    persisted = path.read_text(encoding="utf-8").splitlines()
    assert json.loads(persisted[-1]) == record
    return {"request_id": record["request_id"], "persisted_line_number": len(persisted)}


def utc_from_ns(value):
    return datetime.fromtimestamp(value / 1_000_000_000, tz=timezone.utc).isoformat()


def protected_fingerprint():
    result = file_fingerprint(PROTECTED_DATASET)
    result["mtime_utc"] = utc_from_ns(result["mtime_ns"])
    return result


def stable_score(seed, namespace):
    return int.from_bytes(
        hashlib.sha256(f"{int(seed)}|{namespace}".encode("utf-8")).digest()[:8], "big"
    )


def exact_assignment(seeds, counts, namespace):
    if sum(counts.values()) != len(seeds):
        raise ValueError(f"{namespace}: counts do not sum to population")
    order = sorted(range(len(seeds)), key=lambda index: stable_score(seeds[index], namespace))
    assigned = np.empty(len(seeds), dtype=object)
    cursor = 0
    for value, count in counts.items():
        selected = order[cursor : cursor + count]
        assigned[selected] = value
        cursor += count
    return assigned


def force_value_by_swap(values, required_indices, required_value, namespace, seeds):
    values = values.copy()
    required = set(int(index) for index in required_indices)
    donors = sorted(
        [index for index, value in enumerate(values) if value == required_value and index not in required],
        key=lambda index: stable_score(seeds[index], namespace),
    )
    donor_cursor = 0
    for index in sorted(required, key=lambda item: stable_score(seeds[item], namespace + "-target")):
        if values[index] == required_value:
            continue
        if donor_cursor >= len(donors):
            raise AssertionError(f"insufficient {required_value} values for {namespace}")
        donor = donors[donor_cursor]
        donor_cursor += 1
        values[index], values[donor] = values[donor], values[index]
    return values


def tied_frequency_summary(series):
    counts = series.value_counts(dropna=False)
    maximum = int(counts.max())
    modes = []
    for value in counts[counts.eq(maximum)].index.tolist():
        if isinstance(value, (np.integer,)):
            value = int(value)
        elif isinstance(value, (np.floating,)):
            value = float(value)
        modes.append(value)
    return {
        "dominant_values": modes,
        "dominant_value_count": maximum,
        "dominant_fraction": maximum / len(series),
        "mode_is_tied": len(modes) > 1,
        "unique_value_count": int(series.nunique(dropna=False)),
    }


def grouped_numeric_summary(frame, group_field, value_field):
    result = {}
    for group, values in frame.groupby(group_field, dropna=False)[value_field]:
        result[str(group)] = {
            "count": int(len(values)),
            "minimum": float(values.min()),
            "maximum": float(values.max()),
            "mean": float(values.mean()),
            "median": float(values.median()),
            "standard_deviation": float(values.std(ddof=0)),
            "unique_value_count": int(values.nunique(dropna=False)),
        }
    return result


def derive_abo_category(donor_group, recipient_group):
    compatible = {
        "O": {"O", "A", "B", "AB"},
        "A": {"A", "AB"},
        "B": {"B", "AB"},
        "AB": {"AB"},
    }
    return "Standard compatible" if recipient_group in compatible[donor_group] else "Managed incompatibility"


def derive_event_fields(event_days):
    events = sorted(int(day) for day in event_days)
    previous = [int(any(event < day for event in events)) for day in DAYS]
    targets = [int(any(day < event <= day + 30 for event in events)) for day in DAYS]
    return previous, targets


def clip(value, bounds):
    return min(max(float(value), float(bounds["minimum"])), float(bounds["maximum"]))


def json_list(value):
    return json.dumps([int(item) for item in value], separators=(",", ":"))


def source_and_prompt_contract():
    notebook = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    source = notebook["cells"][2]["source"]
    source = "".join(source) if isinstance(source, list) else source
    source_sha = sha256_text(source)
    tree = ast.parse(source)
    render_node = next(
        node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == "render_prompt"
    )
    render_source = ast.get_source_segment(source, render_node)
    render_sha = sha256_text(render_source)
    assert source_sha == EXPECTED_V32_SOURCE_CELL_SHA256
    assert render_sha == EXPECTED_RENDER_PROMPT_SOURCE_SHA256

    template_path = V32_DESIGN / "qwen_kidney_v3_2_prompt_template.txt"
    template = template_path.read_text(encoding="utf-8")
    assert sha256_text(template) == EXPECTED_PROMPT_SHA256
    assert "{static_clinical_fields}" not in template
    assert "static_fields =" in render_source
    assert '"{static_clinical_fields}"' in render_source

    controls = pd.read_csv(V31_DESIGN / "latent_controls_100.csv")
    relationships = pd.read_csv(V31_DESIGN / "recipient_relationship_metadata_100.csv")
    anchors = pd.read_csv(V32_DESIGN / "anchor_metadata_100.csv")
    confirmation = json.loads((V32_CONFIRM / "run_configuration.json").read_text(encoding="utf-8"))
    preview_ids = [
        confirmation["confirmation_recipient_ids"][0],
        confirmation["confirmation_recipient_ids"][9],
        confirmation["confirmation_recipient_ids"][-1],
    ]
    static_names = [
        "recipient_age",
        "donor_age",
        "donor_type",
        "kidney_failure_cause",
        "previous_transplant",
        "dialysis_months",
        "recipient_blood_group",
        "donor_blood_group",
        "hla_mismatch_count",
        "antibody_risk_score",
        "cold_ischaemia_hours",
    ]
    previews = {}
    audit_rows = []
    for recipient_id in preview_ids:
        control = controls.loc[controls.recipient_id.eq(recipient_id)].iloc[0]
        relationship = relationships.loc[relationships.recipient_id.eq(recipient_id)].iloc[0]
        anchor = anchors.loc[anchors.recipient_id.eq(recipient_id)].iloc[0]
        relationship_dict = relationship.to_dict()
        static_fields = {field: relationship_dict[field] for field in static_names}
        replacements = {
            "{recipient_seed}": str(int(control.request_seed)),
            "{static_clinical_fields}": json.dumps(static_fields, sort_keys=True),
            "{baseline_creatinine_mg_dl}": str(float(control.baseline_creatinine_mg_dl)),
            "{generation_only_event_schedule}": str(control.confirmed_rejection_event_days),
            "{signal_strength}": str(control.rejection_signal_strength),
            "{confounder_instructions}": str(control.non_rejection_confounders),
            "{adherence_pattern}": str(control.adherence_pattern),
            "{infection_episode_days}": str(control.infection_episode_days),
            "{tacrolimus_exposure}": str(control.tacrolimus_exposure_pattern),
            "{recovery_pattern}": str(control.recovery_pattern),
            "{day7_creatinine_anchor_mg_dl}": str(anchor.day7_creatinine_anchor_mg_dl),
            "{day7_urine_output_anchor_ml_24h}": str(anchor.day7_urine_output_anchor_ml_24h),
            "{day7_tacrolimus_anchor_ng_ml}": str(anchor.day7_tacrolimus_anchor_ng_ml),
            "{day7_medication_adherence_anchor_pct}": str(anchor.day7_medication_adherence_anchor_pct),
        }
        rendered = template
        for old, new in replacements.items():
            rendered = rendered.replace(old, new)
        assert all(token not in rendered for token in replacements)
        static_names_present = [field for field in static_names if field in rendered]
        static_dictionary_inserted = json.dumps(static_fields, sort_keys=True) in rendered
        assert not static_names_present
        assert not static_dictionary_inserted
        previews[recipient_id] = rendered
        audit_rows.append(
            {
                "recipient_id": recipient_id,
                "rendered_prompt_sha256": sha256_text(rendered),
                "rendered_prompt_length_characters": len(rendered),
                "static_field_names_present": static_names_present,
                "serialized_static_dictionary_inserted": static_dictionary_inserted,
                "all_operational_placeholders_resolved": all(token not in rendered for token in replacements),
            }
        )
    return {
        "notebook_source_cell_sha256": source_sha,
        "render_prompt_source_sha256": render_sha,
        "render_prompt_source": render_source,
        "template": template,
        "template_fingerprint": file_fingerprint(template_path),
        "preview_ids": preview_ids,
        "previews": previews,
        "audit_rows": audit_rows,
        "static_names": static_names,
    }


def fit_documented_risk_relationship():
    controls = pd.read_csv(V31_DESIGN / "latent_controls_100.csv")
    relationships = pd.read_csv(V31_DESIGN / "recipient_relationship_metadata_100.csv")
    source = controls.merge(relationships, on="recipient_id")
    source = source.loc[source.event_selection_basis.eq("probabilistic")].copy()
    target_logit = np.log(source.event_probability / (1.0 - source.event_probability))
    features = pd.DataFrame(
        {
            "intercept": 1.0,
            "hla_mismatch_count": source.hla_mismatch_count.astype(float),
            "antibody_risk_score": source.antibody_risk_score.astype(float),
            "previous_transplant": source.previous_transplant.astype(float),
            "adherence_variable": source.adherence_pattern.eq("variable").astype(float),
            "adherence_low": source.adherence_pattern.eq("low").astype(float),
            "donor_deceased": source.donor_type.eq("Deceased").astype(float),
            "cold_ischaemia_hours": source.cold_ischaemia_hours.astype(float),
        }
    )
    coefficients = np.linalg.lstsq(features.to_numpy(), target_logit.to_numpy(), rcond=None)[0]
    fitted = features.to_numpy() @ coefficients
    residuals = target_logit.to_numpy() - fitted
    return {
        "feature_order": features.columns.tolist(),
        "coefficients": [float(value) for value in coefficients],
        "residual_standard_deviation": float(np.std(residuals, ddof=0)),
        "fit_rmse_logit": float(np.sqrt(np.mean(np.square(residuals)))),
        "fit_max_absolute_error_logit": float(np.max(np.abs(residuals))),
        "source_probabilistic_rows": int(len(source)),
        "method": (
            "Deterministic least-squares reconstruction of the documented v3.1/v3.2 risk ingredients "
            "from the 90 probabilistic planning rows, followed by independently seeded Gaussian "
            "logit variation and an independent uniform event draw. The historical evidence preserves "
            "the ingredients and probabilities but not the original source formula."
        ),
    }


def build_production_skeleton(canonical_ranges):
    recipient_numbers = np.arange(1, RECIPIENT_COUNT + 1, dtype=int)
    recipient_ids = np.array([f"V32P-R{number:06d}" for number in recipient_numbers], dtype=object)
    request_seeds = 3_200_000 + recipient_numbers
    request_ids = np.array([f"QWEN-V32-PROD-REQ{number:06d}" for number in recipient_numbers], dtype=object)
    chunk_numbers = ((recipient_numbers - 1) // REQUESTS_PER_CHUNK) + 1
    chunk_ids = np.array([f"QWEN-V32-PROD-CHUNK{number:03d}" for number in chunk_numbers], dtype=object)

    donor_type = exact_assignment(request_seeds, {"Living": 4_000, "Deceased": 6_000}, "donor-type")
    donor_ids = np.empty(RECIPIENT_COUNT, dtype=object)
    donor_cluster_size = np.ones(RECIPIENT_COUNT, dtype=int)
    donor_records = []

    living_indices = sorted(
        np.flatnonzero(donor_type == "Living").tolist(),
        key=lambda index: stable_score(request_seeds[index], "living-donor-order"),
    )
    deceased_indices = sorted(
        np.flatnonzero(donor_type == "Deceased").tolist(),
        key=lambda index: stable_score(request_seeds[index], "deceased-donor-order"),
    )
    shared_indices = deceased_indices[:4_000]
    deceased_single_indices = deceased_indices[4_000:]

    living_clusters = [[index] for index in living_indices]
    deceased_single_clusters = [[index] for index in deceased_single_indices]
    shared_clusters = [shared_indices[offset : offset + 2] for offset in range(0, 4_000, 2)]

    blood_groups = ["A", "AB", "B", "O"]
    cluster_groups = {}
    for label, clusters, per_group in [
        ("living", living_clusters, 1_000),
        ("deceased-single", deceased_single_clusters, 500),
        ("deceased-shared", shared_clusters, 500),
    ]:
        ordered_clusters = sorted(
            range(len(clusters)),
            key=lambda cluster_index: stable_score(
                request_seeds[clusters[cluster_index][0]], f"{label}-blood-group"
            ),
        )
        for group_position, blood_group in enumerate(blood_groups):
            for cluster_index in ordered_clusters[
                group_position * per_group : (group_position + 1) * per_group
            ]:
                cluster_groups[(label, cluster_index)] = blood_group

    donor_sequence = 0
    for label, clusters, donor_prefix, donor_kind in [
        ("living", living_clusters, "V32P-DL", "Living"),
        ("deceased-single", deceased_single_clusters, "V32P-DD", "Deceased"),
        ("deceased-shared", shared_clusters, "V32P-DP", "Deceased"),
    ]:
        for cluster_index, members in enumerate(clusters, start=0):
            donor_sequence += 1
            donor_id = f"{donor_prefix}{cluster_index + 1:06d}"
            donor_seed = stable_score(request_seeds[members[0]], "donor-profile") % (2**32)
            donor_rng = np.random.default_rng(donor_seed)
            donor_age = int(donor_rng.integers(25, 71))
            blood_group = cluster_groups[(label, cluster_index)]
            for member in members:
                donor_ids[member] = donor_id
                donor_cluster_size[member] = len(members)
            donor_records.append(
                {
                    "donor_id": donor_id,
                    "donor_type": donor_kind,
                    "donor_age": donor_age,
                    "donor_blood_group": blood_group,
                    "linked_recipient_count": len(members),
                }
            )
    donors = pd.DataFrame(donor_records)
    assert len(donors) == UNIQUE_DONOR_COUNT
    assert len(set(donor_ids)) == UNIQUE_DONOR_COUNT

    donor_lookup = donors.set_index("donor_id")
    donor_age = np.array([int(donor_lookup.loc[donor_id, "donor_age"]) for donor_id in donor_ids])
    donor_blood_group = np.array(
        [str(donor_lookup.loc[donor_id, "donor_blood_group"]) for donor_id in donor_ids], dtype=object
    )

    recipient_blood_group = np.empty(RECIPIENT_COUNT, dtype=object)
    pair_counts = {
        "A": {"AB": 500, "O": 2_000},
        "AB": {"AB": 500, "B": 1_500, "O": 500},
        "B": {"A": 500, "AB": 1_500, "B": 500},
        "O": {"A": 2_000, "B": 500},
    }
    for group, counts in pair_counts.items():
        indices = np.flatnonzero(donor_blood_group == group)
        ordered = sorted(indices.tolist(), key=lambda index: stable_score(request_seeds[index], f"recipient-abo-{group}"))
        cursor = 0
        for recipient_group, count in counts.items():
            recipient_blood_group[ordered[cursor : cursor + count]] = recipient_group
            cursor += count
        assert cursor == len(indices)
    abo_category = np.array(
        [derive_abo_category(donor, recipient) for donor, recipient in zip(donor_blood_group, recipient_blood_group)],
        dtype=object,
    )

    hospital_id = exact_assignment(
        request_seeds,
        {
            "V32P-H01": 1_800,
            "V32P-H02": 1_500,
            "V32P-H03": 1_300,
            "V32P-H04": 1_200,
            "V32P-H05": 1_000,
            "V32P-H06": 900,
            "V32P-H07": 800,
            "V32P-H08": 700,
            "V32P-H09": 500,
            "V32P-H10": 300,
        },
        "hospital",
    )
    kidney_failure_cause = exact_assignment(
        request_seeds,
        {
            "Glomerulonephritis": 2_000,
            "Hypertensive kidney disease": 2_000,
            "Polycystic kidney disease": 2_000,
            "Other": 2_000,
            "Diabetic kidney disease": 2_000,
        },
        "kidney-failure-cause",
    )
    previous_transplant = exact_assignment(request_seeds, {0: 8_800, 1: 1_200}, "previous-transplant").astype(int)
    hla_mismatch_count = exact_assignment(
        request_seeds, {0: 1_400, 1: 1_500, 2: 1_400, 3: 1_500, 4: 1_300, 5: 1_400, 6: 1_500}, "hla"
    ).astype(int)
    recipient_sex = exact_assignment(request_seeds, {"Male": 5_000, "Female": 5_000}, "recipient-sex")
    recipient_ethnicity = exact_assignment(
        request_seeds, {"Black": 2_000, "Mixed": 2_000, "Other": 2_000, "White": 2_000, "Asian": 2_000}, "ethnicity"
    )
    recipient_region = exact_assignment(
        request_seeds,
        {"Scotland": 2_000, "Wales": 2_000, "London": 2_000, "North West": 2_000, "Northern Ireland": 2_000},
        "region",
    )
    recovery_pattern = exact_assignment(request_seeds, {"rapid": 3_600, "gradual": 3_300, "partial": 3_100}, "recovery")
    adherence_pattern = exact_assignment(request_seeds, {"high": 4_800, "variable": 3_900, "low": 1_300}, "adherence")
    tacrolimus_pattern = exact_assignment(request_seeds, {"in_range": 3_900, "variable": 3_400, "high": 2_700}, "tacrolimus")
    infection_schedule = exact_assignment(
        request_seeds, {"[]": 7_400, "[30]": 1_000, "[14]": 700, "[90]": 600, "[60]": 300}, "infection-schedule"
    )
    confounder = exact_assignment(
        request_seeds,
        {"dehydration": 3_700, "tacrolimus_exposure": 3_100, "none": 3_000, "infection": 100, "high_creatinine": 100},
        "confounder",
    )

    recipient_age = np.empty(RECIPIENT_COUNT, dtype=int)
    dialysis_months = np.empty(RECIPIENT_COUNT, dtype=int)
    antibody_risk_score = np.empty(RECIPIENT_COUNT, dtype=float)
    cold_ischaemia_hours = np.empty(RECIPIENT_COUNT, dtype=float)
    distance_km = np.empty(RECIPIENT_COUNT, dtype=float)
    baseline_creatinine = np.empty(RECIPIENT_COUNT, dtype=float)
    event_random_draw = np.empty(RECIPIENT_COUNT, dtype=float)
    for index, seed in enumerate(request_seeds):
        rng = np.random.default_rng(int(seed) + 1_100_000)
        recipient_age[index] = int(rng.integers(22, 79))
        dialysis_months[index] = int(rng.integers(3, 73))
        antibody_risk_score[index] = round(float(rng.uniform(0.06, 0.93)), 3)
        cold_ischaemia_hours[index] = round(float(rng.uniform(2.2, 20.9)), 2)
        distance_km[index] = round(float(rng.uniform(6.76, 143.89)), 2)
        baseline_creatinine[index] = round(float(rng.uniform(0.67, 1.78)), 3)
        event_random_draw[index] = round(float(rng.random()), 8)

    available = set(range(RECIPIENT_COUNT))

    def take_hashed(count, namespace, candidates=None):
        pool = available if candidates is None else available.intersection(set(candidates))
        ordered = sorted(pool, key=lambda index: stable_score(request_seeds[index], namespace))
        selected = ordered[:count]
        if len(selected) != count:
            raise AssertionError(f"not enough candidates for {namespace}")
        available.difference_update(selected)
        return selected

    forced_families = {}
    forced_families["stable_negative"] = take_hashed(300, "arch-stable")
    forced_families["infection_confounder_negative"] = take_hashed(100, "arch-infection")
    forced_families["high_creatinine_confounder_negative"] = take_hashed(100, "arch-high-creatinine")
    forced_families["subtle_positive"] = take_hashed(100, "arch-subtle")
    forced_families["moderate_positive"] = take_hashed(100, "arch-moderate")
    forced_families["rejection_high_adherence_in_range_tacrolimus"] = take_hashed(100, "arch-high-adherence")

    adherence_pattern = force_value_by_swap(
        adherence_pattern,
        forced_families["stable_negative"] + forced_families["rejection_high_adherence_in_range_tacrolimus"],
        "high",
        "force-high-adherence",
        request_seeds,
    )
    tacrolimus_pattern = force_value_by_swap(
        tacrolimus_pattern,
        forced_families["stable_negative"] + forced_families["rejection_high_adherence_in_range_tacrolimus"],
        "in_range",
        "force-in-range-tacrolimus",
        request_seeds,
    )
    confounder = force_value_by_swap(
        confounder, forced_families["stable_negative"], "none", "force-stable-none", request_seeds
    )
    confounder = force_value_by_swap(
        confounder, forced_families["infection_confounder_negative"], "infection", "force-infection", request_seeds
    )
    infection_schedule = force_value_by_swap(
        infection_schedule, forced_families["infection_confounder_negative"], "[30]", "force-infection-day", request_seeds
    )
    confounder = force_value_by_swap(
        confounder,
        forced_families["high_creatinine_confounder_negative"],
        "high_creatinine",
        "force-high-creatinine",
        request_seeds,
    )

    risk_fit = fit_documented_risk_relationship()
    coefficient = dict(zip(risk_fit["feature_order"], risk_fit["coefficients"]))
    logit = (
        coefficient["intercept"]
        + coefficient["hla_mismatch_count"] * hla_mismatch_count
        + coefficient["antibody_risk_score"] * antibody_risk_score
        + coefficient["previous_transplant"] * previous_transplant
        + coefficient["adherence_variable"] * (adherence_pattern == "variable")
        + coefficient["adherence_low"] * (adherence_pattern == "low")
        + coefficient["donor_deceased"] * (donor_type == "Deceased")
        + coefficient["cold_ischaemia_hours"] * cold_ischaemia_hours
    )
    risk_noise = np.array(
        [
            np.random.default_rng(int(seed) + 2_100_000).normal(0.0, risk_fit["residual_standard_deviation"])
            for seed in request_seeds
        ]
    )
    base_production_logit = logit + risk_noise
    uncalibrated_probability = 1.0 / (1.0 + np.exp(-base_production_logit))

    high_risk_candidates = sorted(available, key=lambda index: (-uncalibrated_probability[index], stable_score(request_seeds[index], "arch-high-risk")))
    forced_families["high_risk_without_rejection"] = high_risk_candidates[:100]
    available.difference_update(forced_families["high_risk_without_rejection"])
    shared_candidates = [index for index in available if donor_cluster_size[index] == 2]
    lower_risk_candidates = sorted(shared_candidates, key=lambda index: (uncalibrated_probability[index], stable_score(request_seeds[index], "arch-lower-risk")))
    forced_families["lower_risk_with_rejection_shared_deceased_donor"] = lower_risk_candidates[:100]
    available.difference_update(forced_families["lower_risk_with_rejection_shared_deceased_donor"])

    archetype = np.full(RECIPIENT_COUNT, "probabilistic", dtype=object)
    for family, indices in forced_families.items():
        archetype[indices] = family
    event_selection_basis = np.where(archetype == "probabilistic", "probabilistic", "forced_overlap")
    probabilistic_indices = np.flatnonzero(event_selection_basis == "probabilistic")
    target_probabilistic_events = 3_000
    clipped_draws = np.clip(event_random_draw[probabilistic_indices], 1e-12, 1.0 - 1e-12)
    required_offsets = (
        np.log(clipped_draws / (1.0 - clipped_draws))
        - base_production_logit[probabilistic_indices]
    )
    ordered_offsets = np.sort(required_offsets)
    calibration_offset = float(
        (ordered_offsets[target_probabilistic_events - 1] + ordered_offsets[target_probabilistic_events])
        / 2.0
    )
    calibrated_probability = 1.0 / (1.0 + np.exp(-(base_production_logit + calibration_offset)))
    event_flag = event_random_draw < calibrated_probability
    assert int(event_flag[probabilistic_indices].sum()) == target_probabilistic_events
    event_probability = np.round(calibrated_probability, 8)
    risk_fit["production_pre_generation_calibration"] = {
        "logit_intercept_offset": calibration_offset,
        "purpose": "retain the frozen 30/90 probabilistic-event proportion in the scaled 9,000-recipient probabilistic cohort",
        "target_probabilistic_event_recipients": target_probabilistic_events,
        "calibration_used_classifier_performance": False,
    }
    forced_negative = {
        "stable_negative",
        "infection_confounder_negative",
        "high_creatinine_confounder_negative",
        "high_risk_without_rejection",
    }
    forced_positive = {
        "subtle_positive",
        "moderate_positive",
        "rejection_high_adherence_in_range_tacrolimus",
        "lower_risk_with_rejection_shared_deceased_donor",
    }
    for family in forced_negative:
        event_flag[np.array(forced_families[family], dtype=int)] = False
    for family in forced_positive:
        event_flag[np.array(forced_families[family], dtype=int)] = True

    event_day_values = np.array([20, 35, 75, 110, 150], dtype=int)
    event_day_weights = np.array([3, 9, 5, 8, 9], dtype=float) / 34.0
    signal_values = np.array(["subtle", "moderate", "strong"], dtype=object)
    signal_weights = np.array([9, 14, 11], dtype=float) / 34.0
    event_days = np.empty(RECIPIENT_COUNT, dtype=object)
    signal_strength = np.full(RECIPIENT_COUNT, "none", dtype=object)
    for index, seed in enumerate(request_seeds):
        if not event_flag[index]:
            event_days[index] = "[]"
            continue
        rng = np.random.default_rng(int(seed) + 2_200_000)
        event_day = int(rng.choice(event_day_values, p=event_day_weights))
        event_days[index] = json_list([event_day])
        signal_strength[index] = str(rng.choice(signal_values, p=signal_weights))
    signal_strength[np.array(forced_families["subtle_positive"], dtype=int)] = "subtle"
    signal_strength[np.array(forced_families["moderate_positive"], dtype=int)] = "moderate"
    signal_strength[
        np.array(forced_families["rejection_high_adherence_in_range_tacrolimus"], dtype=int)
    ] = "moderate"
    signal_strength[
        np.array(forced_families["lower_risk_with_rejection_shared_deceased_donor"], dtype=int)
    ] = "subtle"

    retention_group = exact_assignment(request_seeds, {"one": 400, "two": 1_300, "three": 2_000, "none": 6_300}, "retention-group")
    transplant_dates = []
    for index, seed in enumerate(request_seeds):
        rng = np.random.default_rng(int(seed) + 2_300_000)
        group = retention_group[index]
        if group == "one":
            days_before = int(rng.integers(737, 744))
        elif group == "two":
            days_before = int(rng.integers(744, 760))
        elif group == "three":
            days_before = int(rng.integers(760, 790))
        else:
            start = date(2024, 1, 1)
            days_before = None
            transplant = start + timedelta(days=int(rng.integers(0, 366)))
        if days_before is not None:
            transplant = RETENTION_CUTOFF - timedelta(days=days_before)
        transplant_dates.append(transplant)

    invalid_group = exact_assignment(request_seeds, {"one": 1_900, "two": 1_600, "three": 300, "none": 6_200}, "invalid-consent-group")

    relationship = pd.DataFrame(
        {
            "recipient_id": recipient_ids,
            "donor_id": donor_ids,
            "hospital_id": hospital_id,
            "recipient_age": recipient_age,
            "donor_age": donor_age,
            "donor_type": donor_type,
            "kidney_failure_cause": kidney_failure_cause,
            "previous_transplant": previous_transplant,
            "dialysis_months": dialysis_months,
            "donor_blood_group": donor_blood_group,
            "recipient_blood_group": recipient_blood_group,
            "abo_compatibility_category": abo_category,
            "hla_mismatch_count": hla_mismatch_count,
            "antibody_risk_score": antibody_risk_score,
            "cold_ischaemia_hours": cold_ischaemia_hours,
            "recipient_sex": recipient_sex,
            "recipient_ethnicity": recipient_ethnicity,
            "recipient_region": recipient_region,
            "distance_to_transplant_centre_km": distance_km,
            "transplant_date": [value.isoformat() for value in transplant_dates],
        }
    )
    controls = pd.DataFrame(
        {
            "recipient_id": recipient_ids,
            "request_id": request_ids,
            "chunk_id": chunk_ids,
            "request_sequence_number": recipient_numbers,
            "request_seed": request_seeds,
            "event_probability": event_probability,
            "event_random_draw": event_random_draw,
            "event_selection_basis": event_selection_basis,
            "forced_overlap_archetype": archetype,
            "confirmed_rejection_event_days": event_days,
            "rejection_signal_strength": signal_strength,
            "non_rejection_confounders": confounder,
            "adherence_pattern": adherence_pattern,
            "infection_episode_days": infection_schedule,
            "tacrolimus_exposure_pattern": tacrolimus_pattern,
            "recovery_pattern": recovery_pattern,
            "baseline_creatinine_mg_dl": baseline_creatinine,
        }
    )
    recipients = controls.merge(relationship, on="recipient_id", validate="one_to_one")

    anchor_records = []
    for row in recipients.itertuples(index=False):
        seed = int(row.request_seed)
        recipient_number = int(str(row.recipient_id)[-6:])
        rng = np.random.default_rng(seed + 3_200_000)
        events = json.loads(row.confirmed_rejection_event_days)
        infections = json.loads(row.infection_episode_days)
        early_event = any(7 < int(event) <= 37 for event in events)
        early_infection = any(int(day) <= 30 for day in infections)
        recovery_factor = {"rapid": 0.84, "gradual": 0.96, "partial": 1.08}[row.recovery_pattern]
        creatinine_offset = {
            "none": 0.0,
            "dehydration": 0.10,
            "high_creatinine": 0.24,
            "infection": 0.12,
            "tacrolimus_exposure": 0.09,
        }[row.non_rejection_confounders]
        creatinine = (
            float(row.baseline_creatinine_mg_dl) * recovery_factor
            + creatinine_offset
            + (0.07 if early_infection else 0.0)
            + (0.09 if early_event else 0.0)
            + rng.uniform(-0.085, 0.085)
            + (recipient_number % 11) * 0.001
        )
        creatinine = round(clip(creatinine, canonical_ranges["creatinine_mg_dl"]), 3)
        recovery_urine = {"rapid": 170.0, "gradual": 30.0, "partial": -130.0}[row.recovery_pattern]
        confounder_urine = {
            "none": 0.0,
            "dehydration": -220.0,
            "high_creatinine": -180.0,
            "infection": -160.0,
            "tacrolimus_exposure": -90.0,
        }[row.non_rejection_confounders]
        urine = (
            2380.0
            - 360.0 * (creatinine - 1.10)
            + recovery_urine
            + confounder_urine
            - (110.0 if early_infection else 0.0)
            - (125.0 if early_event else 0.0)
            + rng.uniform(-190.0, 190.0)
            + (recipient_number % 13) * 3.7
        )
        urine = round(clip(urine, canonical_ranges["urine_output_ml_24h"]), 1)
        if row.tacrolimus_exposure_pattern == "high":
            tacrolimus = rng.uniform(10.5, 12.4)
        elif row.tacrolimus_exposure_pattern == "in_range":
            tacrolimus = rng.uniform(7.4, 10.0)
        else:
            tacrolimus = rng.uniform(6.3, 11.2)
        tacrolimus += 0.8 if row.non_rejection_confounders == "tacrolimus_exposure" else 0.0
        tacrolimus += 0.2 if early_event else 0.0
        tacrolimus += 0.15 if early_infection else 0.0
        tacrolimus += (recipient_number % 7) * 0.007
        tacrolimus = round(clip(tacrolimus, canonical_ranges["tacrolimus_level_ng_ml"]), 2)
        if row.adherence_pattern == "high":
            adherence = rng.uniform(96.1, 99.85)
        elif row.adherence_pattern == "variable":
            adherence = rng.uniform(79.0, 95.4)
        else:
            adherence = rng.uniform(56.0, 81.5)
        adherence -= 0.9 if early_infection else 0.0
        adherence -= 0.6 if row.non_rejection_confounders == "infection" else 0.0
        adherence -= 0.4 if early_event and row.adherence_pattern != "high" else 0.0
        adherence += (recipient_number % 9) * 0.009
        adherence = round(clip(adherence, canonical_ranges["medication_adherence_pct"]), 2)
        anchor_records.append(
            {
                "recipient_id": row.recipient_id,
                "request_seed": seed,
                "anchor_algorithm_version": ANCHOR_ALGORITHM_VERSION,
                "recovery_pattern": row.recovery_pattern,
                "adherence_pattern": row.adherence_pattern,
                "tacrolimus_exposure_pattern": row.tacrolimus_exposure_pattern,
                "infection_episode_days": row.infection_episode_days,
                "confirmed_rejection_event_days": row.confirmed_rejection_event_days,
                "event_status": int(bool(events)),
                "rejection_signal_strength": row.rejection_signal_strength,
                "non_rejection_confounders": row.non_rejection_confounders,
                "baseline_creatinine_mg_dl": row.baseline_creatinine_mg_dl,
                "day7_creatinine_anchor_mg_dl": creatinine,
                "day7_urine_output_anchor_ml_24h": urine,
                "day7_tacrolimus_anchor_ng_ml": tacrolimus,
                "day7_medication_adherence_anchor_pct": adherence,
            }
        )
    anchors = pd.DataFrame(anchor_records)

    assessment_records = []
    event_audit_records = []
    for index, row in enumerate(recipients.itertuples(index=False)):
        transplant = transplant_dates[index]
        events = json.loads(row.confirmed_rejection_event_days)
        infection_days = set(json.loads(row.infection_episode_days))
        previous, targets = derive_event_fields(events)
        invalid_days_by_group = {
            "none": set(),
            "one": {30},
            "two": {14, 30},
            "three": {7, 14, 30},
        }
        invalid_days = invalid_days_by_group[invalid_group[index]]
        for assessment_offset, day in enumerate(DAYS):
            assessment_date = transplant + timedelta(days=day)
            retention_expiry = assessment_date + timedelta(days=730)
            if day in invalid_days:
                consent_status = "Invalidated"
                consent_version = "RECIPIENT_V3"
            elif invalid_days and day > max(invalid_days):
                consent_status = "Active"
                consent_version = "RECIPIENT_V4"
            else:
                consent_status = "Active"
                consent_version = (
                    "RECIPIENT_V1"
                    if day == 7
                    else "RECIPIENT_V2"
                    if day == 14
                    else "RECIPIENT_V4"
                )
            assessment_id = f"V32P-A{index * 6 + assessment_offset + 1:06d}"
            assessment_records.append(
                {
                    "assessment_id": assessment_id,
                    "recipient_id": row.recipient_id,
                    "donor_id": row.donor_id,
                    "hospital_id": row.hospital_id,
                    "days_since_transplant": day,
                    "assessment_date": assessment_date.isoformat(),
                    "retention_expiry_date": retention_expiry.isoformat(),
                    "recipient_age": row.recipient_age,
                    "donor_age": row.donor_age,
                    "donor_type": row.donor_type,
                    "kidney_failure_cause": row.kidney_failure_cause,
                    "previous_transplant": row.previous_transplant,
                    "dialysis_months": row.dialysis_months,
                    "donor_blood_group": row.donor_blood_group,
                    "recipient_blood_group": row.recipient_blood_group,
                    "abo_compatibility_category": row.abo_compatibility_category,
                    "hla_mismatch_count": row.hla_mismatch_count,
                    "antibody_risk_score": row.antibody_risk_score,
                    "cold_ischaemia_hours": row.cold_ischaemia_hours,
                    "recipient_sex": row.recipient_sex,
                    "recipient_ethnicity": row.recipient_ethnicity,
                    "recipient_region": row.recipient_region,
                    "distance_to_transplant_centre_km": row.distance_to_transplant_centre_km,
                    "training_consent_status": consent_status,
                    "training_consent_version": consent_version,
                    "infection_indicator": int(day in infection_days),
                    "previous_rejection": previous[assessment_offset],
                    "acute_rejection_within_30_days": targets[assessment_offset],
                }
            )
            qualifying = [event for event in events if day < event <= day + 30]
            event_audit_records.append(
                {
                    "recipient_id": row.recipient_id,
                    "assessment_id": assessment_id,
                    "assessment_day": day,
                    "confirmed_rejection_event_days": row.confirmed_rejection_event_days,
                    "qualifying_future_event_days": json_list(qualifying),
                    "previous_rejection": previous[assessment_offset],
                    "acute_rejection_within_30_days": targets[assessment_offset],
                    "target_rule": "assessment_day < event_day <= assessment_day + 30",
                }
            )
    assessments = pd.DataFrame(assessment_records)
    event_audit = pd.DataFrame(event_audit_records)

    identity_records = []
    for index, row in enumerate(recipients.itertuples(index=False), start=1):
        birth_year = transplant_dates[index - 1].year - int(row.recipient_age)
        identity_records.append(
            {
                "entity_id": row.recipient_id,
                "person_id": f"V32P-PR{index:06d}",
                "person_role": "Recipient",
                "full_name": f"SYNTHETIC_PRODUCTION_RECIPIENT_{index:06d}",
                "date_of_birth": date(birth_year, 1, 1).isoformat(),
                "email_address": f"v32p-recipient-{index:06d}@example.invalid",
                "postcode": f"V32R{index:06d}",
                "hospital_number": f"V32P-RHN-{index:06d}",
                "current_consent_status": "Active",
                "current_consent_version": "RECIPIENT_V4",
                "consent_granted_date": "2023-01-01",
                "consent_withdrawal_date": "",
            }
        )
    for index, donor in enumerate(donors.itertuples(index=False), start=1):
        identity_records.append(
            {
                "entity_id": donor.donor_id,
                "person_id": f"V32P-PD{index:06d}",
                "person_role": "Donor",
                "full_name": f"SYNTHETIC_PRODUCTION_DONOR_{index:06d}",
                "date_of_birth": date(2023 - int(donor.donor_age), 1, 1).isoformat(),
                "email_address": f"v32p-donor-{index:06d}@example.invalid",
                "postcode": f"V32D{index:06d}",
                "hospital_number": f"V32P-DHN-{index:06d}",
                "current_consent_status": "Active",
                "current_consent_version": "DONOR_V1",
                "consent_granted_date": "2023-01-01",
                "consent_withdrawal_date": "",
            }
        )
    identity = pd.DataFrame(identity_records)
    return recipients, assessments, identity, anchors, event_audit, donors, risk_fit


def build_deletion_audit(assessments, recipients):
    seeds = recipients.request_seed.to_numpy()
    recipient_order = sorted(range(len(recipients)), key=lambda index: stable_score(seeds[index], "delete-recipient"))
    recipient_ids = set(recipients.iloc[recipient_order[:100]].recipient_id)

    shared_donors = (
        recipients.groupby("donor_id").recipient_id.nunique().loc[lambda values: values.eq(2)].index.tolist()
    )
    shared_donors = sorted(shared_donors, key=lambda donor: sha256_text(f"{donor}|delete-donor"))
    donor_ids = set(shared_donors[:250])
    donor_recipient_ids = set(recipients.loc[recipients.donor_id.isin(donor_ids), "recipient_id"])
    assert len(donor_recipient_ids) == 500

    masks = {
        "recipient_withdrawal": assessments.recipient_id.isin(recipient_ids),
        "donor_withdrawal": assessments.donor_id.isin(donor_ids),
        "hospital_removal": assessments.hospital_id.eq("V32P-H05"),
        "invalid_consent": assessments.training_consent_status.eq("Invalidated")
        & assessments.training_consent_version.eq("RECIPIENT_V3"),
        "retention_expiry": pd.to_datetime(assessments.retention_expiry_date).dt.date.le(RETENTION_CUTOFF),
    }
    expected = {
        "recipient_withdrawal": 600,
        "donor_withdrawal": 3_000,
        "hospital_removal": 6_000,
        "invalid_consent": 6_000,
        "retention_expiry": 9_000,
    }
    scenarios = {}
    for name, mask in masks.items():
        selected = assessments.loc[mask]
        counts = selected.groupby("recipient_id").size()
        scenarios[name] = {
            "matched_assessment_rows": int(len(selected)),
            "matched_fraction": float(len(selected) / len(assessments)),
            "affected_recipients": int(selected.recipient_id.nunique()),
            "affected_donors": int(selected.donor_id.nunique()),
            "affected_hospitals": int(selected.hospital_id.nunique()),
            "complete_recipient_histories": int(counts.eq(6).sum()),
            "records_per_recipient_distribution": {
                str(int(key)): int(value) for key, value in counts.value_counts().sort_index().items()
            },
            "assessment_days": sorted(selected.days_since_transplant.unique().astype(int).tolist()),
            "selection_key": {
                "recipient_withdrawal": "recipient_id",
                "donor_withdrawal": "donor_id (250 shared deceased donors covering 500 recipients)",
                "hospital_removal": "hospital_id=V32P-H05",
                "invalid_consent": "training_consent_status=Invalidated and training_consent_version=RECIPIENT_V3",
                "retention_expiry": f"retention_expiry_date <= {RETENTION_CUTOFF.isoformat()}",
            }[name],
        }
        assert len(selected) == expected[name]
    assert scenarios["recipient_withdrawal"]["complete_recipient_histories"] == 100
    assert scenarios["donor_withdrawal"]["complete_recipient_histories"] == 500
    assert scenarios["hospital_removal"]["complete_recipient_histories"] == 1_000
    assert scenarios["invalid_consent"]["complete_recipient_histories"] == 0
    assert scenarios["retention_expiry"]["complete_recipient_histories"] == 0

    overlap = {}
    names = list(masks)
    for left in names:
        overlap[left] = {}
        for right in names:
            overlap[left][right] = int((masks[left] & masks[right]).sum())
    return {
        "status": "passed",
        "assessment_rows": len(assessments),
        "scenarios": scenarios,
        "pairwise_assessment_row_overlap": overlap,
        "mutual_exclusivity_forced": False,
        "complete_history_rule": (
            "Recipient, donor and hospital selections remove complete histories by design; invalid-consent "
            "and retention-expiry remain record-level and remove no complete history."
        ),
    }


def build_anchor_report(anchors, canonical_ranges):
    anchor_fields = {
        "day7_creatinine_anchor_mg_dl": "creatinine_mg_dl",
        "day7_urine_output_anchor_ml_24h": "urine_output_ml_24h",
        "day7_tacrolimus_anchor_ng_ml": "tacrolimus_level_ng_ml",
        "day7_medication_adherence_anchor_pct": "medication_adherence_pct",
    }
    suffix = anchors.recipient_id.str[-6:].astype(int)
    result = {}
    for anchor_field, range_field in anchor_fields.items():
        values = anchors[anchor_field]
        bounds = canonical_ranges[range_field]
        correlation = float(np.corrcoef(suffix, values)[0, 1])
        result[anchor_field] = {
            "minimum": float(values.min()),
            "maximum": float(values.max()),
            **tied_frequency_summary(values),
            "pearson_correlation_with_recipient_numeric_suffix": correlation,
            "absolute_suffix_correlation": abs(correlation),
            "material_monotonic_association": abs(correlation) >= 0.10,
            "clipped_at_approved_minimum": int(values.eq(bounds["minimum"]).sum()),
            "clipped_at_approved_maximum": int(values.eq(bounds["maximum"]).sum()),
            "approved_bounds": bounds,
            "distribution_by_control_pattern": {
                field: grouped_numeric_summary(anchors, field, anchor_field)
                for field in [
                    "recovery_pattern",
                    "adherence_pattern",
                    "tacrolimus_exposure_pattern",
                    "infection_episode_days",
                    "non_rejection_confounders",
                ]
            },
            "distribution_by_event_status": grouped_numeric_summary(anchors, "event_status", anchor_field),
        }
        assert not result[anchor_field]["material_monotonic_association"]
    return result


def build_target_report(recipients, assessments):
    event_recipients = recipients.confirmed_rejection_event_days.ne("[]")
    grouped_targets = assessments.groupby("recipient_id").acute_rejection_within_30_days.apply(
        lambda values: json_list(values.astype(int).tolist())
    )
    event_days = recipients.loc[event_recipients, "confirmed_rejection_event_days"].map(
        lambda value: int(json.loads(value)[0])
    )
    recipients_positive = assessments.groupby("recipient_id").acute_rejection_within_30_days.max().eq(1)
    no_coverage_ids = set(recipients.loc[event_recipients, "recipient_id"]) - set(recipients_positive.loc[recipients_positive].index)
    no_coverage_events = recipients.loc[recipients.recipient_id.isin(no_coverage_ids), "confirmed_rejection_event_days"].map(
        lambda value: int(json.loads(value)[0])
    )
    result = {
        "target_rule": "assessment_day < event_day <= assessment_day + 30",
        "controlled_rejection_event_recipients": int(event_recipients.sum()),
        "recipients_with_at_least_one_positive_assessment": int(recipients_positive.sum()),
        "positive_assessment_rows": int(assessments.acute_rejection_within_30_days.sum()),
        "assessment_level_target_prevalence": float(assessments.acute_rejection_within_30_days.mean()),
        "recipient_level_event_prevalence": float(event_recipients.mean()),
        "events_without_scheduled_assessment_in_preceding_30_days": int(len(no_coverage_ids)),
        "uncovered_event_day_counts": {str(int(key)): int(value) for key, value in no_coverage_events.value_counts().sort_index().items()},
        "event_day_counts": {str(int(key)): int(value) for key, value in event_days.value_counts().sort_index().items()},
        "six_assessment_target_sequence_counts": {
            key: int(value) for key, value in grouped_targets.value_counts().sort_index().items()
        },
    }
    assert set(no_coverage_events.unique().tolist()) <= {150}
    return result


def build_request_ledger(prompt_contract, api_config, anchors):
    schema = {
        "$schema": "https://json-schema.org/draft/2020-12/schema",
        "title": "Qwen v3.2 append-only prospective request ledger record",
        "type": "object",
        "additionalProperties": False,
        "required": [
            "run_id",
            "chunk_id",
            "request_id",
            "recipient_id",
            "request_sequence_number",
            "attempt_number",
            "deterministic_request_seed",
            "prompt_version",
            "prompt_template_sha256",
            "rendered_prompt_sha256",
            "rendered_message_payload",
            "model",
            "completion_parameters",
            "anchor_algorithm_version",
            "anchor_metadata_sha256",
            "timestamp_utc",
            "status",
        ],
        "properties": {
            "run_id": {"type": "string"},
            "chunk_id": {"type": "string"},
            "request_id": {"type": "string"},
            "recipient_id": {"type": "string", "description": "Pseudonymous production recipient ID; not a direct identifier."},
            "request_sequence_number": {"type": "integer", "minimum": 1},
            "attempt_number": {"const": 1},
            "deterministic_request_seed": {"type": "integer"},
            "prompt_version": {"const": PROMPT_VERSION},
            "prompt_template_sha256": {"type": "string", "pattern": "^[0-9a-f]{64}$"},
            "rendered_prompt_sha256": {"type": "string", "pattern": "^[0-9a-f]{64}$"},
            "rendered_message_payload": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["role", "content"],
                    "properties": {"role": {"enum": ["system", "user"]}, "content": {"type": "string"}},
                },
                "minItems": 2,
                "maxItems": 2,
            },
            "model": {"type": "string"},
            "completion_parameters": {"type": "object"},
            "anchor_algorithm_version": {"const": ANCHOR_ALGORITHM_VERSION},
            "anchor_metadata_sha256": {"type": "string", "pattern": "^[0-9a-f]{64}$"},
            "timestamp_utc": {"type": "string", "format": "date-time"},
            "status": {"const": "preserved_before_network_submission"},
        },
        "append_only_invariants": [
            "serialize and fsync this record before network submission",
            "never overwrite, truncate or reorder requests.jsonl",
            "attempt_number remains 1 because automatic retries are prohibited",
            "skip request IDs already validated by the authoritative checkpoints",
            "responses.jsonl remains separate and stores the complete serialized response before extraction or parsing",
            "direct identifiers are prohibited",
        ],
        "prohibited_direct_identifier_fields": [
            "person_id",
            "full_name",
            "date_of_birth",
            "email_address",
            "postcode",
            "hospital_number",
        ],
    }
    mock_anchor = {
        "recipient_id": "V32P-RMOCK001",
        "request_seed": 9_999_001,
        "anchor_algorithm_version": ANCHOR_ALGORITHM_VERSION,
        "day7_creatinine_anchor_mg_dl": 1.123,
        "day7_urine_output_anchor_ml_24h": 2345.6,
        "day7_tacrolimus_anchor_ng_ml": 8.76,
        "day7_medication_adherence_anchor_pct": 97.65,
    }
    mock_replacements = {
        "{recipient_seed}": "9999001",
        "{static_clinical_fields}": json.dumps(
            {"recipient_age": 50, "donor_age": 45, "donor_type": "Living"}, sort_keys=True
        ),
        "{baseline_creatinine_mg_dl}": "1.234",
        "{generation_only_event_schedule}": "[]",
        "{signal_strength}": "none",
        "{confounder_instructions}": "none",
        "{adherence_pattern}": "high",
        "{infection_episode_days}": "[]",
        "{tacrolimus_exposure}": "in_range",
        "{recovery_pattern}": "gradual",
        "{day7_creatinine_anchor_mg_dl}": "1.123",
        "{day7_urine_output_anchor_ml_24h}": "2345.6",
        "{day7_tacrolimus_anchor_ng_ml}": "8.76",
        "{day7_medication_adherence_anchor_pct}": "97.65",
    }
    mock_prompt = prompt_contract["template"]
    for old, new in mock_replacements.items():
        mock_prompt = mock_prompt.replace(old, new)
    assert all(token not in mock_prompt for token in mock_replacements)
    messages = [
        {"role": "system", "content": "Return only the exact JSON object requested. No prose or Markdown."},
        {"role": "user", "content": mock_prompt},
    ]
    completion = dict(api_config["completion"])
    model = completion.pop("model")
    mock_record = {
        "run_id": "QWEN-V32-PROD-MOCK",
        "chunk_id": "QWEN-V32-PROD-MOCK-CHUNK001",
        "request_id": "QWEN-V32-PROD-MOCK-REQ000001",
        "recipient_id": "V32P-RMOCK001",
        "request_sequence_number": 1,
        "attempt_number": 1,
        "deterministic_request_seed": 9_999_001,
        "prompt_version": PROMPT_VERSION,
        "prompt_template_sha256": EXPECTED_PROMPT_SHA256,
        "rendered_prompt_sha256": sha256_text(mock_prompt),
        "rendered_message_payload": messages,
        "model": model,
        "completion_parameters": completion,
        "anchor_algorithm_version": ANCHOR_ALGORITHM_VERSION,
        "anchor_metadata_sha256": sha256_text(json.dumps(mock_anchor, sort_keys=True)),
        "timestamp_utc": "2026-01-01T00:00:00+00:00",
        "status": "preserved_before_network_submission",
    }
    serialized = json.dumps(mock_record, sort_keys=True, separators=(",", ":")) + "\n"
    round_trip = json.loads(serialized)
    assert round_trip == mock_record
    assert round_trip["rendered_prompt_sha256"] == sha256_text(round_trip["rendered_message_payload"][1]["content"])
    assert not set(schema["prohibited_direct_identifier_fields"]) & set(round_trip)
    validation = {
        "status": "passed",
        "mock_record_count": 1,
        "round_trip_exact": True,
        "rendered_prompt_hash_verified": True,
        "pre_network_status_verified": True,
        "attempt_number_verified_as_one": True,
        "direct_identifier_fields_absent": True,
        "api_requests": 0,
        "network_requests": 0,
    }
    return schema, serialized, validation


def build_chunk_plan():
    chunks = []
    for chunk_number in range(1, CHUNK_COUNT + 1):
        first = (chunk_number - 1) * REQUESTS_PER_CHUNK + 1
        last = chunk_number * REQUESTS_PER_CHUNK
        chunks.append(
            {
                "chunk_number": chunk_number,
                "chunk_id": f"QWEN-V32-PROD-CHUNK{chunk_number:03d}",
                "recipient_sequence_start": first,
                "recipient_sequence_end": last,
                "recipient_count": REQUESTS_PER_CHUNK,
                "maximum_api_calls": REQUESTS_PER_CHUNK,
                "status": "planned_disabled",
            }
        )
    return {
        "status": "prepared_not_executed",
        "run_id": PRODUCTION_RUN_ID,
        "recipient_count": RECIPIENT_COUNT,
        "recipients_per_chunk": REQUESTS_PER_CHUNK,
        "chunk_count": CHUNK_COUNT,
        "maximum_total_api_calls": RECIPIENT_COUNT,
        "request_execution": "sequential unless separately authorised",
        "max_retries": 0,
        "failure_policy": "stop on first failure; preserve all partial request, response and checkpoint evidence",
        "resume_policy": "manual and explicitly gated; skip validated request IDs; never overwrite requests or responses",
        "requests_ledger": "append-only requests.jsonl preserved before each network submission",
        "responses_ledger": "separate append-only responses.jsonl; preserve complete model_dump_json before extraction",
        "proposed_first_chunk_gate": PROPOSED_CHUNK_1_GATE,
        "first_chunk_gate_enabled": False,
        "all_api_and_generation_flags": {
            "RUN_PRODUCTION_CHUNK_001": RUN_PRODUCTION_CHUNK_001,
            "RUN_10000_RECIPIENTS": RUN_10000_RECIPIENTS,
            "RUN_FULL_GENERATION": RUN_FULL_GENERATION,
        },
        "chunks": chunks,
    }


def validate_design(recipients, assessments, identity, anchors, event_audit, donors, target_report, deletion_report, anchor_report, field_lists):
    direct = set(field_lists["direct_identifiers"])
    classifier = set(field_lists["classifier_features"])
    forbidden_classifier = set(
        field_lists["pseudonymous_identifiers"]
        + field_lists["assessment_audit_only"]
        + field_lists["event_control_metadata"]
        + ["assessment_date", "retention_expiry_date", "transplant_date", "training_consent_status", "training_consent_version"]
        + [column for column in anchors.columns if "anchor" in column]
    )
    consent_rank = assessments.training_consent_version.map(
        {"RECIPIENT_V1": 1, "RECIPIENT_V2": 2, "RECIPIENT_V3": 3, "RECIPIENT_V4": 4}
    )
    consent_monotonic = consent_rank.groupby(assessments.recipient_id).apply(
        lambda values: values.diff().dropna().ge(0).all()
    ).all()
    probabilistic = recipients.event_selection_basis.eq("probabilistic")
    probabilistic_schedule_consistent = (
        recipients.loc[probabilistic, "event_random_draw"].lt(
            recipients.loc[probabilistic, "event_probability"]
        )
        == recipients.loc[probabilistic, "confirmed_rejection_event_days"].ne("[]")
    ).all()
    checks = {
        "recipient_metadata_dimensions": recipients.shape[0] == RECIPIENT_COUNT,
        "assessment_metadata_dimensions": assessments.shape[0] == ASSESSMENT_COUNT,
        "identity_dimensions": identity.shape[0] == IDENTITY_COUNT,
        "anchor_dimensions": anchors.shape[0] == RECIPIENT_COUNT,
        "event_target_audit_dimensions": event_audit.shape[0] == ASSESSMENT_COUNT,
        "unique_production_recipient_ids": recipients.recipient_id.is_unique,
        "unique_request_ids": recipients.request_id.is_unique,
        "unique_request_seeds": recipients.request_seed.is_unique,
        "unique_assessment_ids": assessments.assessment_id.is_unique,
        "six_assessments_per_recipient": assessments.groupby("recipient_id").size().eq(6).all(),
        "assessment_days_exact": assessments.groupby("recipient_id").days_since_transplant.apply(list).map(lambda value: value == DAYS).all(),
        "no_duplicate_assessment_metadata": not assessments.duplicated().any(),
        "new_recipient_namespace": recipients.recipient_id.str.startswith("V32P-R").all(),
        "no_v31_recipient_reuse": not recipients.recipient_id.str.startswith("V31-").any(),
        "donor_count": donors.donor_id.nunique() == UNIQUE_DONOR_COUNT,
        "shared_donors_have_constant_attributes": recipients.groupby("donor_id")[["donor_age", "donor_type", "donor_blood_group"]].nunique().le(1).all().all(),
        "abo_derived_exactly": all(
            derive_abo_category(donor, recipient) == category
            for donor, recipient, category in zip(
                recipients.donor_blood_group, recipients.recipient_blood_group, recipients.abo_compatibility_category
            )
        ),
        "direct_identifiers_only_in_identity": all(
            not direct.intersection(frame.columns)
            for frame in [recipients, assessments, anchors, event_audit]
        ),
        "classifier_excludes_ids_consent_events_dates_anchors": not classifier.intersection(forbidden_classifier),
        "target_audit_matches_assessment_metadata": event_audit.acute_rejection_within_30_days.astype(int).equals(
            assessments.acute_rejection_within_30_days.astype(int)
        ),
        "consent_versions_monotonic": consent_monotonic,
        "probabilistic_event_draws_match_stored_probabilities": probabilistic_schedule_consistent,
        "recipient_deletion_exact": deletion_report["scenarios"]["recipient_withdrawal"]["matched_assessment_rows"] == 600,
        "donor_deletion_exact": deletion_report["scenarios"]["donor_withdrawal"]["matched_assessment_rows"] == 3_000,
        "hospital_deletion_exact": deletion_report["scenarios"]["hospital_removal"]["matched_assessment_rows"] == 6_000,
        "invalid_consent_exact": deletion_report["scenarios"]["invalid_consent"]["matched_assessment_rows"] == 6_000,
        "retention_expiry_exact": deletion_report["scenarios"]["retention_expiry"]["matched_assessment_rows"] == 9_000,
        "record_level_deletions_not_complete_histories": deletion_report["scenarios"]["invalid_consent"]["complete_recipient_histories"] == 0
        and deletion_report["scenarios"]["retention_expiry"]["complete_recipient_histories"] == 0,
        "all_anchors_non_material_suffix_correlation": all(
            not report["material_monotonic_association"] for report in anchor_report.values()
        ),
        "target_rule_preserved": target_report["target_rule"] == "assessment_day < event_day <= assessment_day + 30",
        "production_flags_disabled": not RUN_PRODUCTION_CHUNK_001 and not RUN_10000_RECIPIENTS and not RUN_FULL_GENERATION,
        "api_requests_zero": API_REQUESTS_MADE == 0 and NETWORK_REQUESTS_MADE == 0,
    }
    failures = [name for name, passed in checks.items() if not bool(passed)]
    return checks, failures


def run_production_design():
    assert not PRODUCTION_DIR.exists(), "production design directory already exists; overwrite prohibited"

    existing_raw_entries = sorted(
        [path for path in (ROOT / "data/raw").iterdir() if path.name != "KidneyTransplant"]
        + [path for path in RAW_KIDNEY.iterdir() if path != PRODUCTION_DIR],
        key=lambda path: str(path),
    )
    immutable_paths = existing_raw_entries + [
        ROOT / "data/processed",
        ROOT / "models",
        ROOT / "results",
    ]
    immutable_before = {display_path(path): tree_fingerprint(path) for path in immutable_paths}
    notebooks_01_05 = sorted((ROOT / "notebooks/KidneyTransplant").glob("0[1-5]_*.ipynb"))
    notebooks_before = {display_path(path): file_fingerprint(path) for path in notebooks_01_05}
    protected_before = protected_fingerprint()
    assert {key: protected_before[key] for key in EXPECTED_PROTECTED} == EXPECTED_PROTECTED

    evidence_files = sorted(
        [path for folder in [V31_DESIGN, V32_DESIGN, V32_CONFIRM, V32_ADDENDUM] for path in folder.iterdir() if path.is_file()]
    )
    evidence_before = {display_path(path): file_fingerprint(path) for path in evidence_files}
    prompt_contract = source_and_prompt_contract()
    canonical_ranges = json.loads((V31_DESIGN / "canonical_qwen_ranges.json").read_text(encoding="utf-8"))
    field_lists = json.loads((V31_DESIGN / "field_lists.json").read_text(encoding="utf-8"))
    api_config = json.loads((V32_DESIGN / "api_configuration.json").read_text(encoding="utf-8"))
    confirmation_report = json.loads((V32_CONFIRM / "run_report.json").read_text(encoding="utf-8"))

    validation_report = {
        "design_id": DESIGN_ID,
        "status": "running",
        "error": None,
        "local_only": True,
        "api_requests_made": API_REQUESTS_MADE,
        "network_requests_made": NETWORK_REQUESTS_MADE,
        "production_chunk_started": False,
        "full_generation_enabled": False,
        "protected_before": protected_before,
        "immutable_evidence_before": immutable_before,
        "input_evidence_fingerprints_before": evidence_before,
        "notebooks_01_05_before": notebooks_before,
        "preproduction_notebook_sha256": EXPECTED_PREPRODUCTION_NOTEBOOK_SHA256,
    }
    error = None
    created_files = []
    try:
        recipients, assessments, identity, anchors, event_audit, donors, risk_fit = build_production_skeleton(canonical_ranges)
        target_report = build_target_report(recipients, assessments)
        deletion_report = build_deletion_audit(assessments, recipients)
        anchor_report = build_anchor_report(anchors, canonical_ranges)
        ledger_schema, mock_ledger, ledger_validation = build_request_ledger(prompt_contract, api_config, anchors)
        chunk_plan = build_chunk_plan()
        checks, failures = validate_design(
            recipients,
            assessments,
            identity,
            anchors,
            event_audit,
            donors,
            target_report,
            deletion_report,
            anchor_report,
            field_lists,
        )
        if failures:
            raise AssertionError(f"production design validation failures: {failures}")

        direct_identifier_columns = field_lists["direct_identifiers"]
        category_fields = [
            "hospital_id",
            "donor_type",
            "kidney_failure_cause",
            "abo_compatibility_category",
            "training_consent_version",
            "recovery_pattern",
            "adherence_pattern",
            "tacrolimus_exposure_pattern",
            "infection_episode_days",
            "non_rejection_confounders",
            "forced_overlap_archetype",
            "event_selection_basis",
            "confirmed_rejection_event_days",
        ]
        distribution_counts = {}
        for field in category_fields:
            source = assessments if field == "training_consent_version" else recipients
            distribution_counts[field] = {
                str(key): int(value) for key, value in source[field].value_counts(dropna=False).sort_index().items()
            }
        donor_cluster_distribution = {
            str(int(key)): int(value)
            for key, value in recipients.groupby("donor_id").size().value_counts().sort_index().items()
        }
        numerical_fields = [
            "recipient_age",
            "donor_age",
            "dialysis_months",
            "hla_mismatch_count",
            "antibody_risk_score",
            "cold_ischaemia_hours",
            "distance_to_transplant_centre_km",
            "baseline_creatinine_mg_dl",
            "event_probability",
            "event_random_draw",
        ]
        numerical_ranges = {
            field: {
                "minimum": float(recipients[field].min()),
                "maximum": float(recipients[field].max()),
                **tied_frequency_summary(recipients[field]),
            }
            for field in numerical_fields
        }
        distribution_report = {
            "status": "passed",
            "design_id": DESIGN_ID,
            "recipient_count": len(recipients),
            "assessment_metadata_rows": len(assessments),
            "identity_rows": len(identity),
            "unique_donors": int(recipients.donor_id.nunique()),
            "shared_donors": int(recipients.groupby("donor_id").size().eq(2).sum()),
            "donor_cluster_size_distribution": donor_cluster_distribution,
            "exact_category_counts": distribution_counts,
            "numerical_ranges_and_tie_aware_modes": numerical_ranges,
            "target_distribution": target_report,
            "anchor_scalability": anchor_report,
            "risk_relationship": risk_fit,
            "approved_measurement_ranges": canonical_ranges,
            "approved_measurement_range_provenance": (
                "Approved Qwen measurement bounds are aggregate minima/maxima from the protected canonical dataset; "
                "no source records were copied into the production skeleton or any Qwen-generated dataset."
            ),
            "mode_reporting_policy": (
                "All values tied at the maximum frequency are reported in dominant_values; no arbitrary tied value is presented as a unique mode."
            ),
        }

        actual_contract_report = {
            "status": "actual_contract_frozen",
            "design_id": DESIGN_ID,
            "confirmed_generator_version": PROMPT_VERSION,
            "prompt_template_sha256": EXPECTED_PROMPT_SHA256,
            "prompt_template_unchanged": True,
            "render_prompt_function_source_sha256": prompt_contract["render_prompt_source_sha256"],
            "render_prompt_function_unchanged": True,
            "finding": (
                "The exact render_prompt function constructs a static_fields dictionary and includes a replacement for "
                "{static_clinical_fields}, but the saved v3.2 template contains no such placeholder. The replacement is "
                "therefore a no-op. Explicit static clinical field names and values were not inserted into the confirmed Qwen prompts."
            ),
            "actual_frozen_contract": {
                "python_uses_static_clinical_fields_for": [
                    "risk construction",
                    "event construction",
                    "relationship metadata",
                    "audit metadata",
                    "Python-derived ABO compatibility",
                ],
                "qwen_receives": [
                    "pseudonymous deterministic seed",
                    "baseline creatinine",
                    "four generation-only day-7 numerical anchors",
                    "recovery pattern",
                    "event schedule",
                    "rejection signal strength",
                    "infection schedule",
                    "adherence pattern",
                    "tacrolimus-exposure pattern",
                    "non-rejection confounder",
                    "fixed output schema and approved measurement ranges",
                ],
                "qwen_does_not_receive": [
                    "explicit static clinical field names or values",
                    "direct identifiers",
                    "sensitive audit fields such as ethnicity or region",
                    "consent fields",
                    "retention fields",
                    "target labels",
                ],
                "sensitive_audit_fields_influence_qwen_measurements": False,
            },
            "three_rendered_preview_audits": prompt_contract["audit_rows"],
            "response_ledger_limitation": (
                "The immutable v3.2 responses ledger preserves complete responses but not rendered requests; these local previews reconstruct the exact prompts from the frozen function inputs without making an API call."
            ),
            "behavior_change_made": False,
            "static_fields_added_to_prompt": False,
            "api_requests": 0,
        }

        confirmation_tokens = confirmation_report["token_usage_total"]
        response_size = (V32_CONFIRM / "responses.jsonl").stat().st_size
        manifest_size = (V32_CONFIRM / "manifest.jsonl").stat().st_size
        assessment_size = (V32_CONFIRM / "assessment_table.csv").stat().st_size
        identity_size = (V32_CONFIRM / "identity_table.csv").stat().st_size
        storage_estimate = {
            "status": "provisional_planning_estimate",
            "initial_tokens_per_recipient": PROVISIONAL_TOKENS_PER_RECIPIENT,
            "estimated_total_tokens": RECIPIENT_COUNT * PROVISIONAL_TOKENS_PER_RECIPIENT,
            "v3_2_confirmation_observed": {
                "recipients": 20,
                "prompt_tokens": confirmation_tokens["prompt_tokens"],
                "completion_tokens": confirmation_tokens["completion_tokens"],
                "total_tokens": confirmation_tokens["total_tokens"],
                "observed_total_tokens_per_recipient": confirmation_tokens["total_tokens"] / 20,
            },
            "provisional_note": (
                "1,582 tokens per recipient is the initial rounded estimate from the 20-recipient confirmation and must be updated after an authorised first-chunk benchmark."
            ),
            "storage_estimates_bytes": {
                "responses_jsonl": int(math.ceil(response_size / 20 * RECIPIENT_COUNT)),
                "future_manifest_or_status_jsonl_at_confirmation_density": int(math.ceil(manifest_size / 20 * RECIPIENT_COUNT)),
                "validated_assessment_csv_at_confirmation_density": int(math.ceil(assessment_size / 120 * ASSESSMENT_COUNT)),
                "identity_csv_at_confirmation_density": int(math.ceil(identity_size / 40 * IDENTITY_COUNT)),
                "prospective_requests_jsonl_using_mock_line_size": len(mock_ledger.encode("utf-8")) * RECIPIENT_COUNT,
            },
            "estimation_basis_fingerprints": {
                name: file_fingerprint(V32_CONFIRM / name)
                for name in ["responses.jsonl", "manifest.jsonl", "assessment_table.csv", "identity_table.csv", "run_report.json"]
            },
        }

        production_freeze = {
            "status": "frozen_local_design",
            "design_id": DESIGN_ID,
            "production_run_id": PRODUCTION_RUN_ID,
            "generator_version": PROMPT_VERSION,
            "prompt_template_sha256": EXPECTED_PROMPT_SHA256,
            "anchor_algorithm_version": ANCHOR_ALGORITHM_VERSION,
            "prompt_behavior_changed": False,
            "production_dimensions": {
                "recipients": RECIPIENT_COUNT,
                "assessments_per_recipient": len(DAYS),
                "assessment_metadata_rows": ASSESSMENT_COUNT,
                "identity_rows": IDENTITY_COUNT,
                "unique_donors": UNIQUE_DONOR_COUNT,
                "hospitals": int(recipients.hospital_id.nunique()),
            },
            "new_namespaces": {
                "recipient": "V32P-R######",
                "assessment": "V32P-A######",
                "donor": ["V32P-DL######", "V32P-DD######", "V32P-DP######"],
                "request": "QWEN-V32-PROD-REQ######",
            },
            "no_diagnostic_or_confirmation_recipient_reuse": True,
            "independent_generation": (
                "Every recipient has a unique deterministic request seed. Exact scaled categorical allocations are ordered by seed-derived SHA-256 scores; continuous values, event draws and anchors use recipient-specific random generators. The 100-recipient block is not copied or repeated."
            ),
            "target_summary": target_report,
            "deletion_summary": {
                name: report["matched_assessment_rows"] for name, report in deletion_report["scenarios"].items()
            },
            "classifier_feature_contract": {
                "features": field_lists["classifier_features"],
                "target": field_lists["target"],
                "identifiers_consent_event_schedules_dates_and_anchors_excluded": True,
                "anchors_generation_only": True,
            },
            "proposed_first_chunk_gate": PROPOSED_CHUNK_1_GATE,
            "proposed_first_chunk_gate_enabled": False,
            "api_requests": 0,
            "network_requests": 0,
            "full_production_enabled": False,
        }

        PRODUCTION_DIR.mkdir(parents=False, exist_ok=False)
        preview_names = {}
        for recipient_id, rendered in prompt_contract["previews"].items():
            name = f"rendered_prompt_preview_{recipient_id}.txt"
            write_text(PRODUCTION_DIR / name, rendered)
            preview_names[recipient_id] = name
        write_json(PRODUCTION_DIR / "production_freeze_report.json", production_freeze)
        write_json(PRODUCTION_DIR / "actual_prompt_contract_audit.json", actual_contract_report)
        write_json(PRODUCTION_DIR / "request_ledger_schema.json", ledger_schema)
        mock_append_result = append_request_ledger_line(
            PRODUCTION_DIR / "mock_requests.jsonl", mock_ledger
        )
        ledger_validation.update(
            {
                "append_only_file_test": "passed",
                "flush_and_fsync_completed": True,
                "persisted_line_number": mock_append_result["persisted_line_number"],
            }
        )
        write_csv(PRODUCTION_DIR / "production_recipient_metadata.csv", recipients)
        write_csv(PRODUCTION_DIR / "production_assessment_metadata.csv", assessments)
        write_csv(PRODUCTION_DIR / "production_identity_skeleton.csv", identity)
        write_csv(PRODUCTION_DIR / "production_anchor_metadata.csv", anchors)
        write_csv(PRODUCTION_DIR / "production_event_target_audit.csv", event_audit)
        write_json(PRODUCTION_DIR / "production_distribution_report.json", distribution_report)
        write_json(PRODUCTION_DIR / "production_deletion_scenario_audit.json", deletion_report)
        write_json(PRODUCTION_DIR / "production_chunk_plan.json", chunk_plan)
        write_json(PRODUCTION_DIR / "production_storage_and_token_estimate.json", storage_estimate)

        created_files = sorted(path.name for path in PRODUCTION_DIR.iterdir() if path.is_file())
        expected_files = {
            "production_freeze_report.json",
            "actual_prompt_contract_audit.json",
            "request_ledger_schema.json",
            "mock_requests.jsonl",
            "production_recipient_metadata.csv",
            "production_assessment_metadata.csv",
            "production_identity_skeleton.csv",
            "production_anchor_metadata.csv",
            "production_event_target_audit.csv",
            "production_distribution_report.json",
            "production_deletion_scenario_audit.json",
            "production_chunk_plan.json",
            "production_storage_and_token_estimate.json",
            *preview_names.values(),
            "production_validation_report.json",
        }
        assert set(created_files) | {"production_validation_report.json"} == expected_files
        validation_report.update(
            {
                "status": "completed",
                "validation_checks": {key: bool(value) for key, value in checks.items()},
                "validation_failures": failures,
                "skeleton_dimensions": {
                    "recipient_metadata": list(recipients.shape),
                    "assessment_metadata": list(assessments.shape),
                    "identity_skeleton": list(identity.shape),
                    "anchor_metadata": list(anchors.shape),
                    "event_target_audit": list(event_audit.shape),
                },
                "identity_counts": {
                    "recipients": int(identity.person_role.eq("Recipient").sum()),
                    "donors": int(identity.person_role.eq("Donor").sum()),
                    "unique_donors": int(recipients.donor_id.nunique()),
                    "shared_donors": int(recipients.groupby("donor_id").size().eq(2).sum()),
                },
                "target_results": target_report,
                "deletion_results": deletion_report,
                "anchor_diversity_results": anchor_report,
                "prompt_contract_finding": actual_contract_report["finding"],
                "request_ledger_mock_validation": ledger_validation,
                "token_storage_estimate": storage_estimate,
                "created_files": sorted(expected_files),
                "production_chunk_plan_prepared": True,
                "production_chunk_1_executed": False,
                "proposed_chunk_1_gate": PROPOSED_CHUNK_1_GATE,
                "proposed_chunk_1_gate_enabled": False,
            }
        )
    except Exception as exc:
        error = exc
        validation_report["status"] = "failed"
        validation_report["error"] = {"type": type(exc).__name__, "message": str(exc)}
    finally:
        protected_in_finally = protected_fingerprint()
        immutable_in_finally = {display_path(path): tree_fingerprint(path) for path in immutable_paths}
        notebooks_in_finally = {display_path(path): file_fingerprint(path) for path in notebooks_01_05}
        evidence_in_finally = {display_path(path): file_fingerprint(path) for path in evidence_files}
        validation_report.update(
            {
                "protected_in_finally": protected_in_finally,
                "protected_integrity_in_finally": protected_in_finally == protected_before,
                "immutable_evidence_in_finally": immutable_in_finally,
                "immutable_evidence_unchanged_in_finally": immutable_in_finally == immutable_before,
                "notebooks_01_05_in_finally": notebooks_in_finally,
                "notebooks_01_05_unchanged_in_finally": notebooks_in_finally == notebooks_before,
                "input_evidence_fingerprints_in_finally": evidence_in_finally,
                "input_evidence_unchanged_in_finally": evidence_in_finally == evidence_before,
            }
        )
        if PRODUCTION_DIR.exists():
            write_json(PRODUCTION_DIR / "production_validation_report.json", validation_report)
        assert protected_in_finally == protected_before
        assert immutable_in_finally == immutable_before
        assert notebooks_in_finally == notebooks_before
        assert evidence_in_finally == evidence_before
    if error is not None:
        raise error

    protected_after = protected_fingerprint()
    immutable_after = {display_path(path): tree_fingerprint(path) for path in immutable_paths}
    notebooks_after = {display_path(path): file_fingerprint(path) for path in notebooks_01_05}
    evidence_after = {display_path(path): file_fingerprint(path) for path in evidence_files}
    validation_report.update(
        {
            "protected_after_execution": protected_after,
            "protected_integrity_after_execution": protected_after == protected_before,
            "immutable_evidence_after_execution": immutable_after,
            "immutable_evidence_unchanged_after_execution": immutable_after == immutable_before,
            "notebooks_01_05_after_execution": notebooks_after,
            "notebooks_01_05_unchanged_after_execution": notebooks_after == notebooks_before,
            "input_evidence_fingerprints_after_execution": evidence_after,
            "input_evidence_unchanged_after_execution": evidence_after == evidence_before,
            "created_file_fingerprints_excluding_self": {
                path.name: file_fingerprint(path)
                for path in sorted(PRODUCTION_DIR.iterdir())
                if path.is_file() and path.name != "production_validation_report.json"
            },
        }
    )
    write_json(PRODUCTION_DIR / "production_validation_report.json", validation_report)
    assert protected_after == protected_before
    assert immutable_after == immutable_before
    assert notebooks_after == notebooks_before
    assert evidence_after == evidence_before
    assert API_REQUESTS_MADE == 0 and NETWORK_REQUESTS_MADE == 0
    assert not RUN_PRODUCTION_CHUNK_001 and not RUN_10000_RECIPIENTS and not RUN_FULL_GENERATION
    return validation_report


PRODUCTION_DESIGN_VALIDATION = run_production_design()
print(
    json.dumps(
        {
            "status": PRODUCTION_DESIGN_VALIDATION["status"],
            "api_requests_made": PRODUCTION_DESIGN_VALIDATION["api_requests_made"],
            "network_requests_made": PRODUCTION_DESIGN_VALIDATION["network_requests_made"],
            "skeleton_dimensions": PRODUCTION_DESIGN_VALIDATION["skeleton_dimensions"],
            "identity_counts": PRODUCTION_DESIGN_VALIDATION["identity_counts"],
            "target_results": PRODUCTION_DESIGN_VALIDATION["target_results"],
            "deletion_counts": {
                name: report["matched_assessment_rows"]
                for name, report in PRODUCTION_DESIGN_VALIDATION["deletion_results"]["scenarios"].items()
            },
            "prompt_contract_finding": PRODUCTION_DESIGN_VALIDATION["prompt_contract_finding"],
            "request_ledger_mock_validation": PRODUCTION_DESIGN_VALIDATION["request_ledger_mock_validation"],
            "protected_integrity_after_execution": PRODUCTION_DESIGN_VALIDATION["protected_integrity_after_execution"],
            "immutable_evidence_unchanged_after_execution": PRODUCTION_DESIGN_VALIDATION["immutable_evidence_unchanged_after_execution"],
            "production_chunk_1_executed": PRODUCTION_DESIGN_VALIDATION["production_chunk_1_executed"],
            "full_generation_enabled": PRODUCTION_DESIGN_VALIDATION["full_generation_enabled"],
            "created_files": PRODUCTION_DESIGN_VALIDATION["created_files"],
        },
        indent=2,
    )
)


## Guarded v3.2 production chunk 001

This cell executes only with the exact `RUN_QWEN_V32_PRODUCTION_CHUNK_001` gate. It re-verifies the frozen production design and immutable inputs, makes at most one sequential request for each of the first 100 frozen recipients with retries disabled, preserves each request and complete response before extraction, checkpoints each validated recipient, and stops on the first failure. Chunks 002–100 and final dataset assembly remain disabled.

In [ ]:
import hashlib
import json
import math
import os
import time
import traceback
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from openai import OpenAI


ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists())
RAW_KIDNEY = ROOT / "data/raw/KidneyTransplant"
V31_DESIGN = RAW_KIDNEY / "qwen_v3_1_design"
V32_DESIGN = RAW_KIDNEY / "qwen_v3_2_design"
PRODUCTION_DESIGN = RAW_KIDNEY / "qwen_v3_2_production_design_001"
PRODUCTION_ROOT = RAW_KIDNEY / "qwen_v3_2_production_001"
CHUNK_DIR = PRODUCTION_ROOT / "chunk_001"
PROTECTED_DATASET = ROOT / "data/raw/kidney_transplant_unlearning_dataset.csv"

RUN_ID = "QWEN-V32-PROD-001"
CHUNK_ID = "CHUNK-001"
FROZEN_CHUNK_ID = "QWEN-V32-PROD-CHUNK001"
GATE_NAME = "RUN_QWEN_V32_PRODUCTION_CHUNK_001"
GATE_VALUE = "QWEN-V32-PROD-001-CHUNK-001"
BASE_URL = "https://resolution-andreas-alerts-blah.trycloudflare.com/v1"
API_KEY = "local-key"
PROMPT_VERSION = "qwen-kidney-v3.2"
PROMPT_SHA256 = "6cbdec644f63e9905795e2a71d444beab79fc50f8847b6446673df049b36d747"
ANCHOR_ALGORITHM_VERSION = "qwen-kidney-v3.2-anchor-v1"
EXPECTED_PRODUCTION_VALIDATION_SHA256 = "532f8457e0e18f8b7581ac0031532820796bb46acb05746fb4ab4517768d2b73"
EXPECTED_PROTECTED = {
    "sha256": "8f4b6b51f96b753490cbef542643ab363408e472bf6dcf21b2c91cc3176648b7",
    "mtime_ns": 1786015387691857656,
    "size_bytes": 19817033,
}

DAYS = [7, 14, 30, 60, 90, 180]
QWEN_FIELDS = [
    "days_since_transplant",
    "creatinine_mg_dl",
    "urine_output_ml_24h",
    "tacrolimus_level_ng_ml",
    "medication_adherence_pct",
]
ANCHOR_TOLERANCES = {
    "creatinine_mg_dl": 0.12,
    "urine_output_ml_24h": 120.0,
    "tacrolimus_level_ng_ml": 0.6,
    "medication_adherence_pct": 1.5,
}
MAX_API_REQUESTS = 100
EXPECTED_RECIPIENTS = 100
EXPECTED_ROWS = 600
EXPECTED_RETURNED_FIELDS = 3_000
RUN_CHUNKS_002_TO_100 = False
RUN_10000_RECIPIENTS = False
RUN_FULL_GENERATION = False
assert not RUN_CHUNKS_002_TO_100
assert not RUN_10000_RECIPIENTS
assert not RUN_FULL_GENERATION


def sha256_text(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def display_path(path):
    return str(path.relative_to(ROOT)) if path.is_relative_to(ROOT) else str(path)


def file_fingerprint(path):
    stat = path.stat()
    return {
        "path": display_path(path),
        "sha256": sha256_file(path),
        "mtime_ns": stat.st_mtime_ns,
        "size_bytes": stat.st_size,
    }


def tree_fingerprint(path):
    paths = [path] if path.is_file() else sorted(item for item in path.rglob("*") if item.is_file())
    base = path.parent if path.is_file() else path
    records = []
    for item in paths:
        stat = item.stat()
        records.append(
            (
                str(item.relative_to(base)),
                sha256_file(item),
                stat.st_mtime_ns,
                stat.st_size,
            )
        )
    aggregate = sha256_text("\n".join("|".join(map(str, record)) for record in records))
    return {
        "path": display_path(path),
        "file_count": len(records),
        "total_size_bytes": sum(record[3] for record in records),
        "aggregate_sha256_with_metadata": aggregate,
    }


def protected_fingerprint():
    result = file_fingerprint(PROTECTED_DATASET)
    result["mtime_utc"] = datetime.fromtimestamp(
        result["mtime_ns"] / 1_000_000_000, tz=timezone.utc
    ).isoformat()
    return result


def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, (pd.Timestamp, datetime, date)):
        return value.isoformat()
    if pd.isna(value) if not isinstance(value, (str, bytes)) else False:
        return None
    return value


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def write_json_exclusive(path, value):
    with path.open("x", encoding="utf-8", newline="") as handle:
        handle.write(json.dumps(json_safe(value), indent=2) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def write_json_state(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="") as handle:
        handle.write(json.dumps(json_safe(value), indent=2) + "\n")
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def write_csv_exclusive(path, frame):
    with path.open("x", encoding="utf-8", newline="") as handle:
        frame.to_csv(handle, index=False, lineterminator="\n")
        handle.flush()
        os.fsync(handle.fileno())


def initialize_checkpoint(path, columns):
    with path.open("x", encoding="utf-8", newline="") as handle:
        pd.DataFrame(columns=columns).to_csv(handle, index=False, lineterminator="\n")
        handle.flush()
        os.fsync(handle.fileno())


def append_checkpoint(path, frame):
    with path.open("a", encoding="utf-8", newline="") as handle:
        frame.to_csv(handle, index=False, header=False, lineterminator="\n")
        handle.flush()
        os.fsync(handle.fileno())


def append_jsonl(path, record):
    serialized = json.dumps(json_safe(record), separators=(",", ":"), sort_keys=True)
    with path.open("a", encoding="utf-8", newline="") as handle:
        handle.write(serialized + "\n")
        handle.flush()
        os.fsync(handle.fileno())
    return sha256_text(serialized)


def count_jsonl(path):
    if not path.exists():
        return 0
    return sum(1 for line in path.read_text(encoding="utf-8").splitlines() if line.strip())


def is_number(value):
    return (
        isinstance(value, (int, float, np.integer, np.floating))
        and not isinstance(value, (bool, np.bool_))
        and math.isfinite(float(value))
    )


def derive_abo_category(donor_group, recipient_group):
    compatible = {
        "O": {"O", "A", "B", "AB"},
        "A": {"A", "AB"},
        "B": {"B", "AB"},
        "AB": {"AB"},
    }
    return "Standard compatible" if recipient_group in compatible[donor_group] else "Managed incompatibility"


def derive_changes(baseline, values):
    changes = []
    previous = float(baseline)
    for current in values:
        current = float(current)
        changes.append(100.0 * (current - previous) / previous)
        previous = current
    return changes


def derive_event_fields(event_days):
    events = sorted(int(day) for day in event_days)
    previous = [int(any(event < day for event in events)) for day in DAYS]
    targets = [int(any(day < event <= day + 30 for event in events)) for day in DAYS]
    return previous, targets


def tied_mode(series):
    counts = series.value_counts(dropna=False)
    maximum = int(counts.max())
    modes = [json_safe(value) for value in counts[counts.eq(maximum)].index.tolist()]
    return {
        "dominant_values": modes,
        "dominant_count": maximum,
        "dominant_fraction": float(maximum / len(series)),
        "mode_is_tied": len(modes) > 1,
        "unique_values": int(series.nunique(dropna=False)),
    }


def correlation(left, right):
    left = np.asarray(left, dtype=float)
    right = np.asarray(right, dtype=float)
    if len(left) < 2 or np.std(left) == 0 or np.std(right) == 0:
        return None
    return float(np.corrcoef(left, right)[0, 1])


def immutable_inputs():
    raw_entries = sorted(
        [path for path in (ROOT / "data/raw").iterdir() if path.name != "KidneyTransplant"]
        + [path for path in RAW_KIDNEY.iterdir() if path != PRODUCTION_ROOT],
        key=lambda path: str(path),
    )
    return raw_entries + [ROOT / "data/processed", ROOT / "models", ROOT / "results"]


def snapshot_immutable(paths):
    return {display_path(path): tree_fingerprint(path) for path in paths}


def snapshot_notebooks_01_05():
    paths = sorted((ROOT / "notebooks/KidneyTransplant").glob("0[1-5]_*.ipynb"))
    return {display_path(path): file_fingerprint(path) for path in paths}


def verify_frozen_production_design():
    report_path = PRODUCTION_DESIGN / "production_validation_report.json"
    assert sha256_file(report_path) == EXPECTED_PRODUCTION_VALIDATION_SHA256
    report = json.loads(report_path.read_text(encoding="utf-8"))
    expected = report["created_file_fingerprints_excluding_self"]
    checks = {}
    for name, recorded in expected.items():
        path = PRODUCTION_DESIGN / name
        current = file_fingerprint(path)
        checks[name] = all(
            current[key] == recorded[key] for key in ["sha256", "mtime_ns", "size_bytes"]
        )
    checks["production_validation_report.json"] = True
    actual_files = {path.name for path in PRODUCTION_DESIGN.iterdir() if path.is_file()}
    checks["exact_17_artifact_set"] = actual_files == set(expected) | {"production_validation_report.json"}
    assert all(checks.values())
    return checks, report


def render_prompt(template, recipient, anchor):
    static_fields = {
        field: json_safe(recipient[field])
        for field in [
            "recipient_age",
            "donor_age",
            "donor_type",
            "kidney_failure_cause",
            "previous_transplant",
            "dialysis_months",
            "recipient_blood_group",
            "donor_blood_group",
            "hla_mismatch_count",
            "antibody_risk_score",
            "cold_ischaemia_hours",
        ]
    }
    replacements = {
        "{recipient_seed}": str(int(recipient["request_seed"])),
        "{static_clinical_fields}": json.dumps(static_fields, sort_keys=True),
        "{baseline_creatinine_mg_dl}": str(float(recipient["baseline_creatinine_mg_dl"])),
        "{generation_only_event_schedule}": str(recipient["confirmed_rejection_event_days"]),
        "{signal_strength}": str(recipient["rejection_signal_strength"]),
        "{confounder_instructions}": str(recipient["non_rejection_confounders"]),
        "{adherence_pattern}": str(recipient["adherence_pattern"]),
        "{infection_episode_days}": str(recipient["infection_episode_days"]),
        "{tacrolimus_exposure}": str(recipient["tacrolimus_exposure_pattern"]),
        "{recovery_pattern}": str(recipient["recovery_pattern"]),
        "{day7_creatinine_anchor_mg_dl}": str(anchor["day7_creatinine_anchor_mg_dl"]),
        "{day7_urine_output_anchor_ml_24h}": str(anchor["day7_urine_output_anchor_ml_24h"]),
        "{day7_tacrolimus_anchor_ng_ml}": str(anchor["day7_tacrolimus_anchor_ng_ml"]),
        "{day7_medication_adherence_anchor_pct}": str(anchor["day7_medication_adherence_anchor_pct"]),
    }
    rendered = template
    for old, new in replacements.items():
        rendered = rendered.replace(old, new)
    assert all(token not in rendered for token in replacements)
    return rendered


def anchor_row_hash(anchor):
    return sha256_text(json.dumps(json_safe(anchor), sort_keys=True, separators=(",", ":")))


def preflight():
    assert not PRODUCTION_ROOT.exists(), "production root already exists; chunk 1 was previously started"
    assert not CHUNK_DIR.exists(), "chunk_001 already exists; automatic resume prohibited"
    assert os.environ.get(GATE_NAME) == GATE_VALUE, "exact chunk-1 gate mismatch"
    for number in range(2, 101):
        assert not os.environ.get(f"RUN_QWEN_V32_PRODUCTION_CHUNK_{number:03d}"), f"chunk {number} gate must be false"
    for name in [
        "RUN_QWEN_V32_CONFIRM20",
        "RUN_CORRECTED_10",
        "RUN_100_RECIPIENTS",
        "RUN_10000_RECIPIENTS",
        "RUN_FULL_GENERATION",
    ]:
        assert not os.environ.get(name), f"{name} must be false"

    design_checks, design_validation = verify_frozen_production_design()
    prompt_path = V32_DESIGN / "qwen_kidney_v3_2_prompt_template.txt"
    template = prompt_path.read_text(encoding="utf-8")
    assert sha256_text(template) == PROMPT_SHA256
    assert "{static_clinical_fields}" not in template

    api_config = json.loads((V32_DESIGN / "api_configuration.json").read_text(encoding="utf-8"))
    assert api_config["client"]["max_retries"] == 0
    assert api_config["automatic_retry"] is False
    assert api_config["completion"] == {
        "model": "Qwen/Qwen3.6-35B-A3B",
        "max_tokens": 8192,
        "temperature": 0.7,
        "top_p": 0.8,
        "presence_penalty": 1.5,
        "extra_body": {"top_k": 20, "chat_template_kwargs": {"enable_thinking": False}},
    }
    assert MAX_API_REQUESTS == 100

    recipients_all = pd.read_csv(PRODUCTION_DESIGN / "production_recipient_metadata.csv")
    assessments_all = pd.read_csv(PRODUCTION_DESIGN / "production_assessment_metadata.csv")
    identities_all = pd.read_csv(
        PRODUCTION_DESIGN / "production_identity_skeleton.csv", keep_default_na=False
    )
    anchors_all = pd.read_csv(PRODUCTION_DESIGN / "production_anchor_metadata.csv")
    recipients = recipients_all.loc[
        recipients_all.request_sequence_number.between(1, 100)
    ].sort_values("request_sequence_number").reset_index(drop=True)
    recipient_ids = recipients.recipient_id.tolist()
    assessments = assessments_all.loc[
        assessments_all.recipient_id.isin(recipient_ids)
    ].sort_values(["recipient_id", "days_since_transplant"]).reset_index(drop=True)
    anchors = anchors_all.loc[anchors_all.recipient_id.isin(recipient_ids)].sort_values(
        "recipient_id"
    ).reset_index(drop=True)
    entity_ids = set(recipient_ids) | set(recipients.donor_id)
    identities = identities_all.loc[identities_all.entity_id.isin(entity_ids)].copy()

    checks = {
        "frozen_design_artifacts": all(design_checks.values()),
        "prompt_hash": sha256_text(template) == PROMPT_SHA256,
        "anchor_algorithm_version": anchors.anchor_algorithm_version.eq(ANCHOR_ALGORITHM_VERSION).all(),
        "recipient_count": len(recipients) == EXPECTED_RECIPIENTS,
        "new_recipient_ids": recipients.recipient_id.str.startswith("V32P-R").all(),
        "no_prior_recipient_namespace": not recipients.recipient_id.str.startswith(("V31-", "QWEN-")).any(),
        "request_sequence_exact": recipients.request_sequence_number.tolist() == list(range(1, 101)),
        "request_ids_unique": recipients.request_id.is_unique,
        "request_seeds_unique": recipients.request_seed.is_unique,
        "chunk_id_exact": recipients.chunk_id.eq(FROZEN_CHUNK_ID).all(),
        "assessment_metadata_rows": len(assessments) == EXPECTED_ROWS,
        "six_assessments_each": assessments.groupby("recipient_id").size().eq(6).all(),
        "assessment_days_exact": assessments.groupby("recipient_id").days_since_transplant.apply(list).map(lambda values: values == DAYS).all(),
        "anchor_rows": len(anchors) == EXPECTED_RECIPIENTS,
        "attempt_number": 1 == 1,
        "maximum_requests": MAX_API_REQUESTS == EXPECTED_RECIPIENTS,
        "automatic_retries_disabled": api_config["client"]["max_retries"] == 0 and api_config["automatic_retry"] is False,
        "other_gates_false": not RUN_CHUNKS_002_TO_100 and not RUN_10000_RECIPIENTS and not RUN_FULL_GENERATION,
    }
    assert all(checks.values()), [name for name, passed in checks.items() if not passed]

    # Independently reproduce every frozen Python-derived chunk field before any request.
    recipient_lookup = recipients.set_index("recipient_id")
    for row in assessments.itertuples(index=False):
        recipient = recipient_lookup.loc[row.recipient_id]
        transplant = date.fromisoformat(recipient.transplant_date)
        assessment_date = transplant + timedelta(days=int(row.days_since_transplant))
        assert assessment_date.isoformat() == row.assessment_date
        assert (assessment_date + timedelta(days=730)).isoformat() == row.retention_expiry_date
        assert derive_abo_category(recipient.donor_blood_group, recipient.recipient_blood_group) == row.abo_compatibility_category
        infections = set(json.loads(recipient.infection_episode_days))
        previous, targets = derive_event_fields(json.loads(recipient.confirmed_rejection_event_days))
        position = DAYS.index(int(row.days_since_transplant))
        assert int(row.infection_indicator) == int(row.days_since_transplant in infections)
        assert int(row.previous_rejection) == previous[position]
        assert int(row.acute_rejection_within_30_days) == targets[position]

    field_lists = json.loads((V31_DESIGN / "field_lists.json").read_text(encoding="utf-8"))
    canonical_ranges = json.loads((V31_DESIGN / "canonical_qwen_ranges.json").read_text(encoding="utf-8"))
    assessment_columns = pd.read_csv(V31_DESIGN / "assessment_table_schema.csv").field_name.tolist()
    direct = set(field_lists["direct_identifiers"])
    assert direct <= set(identities.columns)
    assert not direct.intersection(recipients.columns)
    assert not direct.intersection(assessments.columns)
    assert not direct.intersection(anchors.columns)
    classifier = set(field_lists["classifier_features"])
    forbidden = set(
        field_lists["pseudonymous_identifiers"]
        + field_lists["assessment_audit_only"]
        + field_lists["event_control_metadata"]
        + ["assessment_date", "retention_expiry_date", "transplant_date", "training_consent_status", "training_consent_version"]
        + [column for column in anchors.columns if "anchor" in column]
    )
    assert not classifier.intersection(forbidden)
    assert len(assessment_columns) == 31

    immutable_paths = immutable_inputs()
    immutable_before = snapshot_immutable(immutable_paths)
    notebooks_before = snapshot_notebooks_01_05()
    protected_before = protected_fingerprint()
    assert {key: protected_before[key] for key in EXPECTED_PROTECTED} == EXPECTED_PROTECTED
    production_design_before = tree_fingerprint(PRODUCTION_DESIGN)
    return {
        "checks": checks,
        "design_validation": design_validation,
        "template": template,
        "api_config": api_config,
        "recipients": recipients,
        "assessments": assessments,
        "identities": identities,
        "anchors": anchors,
        "field_lists": field_lists,
        "canonical_ranges": canonical_ranges,
        "assessment_columns": assessment_columns,
        "immutable_paths": immutable_paths,
        "immutable_before": immutable_before,
        "notebooks_before": notebooks_before,
        "protected_before": protected_before,
        "production_design_before": production_design_before,
    }


def validate_payload(payload, anchor, canonical_ranges):
    errors = []
    warnings = []
    if not isinstance(payload, dict):
        return ["root must be one JSON object"], warnings
    if set(payload) != {"assessments"}:
        errors.append("root must contain only assessments")
    items = payload.get("assessments")
    if not isinstance(items, list):
        return errors + ["assessments must be an array"], warnings
    if len(items) != 6:
        errors.append("assessments must contain exactly six objects")
        return errors, warnings
    for position, item in enumerate(items):
        if not isinstance(item, dict):
            errors.append(f"assessment {position} must be an object")
            continue
        if set(item) != set(QWEN_FIELDS):
            errors.append(f"assessment {position} key set is invalid")
            continue
        if not all(is_number(item[field]) for field in QWEN_FIELDS):
            errors.append(f"assessment {position} contains non-finite or non-numeric values")
            continue
        if int(item["days_since_transplant"]) != DAYS[position] or float(item["days_since_transplant"]) != DAYS[position]:
            errors.append(f"assessment {position} day is invalid")
        for field, bounds in canonical_ranges.items():
            value = float(item[field])
            if not bounds["minimum"] <= value <= bounds["maximum"]:
                errors.append(f"assessment {position} {field} outside approved range")
    if errors:
        return errors, warnings
    day7 = items[0]
    anchor_map = {
        "creatinine_mg_dl": "day7_creatinine_anchor_mg_dl",
        "urine_output_ml_24h": "day7_urine_output_anchor_ml_24h",
        "tacrolimus_level_ng_ml": "day7_tacrolimus_anchor_ng_ml",
        "medication_adherence_pct": "day7_medication_adherence_anchor_pct",
    }
    for field, anchor_field in anchor_map.items():
        difference = abs(float(day7[field]) - float(anchor[anchor_field]))
        if difference > ANCHOR_TOLERANCES[field] + 1e-12:
            errors.append(f"day-7 {field} differs from anchor by {difference}, exceeding tolerance")
    for field, bounds in canonical_ranges.items():
        values = np.array([float(item[field]) for item in items])
        span = float(bounds["maximum"] - bounds["minimum"])
        deltas = np.abs(np.diff(values))
        if (deltas > span + 1e-12).any():
            errors.append(f"{field} contains a range-exceeding longitudinal jump")
        elif (deltas > 0.5 * span).any():
            warnings.append(f"{field} contains a large but range-valid consecutive change")
    return errors, warnings


def build_assessment_rows(recipient, anchor, frozen_metadata, payload, assessment_columns):
    items = payload["assessments"]
    creatinine = [float(item["creatinine_mg_dl"]) for item in items]
    changes = derive_changes(recipient["baseline_creatinine_mg_dl"], creatinine)
    rows = []
    for position, metadata in frozen_metadata.sort_values("days_since_transplant").reset_index(drop=True).iterrows():
        item = items[position]
        row = {
            "assessment_id": metadata.assessment_id,
            "recipient_id": metadata.recipient_id,
            "donor_id": metadata.donor_id,
            "hospital_id": metadata.hospital_id,
            "assessment_date": metadata.assessment_date,
            "training_consent_status": metadata.training_consent_status,
            "training_consent_version": metadata.training_consent_version,
            "retention_expiry_date": metadata.retention_expiry_date,
            "recipient_sex": metadata.recipient_sex,
            "recipient_ethnicity": metadata.recipient_ethnicity,
            "recipient_region": metadata.recipient_region,
            "distance_to_transplant_centre_km": metadata.distance_to_transplant_centre_km,
            "recipient_age": metadata.recipient_age,
            "donor_age": metadata.donor_age,
            "donor_type": metadata.donor_type,
            "kidney_failure_cause": metadata.kidney_failure_cause,
            "previous_transplant": metadata.previous_transplant,
            "dialysis_months": metadata.dialysis_months,
            "abo_compatibility_category": metadata.abo_compatibility_category,
            "hla_mismatch_count": metadata.hla_mismatch_count,
            "antibody_risk_score": metadata.antibody_risk_score,
            "cold_ischaemia_hours": metadata.cold_ischaemia_hours,
            "days_since_transplant": int(item["days_since_transplant"]),
            "creatinine_mg_dl": float(item["creatinine_mg_dl"]),
            "creatinine_change_pct": float(changes[position]),
            "urine_output_ml_24h": float(item["urine_output_ml_24h"]),
            "tacrolimus_level_ng_ml": float(item["tacrolimus_level_ng_ml"]),
            "medication_adherence_pct": float(item["medication_adherence_pct"]),
            "infection_indicator": int(metadata.infection_indicator),
            "previous_rejection": int(metadata.previous_rejection),
            "acute_rejection_within_30_days": int(metadata.acute_rejection_within_30_days),
        }
        assert list(row) == assessment_columns
        rows.append(row)
    return rows


def reconcile_payloads(payloads, assessment):
    mismatches = []
    checked = 0
    for recipient_id, payload in payloads.items():
        rows = assessment.loc[assessment.recipient_id.eq(recipient_id)].sort_values("days_since_transplant")
        for position, item in enumerate(payload["assessments"]):
            row = rows.iloc[position]
            for field in QWEN_FIELDS:
                checked += 1
                left = float(item[field])
                right = float(row[field])
                if left != right:
                    mismatches.append(
                        {
                            "recipient_id": recipient_id,
                            "assessment_position": position,
                            "field": field,
                            "response_value": left,
                            "table_value": right,
                        }
                    )
    return {
        "status": "passed" if not mismatches and checked == EXPECTED_RETURNED_FIELDS else "failed",
        "recipient_payloads": len(payloads),
        "qwen_fields_reconciled": checked,
        "expected_qwen_fields": EXPECTED_RETURNED_FIELDS,
        "mismatch_count": len(mismatches),
        "mismatch_examples": mismatches[:20],
        "all_preserved_response_fields_match_validated_table": not mismatches and checked == EXPECTED_RETURNED_FIELDS,
    }


def validate_complete_chunk(assessment, identities, recipients, anchors, payloads, reconciliation, context):
    assessment_columns = context["assessment_columns"]
    direct = set(context["field_lists"]["direct_identifiers"])
    checks = {
        "dimensions": assessment.shape == (EXPECTED_ROWS, len(assessment_columns)),
        "schema_exact": assessment.columns.tolist() == assessment_columns,
        "recipient_count": assessment.recipient_id.nunique() == EXPECTED_RECIPIENTS,
        "six_rows_each": assessment.groupby("recipient_id").size().eq(6).all(),
        "ordered_days": assessment.groupby("recipient_id").days_since_transplant.apply(list).map(lambda values: values == DAYS).all(),
        "unique_request_ids": recipients.request_id.is_unique,
        "unique_recipient_days": not assessment.duplicated(["recipient_id", "days_since_transplant"]).any(),
        "unique_assessment_ids": assessment.assessment_id.is_unique,
        "no_missing_values": not assessment.isna().any().any(),
        "no_duplicate_rows": not assessment.duplicated().any(),
        "response_payloads": len(payloads) == EXPECTED_RECIPIENTS,
        "reconciliation": reconciliation["status"] == "passed",
        "direct_identifiers_absent": not direct.intersection(assessment.columns),
        "joined_42_column_export_absent": len(assessment.columns) == 31,
    }
    frozen = context["assessments"].set_index("assessment_id", drop=False)
    recipient_lookup = recipients.set_index("recipient_id")
    anchor_lookup = anchors.set_index("recipient_id")
    static_fields = [
        "donor_id",
        "hospital_id",
        "recipient_sex",
        "recipient_ethnicity",
        "recipient_region",
        "distance_to_transplant_centre_km",
        "recipient_age",
        "donor_age",
        "donor_type",
        "kidney_failure_cause",
        "previous_transplant",
        "dialysis_months",
        "abo_compatibility_category",
        "hla_mismatch_count",
        "antibody_risk_score",
        "cold_ischaemia_hours",
    ]
    checks["static_fields_constant"] = bool(
        assessment.groupby("recipient_id")[static_fields].nunique(dropna=False).le(1).all().all()
    )
    derivation_errors = []
    range_errors = []
    anchor_errors = []
    for recipient_id, group in assessment.groupby("recipient_id", sort=True):
        group = group.sort_values("days_since_transplant").reset_index(drop=True)
        recipient = recipient_lookup.loc[recipient_id]
        anchor = anchor_lookup.loc[recipient_id]
        expected_changes = derive_changes(
            recipient.baseline_creatinine_mg_dl, group.creatinine_mg_dl.tolist()
        )
        expected_previous, expected_targets = derive_event_fields(
            json.loads(recipient.confirmed_rejection_event_days)
        )
        expected_infections = [
            int(day in set(json.loads(recipient.infection_episode_days))) for day in DAYS
        ]
        if not np.allclose(group.creatinine_change_pct, expected_changes, rtol=0, atol=1e-12):
            derivation_errors.append(f"{recipient_id}:creatinine_change")
        if group.previous_rejection.astype(int).tolist() != expected_previous:
            derivation_errors.append(f"{recipient_id}:previous_rejection")
        if group.acute_rejection_within_30_days.astype(int).tolist() != expected_targets:
            derivation_errors.append(f"{recipient_id}:target")
        if group.infection_indicator.astype(int).tolist() != expected_infections:
            derivation_errors.append(f"{recipient_id}:infection")
        if derive_abo_category(recipient.donor_blood_group, recipient.recipient_blood_group) != group.loc[0, "abo_compatibility_category"]:
            derivation_errors.append(f"{recipient_id}:abo")
        transplant = date.fromisoformat(recipient.transplant_date)
        for position, day in enumerate(DAYS):
            expected_date = transplant + timedelta(days=day)
            if group.loc[position, "assessment_date"] != expected_date.isoformat():
                derivation_errors.append(f"{recipient_id}:assessment_date:{day}")
            if group.loc[position, "retention_expiry_date"] != (expected_date + timedelta(days=730)).isoformat():
                derivation_errors.append(f"{recipient_id}:retention:{day}")
            frozen_row = frozen.loc[group.loc[position, "assessment_id"]]
            for field in context["assessments"].columns:
                current = group.loc[position, field] if field in group.columns else None
                expected = frozen_row[field]
                if field in group.columns:
                    if isinstance(current, (int, float, np.integer, np.floating)) and isinstance(
                        expected, (int, float, np.integer, np.floating)
                    ):
                        matches_frozen = bool(np.isclose(float(current), float(expected), rtol=0, atol=1e-12))
                    else:
                        matches_frozen = str(current) == str(expected)
                    if not matches_frozen and field not in [
                        "infection_indicator",
                        "previous_rejection",
                        "acute_rejection_within_30_days",
                    ]:
                        derivation_errors.append(f"{recipient_id}:frozen:{field}:{day}")
        for field, bounds in context["canonical_ranges"].items():
            if not group[field].between(bounds["minimum"], bounds["maximum"]).all():
                range_errors.append(f"{recipient_id}:{field}")
        day7 = group.iloc[0]
        for field, anchor_field in {
            "creatinine_mg_dl": "day7_creatinine_anchor_mg_dl",
            "urine_output_ml_24h": "day7_urine_output_anchor_ml_24h",
            "tacrolimus_level_ng_ml": "day7_tacrolimus_anchor_ng_ml",
            "medication_adherence_pct": "day7_medication_adherence_anchor_pct",
        }.items():
            if abs(float(day7[field]) - float(anchor[anchor_field])) > ANCHOR_TOLERANCES[field] + 1e-12:
                anchor_errors.append(f"{recipient_id}:{field}")
    checks["python_derivations_exact"] = not derivation_errors
    checks["approved_ranges"] = not range_errors
    checks["day7_anchor_tolerances"] = not anchor_errors
    checks["identity_separate"] = direct <= set(identities.columns)
    failures = [name for name, passed in checks.items() if not bool(passed)]
    return {
        "status": "passed" if not failures else "failed",
        "checks": {name: bool(value) for name, value in checks.items()},
        "failures": failures,
        "derivation_error_examples": derivation_errors[:20],
        "range_error_examples": range_errors[:20],
        "anchor_error_examples": anchor_errors[:20],
    }


def build_diagnostics(assessment, recipients, anchors):
    measurement_fields = [
        "creatinine_mg_dl",
        "urine_output_ml_24h",
        "tacrolimus_level_ng_ml",
        "medication_adherence_pct",
    ]
    numerical_ranges = {
        field: {
            "minimum": float(assessment[field].min()),
            "maximum": float(assessment[field].max()),
        }
        for field in measurement_fields + ["creatinine_change_pct"]
    }
    per_day = {}
    warnings = []
    for day, frame in assessment.groupby("days_since_transplant"):
        per_day[str(int(day))] = {}
        for field in measurement_fields:
            summary = tied_mode(frame[field])
            per_day[str(int(day))][field] = summary
            if summary["dominant_fraction"] >= 0.20:
                warnings.append(
                    f"day {int(day)} {field} dominant value frequency is {summary['dominant_fraction']:.3f}"
                )
    trajectory_keys = assessment.pivot(index="recipient_id", columns="days_since_transplant", values=measurement_fields)
    trajectory_hashes = trajectory_keys.apply(
        lambda row: sha256_text(json.dumps([float(value) for value in row.tolist()])), axis=1
    )
    trajectory_counts = trajectory_hashes.value_counts()
    identical = {
        "identical_trajectory_groups": int(trajectory_counts.gt(1).sum()),
        "recipients_in_identical_trajectory_groups": int(trajectory_counts.loc[trajectory_counts.gt(1)].sum()),
        "maximum_identical_group_size": int(trajectory_counts.max()),
    }
    if identical["identical_trajectory_groups"]:
        warnings.append("one or more identical complete trajectories detected")

    day7 = assessment.loc[assessment.days_since_transplant.eq(7)].sort_values("recipient_id")
    anchor_sorted = anchors.sort_values("recipient_id")
    suffix = day7.recipient_id.str[-6:].astype(int).to_numpy()
    anchor_output = {}
    for output, anchor_field in {
        "creatinine_mg_dl": "day7_creatinine_anchor_mg_dl",
        "urine_output_ml_24h": "day7_urine_output_anchor_ml_24h",
        "tacrolimus_level_ng_ml": "day7_tacrolimus_anchor_ng_ml",
        "medication_adherence_pct": "day7_medication_adherence_anchor_pct",
    }.items():
        anchor_output[output] = {
            "anchor_output_correlation": correlation(anchor_sorted[anchor_field], day7[output]),
            "output_recipient_suffix_correlation": correlation(suffix, day7[output]),
            "anchor_recipient_suffix_correlation": correlation(suffix, anchor_sorted[anchor_field]),
            "anchor_minimum": float(anchor_sorted[anchor_field].min()),
            "anchor_maximum": float(anchor_sorted[anchor_field].max()),
            "anchor_unique_values": int(anchor_sorted[anchor_field].nunique()),
        }
    canonical_ranges = json.loads((V31_DESIGN / "canonical_qwen_ranges.json").read_text())
    anchor_clipping = {}
    for output, anchor_field in {
        "creatinine_mg_dl": "day7_creatinine_anchor_mg_dl",
        "urine_output_ml_24h": "day7_urine_output_anchor_ml_24h",
        "tacrolimus_level_ng_ml": "day7_tacrolimus_anchor_ng_ml",
        "medication_adherence_pct": "day7_medication_adherence_anchor_pct",
    }.items():
        bounds = canonical_ranges[output]
        anchor_clipping[output] = {
            "at_minimum": int(anchor_sorted[anchor_field].eq(bounds["minimum"]).sum()),
            "at_maximum": int(anchor_sorted[anchor_field].eq(bounds["maximum"]).sum()),
        }

    overlap = {}
    positive = assessment.loc[assessment.acute_rejection_within_30_days.eq(1)]
    negative = assessment.loc[assessment.acute_rejection_within_30_days.eq(0)]
    for field in measurement_fields:
        positive_range = [float(positive[field].min()), float(positive[field].max())]
        negative_range = [float(negative[field].min()), float(negative[field].max())]
        overlaps = max(positive_range[0], negative_range[0]) <= min(positive_range[1], negative_range[1])
        overlap[field] = {
            "positive_range": positive_range,
            "negative_range": negative_range,
            "ranges_overlap": bool(overlaps),
        }
        if not overlaps:
            warnings.append(f"positive and negative ranges do not overlap for {field}")

    target_sequences = assessment.groupby("recipient_id").acute_rejection_within_30_days.apply(
        lambda values: json.dumps(values.astype(int).tolist(), separators=(",", ":"))
    )
    event_count = int(recipients.confirmed_rejection_event_days.ne("[]").sum())
    category_counts = {
        field: {str(key): int(value) for key, value in recipients[field].value_counts().sort_index().items()}
        for field in [
            "hospital_id",
            "donor_type",
            "recovery_pattern",
            "adherence_pattern",
            "tacrolimus_exposure_pattern",
            "infection_episode_days",
            "non_rejection_confounders",
        ]
    }
    return {
        "status": "diagnostic_only_not_used_for_regeneration",
        "classifier_trained": False,
        "regeneration_triggered": False,
        "numerical_ranges": numerical_ranges,
        "day_specific_unique_values_and_tie_aware_modes": per_day,
        "identical_complete_trajectories": identical,
        "anchor_output_and_suffix_correlations": anchor_output,
        "anchor_clipping_counts": anchor_clipping,
        "positive_negative_range_overlap": overlap,
        "target_count": int(assessment.acute_rejection_within_30_days.sum()),
        "target_prevalence": float(assessment.acute_rejection_within_30_days.mean()),
        "target_sequence_counts": {
            key: int(value) for key, value in target_sequences.value_counts().sort_index().items()
        },
        "event_recipients": event_count,
        "non_event_recipients": int(len(recipients) - event_count),
        "category_counts": category_counts,
        "warnings": warnings,
    }


def boundary_integrity(context, sequence):
    protected = protected_fingerprint()
    design = tree_fingerprint(PRODUCTION_DESIGN)
    assert protected == context["protected_before"]
    assert design == context["production_design_before"]
    return {"sequence": int(sequence), "protected_unchanged": True, "production_design_unchanged": True}


def run_chunk_001():
    started = time.monotonic()
    context = preflight()
    recipients = context["recipients"]
    frozen_assessments = context["assessments"]
    identities = context["identities"]
    anchors = context["anchors"]
    api_config = context["api_config"]
    assessment_columns = context["assessment_columns"]

    PRODUCTION_ROOT.mkdir(parents=False, exist_ok=False)
    CHUNK_DIR.mkdir(parents=False, exist_ok=False)
    paths = {
        name: CHUNK_DIR / name
        for name in [
            "requests.jsonl",
            "responses.jsonl",
            "manifest.jsonl",
            "validated_recipient_checkpoint.csv",
            "assessment_chunk_001.csv",
            "identity_chunk_001.csv",
            "anchor_audit_chunk_001.csv",
            "request_response_reconciliation.json",
            "diagnostic_quality_report.json",
            "run_configuration.json",
            "completion_state.json",
            "chunk_001_run_report.json",
        ]
    }
    for name in ["requests.jsonl", "responses.jsonl", "manifest.jsonl"]:
        with paths[name].open("x", encoding="utf-8") as handle:
            handle.flush()
            os.fsync(handle.fileno())
    initialize_checkpoint(paths["validated_recipient_checkpoint.csv"], assessment_columns)
    write_csv_exclusive(paths["identity_chunk_001.csv"], identities)

    completion_parameters = dict(api_config["completion"])
    run_configuration = {
        "run_id": RUN_ID,
        "chunk_id": CHUNK_ID,
        "frozen_chunk_id": FROZEN_CHUNK_ID,
        "status": "started_after_preflight",
        "started_at_utc": now_utc(),
        "base_url": BASE_URL,
        "api_key": "REDACTED",
        "model": completion_parameters["model"],
        "completion_parameters": completion_parameters,
        "client": {"max_retries": 0, "timeout_seconds": 180.0},
        "automatic_retry": False,
        "maximum_api_requests": MAX_API_REQUESTS,
        "request_execution": "sequential",
        "attempt_number": 1,
        "prompt_version": PROMPT_VERSION,
        "prompt_template_sha256": PROMPT_SHA256,
        "anchor_algorithm_version": ANCHOR_ALGORITHM_VERSION,
        "gate": f"{GATE_NAME}={GATE_VALUE}",
        "preflight_checks": context["checks"],
        "protected_before": context["protected_before"],
        "production_design_before": context["production_design_before"],
        "persistent_generation_flags": {
            "chunks_002_to_100": RUN_CHUNKS_002_TO_100,
            "RUN_10000_RECIPIENTS": RUN_10000_RECIPIENTS,
            "RUN_FULL_GENERATION": RUN_FULL_GENERATION,
        },
    }
    write_json_exclusive(paths["run_configuration.json"], run_configuration)
    write_json_state(
        paths["completion_state.json"],
        {
            "run_id": RUN_ID,
            "chunk_id": CHUNK_ID,
            "status": "running",
            "automatic_resume": False,
            "requests_attempted": 0,
            "requests_completed": 0,
            "recipients_validated": 0,
            "checkpoint_rows": 0,
        },
    )

    report = {
        "run_id": RUN_ID,
        "chunk_id": CHUNK_ID,
        "status": "running",
        "api_requests_attempted": 0,
        "api_requests_completed": 0,
        "api_responses_received": 0,
        "recipients_validated": 0,
        "attempts": 0,
        "failures": 0,
        "retries": 0,
        "automatic_resume": False,
        "chunk_2_to_100_started": False,
        "final_dataset_assembly_started": False,
    }
    failure = None
    rows = []
    payloads = {}
    usage_records = []
    finish_records = []
    warnings = []
    boundary_checks = []
    validation = None
    diagnostics = None
    reconciliation = None
    anchor_audit = None
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL, max_retries=0, timeout=180.0)
    try:
        recipient_lookup = recipients.set_index("recipient_id")
        anchor_lookup = anchors.set_index("recipient_id")
        seen_request_ids = set()
        for recipient in recipients.to_dict(orient="records"):
            if report["api_requests_attempted"] >= MAX_API_REQUESTS:
                raise RuntimeError("100-request hard cap reached")
            request_id = recipient["request_id"]
            recipient_id = recipient["recipient_id"]
            if request_id in seen_request_ids:
                raise RuntimeError(f"duplicate request ID: {request_id}")
            seen_request_ids.add(request_id)
            anchor = anchor_lookup.loc[recipient_id].to_dict()
            rendered_prompt = render_prompt(context["template"], recipient, anchor)
            messages = [
                {
                    "role": "system",
                    "content": "Return only the exact JSON object requested. No prose or Markdown.",
                },
                {"role": "user", "content": rendered_prompt},
            ]
            request_record = {
                "run_id": RUN_ID,
                "chunk_id": CHUNK_ID,
                "request_id": request_id,
                "recipient_id": recipient_id,
                "request_sequence_number": int(recipient["request_sequence_number"]),
                "attempt_number": 1,
                "deterministic_request_seed": int(recipient["request_seed"]),
                "prompt_version": PROMPT_VERSION,
                "prompt_template_sha256": PROMPT_SHA256,
                "rendered_prompt": rendered_prompt,
                "rendered_prompt_sha256": sha256_text(rendered_prompt),
                "rendered_message_payload": messages,
                "model": completion_parameters["model"],
                "completion_parameters": {
                    **{key: value for key, value in completion_parameters.items() if key != "model"},
                    "seed": int(recipient["request_seed"]),
                },
                "anchor_algorithm_version": ANCHOR_ALGORITHM_VERSION,
                "anchor_metadata_sha256": anchor_row_hash(anchor),
                "timestamp_utc": now_utc(),
                "status": "request_preserved_before_submission",
            }
            append_jsonl(paths["requests.jsonl"], request_record)
            append_jsonl(
                paths["manifest.jsonl"],
                {
                    "run_id": RUN_ID,
                    "chunk_id": CHUNK_ID,
                    "request_id": request_id,
                    "recipient_id": recipient_id,
                    "request_sequence_number": int(recipient["request_sequence_number"]),
                    "attempt_number": 1,
                    "status": "request_preserved_before_submission",
                    "timestamp_utc": now_utc(),
                },
            )
            report["api_requests_attempted"] += 1
            report["attempts"] += 1
            append_jsonl(
                paths["manifest.jsonl"],
                {
                    "request_id": request_id,
                    "recipient_id": recipient_id,
                    "attempt_number": 1,
                    "status": "network_submission_started",
                    "timestamp_utc": now_utc(),
                },
            )
            try:
                response = client.chat.completions.create(
                    model=completion_parameters["model"],
                    messages=messages,
                    max_tokens=completion_parameters["max_tokens"],
                    temperature=completion_parameters["temperature"],
                    top_p=completion_parameters["top_p"],
                    presence_penalty=completion_parameters["presence_penalty"],
                    seed=int(recipient["request_seed"]),
                    extra_body=completion_parameters["extra_body"],
                )
            except Exception as exc:
                failure = {
                    "stage": "api_request",
                    "request_id": request_id,
                    "recipient_id": recipient_id,
                    "request_sequence_number": int(recipient["request_sequence_number"]),
                    "attempt_number": 1,
                    "exception_type": type(exc).__name__,
                    "message": str(exc),
                }
                report["failures"] += 1
                append_jsonl(paths["manifest.jsonl"], {**failure, "status": "request_failed", "timestamp_utc": now_utc()})
                raise RuntimeError(f"request failed for {request_id}") from exc

            report["api_responses_received"] += 1
            report["api_requests_completed"] += 1
            serialized_response = response.model_dump_json(indent=2)
            response_record = {
                "run_id": RUN_ID,
                "chunk_id": CHUNK_ID,
                "request_id": request_id,
                "recipient_id": recipient_id,
                "request_sequence_number": int(recipient["request_sequence_number"]),
                "attempt_number": 1,
                "received_at_utc": now_utc(),
                "serialized_response_json": serialized_response,
                "serialized_response_sha256": sha256_text(serialized_response),
                "status": "complete_response_preserved_before_extraction",
            }
            append_jsonl(paths["responses.jsonl"], response_record)

            choice = response.choices[0] if response.choices else None
            finish_reason = choice.finish_reason if choice else None
            message = choice.message if choice else None
            content = message.content if message else None
            reasoning = getattr(message, "reasoning_content", None) if message else None
            usage = response.usage.model_dump() if response.usage else {}
            metadata = {
                "request_id": request_id,
                "recipient_id": recipient_id,
                "request_sequence_number": int(recipient["request_sequence_number"]),
                "finish_reason": finish_reason,
                "reasoning_content_status": (
                    "present_nonempty" if reasoning else "present_empty" if reasoning == "" else "absent"
                ),
                "final_content_length": len(content) if isinstance(content, str) else 0,
                "token_usage": usage,
            }
            usage_records.append(usage)
            finish_records.append(metadata)
            append_jsonl(
                paths["manifest.jsonl"],
                {**metadata, "status": "response_preserved", "timestamp_utc": now_utc()},
            )
            if finish_reason == "length":
                failure = {"stage": "finish_reason_length", **metadata}
            elif not content:
                failure = {"stage": "absent_final_content", **metadata}
            else:
                try:
                    payload = json.loads(content)
                except Exception as exc:
                    failure = {
                        "stage": "json_parse",
                        **metadata,
                        "exception_type": type(exc).__name__,
                        "message": str(exc),
                    }
            if failure is not None:
                report["failures"] += 1
                append_jsonl(paths["manifest.jsonl"], {**failure, "status": "validation_failed", "timestamp_utc": now_utc()})
                raise RuntimeError(f"response failed before schema validation for {request_id}")
            errors, soft = validate_payload(payload, anchor, context["canonical_ranges"])
            if errors:
                failure = {
                    "stage": "payload_validation",
                    **metadata,
                    "errors": errors,
                    "warnings": soft,
                }
                report["failures"] += 1
                append_jsonl(paths["manifest.jsonl"], {**failure, "status": "validation_failed", "timestamp_utc": now_utc()})
                raise RuntimeError(f"payload validation failed for {request_id}")
            payloads[recipient_id] = payload
            warnings.extend({"recipient_id": recipient_id, "warning": item} for item in soft)
            frozen = frozen_assessments.loc[frozen_assessments.recipient_id.eq(recipient_id)].copy()
            recipient_rows = build_assessment_rows(
                recipient, anchor, frozen, payload, assessment_columns
            )
            rows.extend(recipient_rows)
            append_checkpoint(
                paths["validated_recipient_checkpoint.csv"],
                pd.DataFrame(recipient_rows, columns=assessment_columns),
            )
            report["recipients_validated"] += 1
            boundary_checks.append(
                boundary_integrity(context, recipient["request_sequence_number"])
            )
            append_jsonl(
                paths["manifest.jsonl"],
                {
                    "request_id": request_id,
                    "recipient_id": recipient_id,
                    "request_sequence_number": int(recipient["request_sequence_number"]),
                    "attempt_number": 1,
                    "status": "validated_and_checkpointed",
                    "checkpoint_recipient_count": report["recipients_validated"],
                    "checkpoint_row_count": len(rows),
                    "soft_warnings": soft,
                    "timestamp_utc": now_utc(),
                },
            )
            write_json_state(
                paths["completion_state.json"],
                {
                    "run_id": RUN_ID,
                    "chunk_id": CHUNK_ID,
                    "status": "running",
                    "automatic_resume": False,
                    "requests_attempted": report["api_requests_attempted"],
                    "requests_completed": report["api_requests_completed"],
                    "recipients_validated": report["recipients_validated"],
                    "checkpoint_rows": len(rows),
                    "last_validated_request_id": request_id,
                    "last_validated_recipient_id": recipient_id,
                },
            )

        assessment = pd.DataFrame(rows, columns=assessment_columns)
        reconciliation = reconcile_payloads(payloads, assessment)
        validation = validate_complete_chunk(
            assessment, identities, recipients, anchors, payloads, reconciliation, context
        )
        if validation["status"] != "passed":
            failure = {"stage": "complete_chunk_validation", "validation": validation}
            report["failures"] += 1
            raise RuntimeError("complete chunk validation failed")
        diagnostics = build_diagnostics(assessment, recipients, anchors)
        day7 = assessment.loc[assessment.days_since_transplant.eq(7)].copy()
        anchor_audit = anchors.merge(
            day7[["recipient_id", "creatinine_mg_dl", "urine_output_ml_24h", "tacrolimus_level_ng_ml", "medication_adherence_pct"]],
            on="recipient_id",
            validate="one_to_one",
        )
        for output, anchor_field in {
            "creatinine_mg_dl": "day7_creatinine_anchor_mg_dl",
            "urine_output_ml_24h": "day7_urine_output_anchor_ml_24h",
            "tacrolimus_level_ng_ml": "day7_tacrolimus_anchor_ng_ml",
            "medication_adherence_pct": "day7_medication_adherence_anchor_pct",
        }.items():
            difference = (anchor_audit[output] - anchor_audit[anchor_field]).abs()
            anchor_audit[f"{output}_absolute_anchor_difference"] = difference
            anchor_audit[f"{output}_within_tolerance"] = difference.le(ANCHOR_TOLERANCES[output])
        write_csv_exclusive(paths["assessment_chunk_001.csv"], assessment)
        write_csv_exclusive(paths["anchor_audit_chunk_001.csv"], anchor_audit)
        write_json_exclusive(paths["request_response_reconciliation.json"], reconciliation)
        write_json_exclusive(paths["diagnostic_quality_report.json"], diagnostics)
        report["status"] = "completed"
        write_json_state(
            paths["completion_state.json"],
            {
                "run_id": RUN_ID,
                "chunk_id": CHUNK_ID,
                "status": "completed",
                "automatic_resume": False,
                "requests_attempted": 100,
                "requests_completed": 100,
                "recipients_validated": 100,
                "checkpoint_rows": 600,
                "assessment_chunk_created": True,
                "chunks_002_to_100_started": False,
                "final_dataset_assembly_started": False,
            },
        )
    except Exception as exc:
        if failure is None:
            failure = {
                "stage": "execution_or_chunk_validation",
                "exception_type": type(exc).__name__,
                "message": str(exc),
                "traceback": traceback.format_exc(),
            }
            report["failures"] += 1
        report["status"] = "failed"
        write_json_state(
            paths["completion_state.json"],
            {
                "run_id": RUN_ID,
                "chunk_id": CHUNK_ID,
                "status": "failed",
                "automatic_resume": False,
                "requests_attempted": report["api_requests_attempted"],
                "requests_completed": report["api_requests_completed"],
                "recipients_validated": report["recipients_validated"],
                "checkpoint_rows": len(rows),
                "failure": failure,
                "retry_performed": False,
                "assessment_chunk_created": paths["assessment_chunk_001.csv"].exists(),
            },
        )
    finally:
        protected_in_finally = protected_fingerprint()
        immutable_in_finally = snapshot_immutable(context["immutable_paths"])
        notebooks_in_finally = snapshot_notebooks_01_05()
        production_design_in_finally = tree_fingerprint(PRODUCTION_DESIGN)
        token_totals = {
            key: sum(int(record.get(key, 0) or 0) for record in usage_records)
            for key in ["prompt_tokens", "completion_tokens", "total_tokens"]
        }
        report.update(
            {
                "failure": failure,
                "retries": 0,
                "elapsed_seconds": time.monotonic() - started,
                "finish_reason_counts": dict(Counter(item["finish_reason"] for item in finish_records)),
                "reasoning_content_status_counts": dict(
                    Counter(item["reasoning_content_status"] for item in finish_records)
                ),
                "finish_records": finish_records,
                "token_usage_total": token_totals,
                "request_records": count_jsonl(paths["requests.jsonl"]),
                "response_records": count_jsonl(paths["responses.jsonl"]),
                "manifest_records": count_jsonl(paths["manifest.jsonl"]),
                "manifest_status_counts": dict(
                    Counter(
                        json.loads(line)["status"]
                        for line in paths["manifest.jsonl"].read_text(encoding="utf-8").splitlines()
                        if line.strip()
                    )
                ),
                "checkpoint_rows": len(rows),
                "chunk_dimensions": [len(rows), len(assessment_columns)] if rows else [0, len(assessment_columns)],
                "validation": validation,
                "reconciliation": reconciliation,
                "diagnostics": diagnostics,
                "soft_warnings": warnings,
                "boundary_integrity_checks": boundary_checks,
                "boundary_integrity_check_count": len(boundary_checks),
                "preflight_checks": context["checks"],
                "protected_before": context["protected_before"],
                "protected_in_finally": protected_in_finally,
                "protected_integrity_in_finally": protected_in_finally == context["protected_before"],
                "immutable_before": context["immutable_before"],
                "immutable_in_finally": immutable_in_finally,
                "immutable_integrity_in_finally": immutable_in_finally == context["immutable_before"],
                "notebooks_01_05_before": context["notebooks_before"],
                "notebooks_01_05_in_finally": notebooks_in_finally,
                "notebooks_01_05_integrity_in_finally": notebooks_in_finally == context["notebooks_before"],
                "production_design_before": context["production_design_before"],
                "production_design_in_finally": production_design_in_finally,
                "production_design_integrity_in_finally": production_design_in_finally == context["production_design_before"],
                "chunks_002_to_100_started": False,
                "final_dataset_assembly_started": False,
            }
        )
        write_json_state(paths["chunk_001_run_report.json"], report)
        assert protected_in_finally == context["protected_before"]
        assert immutable_in_finally == context["immutable_before"]
        assert notebooks_in_finally == context["notebooks_before"]
        assert production_design_in_finally == context["production_design_before"]

    protected_after = protected_fingerprint()
    immutable_after = snapshot_immutable(context["immutable_paths"])
    notebooks_after = snapshot_notebooks_01_05()
    production_design_after = tree_fingerprint(PRODUCTION_DESIGN)
    report.update(
        {
            "protected_after_execution": protected_after,
            "protected_integrity_after_execution": protected_after == context["protected_before"],
            "immutable_after_execution": immutable_after,
            "immutable_integrity_after_execution": immutable_after == context["immutable_before"],
            "notebooks_01_05_after_execution": notebooks_after,
            "notebooks_01_05_integrity_after_execution": notebooks_after == context["notebooks_before"],
            "production_design_after_execution": production_design_after,
            "production_design_integrity_after_execution": production_design_after == context["production_design_before"],
            "created_files": sorted(path.name for path in CHUNK_DIR.iterdir() if path.is_file()),
        }
    )
    write_json_state(paths["chunk_001_run_report.json"], report)
    assert protected_after == context["protected_before"]
    assert immutable_after == context["immutable_before"]
    assert notebooks_after == context["notebooks_before"]
    assert production_design_after == context["production_design_before"]
    assert not RUN_CHUNKS_002_TO_100 and not RUN_10000_RECIPIENTS and not RUN_FULL_GENERATION
    return report


CHUNK_001_RUN_REPORT = run_chunk_001()
print(
    json.dumps(
        {
            "status": CHUNK_001_RUN_REPORT["status"],
            "api_requests_attempted": CHUNK_001_RUN_REPORT["api_requests_attempted"],
            "api_requests_completed": CHUNK_001_RUN_REPORT["api_requests_completed"],
            "recipients_validated": CHUNK_001_RUN_REPORT["recipients_validated"],
            "attempts": CHUNK_001_RUN_REPORT["attempts"],
            "failures": CHUNK_001_RUN_REPORT["failures"],
            "retries": CHUNK_001_RUN_REPORT["retries"],
            "token_usage_total": CHUNK_001_RUN_REPORT["token_usage_total"],
            "finish_reason_counts": CHUNK_001_RUN_REPORT["finish_reason_counts"],
            "reasoning_content_status_counts": CHUNK_001_RUN_REPORT["reasoning_content_status_counts"],
            "request_records": CHUNK_001_RUN_REPORT["request_records"],
            "response_records": CHUNK_001_RUN_REPORT["response_records"],
            "manifest_records": CHUNK_001_RUN_REPORT["manifest_records"],
            "chunk_dimensions": CHUNK_001_RUN_REPORT["chunk_dimensions"],
            "validation": CHUNK_001_RUN_REPORT["validation"],
            "reconciliation": CHUNK_001_RUN_REPORT["reconciliation"],
            "diagnostics": CHUNK_001_RUN_REPORT["diagnostics"],
            "elapsed_seconds": CHUNK_001_RUN_REPORT["elapsed_seconds"],
            "protected_integrity_after_execution": CHUNK_001_RUN_REPORT["protected_integrity_after_execution"],
            "production_design_integrity_after_execution": CHUNK_001_RUN_REPORT["production_design_integrity_after_execution"],
            "created_files": CHUNK_001_RUN_REPORT["created_files"],
            "chunks_002_to_100_started": CHUNK_001_RUN_REPORT["chunks_002_to_100_started"],
            "final_dataset_assembly_started": CHUNK_001_RUN_REPORT["final_dataset_assembly_started"],
            "failure": CHUNK_001_RUN_REPORT["failure"],
        },
        indent=2,
    )
)
if CHUNK_001_RUN_REPORT["status"] != "completed":
    raise RuntimeError("chunk 1 stopped after its first failure; no retry or automatic resume was performed")


## Guarded v3.2 production chunks 002–010

This cell runs only with the exact `RUN_QWEN_V32_PRODUCTION_CHUNKS_002_010` gate. It verifies and preserves completed chunks, refuses automatic resume of any partial chunk, processes only chunks 002–010 in numerical order with one attempt per recipient and zero retries, and stops the entire run on the first failure. On success it creates a read-only aggregate progress checkpoint through chunk 010. Chunks 011–100, combined production tables, model training, generator tuning, and final dataset assembly remain disabled.

In [ ]:
import hashlib
import json
import os
import time
import traceback
from collections import Counter
from datetime import date, timedelta
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd
from openai import OpenAI


# Reuse the frozen chunk-001 validation primitives without executing chunk 001.
_repository_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists()
)
_notebook_path = _repository_root / "notebooks/KidneyTransplant/00_generate_and_validate_dataset.ipynb"
_notebook = nbformat.read(_notebook_path, as_version=4)
_chunk_001_source = next(
    cell.source
    for cell in _notebook.cells
    if cell.cell_type == "code" and "def run_chunk_001():" in cell.source
)
_chunk_001_primitives = _chunk_001_source.split("\ndef boundary_integrity", 1)[0]
exec(compile(_chunk_001_primitives, "frozen-chunk-001-primitives", "exec"), globals())


MULTI_GATE_NAME = "RUN_QWEN_V32_PRODUCTION_CHUNKS_002_010"
MULTI_GATE_VALUE = "QWEN-V32-PROD-001-CHUNKS-002-010"
AUTHORIZED_CHUNKS = list(range(2, 11))
DISABLED_CHUNKS = list(range(11, 101))
EXPECTED_NEW_RECIPIENTS = 900
EXPECTED_NEW_ROWS = 5_400
MAX_NEW_API_REQUESTS = 900
PROGRESS_DIR_NAME = "progress_through_chunk_010"
OUTPUT_PRODUCTION_ROOT = PRODUCTION_ROOT

CHUNK_001_FINGERPRINTS = {
    "anchor_audit_chunk_001.csv": {"size_bytes": 23479, "sha256": "bd2d8216780f942e6daca5bb19666ae58cc162583ba597a7ef11fe546ded52a0"},
    "assessment_chunk_001.csv": {"size_bytes": 147222, "sha256": "dbeda22ccbf2b042221dc27bedc13e0452afaf777f2d25bb32fdd132f9b786b1"},
    "chunk_001_run_report.json": {"size_bytes": 92666, "sha256": "04b3c730b470bd1b4b5bb406d2ec5470d388ac03e076cf4c1031a1f962303bb5"},
    "completion_state.json": {"size_bytes": 349, "sha256": "10c8af8f72e2071e01e8ad5af3f568d27736811619f5cc7d67ddfc67bf55be23"},
    "diagnostic_quality_report.json": {"size_bytes": 10156, "sha256": "cbeb748d03d6bdb1d5c7aa9cf85bef6be1cbe05b9ec7dd6d3f976ae0d076b879"},
    "identity_chunk_001.csv": {"size_bytes": 35287, "sha256": "0775c117dfd054dd95604434b5b6381637214e4f92f2ff64bb67907ee7f5e206"},
    "manifest.jsonl": {"size_bytes": 114451, "sha256": "90ffe07f9a25e4620da4f6e7eacb9df7973ac4dc65d1db9c01e14ff805c0ce35"},
    "request_response_reconciliation.json": {"size_bytes": 233, "sha256": "6689cfee50e6d3028b78a6e81d22e68bf05b068d4d9503f8ab4e851c4e2bc2b2"},
    "requests.jsonl": {"size_bytes": 892434, "sha256": "a26d9926487695a8a079c0e53ca0c1f0275b1ade8ece2086bb7858c20503d849"},
    "responses.jsonl": {"size_bytes": 280971, "sha256": "f4e8564adb82e06fb83f9baed9a1f5f555c71256e7c7bd16ee69209974d2457a"},
    "run_configuration.json": {"size_bytes": 2411, "sha256": "11615f62e085a2b4dd820c2bd5826270e6e822d26cfc68708125e67229f8a443"},
    "validated_recipient_checkpoint.csv": {"size_bytes": 147222, "sha256": "dbeda22ccbf2b042221dc27bedc13e0452afaf777f2d25bb32fdd132f9b786b1"},
}


def chunk_directory(number):
    if number == 1:
        return PRODUCTION_ROOT / "chunk_001"
    return OUTPUT_PRODUCTION_ROOT / f"chunk_{number:03d}"


def progress_directory():
    return OUTPUT_PRODUCTION_ROOT / PROGRESS_DIR_NAME


def expected_chunk_files(number):
    suffix = f"{number:03d}"
    return {
        "requests.jsonl",
        "responses.jsonl",
        "manifest.jsonl",
        "validated_recipient_checkpoint.csv",
        f"assessment_chunk_{suffix}.csv",
        f"identity_chunk_{suffix}.csv",
        f"anchor_audit_chunk_{suffix}.csv",
        "request_response_reconciliation.json",
        "diagnostic_quality_report.json",
        "run_configuration.json",
        "completion_state.json",
        f"chunk_{suffix}_run_report.json",
    }


def line_records(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def chunk_file_fingerprints(number, exclude_report=False):
    directory = chunk_directory(number)
    report_name = f"chunk_{number:03d}_run_report.json"
    return {
        path.name: file_fingerprint(path)
        for path in sorted(directory.iterdir())
        if path.is_file() and (not exclude_report or path.name != report_name)
    }


def verify_chunk_001_immutable():
    directory = chunk_directory(1)
    assert directory.is_dir()
    assert {path.name for path in directory.iterdir() if path.is_file()} == set(CHUNK_001_FINGERPRINTS)
    for name, expected in CHUNK_001_FINGERPRINTS.items():
        current = file_fingerprint(directory / name)
        assert current["sha256"] == expected["sha256"], name
        assert current["size_bytes"] == expected["size_bytes"], name
    state = json.loads((directory / "completion_state.json").read_text(encoding="utf-8"))
    report = json.loads((directory / "chunk_001_run_report.json").read_text(encoding="utf-8"))
    assert state["status"] == "completed" and report["status"] == "completed"
    assert state["requests_completed"] == 100 and state["recipients_validated"] == 100
    assert report["failures"] == 0 and report["retries"] == 0
    assert report["chunk_dimensions"] == [600, 31]
    return tree_fingerprint(directory)


def verify_recorded_completed_chunk(number):
    directory = chunk_directory(number)
    assert directory.is_dir()
    assert {path.name for path in directory.iterdir() if path.is_file()} == expected_chunk_files(number)
    suffix = f"{number:03d}"
    state = json.loads((directory / "completion_state.json").read_text(encoding="utf-8"))
    report = json.loads((directory / f"chunk_{suffix}_run_report.json").read_text(encoding="utf-8"))
    assert state["status"] == "completed" and report["status"] == "completed"
    assert state["requests_completed"] == 100 and state["recipients_validated"] == 100
    assert state["checkpoint_rows"] == 600 and state["assessment_chunk_created"] is True
    assert report["api_requests_completed"] == 100 and report["failures"] == 0 and report["retries"] == 0
    assert report["chunk_dimensions"] == [600, 31]
    recorded = report["artifact_fingerprints_excluding_report"]
    current = chunk_file_fingerprints(number, exclude_report=True)
    assert set(recorded) == set(current)
    for name in recorded:
        for key in ["sha256", "mtime_ns", "size_bytes"]:
            assert recorded[name][key] == current[name][key], f"completed chunk {number} changed: {name}"
    assessment = pd.read_csv(directory / f"assessment_chunk_{suffix}.csv")
    checkpoint = pd.read_csv(directory / "validated_recipient_checkpoint.csv")
    pd.testing.assert_frame_equal(assessment, checkpoint, check_dtype=False)
    assert assessment.shape == (600, 31)
    assert len(line_records(directory / "requests.jsonl")) == 100
    assert len(line_records(directory / "responses.jsonl")) == 100
    assert len(line_records(directory / "manifest.jsonl")) == 400
    return tree_fingerprint(directory)


def frozen_chunk_slice(number, recipients_all, assessments_all, identities_all, anchors_all):
    start = (number - 1) * 100 + 1
    stop = number * 100
    recipients = recipients_all.loc[
        recipients_all.request_sequence_number.between(start, stop)
    ].sort_values("request_sequence_number").reset_index(drop=True)
    recipient_ids = recipients.recipient_id.tolist()
    assessments = assessments_all.loc[
        assessments_all.recipient_id.isin(recipient_ids)
    ].sort_values(["recipient_id", "days_since_transplant"]).reset_index(drop=True)
    anchors = anchors_all.loc[
        anchors_all.recipient_id.isin(recipient_ids)
    ].sort_values("recipient_id").reset_index(drop=True)
    entity_ids = set(recipient_ids) | set(recipients.donor_id)
    identities = identities_all.loc[identities_all.entity_id.isin(entity_ids)].copy()
    return {
        "number": number,
        "recipients": recipients,
        "assessments": assessments,
        "anchors": anchors,
        "identities": identities,
    }


def verify_frozen_slice(slice_context):
    number = slice_context["number"]
    recipients = slice_context["recipients"]
    assessments = slice_context["assessments"]
    anchors = slice_context["anchors"]
    expected_sequences = list(range((number - 1) * 100 + 1, number * 100 + 1))
    assert len(recipients) == 100
    assert recipients.request_sequence_number.tolist() == expected_sequences
    assert recipients.chunk_id.eq(f"QWEN-V32-PROD-CHUNK{number:03d}").all()
    assert recipients.recipient_id.str.startswith("V32P-R").all()
    assert not recipients.recipient_id.str.startswith(("V31-", "QWEN-")).any()
    assert recipients.recipient_id.is_unique and recipients.request_id.is_unique and recipients.request_seed.is_unique
    assert len(assessments) == 600 and assessments.assessment_id.is_unique
    assert assessments.groupby("recipient_id").size().eq(6).all()
    assert assessments.groupby("recipient_id").days_since_transplant.apply(list).map(lambda values: values == DAYS).all()
    assert len(anchors) == 100 and anchors.recipient_id.is_unique
    assert anchors.anchor_algorithm_version.eq(ANCHOR_ALGORITHM_VERSION).all()
    recipient_lookup = recipients.set_index("recipient_id")
    for row in assessments.itertuples(index=False):
        recipient = recipient_lookup.loc[row.recipient_id]
        transplant = date.fromisoformat(recipient.transplant_date)
        assessment_date = transplant + timedelta(days=int(row.days_since_transplant))
        assert row.assessment_date == assessment_date.isoformat()
        assert row.retention_expiry_date == (assessment_date + timedelta(days=730)).isoformat()
        assert row.donor_id == recipient.donor_id and row.hospital_id == recipient.hospital_id
        assert derive_abo_category(recipient.donor_blood_group, recipient.recipient_blood_group) == row.abo_compatibility_category
        position = DAYS.index(int(row.days_since_transplant))
        infections = set(json.loads(recipient.infection_episode_days))
        previous, targets = derive_event_fields(json.loads(recipient.confirmed_rejection_event_days))
        assert int(row.infection_indicator) == int(row.days_since_transplant in infections)
        assert int(row.previous_rejection) == previous[position]
        assert int(row.acute_rejection_within_30_days) == targets[position]


def preflight_chunks_002_010():
    assert os.environ.get(MULTI_GATE_NAME) == MULTI_GATE_VALUE, "exact chunks-002-010 gate mismatch"
    for name in [
        GATE_NAME,
        "RUN_QWEN_V32_CONFIRM20",
        "RUN_CORRECTED_10",
        "RUN_100_RECIPIENTS",
        "RUN_10000_RECIPIENTS",
        "RUN_FULL_GENERATION",
    ]:
        assert not os.environ.get(name), f"{name} must be false"
    for number in range(2, 101):
        assert not os.environ.get(f"RUN_QWEN_V32_PRODUCTION_CHUNK_{number:03d}"), f"individual chunk {number} gate must be false"
    assert not RUN_CHUNKS_002_TO_100 and not RUN_10000_RECIPIENTS and not RUN_FULL_GENERATION
    assert MAX_NEW_API_REQUESTS == EXPECTED_NEW_RECIPIENTS == 900

    chunk_001_before = verify_chunk_001_immutable()
    completed_existing = []
    encountered_unstarted = False
    for number in AUTHORIZED_CHUNKS:
        directory = chunk_directory(number)
        if not directory.exists():
            encountered_unstarted = True
            continue
        assert not encountered_unstarted, "later chunk exists after an unstarted authorised chunk"
        state_path = directory / "completion_state.json"
        assert state_path.is_file(), f"chunk {number:03d} exists without an unambiguous completion state"
        state = json.loads(state_path.read_text(encoding="utf-8"))
        assert state.get("status") == "completed", f"chunk {number:03d} is partial or failed; separate resume authorisation required"
        verify_recorded_completed_chunk(number)
        completed_existing.append(number)
    for number in DISABLED_CHUNKS:
        assert not chunk_directory(number).exists(), f"disabled chunk {number:03d} already exists"
    if progress_directory().exists():
        assert completed_existing == AUTHORIZED_CHUNKS
        expected_progress = {
            "production_progress_report.json",
            "cross_chunk_reconciliation.json",
            "cross_chunk_distribution_report.json",
            "cross_chunk_integrity_report.json",
        }
        assert {path.name for path in progress_directory().iterdir() if path.is_file()} == expected_progress
        progress = json.loads((progress_directory() / "production_progress_report.json").read_text(encoding="utf-8"))
        assert progress["status"] == "completed"

    design_checks, design_validation = verify_frozen_production_design()
    prompt_path = V32_DESIGN / "qwen_kidney_v3_2_prompt_template.txt"
    template = prompt_path.read_text(encoding="utf-8")
    assert sha256_text(template) == PROMPT_SHA256
    api_config = json.loads((V32_DESIGN / "api_configuration.json").read_text(encoding="utf-8"))
    assert api_config["client"]["max_retries"] == 0 and api_config["automatic_retry"] is False
    assert api_config["completion"] == {
        "model": "Qwen/Qwen3.6-35B-A3B",
        "max_tokens": 8192,
        "temperature": 0.7,
        "top_p": 0.8,
        "presence_penalty": 1.5,
        "extra_body": {"top_k": 20, "chat_template_kwargs": {"enable_thinking": False}},
    }

    recipients_all = pd.read_csv(PRODUCTION_DESIGN / "production_recipient_metadata.csv")
    assessments_all = pd.read_csv(PRODUCTION_DESIGN / "production_assessment_metadata.csv")
    identities_all = pd.read_csv(PRODUCTION_DESIGN / "production_identity_skeleton.csv", keep_default_na=False)
    anchors_all = pd.read_csv(PRODUCTION_DESIGN / "production_anchor_metadata.csv")
    slices = {
        number: frozen_chunk_slice(number, recipients_all, assessments_all, identities_all, anchors_all)
        for number in range(1, 11)
    }
    for slice_context in slices.values():
        verify_frozen_slice(slice_context)
    first_ten_recipients = pd.concat([slices[number]["recipients"] for number in range(1, 11)], ignore_index=True)
    first_ten_assessments = pd.concat([slices[number]["assessments"] for number in range(1, 11)], ignore_index=True)
    assert len(first_ten_recipients) == 1_000 and first_ten_recipients.recipient_id.is_unique
    assert first_ten_recipients.request_id.is_unique and first_ten_recipients.request_seed.is_unique
    assert len(first_ten_assessments) == 6_000 and first_ten_assessments.assessment_id.is_unique
    assert not first_ten_assessments.duplicated(["recipient_id", "days_since_transplant"]).any()

    field_lists = json.loads((V31_DESIGN / "field_lists.json").read_text(encoding="utf-8"))
    canonical_ranges = json.loads((V31_DESIGN / "canonical_qwen_ranges.json").read_text(encoding="utf-8"))
    assessment_columns = pd.read_csv(V31_DESIGN / "assessment_table_schema.csv").field_name.tolist()
    direct = set(field_lists["direct_identifiers"])
    assert direct <= set(identities_all.columns)
    assert not direct.intersection(first_ten_recipients.columns)
    assert not direct.intersection(first_ten_assessments.columns)
    assert not direct.intersection(anchors_all.columns)
    classifier = set(field_lists["classifier_features"])
    forbidden = set(
        field_lists["pseudonymous_identifiers"]
        + field_lists["assessment_audit_only"]
        + field_lists["event_control_metadata"]
        + ["assessment_date", "retention_expiry_date", "transplant_date", "training_consent_status", "training_consent_version"]
        + [column for column in anchors_all.columns if "anchor" in column]
    )
    assert not classifier.intersection(forbidden)
    assert len(assessment_columns) == 31

    immutable_paths = immutable_inputs()
    context = {
        "checks": {
            "chunk_001_complete_and_fingerprinted": True,
            "frozen_design_artifacts": all(design_checks.values()),
            "prompt_hash": True,
            "anchor_algorithm_version": True,
            "authorised_chunk_count": len(AUTHORIZED_CHUNKS) == 9,
            "new_recipient_count": sum(len(slices[n]["recipients"]) for n in AUTHORIZED_CHUNKS) == 900,
            "new_assessment_metadata_rows": sum(len(slices[n]["assessments"]) for n in AUTHORIZED_CHUNKS) == 5_400,
            "cross_chunk_identifiers_unique": True,
            "automatic_retries_disabled": True,
            "maximum_new_requests": MAX_NEW_API_REQUESTS == 900,
            "chunks_011_100_disabled": True,
            "final_assembly_disabled": True,
        },
        "design_validation": design_validation,
        "template": template,
        "api_config": api_config,
        "slices": slices,
        "recipients_all": recipients_all,
        "assessments_all": assessments_all,
        "identities_all": identities_all,
        "anchors_all": anchors_all,
        "field_lists": field_lists,
        "canonical_ranges": canonical_ranges,
        "assessment_columns": assessment_columns,
        "completed_existing": completed_existing,
        "immutable_paths": immutable_paths,
        "immutable_before": snapshot_immutable(immutable_paths),
        "notebooks_before": snapshot_notebooks_01_05(),
        "protected_before": protected_fingerprint(),
        "production_design_before": tree_fingerprint(PRODUCTION_DESIGN),
        "chunk_001_before": chunk_001_before,
    }
    assert {key: context["protected_before"][key] for key in EXPECTED_PROTECTED} == EXPECTED_PROTECTED
    assert all(context["checks"].values())
    return context


def assert_global_integrity(context, completed_fingerprints=None):
    result = {
        "protected_unchanged": protected_fingerprint() == context["protected_before"],
        "previous_evidence_unchanged": snapshot_immutable(context["immutable_paths"]) == context["immutable_before"],
        "notebooks_01_05_unchanged": snapshot_notebooks_01_05() == context["notebooks_before"],
        "production_design_unchanged": tree_fingerprint(PRODUCTION_DESIGN) == context["production_design_before"],
        "chunk_001_unchanged": tree_fingerprint(chunk_directory(1)) == context["chunk_001_before"],
        "chunks_011_100_absent": all(not chunk_directory(number).exists() for number in DISABLED_CHUNKS),
        "final_assembly_absent": not (OUTPUT_PRODUCTION_ROOT / "assessment_production.csv").exists(),
    }
    for number, fingerprint in (completed_fingerprints or {}).items():
        result[f"chunk_{number:03d}_unchanged"] = tree_fingerprint(chunk_directory(number)) == fingerprint
    assert all(result.values()), [name for name, passed in result.items() if not passed]
    return result


def run_single_chunk(number, context, completed_fingerprints):
    started = time.monotonic()
    suffix = f"{number:03d}"
    chunk_id = f"CHUNK-{suffix}"
    frozen_chunk_id = f"QWEN-V32-PROD-CHUNK{suffix}"
    directory = chunk_directory(number)
    assert not directory.exists()
    slice_context = context["slices"][number]
    recipients = slice_context["recipients"]
    frozen_assessments = slice_context["assessments"]
    identities = slice_context["identities"]
    anchors = slice_context["anchors"]
    assessment_columns = context["assessment_columns"]
    completion_parameters = dict(context["api_config"]["completion"])

    directory.mkdir(parents=False, exist_ok=False)
    paths = {name: directory / name for name in expected_chunk_files(number)}
    for name in ["requests.jsonl", "responses.jsonl", "manifest.jsonl"]:
        with paths[name].open("x", encoding="utf-8") as handle:
            handle.flush()
            os.fsync(handle.fileno())
    initialize_checkpoint(paths["validated_recipient_checkpoint.csv"], assessment_columns)
    write_csv_exclusive(paths[f"identity_chunk_{suffix}.csv"], identities)
    write_json_exclusive(
        paths["run_configuration.json"],
        {
            "run_id": RUN_ID,
            "chunk_id": chunk_id,
            "frozen_chunk_id": frozen_chunk_id,
            "status": "started_after_preflight",
            "started_at_utc": now_utc(),
            "base_url": BASE_URL,
            "api_key": "REDACTED",
            "model": completion_parameters["model"],
            "completion_parameters": completion_parameters,
            "client": {"max_retries": 0, "timeout_seconds": 180.0},
            "automatic_retry": False,
            "maximum_api_requests": 100,
            "request_execution": "sequential",
            "attempt_number": 1,
            "prompt_version": PROMPT_VERSION,
            "prompt_template_sha256": PROMPT_SHA256,
            "anchor_algorithm_version": ANCHOR_ALGORITHM_VERSION,
            "gate": f"{MULTI_GATE_NAME}={MULTI_GATE_VALUE}",
            "preflight_checks": context["checks"],
            "protected_before": context["protected_before"],
            "production_design_before": context["production_design_before"],
            "persistent_generation_flags": {
                "chunks_011_to_100": False,
                "RUN_10000_RECIPIENTS": False,
                "RUN_FULL_GENERATION": False,
            },
        },
    )
    write_json_state(
        paths["completion_state.json"],
        {
            "run_id": RUN_ID,
            "chunk_id": chunk_id,
            "status": "running",
            "automatic_resume": False,
            "requests_attempted": 0,
            "requests_completed": 0,
            "recipients_validated": 0,
            "checkpoint_rows": 0,
        },
    )
    report = {
        "run_id": RUN_ID,
        "chunk_id": chunk_id,
        "status": "running",
        "api_requests_attempted": 0,
        "api_requests_completed": 0,
        "api_responses_received": 0,
        "recipients_validated": 0,
        "attempts": 0,
        "failures": 0,
        "retries": 0,
        "automatic_resume": False,
        "chunks_011_to_100_started": False,
        "final_dataset_assembly_started": False,
    }
    failure = None
    rows = []
    payloads = {}
    usage_records = []
    finish_records = []
    soft_warnings = []
    validation = None
    diagnostics = None
    reconciliation = None
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL, max_retries=0, timeout=180.0)
    try:
        anchor_lookup = anchors.set_index("recipient_id")
        seen_request_ids = set()
        for recipient in recipients.to_dict(orient="records"):
            if report["api_requests_attempted"] >= 100:
                raise RuntimeError("100-request per-chunk hard cap reached")
            request_id = recipient["request_id"]
            recipient_id = recipient["recipient_id"]
            if request_id in seen_request_ids:
                raise RuntimeError(f"duplicate request ID: {request_id}")
            seen_request_ids.add(request_id)
            anchor = anchor_lookup.loc[recipient_id].to_dict()
            rendered_prompt = render_prompt(context["template"], recipient, anchor)
            messages = [
                {"role": "system", "content": "Return only the exact JSON object requested. No prose or Markdown."},
                {"role": "user", "content": rendered_prompt},
            ]
            request_record = {
                "run_id": RUN_ID,
                "chunk_id": chunk_id,
                "request_id": request_id,
                "recipient_id": recipient_id,
                "request_sequence_number": int(recipient["request_sequence_number"]),
                "attempt_number": 1,
                "deterministic_request_seed": int(recipient["request_seed"]),
                "prompt_version": PROMPT_VERSION,
                "prompt_template_sha256": PROMPT_SHA256,
                "rendered_prompt": rendered_prompt,
                "rendered_prompt_sha256": sha256_text(rendered_prompt),
                "rendered_message_payload": messages,
                "model": completion_parameters["model"],
                "completion_parameters": {
                    **{key: value for key, value in completion_parameters.items() if key != "model"},
                    "seed": int(recipient["request_seed"]),
                },
                "anchor_algorithm_version": ANCHOR_ALGORITHM_VERSION,
                "anchor_metadata_sha256": anchor_row_hash(anchor),
                "timestamp_utc": now_utc(),
                "status": "request_preserved_before_submission",
            }
            append_jsonl(paths["requests.jsonl"], request_record)
            append_jsonl(paths["manifest.jsonl"], {
                "run_id": RUN_ID, "chunk_id": chunk_id, "request_id": request_id,
                "recipient_id": recipient_id, "request_sequence_number": int(recipient["request_sequence_number"]),
                "attempt_number": 1, "status": "request_preserved_before_submission", "timestamp_utc": now_utc(),
            })
            report["api_requests_attempted"] += 1
            report["attempts"] += 1
            append_jsonl(paths["manifest.jsonl"], {
                "request_id": request_id, "recipient_id": recipient_id, "attempt_number": 1,
                "status": "network_submission_started", "timestamp_utc": now_utc(),
            })
            try:
                response = client.chat.completions.create(
                    model=completion_parameters["model"], messages=messages,
                    max_tokens=completion_parameters["max_tokens"],
                    temperature=completion_parameters["temperature"], top_p=completion_parameters["top_p"],
                    presence_penalty=completion_parameters["presence_penalty"],
                    seed=int(recipient["request_seed"]), extra_body=completion_parameters["extra_body"],
                )
            except Exception as exc:
                failure = {
                    "stage": "api_request", "request_id": request_id, "recipient_id": recipient_id,
                    "request_sequence_number": int(recipient["request_sequence_number"]), "attempt_number": 1,
                    "exception_type": type(exc).__name__, "message": str(exc),
                }
                report["failures"] += 1
                append_jsonl(paths["manifest.jsonl"], {**failure, "status": "request_failed", "timestamp_utc": now_utc()})
                raise RuntimeError(f"request failed for {request_id}") from exc

            report["api_responses_received"] += 1
            report["api_requests_completed"] += 1
            serialized_response = response.model_dump_json(indent=2)
            append_jsonl(paths["responses.jsonl"], {
                "run_id": RUN_ID, "chunk_id": chunk_id, "request_id": request_id,
                "recipient_id": recipient_id, "request_sequence_number": int(recipient["request_sequence_number"]),
                "attempt_number": 1, "received_at_utc": now_utc(),
                "serialized_response_json": serialized_response,
                "serialized_response_sha256": sha256_text(serialized_response),
                "status": "complete_response_preserved_before_extraction",
            })
            choice = response.choices[0] if response.choices else None
            finish_reason = choice.finish_reason if choice else None
            message = choice.message if choice else None
            content = message.content if message else None
            reasoning = getattr(message, "reasoning_content", None) if message else None
            usage = response.usage.model_dump() if response.usage else {}
            metadata = {
                "request_id": request_id, "recipient_id": recipient_id,
                "request_sequence_number": int(recipient["request_sequence_number"]),
                "finish_reason": finish_reason,
                "reasoning_content_status": "present_nonempty" if reasoning else "present_empty" if reasoning == "" else "absent",
                "final_content_length": len(content) if isinstance(content, str) else 0,
                "token_usage": usage,
            }
            usage_records.append(usage)
            finish_records.append(metadata)
            append_jsonl(paths["manifest.jsonl"], {**metadata, "status": "response_preserved", "timestamp_utc": now_utc()})
            payload = None
            if finish_reason == "length":
                failure = {"stage": "finish_reason_length", **metadata}
            elif not content:
                failure = {"stage": "absent_final_content", **metadata}
            else:
                try:
                    payload = json.loads(content)
                except Exception as exc:
                    failure = {"stage": "json_parse", **metadata, "exception_type": type(exc).__name__, "message": str(exc)}
            if failure is not None:
                report["failures"] += 1
                append_jsonl(paths["manifest.jsonl"], {**failure, "status": "validation_failed", "timestamp_utc": now_utc()})
                raise RuntimeError(f"response failed before schema validation for {request_id}")
            errors, warnings = validate_payload(payload, anchor, context["canonical_ranges"])
            if errors:
                failure = {"stage": "payload_validation", **metadata, "errors": errors, "warnings": warnings}
                report["failures"] += 1
                append_jsonl(paths["manifest.jsonl"], {**failure, "status": "validation_failed", "timestamp_utc": now_utc()})
                raise RuntimeError(f"payload validation failed for {request_id}")
            payloads[recipient_id] = payload
            soft_warnings.extend({"recipient_id": recipient_id, "warning": warning} for warning in warnings)
            frozen = frozen_assessments.loc[frozen_assessments.recipient_id.eq(recipient_id)].copy()
            recipient_rows = build_assessment_rows(recipient, anchor, frozen, payload, assessment_columns)
            rows.extend(recipient_rows)
            append_checkpoint(paths["validated_recipient_checkpoint.csv"], pd.DataFrame(recipient_rows, columns=assessment_columns))
            report["recipients_validated"] += 1
            append_jsonl(paths["manifest.jsonl"], {
                "request_id": request_id, "recipient_id": recipient_id,
                "request_sequence_number": int(recipient["request_sequence_number"]), "attempt_number": 1,
                "status": "validated_and_checkpointed", "checkpoint_recipient_count": report["recipients_validated"],
                "checkpoint_row_count": len(rows), "soft_warnings": warnings, "timestamp_utc": now_utc(),
            })
            write_json_state(paths["completion_state.json"], {
                "run_id": RUN_ID, "chunk_id": chunk_id, "status": "running", "automatic_resume": False,
                "requests_attempted": report["api_requests_attempted"], "requests_completed": report["api_requests_completed"],
                "recipients_validated": report["recipients_validated"], "checkpoint_rows": len(rows),
                "last_validated_request_id": request_id, "last_validated_recipient_id": recipient_id,
            })

        assessment = pd.DataFrame(rows, columns=assessment_columns)
        reconciliation = reconcile_payloads(payloads, assessment)
        validation_context = {
            **context,
            "recipients": recipients,
            "assessments": frozen_assessments,
            "anchors": anchors,
        }
        validation = validate_complete_chunk(
            assessment, identities, recipients, anchors, payloads, reconciliation, validation_context
        )
        diagnostics = build_diagnostics(assessment, recipients, anchors)
        if diagnostics["identical_complete_trajectories"]["identical_trajectory_groups"] != 0:
            validation["checks"]["no_identical_complete_trajectories"] = False
            validation["failures"].append("no_identical_complete_trajectories")
            validation["status"] = "failed"
        else:
            validation["checks"]["no_identical_complete_trajectories"] = True
        checkpoint = pd.read_csv(paths["validated_recipient_checkpoint.csv"])
        try:
            pd.testing.assert_frame_equal(assessment, checkpoint, check_dtype=False)
            validation["checks"]["checkpoint_assessment_equality"] = True
        except AssertionError:
            validation["checks"]["checkpoint_assessment_equality"] = False
            validation["failures"].append("checkpoint_assessment_equality")
            validation["status"] = "failed"
        if validation["status"] != "passed":
            failure = {"stage": "complete_chunk_validation", "validation": validation}
            report["failures"] += 1
            raise RuntimeError("complete chunk validation failed")

        day7 = assessment.loc[assessment.days_since_transplant.eq(7)].copy()
        anchor_audit = anchors.merge(
            day7[["recipient_id", "creatinine_mg_dl", "urine_output_ml_24h", "tacrolimus_level_ng_ml", "medication_adherence_pct"]],
            on="recipient_id", validate="one_to_one",
        )
        for output, anchor_field in {
            "creatinine_mg_dl": "day7_creatinine_anchor_mg_dl",
            "urine_output_ml_24h": "day7_urine_output_anchor_ml_24h",
            "tacrolimus_level_ng_ml": "day7_tacrolimus_anchor_ng_ml",
            "medication_adherence_pct": "day7_medication_adherence_anchor_pct",
        }.items():
            difference = (anchor_audit[output] - anchor_audit[anchor_field]).abs()
            anchor_audit[f"{output}_absolute_anchor_difference"] = difference
            anchor_audit[f"{output}_within_tolerance"] = difference.le(ANCHOR_TOLERANCES[output])
        write_csv_exclusive(paths[f"assessment_chunk_{suffix}.csv"], assessment)
        write_csv_exclusive(paths[f"anchor_audit_chunk_{suffix}.csv"], anchor_audit)
        write_json_exclusive(paths["request_response_reconciliation.json"], reconciliation)
        write_json_exclusive(paths["diagnostic_quality_report.json"], diagnostics)
        report["status"] = "completed"
        write_json_state(paths["completion_state.json"], {
            "run_id": RUN_ID, "chunk_id": chunk_id, "status": "completed", "automatic_resume": False,
            "requests_attempted": 100, "requests_completed": 100, "recipients_validated": 100,
            "checkpoint_rows": 600, "assessment_chunk_created": True,
            "chunks_011_to_100_started": False, "final_dataset_assembly_started": False,
        })
    except Exception as exc:
        if failure is None:
            failure = {
                "stage": "execution_or_chunk_validation", "exception_type": type(exc).__name__,
                "message": str(exc), "traceback": traceback.format_exc(),
            }
            report["failures"] += 1
        report["status"] = "failed"
        write_json_state(paths["completion_state.json"], {
            "run_id": RUN_ID, "chunk_id": chunk_id, "status": "failed", "automatic_resume": False,
            "requests_attempted": report["api_requests_attempted"], "requests_completed": report["api_requests_completed"],
            "recipients_validated": report["recipients_validated"], "checkpoint_rows": len(rows),
            "failure": failure, "retry_performed": False,
            "assessment_chunk_created": paths[f"assessment_chunk_{suffix}.csv"].exists(),
        })
    finally:
        integrity_in_finally = assert_global_integrity(context, completed_fingerprints)
        token_totals = {
            key: sum(int(record.get(key, 0) or 0) for record in usage_records)
            for key in ["prompt_tokens", "completion_tokens", "total_tokens"]
        }
        report.update({
            "failure": failure,
            "elapsed_seconds": time.monotonic() - started,
            "finish_reason_counts": dict(Counter(item["finish_reason"] for item in finish_records)),
            "reasoning_content_status_counts": dict(Counter(item["reasoning_content_status"] for item in finish_records)),
            "finish_records": finish_records,
            "token_usage_total": token_totals,
            "request_records": count_jsonl(paths["requests.jsonl"]),
            "response_records": count_jsonl(paths["responses.jsonl"]),
            "manifest_records": count_jsonl(paths["manifest.jsonl"]),
            "manifest_status_counts": dict(Counter(record["status"] for record in line_records(paths["manifest.jsonl"]))),
            "checkpoint_rows": len(rows),
            "chunk_dimensions": [len(rows), len(assessment_columns)] if rows else [0, len(assessment_columns)],
            "validation": validation,
            "reconciliation": reconciliation,
            "diagnostics": diagnostics,
            "soft_warnings": soft_warnings,
            "integrity_in_finally": integrity_in_finally,
            "protected_before": context["protected_before"],
            "protected_in_finally": protected_fingerprint(),
            "artifact_fingerprints_excluding_report": chunk_file_fingerprints(number, exclude_report=True),
            "created_files": sorted(path.name for path in directory.iterdir() if path.is_file()) + [f"chunk_{suffix}_run_report.json"],
            "chunks_011_to_100_started": False,
            "final_dataset_assembly_started": False,
        })
        write_json_exclusive(paths[f"chunk_{suffix}_run_report.json"], report)
    assert_global_integrity(context, completed_fingerprints)
    return report


def aggregate_through_chunk_010(context, completed_fingerprints, new_chunk_reports):
    started = time.monotonic()
    directory = progress_directory()
    assert not directory.exists(), "progress checkpoint already exists"
    assessments = []
    identities = []
    anchors = []
    requests = []
    responses = []
    manifests = []
    chunk_reports = {}
    per_chunk_warnings = {}
    for number in range(1, 11):
        suffix = f"{number:03d}"
        chunk = chunk_directory(number)
        assessments.append(pd.read_csv(chunk / f"assessment_chunk_{suffix}.csv"))
        identities.append(pd.read_csv(chunk / f"identity_chunk_{suffix}.csv", keep_default_na=False))
        anchors.append(pd.read_csv(chunk / f"anchor_audit_chunk_{suffix}.csv"))
        requests.extend(line_records(chunk / "requests.jsonl"))
        responses.extend(line_records(chunk / "responses.jsonl"))
        manifests.extend(line_records(chunk / "manifest.jsonl"))
        report = json.loads((chunk / f"chunk_{suffix}_run_report.json").read_text(encoding="utf-8"))
        chunk_reports[number] = report
        diagnostic = json.loads((chunk / "diagnostic_quality_report.json").read_text(encoding="utf-8"))
        per_chunk_warnings[suffix] = {
            "soft_validation_warnings": report.get("soft_warnings", []),
            "diagnostic_warnings": diagnostic.get("warnings", []),
            "failures": report.get("failures", 0),
        }
    assessment = pd.concat(assessments, ignore_index=True)
    identity = pd.concat(identities, ignore_index=True)
    anchor = pd.concat(anchors, ignore_index=True)
    recipients = pd.concat([context["slices"][number]["recipients"] for number in range(1, 11)], ignore_index=True)
    frozen_assessment = pd.concat([context["slices"][number]["assessments"] for number in range(1, 11)], ignore_index=True)
    expected_recipient_ids = set(recipients.recipient_id)
    observed_recipient_ids = set(assessment.recipient_id)
    request_ids = [record["request_id"] for record in requests]
    response_request_ids = [record["request_id"] for record in responses]

    checks = {
        "ten_completed_chunks": all(chunk_reports[number]["status"] == "completed" for number in range(1, 11)),
        "recipients_1000": assessment.recipient_id.nunique() == 1_000,
        "rows_6000": assessment.shape == (6_000, 31),
        "assessment_ids_unique": assessment.assessment_id.is_unique,
        "recipient_day_unique": not assessment.duplicated(["recipient_id", "days_since_transplant"]).any(),
        "request_ids_unique": len(request_ids) == len(set(request_ids)) == 1_000,
        "request_records_1000": len(requests) == 1_000,
        "response_records_1000": len(responses) == 1_000,
        "response_request_ids_unique": len(response_request_ids) == len(set(response_request_ids)) == 1_000,
        "request_response_ids_equal": set(request_ids) == set(response_request_ids),
        "manifest_records_4000": len(manifests) == 4_000,
        "no_duplicate_rows": not assessment.duplicated().any(),
        "no_missing_recipients": observed_recipient_ids == expected_recipient_ids,
        "no_unexpected_recipients": observed_recipient_ids == expected_recipient_ids,
        "six_rows_each": assessment.groupby("recipient_id").size().eq(6).all(),
        "ordered_days": assessment.groupby("recipient_id").days_since_transplant.apply(list).map(lambda values: values == DAYS).all(),
    }

    measurement_fields = QWEN_FIELDS
    response_payloads = {}
    response_parse_errors = []
    for record in responses:
        try:
            complete = json.loads(record["serialized_response_json"])
            content = complete["choices"][0]["message"]["content"]
            response_payloads[record["recipient_id"]] = json.loads(content)
            assert sha256_text(record["serialized_response_json"]) == record["serialized_response_sha256"]
        except Exception as exc:
            response_parse_errors.append({"request_id": record.get("request_id"), "error": str(exc)})
    mismatches = []
    reconciled_fields = 0
    for recipient_id, payload in response_payloads.items():
        rows = assessment.loc[assessment.recipient_id.eq(recipient_id)].sort_values("days_since_transplant")
        for position, item in enumerate(payload.get("assessments", [])):
            for field in measurement_fields:
                reconciled_fields += 1
                left = float(item[field])
                right = float(rows.iloc[position][field])
                matches = left == right if field == "days_since_transplant" else bool(
                    np.isclose(left, right, rtol=0, atol=1e-12)
                )
                if not matches:
                    mismatches.append({"recipient_id": recipient_id, "position": position, "field": field})
    checks["all_complete_responses_parse"] = not response_parse_errors and len(response_payloads) == 1_000
    checks["response_assessment_reconciliation"] = reconciled_fields == 30_000 and not mismatches

    diagnostics = build_diagnostics(assessment, recipients, anchor)
    checks["no_cross_chunk_identical_trajectories"] = diagnostics["identical_complete_trajectories"]["identical_trajectory_groups"] == 0

    frozen_index = frozen_assessment.set_index("assessment_id", drop=False)
    frozen_errors = []
    for row in assessment.itertuples(index=False):
        expected = frozen_index.loc[row.assessment_id]
        for field in frozen_assessment.columns:
            if field not in assessment.columns:
                continue
            current_value = getattr(row, field)
            expected_value = expected[field]
            if isinstance(current_value, (int, float, np.integer, np.floating)) and isinstance(expected_value, (int, float, np.integer, np.floating)):
                matches = bool(np.isclose(float(current_value), float(expected_value), rtol=0, atol=1e-12))
            else:
                matches = str(current_value) == str(expected_value)
            if not matches and field not in ["infection_indicator", "previous_rejection", "acute_rejection_within_30_days"]:
                frozen_errors.append(f"{row.assessment_id}:{field}")
    checks["frozen_assessment_metadata_exact"] = not frozen_errors
    frozen_targets = frozen_index.loc[assessment.assessment_id, "acute_rejection_within_30_days"].astype(int).to_numpy()
    checks["target_count_matches_frozen"] = int(assessment.acute_rejection_within_30_days.sum()) == int(frozen_targets.sum())
    frozen_event_count = int(recipients.confirmed_rejection_event_days.ne("[]").sum())
    checks["event_count_matches_frozen"] = diagnostics["event_recipients"] == frozen_event_count

    recipient_donor = recipients.set_index("recipient_id").donor_id
    checks["donor_relationships_match_frozen"] = all(
        group.donor_id.nunique() == 1 and group.donor_id.iloc[0] == recipient_donor.loc[recipient_id]
        for recipient_id, group in assessment.groupby("recipient_id")
    )
    identity_conflicts = []
    for entity_id, group in identity.groupby("entity_id"):
        normalized = group.drop_duplicates()
        if len(normalized) != 1:
            identity_conflicts.append(entity_id)
    checks["shared_donor_identity_consistent"] = not identity_conflicts
    checks["checkpoint_assessment_equality_all_chunks"] = all(
        pd.read_csv(chunk_directory(number) / "validated_recipient_checkpoint.csv").equals(
            pd.read_csv(chunk_directory(number) / f"assessment_chunk_{number:03d}.csv")
        )
        for number in range(1, 11)
    )
    direct = set(context["field_lists"]["direct_identifiers"])
    forbidden_generation = set(context["field_lists"]["event_control_metadata"]) | {
        column for column in context["anchors_all"].columns if "anchor" in column
    }
    checks["no_direct_identifiers_in_assessment"] = not direct.intersection(assessment.columns)
    checks["no_generation_controls_in_assessment"] = not forbidden_generation.intersection(assessment.columns)
    checks["no_joined_42_column_export"] = len(assessment.columns) == 31

    failures = [name for name, passed in checks.items() if not bool(passed)]
    reconciliation = {
        "status": "passed" if not failures else "failed",
        "checks": {name: bool(value) for name, value in checks.items()},
        "failures": failures,
        "response_parse_errors": response_parse_errors[:20],
        "response_assessment_mismatch_count": len(mismatches),
        "response_assessment_mismatch_examples": mismatches[:20],
        "qwen_fields_reconciled": reconciled_fields,
        "expected_qwen_fields": 30_000,
        "numeric_reconciliation_absolute_tolerance": 1e-12,
        "frozen_metadata_mismatch_count": len(frozen_errors),
        "frozen_metadata_mismatch_examples": frozen_errors[:20],
        "identity_conflict_count": len(identity_conflicts),
        "identity_conflict_examples": identity_conflicts[:20],
    }

    token_totals = {
        key: sum(int(chunk_reports[number]["token_usage_total"].get(key, 0)) for number in range(1, 11))
        for key in ["prompt_tokens", "completion_tokens", "total_tokens"]
    }
    chunk_elapsed = {f"chunk_{number:03d}": float(chunk_reports[number]["elapsed_seconds"]) for number in range(1, 11)}
    distribution = {
        **diagnostics,
        "status": "diagnostic_only_not_used_for_regeneration",
        "per_chunk_warnings_and_failures": per_chunk_warnings,
        "aggregate_token_usage": token_totals,
        "per_chunk_elapsed_seconds": chunk_elapsed,
        "total_chunk_elapsed_seconds": sum(chunk_elapsed.values()),
    }

    integrity_before_progress = assert_global_integrity(context, completed_fingerprints)
    chunk_fingerprints = {f"chunk_{number:03d}": tree_fingerprint(chunk_directory(number)) for number in range(1, 11)}
    integrity = {
        "status": "passed" if all(integrity_before_progress.values()) else "failed",
        "checked_at_utc": now_utc(),
        "protected_before": context["protected_before"],
        "protected_after_chunks": protected_fingerprint(),
        "global_integrity_checks": integrity_before_progress,
        "chunk_fingerprints": chunk_fingerprints,
        "chunk_001_expected_file_fingerprints": CHUNK_001_FINGERPRINTS,
        "chunks_011_to_100_started": False,
        "final_dataset_assembly_started": False,
    }

    directory.mkdir(parents=False, exist_ok=False)
    write_json_exclusive(directory / "cross_chunk_reconciliation.json", reconciliation)
    write_json_exclusive(directory / "cross_chunk_distribution_report.json", distribution)
    write_json_exclusive(directory / "cross_chunk_integrity_report.json", integrity)
    integrity_after_progress = assert_global_integrity(context, completed_fingerprints)
    progress_file_fingerprints = {
        path.name: file_fingerprint(path)
        for path in sorted(directory.iterdir())
        if path.is_file()
    }
    progress = {
        "run_id": RUN_ID,
        "checkpoint": "through_chunk_010",
        "status": "completed" if not failures else "failed",
        "created_at_utc": now_utc(),
        "chunks_present": list(range(1, 11)),
        "chunks_attempted_this_execution": sorted(new_chunk_reports),
        "chunks_completed_this_execution": sorted(number for number, report in new_chunk_reports.items() if report["status"] == "completed"),
        "total_completed_chunks": 10,
        "requests_attempted_this_execution": sum(report["api_requests_attempted"] for report in new_chunk_reports.values()),
        "requests_completed_this_execution": sum(report["api_requests_completed"] for report in new_chunk_reports.values()),
        "aggregate_request_records": len(requests),
        "aggregate_response_records": len(responses),
        "aggregate_manifest_records": len(manifests),
        "aggregate_dimensions": list(assessment.shape),
        "aggregate_token_usage": token_totals,
        "aggregate_chunk_elapsed_seconds": sum(chunk_elapsed.values()),
        "aggregate_checkpoint_elapsed_seconds": time.monotonic() - started,
        "target_count": diagnostics["target_count"],
        "target_prevalence": diagnostics["target_prevalence"],
        "event_recipients": diagnostics["event_recipients"],
        "non_event_recipients": diagnostics["non_event_recipients"],
        "reconciliation_status": reconciliation["status"],
        "reconciliation_failures": failures,
        "per_chunk_warnings_and_failures": per_chunk_warnings,
        "integrity_after_aggregate_checkpoint": integrity_after_progress,
        "created_files": [
            "production_progress_report.json", "cross_chunk_reconciliation.json",
            "cross_chunk_distribution_report.json", "cross_chunk_integrity_report.json",
        ],
        "progress_file_fingerprints_excluding_self": progress_file_fingerprints,
        "chunks_011_to_100_started": False,
        "final_dataset_assembly_started": False,
        "classifier_trained": False,
        "generator_tuned": False,
    }
    write_json_exclusive(directory / "production_progress_report.json", progress)
    assert not failures, failures
    return progress


def execute_chunks_002_010(context=None):
    context = preflight_chunks_002_010() if context is None else context
    if progress_directory().exists():
        return json.loads((progress_directory() / "production_progress_report.json").read_text(encoding="utf-8"))
    completed_fingerprints = {1: context["chunk_001_before"]}
    for number in context["completed_existing"]:
        completed_fingerprints[number] = verify_recorded_completed_chunk(number)
    new_chunk_reports = {}
    try:
        for number in AUTHORIZED_CHUNKS:
            if number in context["completed_existing"]:
                continue
            report = run_single_chunk(number, context, completed_fingerprints)
            new_chunk_reports[number] = report
            if report["status"] != "completed":
                raise RuntimeError(f"chunk {number:03d} failed; execution stopped without retry or resume")
            completed_fingerprints[number] = tree_fingerprint(chunk_directory(number))
            assert_global_integrity(context, completed_fingerprints)
        assert set(completed_fingerprints) == set(range(1, 11))
        return aggregate_through_chunk_010(context, completed_fingerprints, new_chunk_reports)
    finally:
        assert_global_integrity(context, completed_fingerprints)


CHUNKS_002_010_PROGRESS_REPORT = execute_chunks_002_010()
print(json.dumps({
    "status": CHUNKS_002_010_PROGRESS_REPORT["status"],
    "chunks_completed_this_execution": CHUNKS_002_010_PROGRESS_REPORT["chunks_completed_this_execution"],
    "aggregate_dimensions": CHUNKS_002_010_PROGRESS_REPORT["aggregate_dimensions"],
    "aggregate_token_usage": CHUNKS_002_010_PROGRESS_REPORT["aggregate_token_usage"],
    "reconciliation_status": CHUNKS_002_010_PROGRESS_REPORT["reconciliation_status"],
    "chunks_011_to_100_started": CHUNKS_002_010_PROGRESS_REPORT["chunks_011_to_100_started"],
    "final_dataset_assembly_started": CHUNKS_002_010_PROGRESS_REPORT["final_dataset_assembly_started"],
}, indent=2))


## Production validation amendment 001 and guarded resume through chunk 010

This additive cell preserves the failed chunk-006 evidence, retrospectively revalidates all 572 preserved responses, and documents the post-hoc urine-output tolerance amendment from 120 mL to 150 mL. The frozen prompt and generator remain unchanged. It recovers recipient 572 without an API request and resumes only with the exact manual gate `RESUME_QWEN_V32_PRODUCTION_CHUNKS_006_010=QWEN-V32-PROD-001-RESUME-006-010-A001`. It permits at most 428 new sequential, single-attempt requests; chunks 011–100, final assembly, and classifier training remain disabled.

In [ ]:
import hashlib
import json
import os
import time
import traceback
from collections import Counter
from datetime import date, timedelta
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd
from openai import OpenAI


_repository_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists())
_notebook_path = _repository_root / "notebooks/KidneyTransplant/00_generate_and_validate_dataset.ipynb"
_notebook = nbformat.read(_notebook_path, as_version=4)
_multi_source = next(
    cell.source
    for cell in _notebook.cells
    if cell.cell_type == "code" and "def run_single_chunk(" in cell.source
)
_multi_primitives = _multi_source.rsplit("CHUNKS_002_010_PROGRESS_REPORT = execute_chunks_002_010()", 1)[0]
exec(compile(_multi_primitives, "frozen-chunks-002-010-primitives", "exec"), globals())


AMENDMENT_ID = "PROD-VALIDATION-AMENDMENT-001"
AMENDMENT_DIR_NAME = "validation_amendment_001"
RESUME_GATE_NAME = "RESUME_QWEN_V32_PRODUCTION_CHUNKS_006_010"
RESUME_GATE_VALUE = "QWEN-V32-PROD-001-RESUME-006-010-A001"
RESUME_CHUNKS = [6, 7, 8, 9, 10]
MAX_NEW_API_REQUESTS = 428
ORIGINAL_URINE_TOLERANCE = 120.0
AMENDED_URINE_TOLERANCE = 150.0
ORIGINAL_ANCHOR_TOLERANCES = dict(ANCHOR_TOLERANCES)
AMENDED_ANCHOR_TOLERANCES = {**ORIGINAL_ANCHOR_TOLERANCES, "urine_output_ml_24h": AMENDED_URINE_TOLERANCE}
ANCHOR_TOLERANCES = AMENDED_ANCHOR_TOLERANCES
MULTI_GATE_NAME = RESUME_GATE_NAME
MULTI_GATE_VALUE = RESUME_GATE_VALUE

ORIGINAL_CHUNK_006_REPORT_SHA256 = "285ffed8cca3b3cec14b7ddc57a879a07a3949b5f1f846b7e5c95be509d3849b"
ORIGINAL_CHUNK_006_FILES = {
    "chunk_006_run_report.json": "285ffed8cca3b3cec14b7ddc57a879a07a3949b5f1f846b7e5c95be509d3849b",
    "completion_state.json": "e242c3710035a60075d0e0c61edf7bc92fda3dc990869459e60e7362e837da33",
    "identity_chunk_006.csv": "45a0bd6c43cfe0ebf69adef1a4c75e7d9dfeb3a5b9afc6e0a8a6c33bcb4275af",
    "manifest.jsonl": "186346f8564967472c16769be7e9ebd30cfb3aa0044121665ff6233de66776bd",
    "requests.jsonl": "440ac051e4dea81cc8e7b753bf7f31c3a1258920403bcd016efb14435f50bf97",
    "responses.jsonl": "140fb8fbe83ce8c27fb352eba1e92d8f552791cce4f5e80a84e32cccb11e91ec",
    "run_configuration.json": "e35bd773c34280c12be1b2d0c5684988a1347854758d8b2b98d0b9e53f44712a",
    "validated_recipient_checkpoint.csv": "692a08271e7be18c38758482ab70f435736f475b28d5395721cd5e27a535e33d",
}


def amendment_directory():
    return PRODUCTION_ROOT / AMENDMENT_DIR_NAME


def parse_preserved_response(record):
    serialized = record["serialized_response_json"]
    assert sha256_text(serialized) == record["serialized_response_sha256"]
    complete = json.loads(serialized)
    content = complete["choices"][0]["message"]["content"]
    return complete, json.loads(content)


def fingerprint_named_files(directory):
    return {path.name: file_fingerprint(path) for path in sorted(directory.iterdir()) if path.is_file()}


def resume_preflight():
    assert os.environ.get(RESUME_GATE_NAME) == RESUME_GATE_VALUE, "exact amendment resume gate mismatch"
    for name in [GATE_NAME, "RUN_QWEN_V32_PRODUCTION_CHUNKS_002_010", "RUN_QWEN_V32_CONFIRM20", "RUN_CORRECTED_10", "RUN_100_RECIPIENTS", "RUN_10000_RECIPIENTS", "RUN_FULL_GENERATION"]:
        assert not os.environ.get(name), f"{name} must be false"
    for number in range(1, 101):
        assert not os.environ.get(f"RUN_QWEN_V32_PRODUCTION_CHUNK_{number:03d}"), f"individual chunk {number} gate must be false"
    assert not amendment_directory().exists(), "validation amendment already exists; automatic rerun prohibited"
    assert not (chunk_directory(6) / "chunk_006_resume_completion_report.json").exists()
    assert all(not chunk_directory(number).exists() for number in range(7, 101))
    assert not progress_directory().exists()
    assert not RUN_10000_RECIPIENTS and not RUN_FULL_GENERATION

    completed_fingerprints = {1: verify_chunk_001_immutable()}
    for number in range(2, 6):
        completed_fingerprints[number] = verify_recorded_completed_chunk(number)
    design_checks, design_validation = verify_frozen_production_design()
    assert all(design_checks.values())
    prompt_path = V32_DESIGN / "qwen_kidney_v3_2_prompt_template.txt"
    template = prompt_path.read_text(encoding="utf-8")
    assert sha256_text(template) == PROMPT_SHA256
    assert "120" in template and "150" not in template
    api_config = json.loads((V32_DESIGN / "api_configuration.json").read_text(encoding="utf-8"))
    assert api_config["client"]["max_retries"] == 0 and api_config["automatic_retry"] is False
    assert api_config["completion"] == {
        "model": "Qwen/Qwen3.6-35B-A3B", "max_tokens": 8192, "temperature": 0.7,
        "top_p": 0.8, "presence_penalty": 1.5,
        "extra_body": {"top_k": 20, "chat_template_kwargs": {"enable_thinking": False}},
    }

    chunk6 = chunk_directory(6)
    assert {path.name for path in chunk6.iterdir() if path.is_file()} == set(ORIGINAL_CHUNK_006_FILES)
    for name, expected_hash in ORIGINAL_CHUNK_006_FILES.items():
        assert sha256_file(chunk6 / name) == expected_hash, name
    original_report = json.loads((chunk6 / "chunk_006_run_report.json").read_text(encoding="utf-8"))
    assert original_report["status"] == "failed" and original_report["retries"] == 0
    assert original_report["api_requests_attempted"] == original_report["api_requests_completed"] == 72
    assert original_report["recipients_validated"] == 71 and original_report["checkpoint_rows"] == 426
    failure = original_report["failure"]
    assert failure["stage"] == "payload_validation"
    assert failure["request_id"] == "QWEN-V32-PROD-REQ000572" and failure["recipient_id"] == "V32P-R000572"
    assert failure["errors"] == ["day-7 urine_output_ml_24h differs from anchor by 123.80000000000018, exceeding tolerance"]

    requests = line_records(chunk6 / "requests.jsonl")
    responses = line_records(chunk6 / "responses.jsonl")
    manifest = line_records(chunk6 / "manifest.jsonl")
    checkpoint = pd.read_csv(chunk6 / "validated_recipient_checkpoint.csv")
    assert len(requests) == len(responses) == 72
    assert len({record["request_id"] for record in requests}) == 72
    assert len({record["request_id"] for record in responses}) == 72
    assert [record["request_sequence_number"] for record in requests] == list(range(501, 573))
    assert requests[-1]["request_id"] == "QWEN-V32-PROD-REQ000572"
    assert sum(record["status"] == "validation_failed" for record in manifest) == 1
    assert [record for record in manifest if record["status"] == "validation_failed"][0]["recipient_id"] == "V32P-R000572"
    assert not any(record["status"] == "request_failed" for record in manifest)
    assert checkpoint.shape == (426, 31)
    assert checkpoint.recipient_id.nunique() == 71
    assert set(checkpoint.recipient_id) == {f"V32P-R{number:06d}" for number in range(501, 572)}
    assert "V32P-R000572" not in set(checkpoint.recipient_id)

    recipients_all = pd.read_csv(PRODUCTION_DESIGN / "production_recipient_metadata.csv")
    assessments_all = pd.read_csv(PRODUCTION_DESIGN / "production_assessment_metadata.csv")
    identities_all = pd.read_csv(PRODUCTION_DESIGN / "production_identity_skeleton.csv", keep_default_na=False)
    anchors_all = pd.read_csv(PRODUCTION_DESIGN / "production_anchor_metadata.csv")
    slices = {number: frozen_chunk_slice(number, recipients_all, assessments_all, identities_all, anchors_all) for number in range(1, 11)}
    for slice_context in slices.values():
        verify_frozen_slice(slice_context)
    field_lists = json.loads((V31_DESIGN / "field_lists.json").read_text(encoding="utf-8"))
    canonical_ranges = json.loads((V31_DESIGN / "canonical_qwen_ranges.json").read_text(encoding="utf-8"))
    assessment_columns = pd.read_csv(V31_DESIGN / "assessment_table_schema.csv").field_name.tolist()

    # Retrospectively validate every preserved production response before any new request.
    all_request_records = []
    all_response_records = []
    all_manifest_records = []
    for number in range(1, 7):
        directory = chunk_directory(number)
        all_request_records.extend(line_records(directory / "requests.jsonl"))
        all_response_records.extend(line_records(directory / "responses.jsonl"))
        all_manifest_records.extend(line_records(directory / "manifest.jsonl"))
    assert len(all_request_records) == len(all_response_records) == 572
    assert len({record["request_id"] for record in all_request_records}) == 572
    assert len({record["request_id"] for record in all_response_records}) == 572
    request_lookup = {record["request_id"]: record for record in all_request_records}
    recipient_lookup = recipients_all.set_index("recipient_id")
    anchor_lookup = anchors_all.set_index("recipient_id")
    frozen_by_recipient = {recipient_id: frame.copy() for recipient_id, frame in assessments_all.groupby("recipient_id")}
    response_payloads = {}
    retrospective_rows = []
    structural_errors = []
    non_urine_errors = []
    derived_errors = []
    for response_record in all_response_records:
        request_id = response_record["request_id"]
        request_record = request_lookup[request_id]
        recipient_id = request_record["recipient_id"]
        _, payload = parse_preserved_response(response_record)
        response_payloads[recipient_id] = payload
        anchor = anchor_lookup.loc[recipient_id].to_dict()
        errors, warnings = validate_payload(payload, anchor, canonical_ranges)
        if errors:
            structural_errors.append({"recipient_id": recipient_id, "errors": errors, "warnings": warnings})
        day7 = payload["assessments"][0]
        differences = {
            "creatinine_mg_dl": abs(float(day7["creatinine_mg_dl"]) - float(anchor["day7_creatinine_anchor_mg_dl"])),
            "urine_output_ml_24h": abs(float(day7["urine_output_ml_24h"]) - float(anchor["day7_urine_output_anchor_ml_24h"])),
            "tacrolimus_level_ng_ml": abs(float(day7["tacrolimus_level_ng_ml"]) - float(anchor["day7_tacrolimus_anchor_ng_ml"])),
            "medication_adherence_pct": abs(float(day7["medication_adherence_pct"]) - float(anchor["day7_medication_adherence_anchor_pct"])),
        }
        for field in ["creatinine_mg_dl", "tacrolimus_level_ng_ml", "medication_adherence_pct"]:
            if differences[field] > ORIGINAL_ANCHOR_TOLERANCES[field] + 1e-12:
                non_urine_errors.append(f"{recipient_id}:{field}:{differences[field]}")
        recipient = recipient_lookup.loc[recipient_id].to_dict()
        try:
            built = build_assessment_rows(
                recipient, anchor, frozen_by_recipient[recipient_id], payload, assessment_columns
            )
            assert len(built) == 6
        except Exception as exc:
            derived_errors.append({"recipient_id": recipient_id, "error": str(exc)})
        retrospective_rows.append({
            "request_id": request_id,
            "recipient_id": recipient_id,
            "request_sequence_number": int(request_record["request_sequence_number"]),
            "attempt_number": int(request_record["attempt_number"]),
            "creatinine_day7_absolute_anchor_difference": differences["creatinine_mg_dl"],
            "urine_output_day7_absolute_anchor_difference_ml_24h": differences["urine_output_ml_24h"],
            "tacrolimus_day7_absolute_anchor_difference_ng_ml": differences["tacrolimus_level_ng_ml"],
            "adherence_day7_absolute_anchor_difference_pct_points": differences["medication_adherence_pct"],
            "passes_original_urine_tolerance_120_ml": differences["urine_output_ml_24h"] <= ORIGINAL_URINE_TOLERANCE + 1e-12,
            "passes_amended_urine_tolerance_150_ml": differences["urine_output_ml_24h"] <= AMENDED_URINE_TOLERANCE + 1e-12,
            "passes_all_non_urine_anchor_tolerances": all(
                differences[field] <= ORIGINAL_ANCHOR_TOLERANCES[field] + 1e-12
                for field in ["creatinine_mg_dl", "tacrolimus_level_ng_ml", "medication_adherence_pct"]
            ),
            "structurally_and_clinically_valid_under_amendment": not errors,
        })
    retrospective = pd.DataFrame(retrospective_rows).sort_values("request_sequence_number").reset_index(drop=True)
    urine = retrospective.urine_output_day7_absolute_anchor_difference_ml_24h
    assert len(retrospective) == 572
    assert int(urine.gt(ORIGINAL_URINE_TOLERANCE + 1e-12).sum()) == 1
    assert int(urine.gt(AMENDED_URINE_TOLERANCE + 1e-12).sum()) == 0
    assert not structural_errors and not non_urine_errors and not derived_errors
    row572 = retrospective.loc[retrospective.recipient_id.eq("V32P-R000572")].iloc[0]
    assert np.isclose(row572.urine_output_day7_absolute_anchor_difference_ml_24h, 123.8, rtol=0, atol=1e-9)
    payload572 = response_payloads["V32P-R000572"]
    recipient572 = recipient_lookup.loc["V32P-R000572"].to_dict()
    anchor572 = anchor_lookup.loc["V32P-R000572"].to_dict()
    recovery_rows = build_assessment_rows(
        recipient572, anchor572, frozen_by_recipient["V32P-R000572"], payload572, assessment_columns
    )
    assert len(recovery_rows) == 6

    immutable_paths = immutable_inputs()
    context = {
        "checks": {
            "completed_chunks_001_005_immutable": True,
            "chunk_006_exact_partial_state": True,
            "frozen_design_artifacts": all(design_checks.values()),
            "prompt_hash_unchanged": True,
            "prompt_retains_120_ml_instruction": True,
            "anchor_algorithm_unchanged": anchors_all.anchor_algorithm_version.eq(ANCHOR_ALGORITHM_VERSION).all(),
            "preserved_responses_recalculated": len(retrospective) == 572,
            "exactly_one_exceeds_120_ml": int(urine.gt(120.0 + 1e-12).sum()) == 1,
            "none_exceeds_150_ml": int(urine.gt(150.0 + 1e-12).sum()) == 0,
            "all_non_urine_anchor_checks_pass": not non_urine_errors,
            "all_preserved_payloads_valid_under_amendment": not structural_errors,
            "all_python_derivations_available": not derived_errors,
            "recipient_572_valid_under_amendment": bool(row572.structurally_and_clinically_valid_under_amendment),
            "recipient_572_not_retried": sum(record["recipient_id"] == "V32P-R000572" for record in requests) == 1,
            "next_request_is_573": recipients_all.loc[recipients_all.request_sequence_number.eq(573), "recipient_id"].item() == "V32P-R000573",
            "chunks_007_010_absent": all(not chunk_directory(number).exists() for number in range(7, 11)),
            "chunks_011_100_disabled": all(not chunk_directory(number).exists() for number in range(11, 101)),
            "maximum_new_requests_428": MAX_NEW_API_REQUESTS == 428,
            "automatic_retries_disabled": api_config["client"]["max_retries"] == 0,
            "final_assembly_disabled": True,
        },
        "design_validation": design_validation,
        "template": template,
        "api_config": api_config,
        "slices": slices,
        "recipients_all": recipients_all,
        "assessments_all": assessments_all,
        "identities_all": identities_all,
        "anchors_all": anchors_all,
        "field_lists": field_lists,
        "canonical_ranges": canonical_ranges,
        "assessment_columns": assessment_columns,
        "response_payloads": response_payloads,
        "retrospective": retrospective,
        "recovery_rows": recovery_rows,
        "completed_fingerprints": completed_fingerprints,
        "immutable_paths": immutable_paths,
        "immutable_before": snapshot_immutable(immutable_paths),
        "notebooks_before": snapshot_notebooks_01_05(),
        "protected_before": protected_fingerprint(),
        "production_design_before": tree_fingerprint(PRODUCTION_DESIGN),
        "chunk_006_partial_before": tree_fingerprint(chunk6),
        "chunk_006_original_report_before": file_fingerprint(chunk6 / "chunk_006_run_report.json"),
        "request_572_hash_before": sha256_file(chunk6 / "requests.jsonl"),
        "response_572_hash_before": sha256_file(chunk6 / "responses.jsonl"),
    }
    assert {key: context["protected_before"][key] for key in EXPECTED_PROTECTED} == EXPECTED_PROTECTED
    assert all(context["checks"].values())
    return context


def difference_distribution(series):
    return {
        "count": int(series.count()), "minimum": float(series.min()), "maximum": float(series.max()),
        "mean": float(series.mean()), "standard_deviation": float(series.std(ddof=0)),
        "median": float(series.median()), "p90": float(series.quantile(0.90)),
        "p95": float(series.quantile(0.95)), "p99": float(series.quantile(0.99)),
    }


def create_validation_amendment(context):
    directory = amendment_directory()
    directory.mkdir(parents=False, exist_ok=False)
    retrospective = context["retrospective"]
    urine = retrospective.urine_output_day7_absolute_anchor_difference_ml_24h
    amendment = {
        "amendment_id": AMENDMENT_ID,
        "status": "active_for_production_validation_from_resume_authorisation",
        "created_at_utc": now_utc(),
        "post_hoc_origin_disclosed": True,
        "origin": {
            "introduced_after_observed_boundary_failure": True,
            "request_id": "QWEN-V32-PROD-REQ000572", "recipient_id": "V32P-R000572",
            "urine_anchor_ml_24h": 2173.8, "returned_value_ml_24h": 2050.0,
            "absolute_difference_ml": 123.8, "original_tolerance_ml": 120.0,
        },
        "change": {
            "validator_field": "day-7 urine_output_ml_24h absolute anchor difference",
            "original_tolerance_ml": 120.0, "amended_tolerance_ml": 150.0,
            "rationale": "validation allowance for reasonable numerical rounding at the observed boundary",
        },
        "unchanged_contract": {
            "qwen_prompt_instruction_ml": 120.0,
            "prompt_sha256": PROMPT_SHA256,
            "anchor_algorithm_version": ANCHOR_ALGORITHM_VERSION,
            "other_anchor_tolerances": {key: value for key, value in ORIGINAL_ANCHOR_TOLERANCES.items() if key != "urine_output_ml_24h"},
            "numerical_ranges_unchanged": True, "generation_rules_unchanged": True,
            "no_retry_or_regeneration_of_recipient_572": True,
        },
        "application_scope": "consistent and retrospective across all production records",
    }
    summary = {
        "amendment_id": AMENDMENT_ID, "status": "passed", "api_requests": 0,
        "preserved_responses_revalidated": len(retrospective),
        "urine_difference_distribution_ml_24h": difference_distribution(urine),
        "complete_difference_distributions": {
            "creatinine_mg_dl": difference_distribution(retrospective.creatinine_day7_absolute_anchor_difference),
            "urine_output_ml_24h": difference_distribution(urine),
            "tacrolimus_level_ng_ml": difference_distribution(retrospective.tacrolimus_day7_absolute_anchor_difference_ng_ml),
            "medication_adherence_pct": difference_distribution(retrospective.adherence_day7_absolute_anchor_difference_pct_points),
        },
        "exceeds_original_120_ml": int(urine.gt(120.0 + 1e-12).sum()),
        "exceeds_amended_150_ml": int(urine.gt(150.0 + 1e-12).sum()),
        "non_urine_anchor_failures": 0, "other_validation_failures": 0,
        "recipient_572": {
            "structural_json_valid": True, "approved_ranges_valid": True,
            "python_derived_fields_available": True, "request_preserved": True,
            "response_preserved": True, "accepted_under_amendment": True,
        },
        "ambiguous_or_other_failed_records": 0,
    }
    write_json_exclusive(directory / "validation_amendment.json", amendment)
    write_csv_exclusive(directory / "retrospective_anchor_revalidation.csv", retrospective)
    write_json_exclusive(directory / "retrospective_anchor_revalidation_summary.json", summary)
    completed_now = {number: tree_fingerprint(chunk_directory(number)) == fingerprint for number, fingerprint in context["completed_fingerprints"].items()}
    integrity = {
        "amendment_id": AMENDMENT_ID, "status": "passed", "api_requests": 0,
        "checked_at_utc": now_utc(), "pre_amendment_checks": context["checks"],
        "protected_before": context["protected_before"], "protected_after_amendment": protected_fingerprint(),
        "completed_chunks_001_005_unchanged": completed_now,
        "chunk_006_partial_unchanged_before_recovery": tree_fingerprint(chunk_directory(6)) == context["chunk_006_partial_before"],
        "original_failed_report_unchanged": file_fingerprint(chunk_directory(6) / "chunk_006_run_report.json") == context["chunk_006_original_report_before"],
        "previous_evidence_unchanged": snapshot_immutable(context["immutable_paths"]) == context["immutable_before"],
        "notebooks_01_05_unchanged": snapshot_notebooks_01_05() == context["notebooks_before"],
        "production_design_unchanged": tree_fingerprint(PRODUCTION_DESIGN) == context["production_design_before"],
        "created_file_fingerprints_excluding_self": {
            name: file_fingerprint(directory / name)
            for name in ["validation_amendment.json", "retrospective_anchor_revalidation.csv", "retrospective_anchor_revalidation_summary.json"]
        },
    }
    assert all(completed_now.values())
    assert integrity["protected_before"] == integrity["protected_after_amendment"]
    assert all(value for key, value in integrity.items() if key.endswith("unchanged"))
    write_json_exclusive(directory / "amendment_integrity_report.json", integrity)
    return amendment, summary, tree_fingerprint(directory)


def assert_resume_integrity(context, completed_fingerprints=None):
    result = {
        "protected_unchanged": protected_fingerprint() == context["protected_before"],
        "previous_evidence_unchanged": snapshot_immutable(context["immutable_paths"]) == context["immutable_before"],
        "notebooks_01_05_unchanged": snapshot_notebooks_01_05() == context["notebooks_before"],
        "production_design_unchanged": tree_fingerprint(PRODUCTION_DESIGN) == context["production_design_before"],
        "original_chunk_006_failed_report_unchanged": sha256_file(chunk_directory(6) / "chunk_006_run_report.json") == ORIGINAL_CHUNK_006_REPORT_SHA256,
        "validation_amendment_unchanged": tree_fingerprint(amendment_directory()) == context["amendment_fingerprint"],
        "chunks_011_100_absent": all(not chunk_directory(number).exists() for number in range(11, 101)),
        "final_assembly_absent": not (PRODUCTION_ROOT / "assessment_production.csv").exists(),
    }
    for number, fingerprint in (completed_fingerprints or {}).items():
        result[f"chunk_{number:03d}_unchanged"] = tree_fingerprint(chunk_directory(number)) == fingerprint
    assert all(result.values()), [name for name, passed in result.items() if not passed]
    return result


assert_global_integrity = assert_resume_integrity


def recover_recipient_572(context):
    chunk6 = chunk_directory(6)
    requests_before = sha256_file(chunk6 / "requests.jsonl")
    responses_before = sha256_file(chunk6 / "responses.jsonl")
    checkpoint = pd.read_csv(chunk6 / "validated_recipient_checkpoint.csv")
    assert checkpoint.shape == (426, 31) and "V32P-R000572" not in set(checkpoint.recipient_id)
    recovery = pd.DataFrame(context["recovery_rows"], columns=context["assessment_columns"])
    append_checkpoint(chunk6 / "validated_recipient_checkpoint.csv", recovery)
    append_jsonl(chunk6 / "manifest.jsonl", {
        "run_id": RUN_ID, "chunk_id": "CHUNK-006", "request_id": "QWEN-V32-PROD-REQ000572",
        "recipient_id": "V32P-R000572", "request_sequence_number": 572, "attempt_number": 1,
        "status": "validated_under_amendment", "amendment_id": AMENDMENT_ID,
        "historical_status_preserved": "validation_failed", "api_request_made_for_recovery": False,
        "checkpoint_recipient_count": 72, "checkpoint_row_count": 432, "timestamp_utc": now_utc(),
    })
    write_json_state(chunk6 / "completion_state.json", {
        "run_id": RUN_ID, "chunk_id": "CHUNK-006", "status": "manual_resume_running",
        "amendment_id": AMENDMENT_ID, "automatic_resume": False,
        "requests_attempted_total": 72, "requests_completed_total": 72,
        "new_requests_attempted": 0, "new_requests_completed": 0,
        "recipients_validated": 72, "checkpoint_rows": 432,
        "last_validated_request_id": "QWEN-V32-PROD-REQ000572",
        "last_validated_recipient_id": "V32P-R000572",
    })
    assert sha256_file(chunk6 / "requests.jsonl") == requests_before == context["request_572_hash_before"]
    assert sha256_file(chunk6 / "responses.jsonl") == responses_before == context["response_572_hash_before"]
    manifest = line_records(chunk6 / "manifest.jsonl")
    assert sum(record.get("recipient_id") == "V32P-R000572" and record["status"] == "validation_failed" for record in manifest) == 1
    assert sum(record.get("recipient_id") == "V32P-R000572" and record["status"] == "validated_under_amendment" for record in manifest) == 1
    return recovery


def resume_chunk_006(context, completed_fingerprints):
    started = time.monotonic()
    chunk6 = chunk_directory(6)
    report_path = chunk6 / "chunk_006_resume_completion_report.json"
    assert not report_path.exists()
    completion_parameters = dict(context["api_config"]["completion"])
    recipients = context["slices"][6]["recipients"]
    remaining = recipients.loc[recipients.request_sequence_number.between(573, 600)].copy()
    anchors = context["slices"][6]["anchors"]
    frozen_assessments = context["slices"][6]["assessments"]
    identities = context["slices"][6]["identities"]
    anchor_lookup = anchors.set_index("recipient_id")
    usage_records = []
    finish_records = []
    soft_warnings = []
    new_rows = []
    failure = None
    new_attempted = 0
    new_completed = 0
    new_validated = 0
    validation = None
    reconciliation = None
    diagnostics = None
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL, max_retries=0, timeout=180.0)
    try:
        for recipient in remaining.to_dict(orient="records"):
            if new_attempted >= 28:
                raise RuntimeError("28-request chunk-006 resume hard cap reached")
            request_id = recipient["request_id"]
            recipient_id = recipient["recipient_id"]
            anchor = anchor_lookup.loc[recipient_id].to_dict()
            rendered_prompt = render_prompt(context["template"], recipient, anchor)
            messages = [
                {"role": "system", "content": "Return only the exact JSON object requested. No prose or Markdown."},
                {"role": "user", "content": rendered_prompt},
            ]
            append_jsonl(chunk6 / "requests.jsonl", {
                "run_id": RUN_ID, "chunk_id": "CHUNK-006", "request_id": request_id,
                "recipient_id": recipient_id, "request_sequence_number": int(recipient["request_sequence_number"]),
                "attempt_number": 1, "deterministic_request_seed": int(recipient["request_seed"]),
                "prompt_version": PROMPT_VERSION, "prompt_template_sha256": PROMPT_SHA256,
                "rendered_prompt": rendered_prompt, "rendered_prompt_sha256": sha256_text(rendered_prompt),
                "rendered_message_payload": messages, "model": completion_parameters["model"],
                "completion_parameters": {**{key: value for key, value in completion_parameters.items() if key != "model"}, "seed": int(recipient["request_seed"])},
                "anchor_algorithm_version": ANCHOR_ALGORITHM_VERSION, "anchor_metadata_sha256": anchor_row_hash(anchor),
                "validation_amendment_id": AMENDMENT_ID, "timestamp_utc": now_utc(),
                "status": "request_preserved_before_submission",
            })
            append_jsonl(chunk6 / "manifest.jsonl", {
                "run_id": RUN_ID, "chunk_id": "CHUNK-006", "request_id": request_id,
                "recipient_id": recipient_id, "request_sequence_number": int(recipient["request_sequence_number"]),
                "attempt_number": 1, "status": "request_preserved_before_submission", "timestamp_utc": now_utc(),
            })
            new_attempted += 1
            append_jsonl(chunk6 / "manifest.jsonl", {
                "request_id": request_id, "recipient_id": recipient_id, "attempt_number": 1,
                "status": "network_submission_started", "timestamp_utc": now_utc(),
            })
            try:
                response = client.chat.completions.create(
                    model=completion_parameters["model"], messages=messages,
                    max_tokens=completion_parameters["max_tokens"], temperature=completion_parameters["temperature"],
                    top_p=completion_parameters["top_p"], presence_penalty=completion_parameters["presence_penalty"],
                    seed=int(recipient["request_seed"]), extra_body=completion_parameters["extra_body"],
                )
            except Exception as exc:
                failure = {"stage": "api_request", "request_id": request_id, "recipient_id": recipient_id, "request_sequence_number": int(recipient["request_sequence_number"]), "attempt_number": 1, "exception_type": type(exc).__name__, "message": str(exc)}
                append_jsonl(chunk6 / "manifest.jsonl", {**failure, "status": "request_failed", "timestamp_utc": now_utc()})
                raise RuntimeError(f"request failed for {request_id}") from exc
            new_completed += 1
            serialized = response.model_dump_json(indent=2)
            append_jsonl(chunk6 / "responses.jsonl", {
                "run_id": RUN_ID, "chunk_id": "CHUNK-006", "request_id": request_id,
                "recipient_id": recipient_id, "request_sequence_number": int(recipient["request_sequence_number"]),
                "attempt_number": 1, "received_at_utc": now_utc(), "serialized_response_json": serialized,
                "serialized_response_sha256": sha256_text(serialized), "status": "complete_response_preserved_before_extraction",
            })
            choice = response.choices[0] if response.choices else None
            message = choice.message if choice else None
            content = message.content if message else None
            reasoning = getattr(message, "reasoning_content", None) if message else None
            usage = response.usage.model_dump() if response.usage else {}
            metadata = {
                "request_id": request_id, "recipient_id": recipient_id,
                "request_sequence_number": int(recipient["request_sequence_number"]),
                "finish_reason": choice.finish_reason if choice else None,
                "reasoning_content_status": "present_nonempty" if reasoning else "present_empty" if reasoning == "" else "absent",
                "final_content_length": len(content) if isinstance(content, str) else 0, "token_usage": usage,
            }
            usage_records.append(usage); finish_records.append(metadata)
            append_jsonl(chunk6 / "manifest.jsonl", {**metadata, "status": "response_preserved", "timestamp_utc": now_utc()})
            payload = None
            if metadata["finish_reason"] == "length": failure = {"stage": "finish_reason_length", **metadata}
            elif not content: failure = {"stage": "absent_final_content", **metadata}
            else:
                try: payload = json.loads(content)
                except Exception as exc: failure = {"stage": "json_parse", **metadata, "exception_type": type(exc).__name__, "message": str(exc)}
            if failure is not None:
                append_jsonl(chunk6 / "manifest.jsonl", {**failure, "status": "validation_failed", "timestamp_utc": now_utc()})
                raise RuntimeError(f"response failed before validation for {request_id}")
            errors, warnings = validate_payload(payload, anchor, context["canonical_ranges"])
            if errors:
                failure = {"stage": "payload_validation", **metadata, "errors": errors, "warnings": warnings, "amendment_id": AMENDMENT_ID}
                append_jsonl(chunk6 / "manifest.jsonl", {**failure, "status": "validation_failed", "timestamp_utc": now_utc()})
                raise RuntimeError(f"payload validation failed for {request_id}")
            context["response_payloads"][recipient_id] = payload
            frozen = frozen_assessments.loc[frozen_assessments.recipient_id.eq(recipient_id)].copy()
            recipient_rows = build_assessment_rows(recipient, anchor, frozen, payload, context["assessment_columns"])
            append_checkpoint(chunk6 / "validated_recipient_checkpoint.csv", pd.DataFrame(recipient_rows, columns=context["assessment_columns"]))
            new_rows.extend(recipient_rows); new_validated += 1
            soft_warnings.extend({"recipient_id": recipient_id, "warning": warning} for warning in warnings)
            append_jsonl(chunk6 / "manifest.jsonl", {
                "request_id": request_id, "recipient_id": recipient_id, "request_sequence_number": int(recipient["request_sequence_number"]),
                "attempt_number": 1, "status": "validated_and_checkpointed", "validation_amendment_id": AMENDMENT_ID,
                "checkpoint_recipient_count": 72 + new_validated, "checkpoint_row_count": 432 + 6 * new_validated,
                "soft_warnings": warnings, "timestamp_utc": now_utc(),
            })
            write_json_state(chunk6 / "completion_state.json", {
                "run_id": RUN_ID, "chunk_id": "CHUNK-006", "status": "manual_resume_running", "amendment_id": AMENDMENT_ID,
                "automatic_resume": False, "requests_attempted_total": 72 + new_attempted,
                "requests_completed_total": 72 + new_completed, "new_requests_attempted": new_attempted,
                "new_requests_completed": new_completed, "recipients_validated": 72 + new_validated,
                "checkpoint_rows": 432 + 6 * new_validated, "last_validated_request_id": request_id,
                "last_validated_recipient_id": recipient_id,
            })

        assessment = pd.read_csv(chunk6 / "validated_recipient_checkpoint.csv")
        assert assessment.shape == (600, 31) and assessment.recipient_id.nunique() == 100
        requests = line_records(chunk6 / "requests.jsonl")
        responses = line_records(chunk6 / "responses.jsonl")
        manifest = line_records(chunk6 / "manifest.jsonl")
        assert len(requests) == len(responses) == 100
        assert len({record["request_id"] for record in requests}) == 100
        assert sum(record["recipient_id"] == "V32P-R000572" for record in requests) == 1
        assert sum(record["recipient_id"] == "V32P-R000572" for record in responses) == 1
        assert len(manifest) == 401
        assert sum(record["status"] == "validation_failed" for record in manifest) == 1
        assert sum(record["status"] == "validated_under_amendment" for record in manifest) == 1
        payloads = {}
        for record in responses:
            _, payload = parse_preserved_response(record)
            payloads[record["recipient_id"]] = payload
        reconciliation = reconcile_payloads(payloads, assessment)
        validation_context = {**context, "recipients": recipients, "assessments": frozen_assessments, "anchors": anchors}
        validation = validate_complete_chunk(assessment, identities, recipients, anchors, payloads, reconciliation, validation_context)
        diagnostics = build_diagnostics(assessment, recipients, anchors)
        validation["checks"]["no_identical_complete_trajectories"] = diagnostics["identical_complete_trajectories"]["identical_trajectory_groups"] == 0
        if not validation["checks"]["no_identical_complete_trajectories"]:
            validation["failures"].append("no_identical_complete_trajectories"); validation["status"] = "failed"
        validation["checks"]["checkpoint_assessment_equality"] = True
        if validation["status"] != "passed":
            failure = {"stage": "complete_chunk_validation", "validation": validation}
            raise RuntimeError("chunk 006 complete validation failed")
        day7 = assessment.loc[assessment.days_since_transplant.eq(7)].copy()
        anchor_audit = anchors.merge(day7[["recipient_id", "creatinine_mg_dl", "urine_output_ml_24h", "tacrolimus_level_ng_ml", "medication_adherence_pct"]], on="recipient_id", validate="one_to_one")
        for output, anchor_field in {"creatinine_mg_dl": "day7_creatinine_anchor_mg_dl", "urine_output_ml_24h": "day7_urine_output_anchor_ml_24h", "tacrolimus_level_ng_ml": "day7_tacrolimus_anchor_ng_ml", "medication_adherence_pct": "day7_medication_adherence_anchor_pct"}.items():
            difference = (anchor_audit[output] - anchor_audit[anchor_field]).abs()
            anchor_audit[f"{output}_absolute_anchor_difference"] = difference
            anchor_audit[f"{output}_within_tolerance"] = difference.le(ANCHOR_TOLERANCES[output] + 1e-12)
        write_csv_exclusive(chunk6 / "assessment_chunk_006.csv", assessment)
        write_csv_exclusive(chunk6 / "anchor_audit_chunk_006.csv", anchor_audit)
        write_json_exclusive(chunk6 / "request_response_reconciliation.json", reconciliation)
        write_json_exclusive(chunk6 / "diagnostic_quality_report.json", diagnostics)
        write_json_state(chunk6 / "completion_state.json", {
            "run_id": RUN_ID, "chunk_id": "CHUNK-006", "status": "completed_under_amendment",
            "amendment_id": AMENDMENT_ID, "automatic_resume": False,
            "requests_attempted": 100, "requests_completed": 100, "recipients_validated": 100,
            "checkpoint_rows": 600, "assessment_chunk_created": True,
            "historical_validation_failures_preserved": 1, "amendment_validation_transitions": 1,
            "chunks_011_to_100_started": False, "final_dataset_assembly_started": False,
        })
    except Exception as exc:
        if failure is None:
            failure = {"stage": "resume_execution_or_validation", "exception_type": type(exc).__name__, "message": str(exc), "traceback": traceback.format_exc()}
        write_json_state(chunk6 / "completion_state.json", {
            "run_id": RUN_ID, "chunk_id": "CHUNK-006", "status": "resume_failed", "amendment_id": AMENDMENT_ID,
            "automatic_resume": False, "new_requests_attempted": new_attempted, "new_requests_completed": new_completed,
            "recipients_validated": 72 + new_validated, "checkpoint_rows": 432 + 6 * new_validated,
            "failure": failure, "retry_performed": False, "assessment_chunk_created": (chunk6 / "assessment_chunk_006.csv").exists(),
        })
    finally:
        original = json.loads((chunk6 / "chunk_006_run_report.json").read_text(encoding="utf-8"))
        new_tokens = {key: sum(int(record.get(key, 0) or 0) for record in usage_records) for key in ["prompt_tokens", "completion_tokens", "total_tokens"]}
        total_tokens = {key: int(original["token_usage_total"][key]) + new_tokens[key] for key in new_tokens}
        resume_report = {
            "run_id": RUN_ID, "chunk_id": "CHUNK-006", "resume_id": "QWEN-V32-PROD-001-RESUME-006-A001",
            "amendment_id": AMENDMENT_ID, "status": "completed" if failure is None else "failed",
            "original_failed_run_report": "chunk_006_run_report.json",
            "original_failed_run_report_sha256": ORIGINAL_CHUNK_006_REPORT_SHA256,
            "historical_failure_preserved": True, "historical_validation_failure_count": 1,
            "amendment_validation_transition_count": 1, "recipient_572_api_requests_during_recovery": 0,
            "new_api_requests_attempted": new_attempted, "new_api_requests_completed": new_completed,
            "new_recipients_validated": new_validated, "new_attempts": new_attempted, "new_failures": 0 if failure is None else 1,
            "new_retries": 0, "failure": failure, "finish_reason_counts": dict(Counter(item["finish_reason"] for item in finish_records)),
            "reasoning_content_status_counts": dict(Counter(item["reasoning_content_status"] for item in finish_records)),
            "new_token_usage": new_tokens, "token_usage_total": total_tokens,
            "new_elapsed_seconds": time.monotonic() - started,
            "elapsed_seconds": float(original["elapsed_seconds"]) + (time.monotonic() - started),
            "request_records": count_jsonl(chunk6 / "requests.jsonl"), "response_records": count_jsonl(chunk6 / "responses.jsonl"),
            "manifest_records": count_jsonl(chunk6 / "manifest.jsonl"),
            "manifest_status_counts": dict(Counter(record["status"] for record in line_records(chunk6 / "manifest.jsonl"))),
            "recipients_validated": int(pd.read_csv(chunk6 / "validated_recipient_checkpoint.csv").recipient_id.nunique()),
            "checkpoint_rows": len(pd.read_csv(chunk6 / "validated_recipient_checkpoint.csv")),
            "chunk_dimensions": [600, 31] if failure is None else [len(pd.read_csv(chunk6 / "validated_recipient_checkpoint.csv")), 31],
            "validation": validation, "reconciliation": reconciliation, "diagnostics": diagnostics,
            "soft_warnings": soft_warnings, "integrity_in_finally": assert_resume_integrity(context, completed_fingerprints),
            "chunks_011_to_100_started": False, "final_dataset_assembly_started": False,
            "created_files": sorted(path.name for path in chunk6.iterdir() if path.is_file()) + ["chunk_006_resume_completion_report.json"],
            "artifact_fingerprints_excluding_resume_report": fingerprint_named_files(chunk6),
        }
        write_json_exclusive(report_path, resume_report)
    return resume_report


def aggregate_resume_through_chunk_010(context, completed_fingerprints, resume_reports):
    started = time.monotonic()
    directory = progress_directory()
    assert not directory.exists()
    assessments=[]; identities=[]; anchors=[]; requests=[]; responses=[]; manifests=[]; reports={}; per_chunk={}
    for number in range(1, 11):
        suffix=f"{number:03d}"; chunk=chunk_directory(number)
        assessments.append(pd.read_csv(chunk/f"assessment_chunk_{suffix}.csv")); identities.append(pd.read_csv(chunk/f"identity_chunk_{suffix}.csv",keep_default_na=False)); anchors.append(pd.read_csv(chunk/f"anchor_audit_chunk_{suffix}.csv"))
        requests.extend(line_records(chunk/"requests.jsonl")); responses.extend(line_records(chunk/"responses.jsonl")); manifests.extend(line_records(chunk/"manifest.jsonl"))
        report_name="chunk_006_resume_completion_report.json" if number==6 else f"chunk_{suffix}_run_report.json"
        report=json.loads((chunk/report_name).read_text(encoding="utf-8")); reports[number]=report
        diagnostic=json.loads((chunk/"diagnostic_quality_report.json").read_text(encoding="utf-8"))
        per_chunk[suffix]={"status":report["status"],"soft_validation_warnings":report.get("soft_warnings",[]),"diagnostic_warnings":diagnostic.get("warnings",[]),"historical_failures_preserved":report.get("historical_validation_failure_count",0),"new_failures":report.get("new_failures",report.get("failures",0))}
    assessment=pd.concat(assessments,ignore_index=True); identity=pd.concat(identities,ignore_index=True); anchor=pd.concat(anchors,ignore_index=True)
    recipients=pd.concat([context["slices"][n]["recipients"] for n in range(1,11)],ignore_index=True); frozen=pd.concat([context["slices"][n]["assessments"] for n in range(1,11)],ignore_index=True)
    request_ids=[r["request_id"] for r in requests]; response_ids=[r["request_id"] for r in responses]
    checks={
        "ten_completed_chunks": all(reports[n]["status"]=="completed" for n in range(1,11)),
        "recipients_1000": assessment.recipient_id.nunique()==1000,
        "rows_6000": assessment.shape==(6000,31), "assessment_ids_unique":assessment.assessment_id.is_unique,
        "recipient_day_unique":not assessment.duplicated(["recipient_id","days_since_transplant"]).any(),
        "request_records_1000":len(requests)==1000,"response_records_1000":len(responses)==1000,
        "request_ids_unique":len(set(request_ids))==1000,"response_ids_unique":len(set(response_ids))==1000,
        "request_response_ids_equal":set(request_ids)==set(response_ids),"manifest_records_4001":len(manifests)==4001,
        "recipient_572_one_request":sum(r["recipient_id"]=="V32P-R000572" for r in requests)==1,
        "recipient_572_one_response":sum(r["recipient_id"]=="V32P-R000572" for r in responses)==1,
        "historical_failure_preserved":sum(r.get("recipient_id")=="V32P-R000572" and r["status"]=="validation_failed" for r in manifests)==1,
        "amendment_transition_present":sum(r.get("recipient_id")=="V32P-R000572" and r["status"]=="validated_under_amendment" for r in manifests)==1,
        "no_duplicate_rows":not assessment.duplicated().any(),"six_rows_each":assessment.groupby("recipient_id").size().eq(6).all(),
        "ordered_days":assessment.groupby("recipient_id").days_since_transplant.apply(list).map(lambda values:values==DAYS).all(),
    }
    payloads={}; parse_errors=[]
    for record in responses:
        try: _,payload=parse_preserved_response(record); payloads[record["recipient_id"]]=payload
        except Exception as exc: parse_errors.append({"request_id":record.get("request_id"),"error":str(exc)})
    mismatches=[]; reconciled=0
    for rid,payload in payloads.items():
        rows=assessment.loc[assessment.recipient_id.eq(rid)].sort_values("days_since_transplant")
        for i,item in enumerate(payload["assessments"]):
            for field in QWEN_FIELDS:
                reconciled+=1
                if not np.isclose(float(item[field]),float(rows.iloc[i][field]),rtol=0,atol=1e-12): mismatches.append({"recipient_id":rid,"position":i,"field":field})
    checks["complete_raw_response_reconciliation"]=not parse_errors and reconciled==30000 and not mismatches
    diagnostics=build_diagnostics(assessment,recipients,anchor)
    checks["no_identical_complete_trajectories"]=diagnostics["identical_complete_trajectories"]["identical_trajectory_groups"]==0
    frozen_index=frozen.set_index("assessment_id")
    assessment_index=assessment.set_index("assessment_id")
    frozen_shared_columns=[column for column in frozen.columns if column != "assessment_id" and column in assessment.columns]
    frozen_metadata_exact=True
    for column in frozen_shared_columns:
        expected=frozen_index.loc[assessment_index.index,column]
        actual=assessment_index[column]
        if pd.api.types.is_numeric_dtype(expected) and pd.api.types.is_numeric_dtype(actual):
            matches=np.allclose(actual.astype(float).to_numpy(),expected.astype(float).to_numpy(),rtol=0,atol=1e-12,equal_nan=True)
        else:
            matches=actual.fillna("<NA>").astype(str).equals(expected.fillna("<NA>").astype(str))
        frozen_metadata_exact=frozen_metadata_exact and bool(matches)
    checks["frozen_schedule_audit_and_derived_fields_exact"]=frozen_metadata_exact
    event_target_fields=["infection_indicator","previous_rejection","acute_rejection_within_30_days"]
    checks["event_and_target_derivation_exact"]=all(
        np.array_equal(
            assessment_index[column].astype(int).to_numpy(),
            frozen_index.loc[assessment_index.index,column].astype(int).to_numpy(),
        )
        for column in event_target_fields
    )
    checks["targets_match_frozen"]=checks["event_and_target_derivation_exact"]
    recipient_donor=recipients.set_index("recipient_id").donor_id
    checks["donor_relationships_consistent"]=all(g.donor_id.nunique()==1 and g.donor_id.iloc[0]==recipient_donor.loc[rid] for rid,g in assessment.groupby("recipient_id"))
    checks["shared_identity_records_consistent"]=all(len(g.drop_duplicates())==1 for _,g in identity.groupby("entity_id"))
    day7=assessment.loc[assessment.days_since_transplant.eq(7)].set_index("recipient_id"); anchor_idx=context["anchors_all"].set_index("recipient_id")
    anchor_failures=[]
    for rid,row in day7.iterrows():
        for field,af in {"creatinine_mg_dl":"day7_creatinine_anchor_mg_dl","urine_output_ml_24h":"day7_urine_output_anchor_ml_24h","tacrolimus_level_ng_ml":"day7_tacrolimus_anchor_ng_ml","medication_adherence_pct":"day7_medication_adherence_anchor_pct"}.items():
            diff=abs(float(row[field])-float(anchor_idx.loc[rid,af]))
            if diff>ANCHOR_TOLERANCES[field]+1e-12: anchor_failures.append({"recipient_id":rid,"field":field,"difference":diff})
    checks["all_anchor_checks_pass_amendment"]=not anchor_failures
    failures=[name for name,value in checks.items() if not bool(value)]
    reconciliation={"status":"passed" if not failures else "failed","checks":{k:bool(v) for k,v in checks.items()},"failures":failures,"qwen_fields_reconciled":reconciled,"expected_qwen_fields":30000,"mismatch_count":len(mismatches),"parse_errors":parse_errors[:20],"anchor_failure_count":len(anchor_failures),"amendment_id":AMENDMENT_ID,"expected_manifest_records":4001,"extra_manifest_record_explanation":"recipient 572 retains its original validation_failed stage and adds one validated_under_amendment stage"}
    tokens={key:sum(int(reports[n]["token_usage_total"][key]) for n in range(1,11)) for key in ["prompt_tokens","completion_tokens","total_tokens"]}
    elapsed={f"chunk_{n:03d}":float(reports[n]["elapsed_seconds"]) for n in range(1,11)}
    distribution={**diagnostics,"status":"diagnostic_only_not_used_for_regeneration","amendment_id":AMENDMENT_ID,"per_chunk_warnings_and_failures":per_chunk,"aggregate_token_usage":tokens,"per_chunk_elapsed_seconds":elapsed,"total_chunk_elapsed_seconds":sum(elapsed.values())}
    integrity_checks=assert_resume_integrity(context,completed_fingerprints)
    integrity={"status":"passed","checked_at_utc":now_utc(),"amendment_id":AMENDMENT_ID,"global_integrity_checks":integrity_checks,"chunk_fingerprints":{f"chunk_{n:03d}":tree_fingerprint(chunk_directory(n)) for n in range(1,11)},"validation_amendment_fingerprint":tree_fingerprint(amendment_directory()),"original_chunk_006_failed_report_sha256":sha256_file(chunk_directory(6)/"chunk_006_run_report.json"),"chunks_011_100_started":False,"final_dataset_assembly_started":False}
    directory.mkdir(parents=False,exist_ok=False)
    write_json_exclusive(directory/"cross_chunk_reconciliation.json",reconciliation); write_json_exclusive(directory/"cross_chunk_distribution_report.json",distribution); write_json_exclusive(directory/"cross_chunk_integrity_report.json",integrity)
    integrity_after=assert_resume_integrity(context,completed_fingerprints)
    progress={"run_id":RUN_ID,"checkpoint":"through_chunk_010","status":"completed" if not failures else "failed","amendment_id":AMENDMENT_ID,"created_at_utc":now_utc(),"chunks_present":list(range(1,11)),"chunks_completed_by_resume":[6,7,8,9,10],"new_api_requests_attempted":sum(r.get("new_api_requests_attempted",r.get("api_requests_attempted",0)) for r in resume_reports.values()),"new_api_requests_completed":sum(r.get("new_api_requests_completed",r.get("api_requests_completed",0)) for r in resume_reports.values()),"new_retries":0,"aggregate_request_records":len(requests),"aggregate_response_records":len(responses),"aggregate_manifest_records":len(manifests),"expected_extra_manifest_records":1,"aggregate_dimensions":list(assessment.shape),"aggregate_token_usage":tokens,"aggregate_chunk_elapsed_seconds":sum(elapsed.values()),"aggregate_checkpoint_elapsed_seconds":time.monotonic()-started,"target_count":diagnostics["target_count"],"target_prevalence":diagnostics["target_prevalence"],"event_recipients":diagnostics["event_recipients"],"non_event_recipients":diagnostics["non_event_recipients"],"reconciliation_status":reconciliation["status"],"reconciliation_failures":failures,"per_chunk_warnings_and_failures":per_chunk,"integrity_after_aggregate_checkpoint":integrity_after,"created_files":["production_progress_report.json","cross_chunk_reconciliation.json","cross_chunk_distribution_report.json","cross_chunk_integrity_report.json"],"progress_file_fingerprints_excluding_self":{p.name:file_fingerprint(p) for p in sorted(directory.iterdir()) if p.is_file()},"chunks_011_to_100_started":False,"final_dataset_assembly_started":False,"classifier_trained":False,"generator_tuned":False}
    write_json_exclusive(directory/"production_progress_report.json",progress)
    assert not failures,failures
    return progress


def execute_amendment_and_resume():
    context=resume_preflight()
    amendment,amendment_summary,amendment_fingerprint=create_validation_amendment(context)
    context["amendment_fingerprint"]=amendment_fingerprint
    recover_recipient_572(context)
    completed_fingerprints=dict(context["completed_fingerprints"])
    resume_reports={}
    try:
        report6=resume_chunk_006(context,completed_fingerprints); resume_reports[6]=report6
        if report6["status"]!="completed": raise RuntimeError("chunk 006 resume failed; stopped without retry")
        completed_fingerprints[6]=tree_fingerprint(chunk_directory(6))
        assert_resume_integrity(context,completed_fingerprints)
        for number in range(7,11):
            report=run_single_chunk(number,context,completed_fingerprints); resume_reports[number]=report
            if report["status"]!="completed": raise RuntimeError(f"chunk {number:03d} failed; stopped without retry")
            completed_fingerprints[number]=tree_fingerprint(chunk_directory(number)); assert_resume_integrity(context,completed_fingerprints)
        assert set(completed_fingerprints)==set(range(1,11))
        progress=aggregate_resume_through_chunk_010(context,completed_fingerprints,resume_reports)
        return {"status":"completed","amendment_summary":amendment_summary,"resume_reports":{str(k):{"status":v["status"],"new_api_requests":v.get("new_api_requests_completed",v.get("api_requests_completed")),"token_usage":v.get("new_token_usage",v.get("token_usage_total"))} for k,v in resume_reports.items()},"progress":progress}
    finally:
        assert_resume_integrity(context,completed_fingerprints)


AMENDMENT_RESUME_RESULT = execute_amendment_and_resume()
print(json.dumps({"status":AMENDMENT_RESUME_RESULT["status"],"amendment_id":AMENDMENT_ID,"resume_reports":AMENDMENT_RESUME_RESULT["resume_reports"],"aggregate_dimensions":AMENDMENT_RESUME_RESULT["progress"]["aggregate_dimensions"],"aggregate_manifest_records":AMENDMENT_RESUME_RESULT["progress"]["aggregate_manifest_records"],"reconciliation_status":AMENDMENT_RESUME_RESULT["progress"]["reconciliation_status"],"chunks_011_to_100_started":False,"final_dataset_assembly_started":False},indent=2))


## Production validation-policy amendment 002 and guarded resume through chunk 010

This additive cell transparently changes individual day-7 anchor proximity from a hard record-validity rule to an `anchor_deviation_warning`, while preserving the frozen prompt and its original tolerances. All structural, clinical-range, derivation, leakage, reconciliation, checkpoint, and evidence-integrity checks remain hard. It retrospectively checks all 627 preserved responses, recovers recipient 627 without another request, evaluates explicit generation-quality gates after each completed 100-recipient chunk, and resumes only with `RESUME_QWEN_V32_PRODUCTION_CHUNKS_007_010=QWEN-V32-PROD-001-RESUME-007-010-A002`. At most 373 new sequential single-attempt requests are permitted; chunks 011–100, final assembly, and classifier training remain disabled.

In [ ]:
import hashlib
import json
import os
import time
import traceback
from collections import Counter
from datetime import date, timedelta
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd
from openai import OpenAI


_repository_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists())
_notebook_path = _repository_root / "notebooks/KidneyTransplant/00_generate_and_validate_dataset.ipynb"
_notebook = nbformat.read(_notebook_path, as_version=4)
_amendment_001_source = next(
    cell.source
    for cell in _notebook.cells
    if cell.cell_type == "code" and "AMENDMENT_ID = \"PROD-VALIDATION-AMENDMENT-001\"" in cell.source
)
_amendment_001_primitives = _amendment_001_source.rsplit(
    "AMENDMENT_RESUME_RESULT = execute_amendment_and_resume()", 1
)[0]
exec(compile(_amendment_001_primitives, "amendment-001-primitives", "exec"), globals())


AMENDMENT_002_ID = "PROD-VALIDATION-AMENDMENT-002"
AMENDMENT_002_DIR_NAME = "validation_amendment_002"
RESUME_002_GATE_NAME = "RESUME_QWEN_V32_PRODUCTION_CHUNKS_007_010"
RESUME_002_GATE_VALUE = "QWEN-V32-PROD-001-RESUME-007-010-A002"
RESUME_002_CHUNKS = [7, 8, 9, 10]
MAX_NEW_API_REQUESTS_002 = 373
ORIGINAL_PROMPT_TOLERANCES = {
    "creatinine_mg_dl": 0.12,
    "urine_output_ml_24h": 120.0,
    "tacrolimus_level_ng_ml": 0.6,
    "medication_adherence_pct": 1.5,
}
ANCHOR_FIELD_MAP = {
    "creatinine_mg_dl": "day7_creatinine_anchor_mg_dl",
    "urine_output_ml_24h": "day7_urine_output_anchor_ml_24h",
    "tacrolimus_level_ng_ml": "day7_tacrolimus_anchor_ng_ml",
    "medication_adherence_pct": "day7_medication_adherence_anchor_pct",
}
IDENTIFIER_MATERIALITY_ABS_CORRELATION = 0.20
MULTI_GATE_NAME = RESUME_002_GATE_NAME
MULTI_GATE_VALUE = RESUME_002_GATE_VALUE
ANCHOR_TOLERANCES = dict(ORIGINAL_PROMPT_TOLERANCES)

CHUNK_006_SHA256 = {
    "anchor_audit_chunk_006.csv": "02a8891bb30acd75f9593b9447e845a0b3dc99f87576c5e85a83661461502ae1",
    "assessment_chunk_006.csv": "08611976d08932c56eee1ac06b9e8cc9eb61874163eb22d316d0ecdea61c4397",
    "chunk_006_resume_completion_report.json": "98bf36522ca480d9f256218b51f64058aae6bad082f794c491bd77353af55b41",
    "chunk_006_run_report.json": "285ffed8cca3b3cec14b7ddc57a879a07a3949b5f1f846b7e5c95be509d3849b",
    "completion_state.json": "5a62ca3ef9803035e049f10d469d6fb21e5257d89136ca8a6d0b97caffc3c556",
    "diagnostic_quality_report.json": "03f2d32defb44447c99d552a1827e7e11f29f804391726a86414bb3b4ae25cab",
    "identity_chunk_006.csv": "45a0bd6c43cfe0ebf69adef1a4c75e7d9dfeb3a5b9afc6e0a8a6c33bcb4275af",
    "manifest.jsonl": "705dbd9878dc063e4b7704583b75df7ce37d7e3ee8aee9290156a8f61f27ca9c",
    "request_response_reconciliation.json": "6689cfee50e6d3028b78a6e81d22e68bf05b068d4d9503f8ab4e851c4e2bc2b2",
    "requests.jsonl": "23182282e124dc7eea97e44bb2ed8d04b8c9d90fcd917ccec4b819095489bf2b",
    "responses.jsonl": "2777133ecb001d68f1756883fed6ef53bd618e165436038235ae7f2c7a368417",
    "run_configuration.json": "e35bd773c34280c12be1b2d0c5684988a1347854758d8b2b98d0b9e53f44712a",
    "validated_recipient_checkpoint.csv": "2ece5fab3cf1d194412fea1afe448a93be3dd299e31cf4d22c82de83a46e2a98",
}
CHUNK_007_ORIGINAL_SHA256 = {
    "chunk_007_run_report.json": "a0b70c4b49ea3932cb7f1059cc7d0411f1e825ac656c900ec4ed3854a4f75146",
    "completion_state.json": "6520c7f2319b482d566f389f248058f42913cd64d147c26d41067524f696e182",
    "identity_chunk_007.csv": "96fc90dae5b372118ad1d8f91ad890dd2f032233480ca409159c3b8e4c999ee0",
    "manifest.jsonl": "e354fc68dd22c7c84d5b42ec08f04be6950519f6e6903d55f0f4177e4e3ce89d",
    "requests.jsonl": "dbdefeed772c683eec08842f46faaedb21bd8beaf21f6408ad023e9914df493f",
    "responses.jsonl": "fd1efa09baccd0d0d8467d05b78831ae4df3c4fa5dd2ba3643a9b9e660ca1f7b",
    "run_configuration.json": "ce78e5e3e6cc19237fad8cab968222890e2fea517ebbf53df167acf067bc5de1",
    "validated_recipient_checkpoint.csv": "30f8b4fff3dbb90891c5240ab09fc635e774dfce327d582dc270d7f7578af88c",
}
AMENDMENT_001_SHA256 = {
    "amendment_integrity_report.json": "f80635e542b10a5563599284dae5d480a1ff6a7aacb777d3d90a7aac67b52222",
    "retrospective_anchor_revalidation.csv": "f9de3152c5a5959e9f9ea18e7289796ee5ca7a56ab7f33a32f86aaf396e2a6cf",
    "retrospective_anchor_revalidation_summary.json": "5f383eeef19fd1ecea78e07924b03c980c65843189fcca52b3d6f8a246e87c76",
    "validation_amendment.json": "52f790e506897b1b43b3e4c3ee7f7f933fd79dc5c9a5d6b19b6184a86058c637",
}


def amendment_002_directory():
    return PRODUCTION_ROOT / AMENDMENT_002_DIR_NAME


def exact_file_hashes(directory):
    return {path.name: sha256_file(path) for path in sorted(directory.iterdir()) if path.is_file()}


def validate_exact_hashes(directory, expected):
    assert directory.is_dir()
    actual = exact_file_hashes(directory)
    assert set(actual) == set(expected), f"unexpected file set in {display_path(directory)}"
    assert actual == expected, f"fingerprint mismatch in {display_path(directory)}"
    return tree_fingerprint(directory)


def file_prefix_sha256(path, byte_count):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        digest.update(handle.read(byte_count))
    return digest.hexdigest()


def anchor_deviation_records(recipient_id, request_id, payload, anchor):
    records = []
    day7 = payload["assessments"][0]
    for field, anchor_field in ANCHOR_FIELD_MAP.items():
        output = float(day7[field])
        anchor_value = float(anchor[anchor_field])
        difference = abs(output - anchor_value)
        tolerance = float(ORIGINAL_PROMPT_TOLERANCES[field])
        records.append({
            "request_id": request_id,
            "recipient_id": recipient_id,
            "measurement": field,
            "day7_anchor_value": anchor_value,
            "day7_output_value": output,
            "absolute_difference": difference,
            "original_prompt_tolerance": tolerance,
            "exceeds_original_prompt_tolerance": difference > tolerance + 1e-12,
            "validation_policy": "diagnostic_warning_not_record_rejection",
            "warning_code": "anchor_deviation_warning" if difference > tolerance + 1e-12 else "",
        })
    return records


def validate_payload_policy_002(payload, anchor, canonical_ranges):
    errors = []
    warnings = []
    if not isinstance(payload, dict):
        return ["root must be one JSON object"], warnings
    if set(payload) != {"assessments"}:
        errors.append("root must contain only assessments")
    items = payload.get("assessments")
    if not isinstance(items, list):
        return errors + ["assessments must be an array"], warnings
    if len(items) != 6:
        errors.append("assessments must contain exactly six objects")
        return errors, warnings
    for position, item in enumerate(items):
        if not isinstance(item, dict):
            errors.append(f"assessment {position} must be an object")
            continue
        if set(item) != set(QWEN_FIELDS):
            errors.append(f"assessment {position} key set is invalid")
            continue
        if not all(is_number(item[field]) for field in QWEN_FIELDS):
            errors.append(f"assessment {position} contains non-finite or non-numeric values")
            continue
        if int(item["days_since_transplant"]) != DAYS[position] or float(item["days_since_transplant"]) != DAYS[position]:
            errors.append(f"assessment {position} day is invalid")
        for field, bounds in canonical_ranges.items():
            value = float(item[field])
            if not bounds["minimum"] <= value <= bounds["maximum"]:
                errors.append(f"assessment {position} {field} outside approved range")
    if errors:
        return errors, warnings
    for record in anchor_deviation_records("", "", payload, anchor):
        if record["exceeds_original_prompt_tolerance"]:
            warnings.append(
                f"anchor_deviation_warning:{record['measurement']}:difference={record['absolute_difference']}:"
                f"prompt_tolerance={record['original_prompt_tolerance']}"
            )
    for field, bounds in canonical_ranges.items():
        values = np.array([float(item[field]) for item in items])
        span = float(bounds["maximum"] - bounds["minimum"])
        deltas = np.abs(np.diff(values))
        if (deltas > span + 1e-12).any():
            errors.append(f"{field} contains a range-exceeding longitudinal jump")
        elif (deltas > 0.5 * span).any():
            warnings.append(f"{field} contains a large but range-valid consecutive change")
    return errors, warnings


_validate_complete_chunk_hard_base = validate_complete_chunk


def validate_complete_chunk_policy_002(assessment, identities, recipients, anchors, payloads, reconciliation, context):
    global ANCHOR_TOLERANCES
    current = ANCHOR_TOLERANCES
    ANCHOR_TOLERANCES = {field: float("inf") for field in ORIGINAL_PROMPT_TOLERANCES}
    try:
        result = _validate_complete_chunk_hard_base(
            assessment, identities, recipients, anchors, payloads, reconciliation, context
        )
    finally:
        ANCHOR_TOLERANCES = current
    result["checks"].pop("day7_anchor_tolerances", None)
    result["checks"]["anchor_proximity_diagnostic_only"] = True
    result["failures"] = [name for name in result["failures"] if name != "day7_anchor_tolerances"]
    result["status"] = "passed" if not result["failures"] else "failed"
    result["anchor_deviation_warnings"] = [
        warning
        for recipient_id, payload in payloads.items()
        for warning in anchor_deviation_records(
            recipient_id,
            recipients.set_index("recipient_id").loc[recipient_id, "request_id"],
            payload,
            anchors.set_index("recipient_id").loc[recipient_id],
        )
        if warning["exceeds_original_prompt_tolerance"]
    ]
    return result


validate_payload = validate_payload_policy_002
validate_complete_chunk = validate_complete_chunk_policy_002


def anchor_quality_report(assessment, recipients, anchors, scope):
    diagnostics = build_diagnostics(assessment, recipients, anchors)
    day7 = assessment.loc[assessment.days_since_transplant.eq(7)].set_index("recipient_id")
    anchor_index = anchors.set_index("recipient_id")
    difference_reports = {}
    exceedance_counts = {}
    for field, anchor_field in ANCHOR_FIELD_MAP.items():
        differences = (
            day7.loc[anchor_index.index, field].astype(float) - anchor_index[anchor_field].astype(float)
        ).abs()
        difference_reports[field] = difference_distribution(differences)
        exceedance_counts[field] = int(differences.gt(ORIGINAL_PROMPT_TOLERANCES[field] + 1e-12).sum())
    correlations = diagnostics["anchor_output_and_suffix_correlations"]
    day7_modes = diagnostics["day_specific_unique_values_and_tie_aware_modes"]["7"]
    gates = {
        f"{field}_anchor_output_correlation_at_least_0_90": correlations[field]["anchor_output_correlation"] >= 0.90
        for field in ANCHOR_FIELD_MAP
    }
    gates.update({
        "day7_urine_unique_values_at_least_30": day7_modes["urine_output_ml_24h"]["unique_values"] >= 30,
        "day7_adherence_unique_values_at_least_30": day7_modes["medication_adherence_pct"]["unique_values"] >= 30,
        "day7_urine_dominant_fraction_at_most_0_20": day7_modes["urine_output_ml_24h"]["dominant_fraction"] <= 0.20,
        "day7_adherence_dominant_fraction_at_most_0_20": day7_modes["medication_adherence_pct"]["dominant_fraction"] <= 0.20,
        "anchor_recipient_suffix_correlations_below_materiality_threshold": all(
            abs(float(correlations[field]["anchor_recipient_suffix_correlation"])) < IDENTIFIER_MATERIALITY_ABS_CORRELATION
            for field in ANCHOR_FIELD_MAP
        ),
        "no_identical_complete_trajectories": diagnostics["identical_complete_trajectories"]["identical_trajectory_groups"] == 0,
    })
    later_day_warnings = [
        warning for warning in diagnostics["warnings"]
        if warning.startswith("day ") and not warning.startswith("day 7 ")
    ]
    return {
        "scope": scope,
        "status": "passed" if all(gates.values()) else "manual_review_required",
        "policy": "chunk_level_generation_quality_gate_not_individual_record_rejection",
        "original_prompt_tolerances": ORIGINAL_PROMPT_TOLERANCES,
        "identifier_materiality_absolute_correlation_threshold": IDENTIFIER_MATERIALITY_ABS_CORRELATION,
        "anchor_output_and_recipient_suffix_correlations": correlations,
        "anchor_difference_distributions": difference_reports,
        "anchor_deviation_warning_counts": exceedance_counts,
        "day7_unique_values_and_tied_modes": day7_modes,
        "identical_complete_trajectories": diagnostics["identical_complete_trajectories"],
        "anchor_clipping_counts": diagnostics["anchor_clipping_counts"],
        "later_day_repetition_warnings": later_day_warnings,
        "gates": {key: bool(value) for key, value in gates.items()},
        "failed_gates": [key for key, value in gates.items() if not bool(value)],
    }


def validate_partial_checkpoint_against_built(existing, built):
    expected = built.loc[built.recipient_id.isin(existing.recipient_id)].copy()
    expected = expected.sort_values(["recipient_id", "days_since_transplant"]).reset_index(drop=True)
    actual = existing.sort_values(["recipient_id", "days_since_transplant"]).reset_index(drop=True)
    pd.testing.assert_frame_equal(actual, expected[actual.columns], check_dtype=False, atol=1e-12, rtol=0)
    return True


def resume_002_preflight():
    assert os.environ.get(RESUME_002_GATE_NAME) == RESUME_002_GATE_VALUE, "exact amendment-002 resume gate mismatch"
    for name in [
        GATE_NAME,
        "RUN_QWEN_V32_PRODUCTION_CHUNKS_002_010",
        "RESUME_QWEN_V32_PRODUCTION_CHUNKS_006_010",
        "RUN_QWEN_V32_CONFIRM20",
        "RUN_CORRECTED_10",
        "RUN_100_RECIPIENTS",
        "RUN_10000_RECIPIENTS",
        "RUN_FULL_GENERATION",
    ]:
        assert not os.environ.get(name), f"{name} must be false"
    for number in range(1, 101):
        assert not os.environ.get(f"RUN_QWEN_V32_PRODUCTION_CHUNK_{number:03d}")
    assert not amendment_002_directory().exists(), "amendment 002 already exists; automatic rerun prohibited"
    assert not (chunk_directory(7) / "chunk_007_resume_completion_report.json").exists()
    assert all(not chunk_directory(number).exists() for number in range(8, 101))
    assert not progress_directory().exists()
    assert not RUN_10000_RECIPIENTS and not RUN_FULL_GENERATION

    completed_fingerprints = {1: verify_chunk_001_immutable()}
    for number in range(2, 6):
        completed_fingerprints[number] = verify_recorded_completed_chunk(number)
    chunk6 = chunk_directory(6)
    completed_fingerprints[6] = validate_exact_hashes(chunk6, CHUNK_006_SHA256)
    chunk6_report = json.loads((chunk6 / "chunk_006_resume_completion_report.json").read_text(encoding="utf-8"))
    assert chunk6_report["status"] == "completed" and chunk6_report["request_records"] == 100
    assert chunk6_report["response_records"] == 100 and chunk6_report["recipients_validated"] == 100
    assert chunk6_report["checkpoint_rows"] == 600 and chunk6_report["new_retries"] == 0

    amendment1 = PRODUCTION_ROOT / "validation_amendment_001"
    amendment1_fingerprint = validate_exact_hashes(amendment1, AMENDMENT_001_SHA256)
    chunk7 = chunk_directory(7)
    chunk7_partial_fingerprint = validate_exact_hashes(chunk7, CHUNK_007_ORIGINAL_SHA256)
    original_chunk7_report = json.loads((chunk7 / "chunk_007_run_report.json").read_text(encoding="utf-8"))
    assert original_chunk7_report["status"] == "failed" and original_chunk7_report["retries"] == 0
    assert original_chunk7_report["api_requests_attempted"] == original_chunk7_report["api_requests_completed"] == 27
    assert original_chunk7_report["recipients_validated"] == 26 and original_chunk7_report["checkpoint_rows"] == 156
    failure = original_chunk7_report["failure"]
    assert failure["stage"] == "payload_validation"
    assert failure["request_id"] == "QWEN-V32-PROD-REQ000627" and failure["recipient_id"] == "V32P-R000627"
    assert failure["errors"] == ["day-7 medication_adherence_pct differs from anchor by 1.759999999999998, exceeding tolerance"]

    requests7 = line_records(chunk7 / "requests.jsonl")
    responses7 = line_records(chunk7 / "responses.jsonl")
    manifest7 = line_records(chunk7 / "manifest.jsonl")
    checkpoint7 = pd.read_csv(chunk7 / "validated_recipient_checkpoint.csv")
    assert len(requests7) == len(responses7) == 27
    assert [record["request_sequence_number"] for record in requests7] == list(range(601, 628))
    assert sum(record["recipient_id"] == "V32P-R000627" for record in requests7) == 1
    assert sum(record["recipient_id"] == "V32P-R000627" for record in responses7) == 1
    assert sum(record["status"] == "validation_failed" for record in manifest7) == 1
    assert checkpoint7.shape == (156, 31) and checkpoint7.recipient_id.nunique() == 26
    assert set(checkpoint7.recipient_id) == {f"V32P-R{number:06d}" for number in range(601, 627)}

    design_checks, design_validation = verify_frozen_production_design()
    assert all(design_checks.values())
    template = (V32_DESIGN / "qwen_kidney_v3_2_prompt_template.txt").read_text(encoding="utf-8")
    assert sha256_text(template) == PROMPT_SHA256
    for value in ["0.12", "120", "0.6", "1.5"]:
        assert value in template
    api_config = json.loads((V32_DESIGN / "api_configuration.json").read_text(encoding="utf-8"))
    assert api_config["client"]["max_retries"] == 0 and api_config["automatic_retry"] is False

    recipients_all = pd.read_csv(PRODUCTION_DESIGN / "production_recipient_metadata.csv")
    assessments_all = pd.read_csv(PRODUCTION_DESIGN / "production_assessment_metadata.csv")
    identities_all = pd.read_csv(PRODUCTION_DESIGN / "production_identity_skeleton.csv", keep_default_na=False)
    anchors_all = pd.read_csv(PRODUCTION_DESIGN / "production_anchor_metadata.csv")
    slices = {
        number: frozen_chunk_slice(number, recipients_all, assessments_all, identities_all, anchors_all)
        for number in range(1, 11)
    }
    for slice_context in slices.values():
        verify_frozen_slice(slice_context)
    field_lists = json.loads((V31_DESIGN / "field_lists.json").read_text(encoding="utf-8"))
    canonical_ranges = json.loads((V31_DESIGN / "canonical_qwen_ranges.json").read_text(encoding="utf-8"))
    assessment_columns = pd.read_csv(V31_DESIGN / "assessment_table_schema.csv").field_name.tolist()

    all_requests = []
    all_responses = []
    all_manifests = []
    for number in range(1, 8):
        directory = chunk_directory(number)
        all_requests.extend(line_records(directory / "requests.jsonl"))
        all_responses.extend(line_records(directory / "responses.jsonl"))
        all_manifests.extend(line_records(directory / "manifest.jsonl"))
    assert len(all_requests) == len(all_responses) == 627
    assert len({record["request_id"] for record in all_requests}) == 627
    assert len({record["request_id"] for record in all_responses}) == 627
    assert {record["request_id"] for record in all_requests} == {record["request_id"] for record in all_responses}
    request_lookup = {record["request_id"]: record for record in all_requests}
    recipient_lookup = recipients_all.set_index("recipient_id")
    anchor_lookup = anchors_all.set_index("recipient_id")
    frozen_by_recipient = {recipient_id: frame.copy() for recipient_id, frame in assessments_all.groupby("recipient_id")}
    payloads = {}
    built_frames = []
    diagnostic_records = []
    hard_failures = []
    build_failures = []
    for response_record in all_responses:
        request_id = response_record["request_id"]
        request_record = request_lookup[request_id]
        recipient_id = request_record["recipient_id"]
        try:
            _, payload = parse_preserved_response(response_record)
        except Exception as exc:
            hard_failures.append({"request_id": request_id, "stage": "preserved_response_parse", "error": str(exc)})
            continue
        anchor = anchor_lookup.loc[recipient_id].to_dict()
        errors, warnings = validate_payload_policy_002(payload, anchor, canonical_ranges)
        if errors:
            hard_failures.append({"request_id": request_id, "recipient_id": recipient_id, "errors": errors})
            continue
        payloads[recipient_id] = payload
        diagnostic_records.extend(anchor_deviation_records(recipient_id, request_id, payload, anchor))
        try:
            rows = build_assessment_rows(
                recipient_lookup.loc[recipient_id].to_dict(), anchor,
                frozen_by_recipient[recipient_id], payload, assessment_columns
            )
            built_frames.append(pd.DataFrame(rows, columns=assessment_columns))
        except Exception as exc:
            build_failures.append({"recipient_id": recipient_id, "error": str(exc)})
    assert not hard_failures and not build_failures
    assert len(payloads) == 627 and len(built_frames) == 627
    built_assessments = pd.concat(built_frames, ignore_index=True)
    assert built_assessments.shape == (3762, 31)
    assert built_assessments.assessment_id.is_unique
    assert not built_assessments.duplicated(["recipient_id", "days_since_transplant"]).any()
    assert not built_assessments.duplicated().any()

    existing_validated = pd.concat(
        [pd.read_csv(chunk_directory(number) / f"assessment_chunk_{number:03d}.csv") for number in range(1, 7)]
        + [checkpoint7], ignore_index=True
    )
    assert existing_validated.shape == (3756, 31)
    assert validate_partial_checkpoint_against_built(existing_validated, built_assessments)
    recovery_rows = built_assessments.loc[built_assessments.recipient_id.eq("V32P-R000627")].copy()
    assert recovery_rows.shape == (6, 31)

    diagnostics = pd.DataFrame(diagnostic_records).sort_values(["request_id", "measurement"]).reset_index(drop=True)
    warnings = diagnostics.loc[diagnostics.exceeds_original_prompt_tolerance].copy()
    assert set(zip(warnings.recipient_id, warnings.measurement)) == {
        ("V32P-R000572", "urine_output_ml_24h"),
        ("V32P-R000627", "medication_adherence_pct"),
    }
    validation_failures = [record for record in all_manifests if record["status"] == "validation_failed"]
    assert len(validation_failures) == 2
    assert {record["recipient_id"] for record in validation_failures} == {"V32P-R000572", "V32P-R000627"}
    assert all(record.get("stage") == "payload_validation" for record in validation_failures)
    assert not any(record["status"] == "request_failed" for record in all_manifests)

    existing_quality = {}
    for number in range(1, 7):
        assessment = pd.read_csv(chunk_directory(number) / f"assessment_chunk_{number:03d}.csv")
        quality = anchor_quality_report(assessment, slices[number]["recipients"], slices[number]["anchors"], f"chunk_{number:03d}")
        assert quality["status"] == "passed", (number, quality["failed_gates"])
        existing_quality[f"chunk_{number:03d}"] = quality
    preserved_recipients = recipients_all.loc[recipients_all.request_sequence_number.le(627)].copy()
    preserved_anchors = anchors_all.loc[anchors_all.recipient_id.isin(preserved_recipients.recipient_id)].copy()
    aggregate_preserved_quality = anchor_quality_report(
        built_assessments, preserved_recipients, preserved_anchors, "all_627_preserved_responses"
    )
    assert all(
        aggregate_preserved_quality["anchor_output_and_recipient_suffix_correlations"][field]["anchor_output_correlation"] >= 0.90
        for field in ANCHOR_FIELD_MAP
    )

    immutable_paths = immutable_inputs()
    chunk7_prefixes = {
        name: {"size": (chunk7 / name).stat().st_size, "sha256": sha256_file(chunk7 / name)}
        for name in ["requests.jsonl", "responses.jsonl", "manifest.jsonl", "validated_recipient_checkpoint.csv"]
    }
    context = {
        "checks": {
            "all_627_preserved_responses_hard_valid": not hard_failures,
            "all_python_derived_rows_available": not build_failures,
            "existing_checkpoints_reconcile": True,
            "only_572_and_627_historical_anchor_failures": len(validation_failures) == 2,
            "only_two_anchor_deviation_warnings": len(warnings) == 2,
            "aggregate_anchor_correlations_at_least_0_90": all(
                aggregate_preserved_quality["anchor_output_and_recipient_suffix_correlations"][field]["anchor_output_correlation"] >= 0.90
                for field in ANCHOR_FIELD_MAP
            ),
            "no_existing_record_requires_regeneration": True,
            "chunks_001_006_complete_immutable": True,
            "chunk_007_exact_partial_state": True,
            "recipient_627_one_request_one_response": True,
            "next_request_is_628": recipients_all.loc[recipients_all.request_sequence_number.eq(628), "recipient_id"].item() == "V32P-R000628",
            "chunks_008_010_absent": all(not chunk_directory(number).exists() for number in range(8, 11)),
            "chunks_011_100_disabled": all(not chunk_directory(number).exists() for number in range(11, 101)),
            "maximum_new_requests_373": MAX_NEW_API_REQUESTS_002 == 373,
            "automatic_retries_disabled": api_config["client"]["max_retries"] == 0,
            "prompt_and_hash_unchanged": sha256_text(template) == PROMPT_SHA256,
            "final_assembly_disabled": True,
        },
        "design_validation": design_validation,
        "template": template,
        "api_config": api_config,
        "slices": slices,
        "recipients_all": recipients_all,
        "assessments_all": assessments_all,
        "identities_all": identities_all,
        "anchors_all": anchors_all,
        "field_lists": field_lists,
        "canonical_ranges": canonical_ranges,
        "assessment_columns": assessment_columns,
        "payloads": payloads,
        "built_assessments": built_assessments,
        "recovery_rows": recovery_rows,
        "retrospective_diagnostics": diagnostics,
        "anchor_warnings": warnings,
        "existing_quality": existing_quality,
        "aggregate_preserved_quality": aggregate_preserved_quality,
        "completed_fingerprints": completed_fingerprints,
        "immutable_paths": immutable_paths,
        "immutable_before": snapshot_immutable(immutable_paths),
        "notebooks_before": snapshot_notebooks_01_05(),
        "protected_before": protected_fingerprint(),
        "production_design_before": tree_fingerprint(PRODUCTION_DESIGN),
        "amendment_001_fingerprint": amendment1_fingerprint,
        "chunk_007_partial_before": chunk7_partial_fingerprint,
        "chunk_007_original_report_before": file_fingerprint(chunk7 / "chunk_007_run_report.json"),
        "chunk_007_prefixes": chunk7_prefixes,
    }
    assert {key: context["protected_before"][key] for key in EXPECTED_PROTECTED} == EXPECTED_PROTECTED
    assert all(context["checks"].values())
    return context


def create_amendment_002(context):
    directory = amendment_002_directory()
    directory.mkdir(parents=False, exist_ok=False)
    diagnostics = context["retrospective_diagnostics"]
    warnings = context["anchor_warnings"]
    amendment = {
        "amendment_id": AMENDMENT_002_ID,
        "status": "active_for_production_validation_from_manual_resume_authorisation",
        "created_at_utc": now_utc(),
        "post_hoc_origin_disclosed": True,
        "rationale": {
            "anchors_are_generation_controls_not_observed_clinical_ground_truth": True,
            "selective_censoring_avoided": True,
            "observed_boundary_cases": [
                {"recipient_id": "V32P-R000572", "measurement": "urine_output_ml_24h", "difference": 123.8, "prompt_tolerance": 120.0},
                {"recipient_id": "V32P-R000627", "measurement": "medication_adherence_pct", "difference": 1.76, "prompt_tolerance": 1.5},
            ],
        },
        "policy_change": {
            "anchor_proximity": "diagnostic_anchor_deviation_warning_not_hard_record_validity",
            "applies_consistently_and_retrospectively": True,
            "individual_anchor_deviation_does_not_trigger_retry_regeneration_or_rejection": True,
            "chunk_level_generation_quality_gates_remain_enforced": True,
        },
        "unchanged": {
            "prompt_tolerances": ORIGINAL_PROMPT_TOLERANCES,
            "prompt_sha256": PROMPT_SHA256,
            "anchor_algorithm_version": ANCHOR_ALGORITHM_VERSION,
            "frozen_prompt": True,
            "frozen_api_configuration": True,
            "hard_validation_rules": [
                "response presence and JSON", "exact assessment schema/count/order", "finite numeric types",
                "approved clinical ranges", "unique recipients/assessments/recipient-days", "dates and retention dates",
                "ABO/infection/creatinine-change/previous-rejection/target derivations", "static-field consistency",
                "identifier/audit/event-control leakage", "raw-response reconciliation", "checkpoint integrity",
                "evidence integrity",
            ],
        },
        "chunk_quality_gate": {
            "minimum_anchor_output_correlation": 0.90,
            "minimum_unique_day7_urine_values": 30,
            "minimum_unique_day7_adherence_values": 30,
            "maximum_day7_urine_or_adherence_dominant_fraction": 0.20,
            "identifier_materiality_absolute_correlation_threshold": IDENTIFIER_MATERIALITY_ABS_CORRELATION,
            "identical_complete_trajectory_groups_allowed": 0,
            "failure_action": "preserve completed valid chunk and stop before next chunk for manual review",
        },
    }
    summary = {
        "amendment_id": AMENDMENT_002_ID,
        "status": "passed",
        "api_requests": 0,
        "preserved_responses_revalidated": 627,
        "hard_validation_failures": 0,
        "records_requiring_regeneration": 0,
        "anchor_deviation_warning_count": len(warnings),
        "anchor_deviation_warning_recipients": sorted(warnings.recipient_id.unique().tolist()),
        "warning_counts_by_measurement": {
            field: int(warnings.measurement.eq(field).sum()) for field in ANCHOR_FIELD_MAP
        },
        "difference_distributions": {
            field: difference_distribution(diagnostics.loc[diagnostics.measurement.eq(field), "absolute_difference"])
            for field in ANCHOR_FIELD_MAP
        },
        "aggregate_preserved_anchor_quality": context["aggregate_preserved_quality"],
        "completed_chunk_quality_gates_001_006": context["existing_quality"],
        "historical_anchor_failures": ["V32P-R000572", "V32P-R000627"],
    }
    write_json_exclusive(directory / "validation_policy_amendment_002.json", amendment)
    write_csv_exclusive(directory / "retrospective_anchor_diagnostics.csv", diagnostics)
    write_json_exclusive(directory / "retrospective_anchor_diagnostics_summary.json", summary)
    integrity = {
        "amendment_id": AMENDMENT_002_ID,
        "status": "passed",
        "api_requests": 0,
        "checked_at_utc": now_utc(),
        "pre_api_checks": context["checks"],
        "protected_before": context["protected_before"],
        "protected_after_amendment": protected_fingerprint(),
        "chunks_001_006_unchanged": {
            f"chunk_{number:03d}": tree_fingerprint(chunk_directory(number)) == context["completed_fingerprints"][number]
            for number in range(1, 7)
        },
        "chunk_007_partial_unchanged_before_recovery": tree_fingerprint(chunk_directory(7)) == context["chunk_007_partial_before"],
        "amendment_001_unchanged": tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_001") == context["amendment_001_fingerprint"],
        "previous_evidence_unchanged": snapshot_immutable(context["immutable_paths"]) == context["immutable_before"],
        "notebooks_01_05_unchanged": snapshot_notebooks_01_05() == context["notebooks_before"],
        "production_design_unchanged": tree_fingerprint(PRODUCTION_DESIGN) == context["production_design_before"],
        "created_file_fingerprints_excluding_self": {
            name: file_fingerprint(directory / name)
            for name in [
                "validation_policy_amendment_002.json",
                "retrospective_anchor_diagnostics.csv",
                "retrospective_anchor_diagnostics_summary.json",
            ]
        },
    }
    assert integrity["protected_before"] == integrity["protected_after_amendment"]
    assert all(integrity["chunks_001_006_unchanged"].values())
    assert all(bool(integrity[key]) for key in [
        "chunk_007_partial_unchanged_before_recovery", "amendment_001_unchanged",
        "previous_evidence_unchanged", "notebooks_01_05_unchanged", "production_design_unchanged",
    ])
    write_json_exclusive(directory / "amendment_002_integrity_report.json", integrity)
    return amendment, summary, tree_fingerprint(directory)


def assert_amendment_002_integrity(context, completed_fingerprints=None):
    chunk7 = chunk_directory(7)
    prefix_checks = {
        name: (chunk7 / name).stat().st_size >= record["size"]
        and file_prefix_sha256(chunk7 / name, record["size"]) == record["sha256"]
        for name, record in context["chunk_007_prefixes"].items()
    }
    result = {
        "protected_unchanged": protected_fingerprint() == context["protected_before"],
        "previous_evidence_unchanged": snapshot_immutable(context["immutable_paths"]) == context["immutable_before"],
        "notebooks_01_05_unchanged": snapshot_notebooks_01_05() == context["notebooks_before"],
        "production_design_unchanged": tree_fingerprint(PRODUCTION_DESIGN) == context["production_design_before"],
        "validation_amendment_001_unchanged": tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_001") == context["amendment_001_fingerprint"],
        "validation_amendment_002_unchanged": tree_fingerprint(amendment_002_directory()) == context["amendment_002_fingerprint"],
        "original_chunk_007_failed_report_unchanged": file_fingerprint(chunk7 / "chunk_007_run_report.json") == context["chunk_007_original_report_before"],
        "chunk_007_request_response_manifest_checkpoint_prefixes_preserved": all(prefix_checks.values()),
        "chunk_007_identity_unchanged": sha256_file(chunk7 / "identity_chunk_007.csv") == CHUNK_007_ORIGINAL_SHA256["identity_chunk_007.csv"],
        "chunk_007_run_configuration_unchanged": sha256_file(chunk7 / "run_configuration.json") == CHUNK_007_ORIGINAL_SHA256["run_configuration.json"],
        "chunks_011_100_absent": all(not chunk_directory(number).exists() for number in range(11, 101)),
        "final_assembly_absent": not (PRODUCTION_ROOT / "assessment_production.csv").exists(),
    }
    for number, fingerprint in (completed_fingerprints or {}).items():
        result[f"chunk_{number:03d}_unchanged"] = tree_fingerprint(chunk_directory(number)) == fingerprint
    assert all(result.values()), [name for name, passed in result.items() if not passed]
    return result


assert_global_integrity = assert_amendment_002_integrity


def recover_recipient_627(context):
    chunk7 = chunk_directory(7)
    requests_before = sha256_file(chunk7 / "requests.jsonl")
    responses_before = sha256_file(chunk7 / "responses.jsonl")
    checkpoint = pd.read_csv(chunk7 / "validated_recipient_checkpoint.csv")
    assert checkpoint.shape == (156, 31) and "V32P-R000627" not in set(checkpoint.recipient_id)
    recovery = context["recovery_rows"].copy()
    append_checkpoint(chunk7 / "validated_recipient_checkpoint.csv", recovery)
    warning = context["anchor_warnings"].loc[context["anchor_warnings"].recipient_id.eq("V32P-R000627")].to_dict(orient="records")
    assert len(warning) == 1 and warning[0]["measurement"] == "medication_adherence_pct"
    append_jsonl(chunk7 / "manifest.jsonl", {
        "run_id": RUN_ID,
        "chunk_id": "CHUNK-007",
        "request_id": "QWEN-V32-PROD-REQ000627",
        "recipient_id": "V32P-R000627",
        "request_sequence_number": 627,
        "attempt_number": 1,
        "status": "validated_under_amendment_002",
        "amendment_id": AMENDMENT_002_ID,
        "historical_status_preserved": "validation_failed",
        "api_request_made_for_recovery": False,
        "anchor_deviation_warnings": warning,
        "checkpoint_recipient_count": 27,
        "checkpoint_row_count": 162,
        "timestamp_utc": now_utc(),
    })
    write_json_state(chunk7 / "completion_state.json", {
        "run_id": RUN_ID, "chunk_id": "CHUNK-007", "status": "manual_resume_running",
        "amendment_id": AMENDMENT_002_ID, "automatic_resume": False,
        "requests_attempted_total": 27, "requests_completed_total": 27,
        "new_requests_attempted": 0, "new_requests_completed": 0,
        "recipients_validated": 27, "checkpoint_rows": 162,
        "last_validated_request_id": "QWEN-V32-PROD-REQ000627",
        "last_validated_recipient_id": "V32P-R000627",
    })
    assert sha256_file(chunk7 / "requests.jsonl") == requests_before
    assert sha256_file(chunk7 / "responses.jsonl") == responses_before
    manifest = line_records(chunk7 / "manifest.jsonl")
    assert sum(record.get("recipient_id") == "V32P-R000627" and record["status"] == "validation_failed" for record in manifest) == 1
    assert sum(record.get("recipient_id") == "V32P-R000627" and record["status"] == "validated_under_amendment_002" for record in manifest) == 1
    return recovery


def resume_chunk_007_under_amendment_002(context, completed_fingerprints):
    started = time.monotonic()
    chunk7 = chunk_directory(7)
    report_path = chunk7 / "chunk_007_resume_completion_report.json"
    assert not report_path.exists()
    completion_parameters = dict(context["api_config"]["completion"])
    recipients = context["slices"][7]["recipients"]
    remaining = recipients.loc[recipients.request_sequence_number.between(628, 700)].copy()
    anchors = context["slices"][7]["anchors"]
    frozen_assessments = context["slices"][7]["assessments"]
    identities = context["slices"][7]["identities"]
    anchor_lookup = anchors.set_index("recipient_id")
    usage_records = []
    finish_records = []
    soft_warnings = []
    failure = None
    new_attempted = 0
    new_completed = 0
    new_validated = 0
    validation = None
    reconciliation = None
    diagnostics = None
    quality = None
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL, max_retries=0, timeout=180.0)
    try:
        for recipient in remaining.to_dict(orient="records"):
            if new_attempted >= 73:
                raise RuntimeError("73-request chunk-007 resume hard cap reached")
            request_id = recipient["request_id"]
            recipient_id = recipient["recipient_id"]
            anchor = anchor_lookup.loc[recipient_id].to_dict()
            rendered_prompt = render_prompt(context["template"], recipient, anchor)
            messages = [
                {"role": "system", "content": "Return only the exact JSON object requested. No prose or Markdown."},
                {"role": "user", "content": rendered_prompt},
            ]
            append_jsonl(chunk7 / "requests.jsonl", {
                "run_id": RUN_ID, "chunk_id": "CHUNK-007", "request_id": request_id,
                "recipient_id": recipient_id, "request_sequence_number": int(recipient["request_sequence_number"]),
                "attempt_number": 1, "deterministic_request_seed": int(recipient["request_seed"]),
                "prompt_version": PROMPT_VERSION, "prompt_template_sha256": PROMPT_SHA256,
                "rendered_prompt": rendered_prompt, "rendered_prompt_sha256": sha256_text(rendered_prompt),
                "rendered_message_payload": messages, "model": completion_parameters["model"],
                "completion_parameters": {**{key: value for key, value in completion_parameters.items() if key != "model"}, "seed": int(recipient["request_seed"])},
                "anchor_algorithm_version": ANCHOR_ALGORITHM_VERSION, "anchor_metadata_sha256": anchor_row_hash(anchor),
                "validation_amendment_id": AMENDMENT_002_ID, "timestamp_utc": now_utc(),
                "status": "request_preserved_before_submission",
            })
            append_jsonl(chunk7 / "manifest.jsonl", {
                "run_id": RUN_ID, "chunk_id": "CHUNK-007", "request_id": request_id,
                "recipient_id": recipient_id, "request_sequence_number": int(recipient["request_sequence_number"]),
                "attempt_number": 1, "status": "request_preserved_before_submission", "timestamp_utc": now_utc(),
            })
            new_attempted += 1
            append_jsonl(chunk7 / "manifest.jsonl", {
                "request_id": request_id, "recipient_id": recipient_id, "attempt_number": 1,
                "status": "network_submission_started", "timestamp_utc": now_utc(),
            })
            try:
                response = client.chat.completions.create(
                    model=completion_parameters["model"], messages=messages,
                    max_tokens=completion_parameters["max_tokens"], temperature=completion_parameters["temperature"],
                    top_p=completion_parameters["top_p"], presence_penalty=completion_parameters["presence_penalty"],
                    seed=int(recipient["request_seed"]), extra_body=completion_parameters["extra_body"],
                )
            except Exception as exc:
                failure = {
                    "stage": "api_request", "request_id": request_id, "recipient_id": recipient_id,
                    "request_sequence_number": int(recipient["request_sequence_number"]), "attempt_number": 1,
                    "exception_type": type(exc).__name__, "message": str(exc),
                }
                append_jsonl(chunk7 / "manifest.jsonl", {**failure, "status": "request_failed", "timestamp_utc": now_utc()})
                raise RuntimeError(f"request failed for {request_id}") from exc
            new_completed += 1
            serialized = response.model_dump_json(indent=2)
            append_jsonl(chunk7 / "responses.jsonl", {
                "run_id": RUN_ID, "chunk_id": "CHUNK-007", "request_id": request_id,
                "recipient_id": recipient_id, "request_sequence_number": int(recipient["request_sequence_number"]),
                "attempt_number": 1, "received_at_utc": now_utc(), "serialized_response_json": serialized,
                "serialized_response_sha256": sha256_text(serialized), "status": "complete_response_preserved_before_extraction",
            })
            choice = response.choices[0] if response.choices else None
            message = choice.message if choice else None
            content = message.content if message else None
            reasoning = getattr(message, "reasoning_content", None) if message else None
            usage = response.usage.model_dump() if response.usage else {}
            metadata = {
                "request_id": request_id, "recipient_id": recipient_id,
                "request_sequence_number": int(recipient["request_sequence_number"]),
                "finish_reason": choice.finish_reason if choice else None,
                "reasoning_content_status": "present_nonempty" if reasoning else "present_empty" if reasoning == "" else "absent",
                "final_content_length": len(content) if isinstance(content, str) else 0,
                "token_usage": usage,
            }
            usage_records.append(usage)
            finish_records.append(metadata)
            append_jsonl(chunk7 / "manifest.jsonl", {**metadata, "status": "response_preserved", "timestamp_utc": now_utc()})
            payload = None
            if metadata["finish_reason"] == "length":
                failure = {"stage": "finish_reason_length", **metadata}
            elif not content:
                failure = {"stage": "absent_final_content", **metadata}
            else:
                try:
                    payload = json.loads(content)
                except Exception as exc:
                    failure = {"stage": "json_parse", **metadata, "exception_type": type(exc).__name__, "message": str(exc)}
            if failure is not None:
                append_jsonl(chunk7 / "manifest.jsonl", {**failure, "status": "validation_failed", "timestamp_utc": now_utc()})
                raise RuntimeError(f"response failed before validation for {request_id}")
            errors, warnings = validate_payload_policy_002(payload, anchor, context["canonical_ranges"])
            if errors:
                failure = {"stage": "hard_payload_validation", **metadata, "errors": errors, "warnings": warnings, "amendment_id": AMENDMENT_002_ID}
                append_jsonl(chunk7 / "manifest.jsonl", {**failure, "status": "validation_failed", "timestamp_utc": now_utc()})
                raise RuntimeError(f"hard payload validation failed for {request_id}")
            context["payloads"][recipient_id] = payload
            frozen = frozen_assessments.loc[frozen_assessments.recipient_id.eq(recipient_id)].copy()
            recipient_rows = build_assessment_rows(recipient, anchor, frozen, payload, context["assessment_columns"])
            append_checkpoint(chunk7 / "validated_recipient_checkpoint.csv", pd.DataFrame(recipient_rows, columns=context["assessment_columns"]))
            new_validated += 1
            soft_warnings.extend({"recipient_id": recipient_id, "warning": warning} for warning in warnings)
            append_jsonl(chunk7 / "manifest.jsonl", {
                "request_id": request_id, "recipient_id": recipient_id,
                "request_sequence_number": int(recipient["request_sequence_number"]), "attempt_number": 1,
                "status": "validated_and_checkpointed", "validation_amendment_id": AMENDMENT_002_ID,
                "checkpoint_recipient_count": 27 + new_validated, "checkpoint_row_count": 162 + 6 * new_validated,
                "soft_warnings": warnings, "timestamp_utc": now_utc(),
            })
            write_json_state(chunk7 / "completion_state.json", {
                "run_id": RUN_ID, "chunk_id": "CHUNK-007", "status": "manual_resume_running",
                "amendment_id": AMENDMENT_002_ID, "automatic_resume": False,
                "requests_attempted_total": 27 + new_attempted, "requests_completed_total": 27 + new_completed,
                "new_requests_attempted": new_attempted, "new_requests_completed": new_completed,
                "recipients_validated": 27 + new_validated, "checkpoint_rows": 162 + 6 * new_validated,
                "last_validated_request_id": request_id, "last_validated_recipient_id": recipient_id,
            })

        assessment = pd.read_csv(chunk7 / "validated_recipient_checkpoint.csv")
        assert assessment.shape == (600, 31) and assessment.recipient_id.nunique() == 100
        requests = line_records(chunk7 / "requests.jsonl")
        responses = line_records(chunk7 / "responses.jsonl")
        manifest = line_records(chunk7 / "manifest.jsonl")
        assert len(requests) == len(responses) == 100
        assert len({record["request_id"] for record in requests}) == 100
        assert sum(record["recipient_id"] == "V32P-R000627" for record in requests) == 1
        assert sum(record["recipient_id"] == "V32P-R000627" for record in responses) == 1
        assert len(manifest) == 401
        assert sum(record["status"] == "validation_failed" for record in manifest) == 1
        assert sum(record["status"] == "validated_under_amendment_002" for record in manifest) == 1
        payloads = {}
        for record in responses:
            _, payload = parse_preserved_response(record)
            payloads[record["recipient_id"]] = payload
        expected_returned = EXPECTED_RETURNED_FIELDS
        globals()["EXPECTED_RETURNED_FIELDS"] = 3000
        try:
            reconciliation = reconcile_payloads(payloads, assessment)
        finally:
            globals()["EXPECTED_RETURNED_FIELDS"] = expected_returned
        validation_context = {**context, "recipients": recipients, "assessments": frozen_assessments, "anchors": anchors}
        validation = validate_complete_chunk_policy_002(
            assessment, identities, recipients, anchors, payloads, reconciliation, validation_context
        )
        diagnostics = build_diagnostics(assessment, recipients, anchors)
        validation["checks"]["no_identical_complete_trajectories"] = diagnostics["identical_complete_trajectories"]["identical_trajectory_groups"] == 0
        validation["checks"]["checkpoint_assessment_equality"] = True
        validation["failures"] = [key for key, value in validation["checks"].items() if not bool(value)]
        validation["status"] = "passed" if not validation["failures"] else "failed"
        if validation["status"] != "passed":
            failure = {"stage": "complete_chunk_hard_validation", "validation": validation}
            raise RuntimeError("chunk 007 complete hard validation failed")
        quality = anchor_quality_report(assessment, recipients, anchors, "chunk_007")
        day7 = assessment.loc[assessment.days_since_transplant.eq(7)].copy()
        anchor_audit = anchors.merge(day7[["recipient_id", *ANCHOR_FIELD_MAP]], on="recipient_id", validate="one_to_one")
        for output, anchor_field in ANCHOR_FIELD_MAP.items():
            difference = (anchor_audit[output] - anchor_audit[anchor_field]).abs()
            anchor_audit[f"{output}_absolute_anchor_difference"] = difference
            anchor_audit[f"{output}_within_prompt_tolerance"] = difference.le(ORIGINAL_PROMPT_TOLERANCES[output] + 1e-12)
            anchor_audit[f"{output}_anchor_deviation_warning"] = difference.gt(ORIGINAL_PROMPT_TOLERANCES[output] + 1e-12)
        write_csv_exclusive(chunk7 / "assessment_chunk_007.csv", assessment)
        write_csv_exclusive(chunk7 / "anchor_audit_chunk_007.csv", anchor_audit)
        write_json_exclusive(chunk7 / "request_response_reconciliation.json", reconciliation)
        write_json_exclusive(chunk7 / "diagnostic_quality_report.json", {**diagnostics, "anchor_generation_quality": quality})
        write_json_state(chunk7 / "completion_state.json", {
            "run_id": RUN_ID, "chunk_id": "CHUNK-007", "status": "completed_under_amendment_002",
            "amendment_id": AMENDMENT_002_ID, "automatic_resume": False,
            "requests_attempted": 100, "requests_completed": 100, "recipients_validated": 100,
            "checkpoint_rows": 600, "assessment_chunk_created": True,
            "historical_validation_failures_preserved": 1, "amendment_validation_transitions": 1,
            "chunk_quality_gate_status": quality["status"],
            "chunks_011_to_100_started": False, "final_dataset_assembly_started": False,
        })
    except Exception as exc:
        if failure is None:
            failure = {"stage": "resume_execution_or_validation", "exception_type": type(exc).__name__, "message": str(exc), "traceback": traceback.format_exc()}
        write_json_state(chunk7 / "completion_state.json", {
            "run_id": RUN_ID, "chunk_id": "CHUNK-007", "status": "resume_failed",
            "amendment_id": AMENDMENT_002_ID, "automatic_resume": False,
            "new_requests_attempted": new_attempted, "new_requests_completed": new_completed,
            "recipients_validated": 27 + new_validated, "checkpoint_rows": 162 + 6 * new_validated,
            "failure": failure, "retry_performed": False,
            "assessment_chunk_created": (chunk7 / "assessment_chunk_007.csv").exists(),
        })
    finally:
        original = json.loads((chunk7 / "chunk_007_run_report.json").read_text(encoding="utf-8"))
        new_tokens = {key: sum(int(record.get(key, 0) or 0) for record in usage_records) for key in ["prompt_tokens", "completion_tokens", "total_tokens"]}
        total_tokens = {key: int(original["token_usage_total"][key]) + new_tokens[key] for key in new_tokens}
        resume_report = {
            "run_id": RUN_ID, "chunk_id": "CHUNK-007", "resume_id": "QWEN-V32-PROD-001-RESUME-007-A002",
            "amendment_id": AMENDMENT_002_ID, "status": "completed" if failure is None else "failed",
            "original_failed_run_report": "chunk_007_run_report.json",
            "original_failed_run_report_sha256": CHUNK_007_ORIGINAL_SHA256["chunk_007_run_report.json"],
            "historical_failure_preserved": True, "historical_validation_failure_count": 1,
            "amendment_validation_transition_count": 1, "recipient_627_api_requests_during_recovery": 0,
            "new_api_requests_attempted": new_attempted, "new_api_requests_completed": new_completed,
            "new_recipients_validated": new_validated, "new_attempts": new_attempted,
            "new_failures": 0 if failure is None else 1, "new_retries": 0, "failure": failure,
            "finish_reason_counts": dict(Counter(item["finish_reason"] for item in finish_records)),
            "reasoning_content_status_counts": dict(Counter(item["reasoning_content_status"] for item in finish_records)),
            "new_token_usage": new_tokens, "token_usage_total": total_tokens,
            "new_elapsed_seconds": time.monotonic() - started,
            "elapsed_seconds": float(original["elapsed_seconds"]) + (time.monotonic() - started),
            "request_records": count_jsonl(chunk7 / "requests.jsonl"),
            "response_records": count_jsonl(chunk7 / "responses.jsonl"),
            "manifest_records": count_jsonl(chunk7 / "manifest.jsonl"),
            "manifest_status_counts": dict(Counter(record["status"] for record in line_records(chunk7 / "manifest.jsonl"))),
            "recipients_validated": int(pd.read_csv(chunk7 / "validated_recipient_checkpoint.csv").recipient_id.nunique()),
            "checkpoint_rows": len(pd.read_csv(chunk7 / "validated_recipient_checkpoint.csv")),
            "chunk_dimensions": [600, 31] if failure is None else [len(pd.read_csv(chunk7 / "validated_recipient_checkpoint.csv")), 31],
            "validation": validation, "reconciliation": reconciliation, "diagnostics": diagnostics,
            "anchor_generation_quality": quality, "soft_warnings": soft_warnings,
            "anchor_deviation_warning_count": sum("anchor_deviation_warning" in item["warning"] for item in soft_warnings) + 1,
            "integrity_in_finally": assert_amendment_002_integrity(context, completed_fingerprints),
            "chunks_011_to_100_started": False, "final_dataset_assembly_started": False,
            "created_files": sorted(path.name for path in chunk7.iterdir() if path.is_file()) + ["chunk_007_resume_completion_report.json"],
            "artifact_fingerprints_excluding_resume_report": fingerprint_named_files(chunk7),
        }
        write_json_exclusive(report_path, resume_report)
    return resume_report


def enrich_completed_chunk_report_with_quality(number, context):
    suffix = f"{number:03d}"
    directory = chunk_directory(number)
    assessment = pd.read_csv(directory / f"assessment_chunk_{suffix}.csv")
    quality = anchor_quality_report(
        assessment, context["slices"][number]["recipients"], context["slices"][number]["anchors"], f"chunk_{suffix}"
    )
    report_path = directory / f"chunk_{suffix}_run_report.json"
    report = json.loads(report_path.read_text(encoding="utf-8"))
    assert report["status"] == "completed"
    report["validation_policy_amendment_id"] = AMENDMENT_002_ID
    report["anchor_generation_quality"] = quality
    report["anchor_deviation_warning_count"] = sum(quality["anchor_deviation_warning_counts"].values())
    write_json_state(report_path, report)
    return report, quality


def aggregate_amendment_002_through_chunk_010(context, completed_fingerprints, resume_reports, quality_reports):
    started = time.monotonic()
    directory = progress_directory()
    assert not directory.exists()
    assessment_frames = []
    identity_frames = []
    anchor_frames = []
    requests = []
    responses = []
    manifests = []
    reports = {}
    per_chunk = {}
    for number in range(1, 11):
        suffix = f"{number:03d}"
        chunk = chunk_directory(number)
        assessment_frames.append(pd.read_csv(chunk / f"assessment_chunk_{suffix}.csv"))
        identity_frames.append(pd.read_csv(chunk / f"identity_chunk_{suffix}.csv", keep_default_na=False))
        anchor_frames.append(pd.read_csv(chunk / f"anchor_audit_chunk_{suffix}.csv"))
        requests.extend(line_records(chunk / "requests.jsonl"))
        responses.extend(line_records(chunk / "responses.jsonl"))
        manifests.extend(line_records(chunk / "manifest.jsonl"))
        if number == 6:
            report_name = "chunk_006_resume_completion_report.json"
        elif number == 7:
            report_name = "chunk_007_resume_completion_report.json"
        else:
            report_name = f"chunk_{suffix}_run_report.json"
        report = json.loads((chunk / report_name).read_text(encoding="utf-8"))
        reports[number] = report
        per_chunk[suffix] = {
            "status": report["status"],
            "anchor_generation_quality": quality_reports[f"chunk_{suffix}"],
            "soft_warnings": report.get("soft_warnings", []),
            "historical_validation_failure_count": report.get("historical_validation_failure_count", 0),
            "new_failures": report.get("new_failures", report.get("failures", 0)),
        }
    assessment = pd.concat(assessment_frames, ignore_index=True)
    identity = pd.concat(identity_frames, ignore_index=True)
    anchor = pd.concat(anchor_frames, ignore_index=True)
    recipients = pd.concat([context["slices"][number]["recipients"] for number in range(1, 11)], ignore_index=True)
    frozen = pd.concat([context["slices"][number]["assessments"] for number in range(1, 11)], ignore_index=True)
    request_ids = [record["request_id"] for record in requests]
    response_ids = [record["request_id"] for record in responses]
    checks = {
        "ten_completed_chunks": all(reports[number]["status"] == "completed" for number in range(1, 11)),
        "recipients_1000": assessment.recipient_id.nunique() == 1000,
        "rows_6000": assessment.shape == (6000, 31),
        "assessment_ids_unique": assessment.assessment_id.is_unique,
        "recipient_day_unique": not assessment.duplicated(["recipient_id", "days_since_transplant"]).any(),
        "no_duplicate_rows": not assessment.duplicated().any(),
        "six_rows_each": assessment.groupby("recipient_id").size().eq(6).all(),
        "ordered_days": assessment.groupby("recipient_id").days_since_transplant.apply(list).map(lambda values: values == DAYS).all(),
        "request_records_1000": len(requests) == 1000,
        "response_records_1000": len(responses) == 1000,
        "request_ids_unique": len(set(request_ids)) == 1000,
        "response_ids_unique": len(set(response_ids)) == 1000,
        "request_response_ids_equal": set(request_ids) == set(response_ids),
        "manifest_records_4002": len(manifests) == 4002,
        "recipient_572_one_request": sum(record["recipient_id"] == "V32P-R000572" for record in requests) == 1,
        "recipient_627_one_request": sum(record["recipient_id"] == "V32P-R000627" for record in requests) == 1,
        "recipient_572_one_response": sum(record["recipient_id"] == "V32P-R000572" for record in responses) == 1,
        "recipient_627_one_response": sum(record["recipient_id"] == "V32P-R000627" for record in responses) == 1,
        "historical_failures_preserved": sum(record["status"] == "validation_failed" for record in manifests) == 2,
        "amendment_001_transition_present": sum(record["status"] == "validated_under_amendment" for record in manifests) == 1,
        "amendment_002_transition_present": sum(record["status"] == "validated_under_amendment_002" for record in manifests) == 1,
        "all_chunk_quality_gates_pass": all(report["status"] == "passed" for report in quality_reports.values()),
    }
    payloads = {}
    parse_errors = []
    for record in responses:
        try:
            _, payload = parse_preserved_response(record)
            payloads[record["recipient_id"]] = payload
        except Exception as exc:
            parse_errors.append({"request_id": record.get("request_id"), "error": str(exc)})
    mismatches = []
    reconciled = 0
    for recipient_id, payload in payloads.items():
        rows = assessment.loc[assessment.recipient_id.eq(recipient_id)].sort_values("days_since_transplant")
        for position, item in enumerate(payload["assessments"]):
            for field in QWEN_FIELDS:
                reconciled += 1
                if float(item[field]) != float(rows.iloc[position][field]):
                    mismatches.append({"recipient_id": recipient_id, "position": position, "field": field})
    checks["complete_raw_response_reconciliation"] = not parse_errors and reconciled == 30000 and not mismatches
    diagnostics = build_diagnostics(assessment, recipients, anchor)
    checks["no_identical_complete_trajectories"] = diagnostics["identical_complete_trajectories"]["identical_trajectory_groups"] == 0
    frozen_index = frozen.set_index("assessment_id")
    assessment_index = assessment.set_index("assessment_id")
    frozen_exact = True
    for column in [column for column in frozen.columns if column != "assessment_id" and column in assessment.columns]:
        expected = frozen_index.loc[assessment_index.index, column]
        actual = assessment_index[column]
        if pd.api.types.is_numeric_dtype(expected) and pd.api.types.is_numeric_dtype(actual):
            matches = np.allclose(actual.astype(float), expected.astype(float), rtol=0, atol=1e-12, equal_nan=True)
        else:
            matches = actual.fillna("<NA>").astype(str).equals(expected.fillna("<NA>").astype(str))
        frozen_exact = frozen_exact and bool(matches)
    checks["frozen_schedule_audit_and_derivations_exact"] = frozen_exact
    for column in ["infection_indicator", "previous_rejection", "acute_rejection_within_30_days"]:
        checks[f"{column}_exact"] = np.array_equal(
            assessment_index[column].astype(int), frozen_index.loc[assessment_index.index, column].astype(int)
        )
    recipient_donor = recipients.set_index("recipient_id").donor_id
    checks["donor_relationships_consistent"] = all(
        group.donor_id.nunique() == 1 and group.donor_id.iloc[0] == recipient_donor.loc[recipient_id]
        for recipient_id, group in assessment.groupby("recipient_id")
    )
    checks["shared_identity_records_consistent"] = all(len(group.drop_duplicates()) == 1 for _, group in identity.groupby("entity_id"))
    direct = set(context["field_lists"]["direct_identifiers"])
    checks["direct_identifiers_absent"] = not direct.intersection(assessment.columns)
    checks["joined_42_column_export_absent"] = assessment.shape[1] == 31
    aggregate_quality = anchor_quality_report(assessment, recipients, context["anchors_all"].iloc[:1000].copy(), "chunks_001_010")
    checks["aggregate_anchor_quality_passes"] = aggregate_quality["status"] == "passed"
    anchor_warning_counts = {
        field: sum(report["anchor_deviation_warning_counts"][field] for report in quality_reports.values())
        for field in ANCHOR_FIELD_MAP
    }
    failures = [name for name, value in checks.items() if not bool(value)]
    reconciliation = {
        "status": "passed" if not failures else "failed",
        "checks": {key: bool(value) for key, value in checks.items()},
        "failures": failures,
        "qwen_fields_reconciled": reconciled,
        "expected_qwen_fields": 30000,
        "mismatch_count": len(mismatches),
        "parse_errors": parse_errors[:20],
        "amendment_id": AMENDMENT_002_ID,
        "expected_manifest_records": 4002,
        "extra_manifest_record_explanation": "one amendment-validation transition each for recipients 572 and 627",
    }
    tokens = {
        key: sum(int(reports[number]["token_usage_total"][key]) for number in range(1, 11))
        for key in ["prompt_tokens", "completion_tokens", "total_tokens"]
    }
    elapsed = {f"chunk_{number:03d}": float(reports[number]["elapsed_seconds"]) for number in range(1, 11)}
    distribution = {
        **diagnostics,
        "status": "diagnostic_only_not_used_for_regeneration",
        "amendment_id": AMENDMENT_002_ID,
        "aggregate_anchor_generation_quality": aggregate_quality,
        "per_chunk_anchor_generation_quality": quality_reports,
        "anchor_deviation_warning_counts": anchor_warning_counts,
        "per_chunk_reports": per_chunk,
        "aggregate_token_usage": tokens,
        "per_chunk_elapsed_seconds": elapsed,
        "total_chunk_elapsed_seconds": sum(elapsed.values()),
    }
    integrity_checks = assert_amendment_002_integrity(context, completed_fingerprints)
    integrity = {
        "status": "passed", "checked_at_utc": now_utc(), "amendment_id": AMENDMENT_002_ID,
        "global_integrity_checks": integrity_checks,
        "chunk_fingerprints": {f"chunk_{number:03d}": tree_fingerprint(chunk_directory(number)) for number in range(1, 11)},
        "validation_amendment_001_fingerprint": tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_001"),
        "validation_amendment_002_fingerprint": tree_fingerprint(amendment_002_directory()),
        "chunks_011_100_started": False, "final_dataset_assembly_started": False,
    }
    directory.mkdir(parents=False, exist_ok=False)
    write_json_exclusive(directory / "cross_chunk_reconciliation.json", reconciliation)
    write_json_exclusive(directory / "cross_chunk_distribution_report.json", distribution)
    write_json_exclusive(directory / "cross_chunk_integrity_report.json", integrity)
    integrity_after = assert_amendment_002_integrity(context, completed_fingerprints)
    progress = {
        "run_id": RUN_ID, "checkpoint": "through_chunk_010", "status": "completed" if not failures else "failed",
        "amendment_id": AMENDMENT_002_ID, "created_at_utc": now_utc(), "chunks_present": list(range(1, 11)),
        "chunks_completed_by_resume": [7, 8, 9, 10],
        "new_api_requests_attempted": sum(report.get("new_api_requests_attempted", report.get("api_requests_attempted", 0)) for report in resume_reports.values()),
        "new_api_requests_completed": sum(report.get("new_api_requests_completed", report.get("api_requests_completed", 0)) for report in resume_reports.values()),
        "new_retries": 0, "aggregate_request_records": len(requests), "aggregate_response_records": len(responses),
        "aggregate_manifest_records": len(manifests), "expected_extra_manifest_records": 2,
        "aggregate_dimensions": list(assessment.shape), "aggregate_token_usage": tokens,
        "aggregate_chunk_elapsed_seconds": sum(elapsed.values()), "aggregate_checkpoint_elapsed_seconds": time.monotonic() - started,
        "target_count": diagnostics["target_count"], "target_prevalence": diagnostics["target_prevalence"],
        "event_recipients": diagnostics["event_recipients"], "non_event_recipients": diagnostics["non_event_recipients"],
        "anchor_deviation_warning_counts": anchor_warning_counts,
        "chunk_quality_gate_statuses": {key: value["status"] for key, value in quality_reports.items()},
        "reconciliation_status": reconciliation["status"], "reconciliation_failures": failures,
        "integrity_after_aggregate_checkpoint": integrity_after,
        "created_files": ["production_progress_report.json", "cross_chunk_reconciliation.json", "cross_chunk_distribution_report.json", "cross_chunk_integrity_report.json"],
        "progress_file_fingerprints_excluding_self": {path.name: file_fingerprint(path) for path in sorted(directory.iterdir()) if path.is_file()},
        "chunks_011_to_100_started": False, "final_dataset_assembly_started": False,
        "classifier_trained": False, "generator_tuned": False,
    }
    write_json_exclusive(directory / "production_progress_report.json", progress)
    assert not failures, failures
    return progress


def execute_amendment_002_and_resume():
    context = resume_002_preflight()
    _, amendment_summary, amendment_fingerprint = create_amendment_002(context)
    context["amendment_002_fingerprint"] = amendment_fingerprint
    recover_recipient_627(context)
    completed_fingerprints = dict(context["completed_fingerprints"])
    resume_reports = {}
    quality_reports = dict(context["existing_quality"])
    try:
        report7 = resume_chunk_007_under_amendment_002(context, completed_fingerprints)
        resume_reports[7] = report7
        if report7["status"] != "completed":
            raise RuntimeError("chunk 007 failed; stopped without retry")
        quality7 = report7["anchor_generation_quality"]
        quality_reports["chunk_007"] = quality7
        completed_fingerprints[7] = tree_fingerprint(chunk_directory(7))
        assert_amendment_002_integrity(context, completed_fingerprints)
        if quality7["status"] != "passed":
            raise RuntimeError("chunk 007 completed but failed generation-quality gate; stopped for manual review")

        total_new_requests = 73
        for number in range(8, 11):
            if total_new_requests + 100 > MAX_NEW_API_REQUESTS_002:
                raise RuntimeError("373-request hard cap would be exceeded")
            report = run_single_chunk(number, context, completed_fingerprints)
            if report["status"] != "completed":
                resume_reports[number] = report
                raise RuntimeError(f"chunk {number:03d} failed; stopped without retry")
            report, quality = enrich_completed_chunk_report_with_quality(number, context)
            resume_reports[number] = report
            quality_reports[f"chunk_{number:03d}"] = quality
            total_new_requests += 100
            completed_fingerprints[number] = tree_fingerprint(chunk_directory(number))
            assert_amendment_002_integrity(context, completed_fingerprints)
            if quality["status"] != "passed":
                raise RuntimeError(f"chunk {number:03d} completed but failed generation-quality gate; stopped for manual review")
        assert total_new_requests == MAX_NEW_API_REQUESTS_002
        assert set(completed_fingerprints) == set(range(1, 11))
        progress = aggregate_amendment_002_through_chunk_010(
            context, completed_fingerprints, resume_reports, quality_reports
        )
        return {
            "status": "completed",
            "amendment_summary": amendment_summary,
            "resume_reports": {
                str(number): {
                    "status": report["status"],
                    "new_api_requests": report.get("new_api_requests_completed", report.get("api_requests_completed")),
                    "token_usage": report.get("new_token_usage", report.get("token_usage_total")),
                    "anchor_quality_status": quality_reports[f"chunk_{number:03d}"]["status"],
                }
                for number, report in resume_reports.items()
            },
            "progress": progress,
        }
    finally:
        assert_amendment_002_integrity(context, completed_fingerprints)


AMENDMENT_002_RESUME_RESULT = execute_amendment_002_and_resume()
print(json.dumps({
    "status": AMENDMENT_002_RESUME_RESULT["status"],
    "amendment_id": AMENDMENT_002_ID,
    "resume_reports": AMENDMENT_002_RESUME_RESULT["resume_reports"],
    "aggregate_dimensions": AMENDMENT_002_RESUME_RESULT["progress"]["aggregate_dimensions"],
    "aggregate_manifest_records": AMENDMENT_002_RESUME_RESULT["progress"]["aggregate_manifest_records"],
    "anchor_deviation_warning_counts": AMENDMENT_002_RESUME_RESULT["progress"]["anchor_deviation_warning_counts"],
    "reconciliation_status": AMENDMENT_002_RESUME_RESULT["progress"]["reconciliation_status"],
    "chunks_011_to_100_started": False,
    "final_dataset_assembly_started": False,
}, indent=2))


## Guarded v3.2 production chunks 011–020 under amendment 002

This additive cell verifies the immutable chunks-001–010 checkpoint, both validation amendments, the frozen prompt/anchors, and the exact frozen recipient slice 1001–2000 before permitting API access. It runs only with `RUN_QWEN_V32_PRODUCTION_CHUNKS_011_020=QWEN-V32-PROD-001-CHUNKS-011-020-A002`, permits at most 1,000 sequential single-attempt requests, applies amendment-002 hard validation and per-chunk quality gates, and creates the progress-through-chunk-020 reports. A completed final chunk that fails a generation-quality gate is preserved and reported as `manual_review_required`; chunks 021–100, final assembly, and classifier training remain disabled.

In [ ]:
import json
import os
import time
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd


_repository_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists())
_notebook_path = _repository_root / "notebooks/KidneyTransplant/00_generate_and_validate_dataset.ipynb"
_notebook = nbformat.read(_notebook_path, as_version=4)
_amendment_002_source = next(
    cell.source
    for cell in _notebook.cells
    if cell.cell_type == "code" and "AMENDMENT_002_ID = \"PROD-VALIDATION-AMENDMENT-002\"" in cell.source
)
_amendment_002_primitives = _amendment_002_source.rsplit(
    "AMENDMENT_002_RESUME_RESULT = execute_amendment_002_and_resume()", 1
)[0]
exec(compile(_amendment_002_primitives, "amendment-002-primitives", "exec"), globals())


CHUNKS_011_020_GATE_NAME = "RUN_QWEN_V32_PRODUCTION_CHUNKS_011_020"
CHUNKS_011_020_GATE_VALUE = "QWEN-V32-PROD-001-CHUNKS-011-020-A002"
AUTHORIZED_CHUNKS = list(range(11, 21))
DISABLED_CHUNKS = list(range(21, 101))
EXPECTED_NEW_RECIPIENTS = 1_000
EXPECTED_NEW_ROWS = 6_000
MAX_NEW_API_REQUESTS = 1_000
PROGRESS_010_DIR_NAME = "progress_through_chunk_010"
PROGRESS_DIR_NAME = "progress_through_chunk_020"
MULTI_GATE_NAME = CHUNKS_011_020_GATE_NAME
MULTI_GATE_VALUE = CHUNKS_011_020_GATE_VALUE

PROGRESS_010_SHA256 = {
    "cross_chunk_distribution_report.json": "a3a5e155ff7401b36df9c777927fb1ce3e55858b50a164cb318c22855bcab21e",
    "cross_chunk_integrity_report.json": "d30cfe73cf7252fcce987fd2d71e8b76d68a6a517994d16c2d8517689778d815",
    "cross_chunk_reconciliation.json": "75b72faef6a31c8a68c0531e5f29da69b87179af6dd067050d3173f953fbd0e6",
    "production_progress_report.json": "2fa30da8b33b1d24fe66d996b014cfd0caba1228d1b51aca04de3847facdb73d",
}
AMENDMENT_002_FINAL_SHA256 = {
    "amendment_002_integrity_report.json": "644fec37198ffb2f2e7eb08d326ba6c5ad38980013ad5c9bfdb9093808e44773",
    "retrospective_anchor_diagnostics.csv": "2c655e3a6bdfd37c4173b26cdcf25ef57af2bca055a2aa19967b1a9bf733a6fc",
    "retrospective_anchor_diagnostics_summary.json": "2b53b3f2b16ae3016ef142b101abe2adda0eef92f21241d2ac36c334e774328e",
    "validation_policy_amendment_002.json": "34da9644240a6168bbe1374f36909796a3cba12c9f574ad404146156c296527e",
}
FROZEN_ANCHOR_METADATA_SHA256 = "d09d836685fe0451ece3015da53d23eceb00a186edd43214e64986dfbd934dd6"


def progress_010_directory():
    return PRODUCTION_ROOT / PROGRESS_010_DIR_NAME


def completed_report_path(number):
    directory = chunk_directory(number)
    if number == 6:
        return directory / "chunk_006_resume_completion_report.json"
    if number == 7:
        return directory / "chunk_007_resume_completion_report.json"
    return directory / f"chunk_{number:03d}_run_report.json"


def same_tree_fingerprint(left, right):
    keys = ["file_count", "total_size_bytes", "aggregate_sha256_with_metadata"]
    return all(left[key] == right[key] for key in keys)


def verify_completed_chunk_general(number, recorded_fingerprint):
    directory = chunk_directory(number)
    assert same_tree_fingerprint(tree_fingerprint(directory), recorded_fingerprint), f"chunk {number:03d} fingerprint mismatch"
    assessment = pd.read_csv(directory / f"assessment_chunk_{number:03d}.csv")
    checkpoint = pd.read_csv(directory / "validated_recipient_checkpoint.csv")
    pd.testing.assert_frame_equal(assessment, checkpoint, check_dtype=False)
    report = json.loads(completed_report_path(number).read_text(encoding="utf-8"))
    state = json.loads((directory / "completion_state.json").read_text(encoding="utf-8"))
    assert report["status"] == "completed"
    assert report.get("request_records", report.get("api_requests_completed")) == 100
    assert report.get("response_records", report.get("api_requests_completed")) == 100
    assert report["recipients_validated"] == 100 and report["checkpoint_rows"] == 600
    assert report.get("new_retries", report.get("retries")) == 0
    assert state["requests_completed"] == 100 and state["recipients_validated"] == 100
    assert state["checkpoint_rows"] == 600 and state["assessment_chunk_created"] is True
    assert assessment.shape == (600, 31) and assessment.recipient_id.nunique() == 100
    assert len(line_records(directory / "requests.jsonl")) == 100
    assert len(line_records(directory / "responses.jsonl")) == 100
    expected_manifest = 401 if number in [6, 7] else 400
    assert len(line_records(directory / "manifest.jsonl")) == expected_manifest
    return tree_fingerprint(directory)


def preflight_chunks_011_020():
    assert os.environ.get(CHUNKS_011_020_GATE_NAME) == CHUNKS_011_020_GATE_VALUE, "exact chunks-011-020 gate mismatch"
    for name in [
        GATE_NAME,
        "RUN_QWEN_V32_PRODUCTION_CHUNKS_002_010",
        "RESUME_QWEN_V32_PRODUCTION_CHUNKS_006_010",
        "RESUME_QWEN_V32_PRODUCTION_CHUNKS_007_010",
        "RUN_QWEN_V32_CONFIRM20",
        "RUN_CORRECTED_10",
        "RUN_100_RECIPIENTS",
        "RUN_10000_RECIPIENTS",
        "RUN_FULL_GENERATION",
    ]:
        assert not os.environ.get(name), f"{name} must be false"
    for number in range(1, 101):
        assert not os.environ.get(f"RUN_QWEN_V32_PRODUCTION_CHUNK_{number:03d}"), f"individual chunk {number} gate must be false"
    assert all(not chunk_directory(number).exists() for number in range(11, 101))
    assert not progress_directory().exists()
    assert not (PRODUCTION_ROOT / "assessment_production.csv").exists()
    assert not RUN_10000_RECIPIENTS and not RUN_FULL_GENERATION

    progress010 = progress_010_directory()
    progress010_fingerprint = validate_exact_hashes(progress010, PROGRESS_010_SHA256)
    progress010_report = json.loads((progress010 / "production_progress_report.json").read_text(encoding="utf-8"))
    reconciliation010 = json.loads((progress010 / "cross_chunk_reconciliation.json").read_text(encoding="utf-8"))
    integrity010 = json.loads((progress010 / "cross_chunk_integrity_report.json").read_text(encoding="utf-8"))
    distribution010 = json.loads((progress010 / "cross_chunk_distribution_report.json").read_text(encoding="utf-8"))
    assert progress010_report["status"] == "completed" and reconciliation010["status"] == "passed"
    assert integrity010["status"] == "passed" and not reconciliation010["failures"]
    assert progress010_report["aggregate_dimensions"] == [6000, 31]
    assert progress010_report["aggregate_request_records"] == progress010_report["aggregate_response_records"] == 1000
    assert progress010_report["aggregate_manifest_records"] == 4002
    assert all(progress010_report["chunk_quality_gate_statuses"][f"chunk_{number:03d}"] == "passed" for number in range(1, 11))

    amendment1_fingerprint = validate_exact_hashes(PRODUCTION_ROOT / "validation_amendment_001", AMENDMENT_001_SHA256)
    amendment2_fingerprint = validate_exact_hashes(PRODUCTION_ROOT / "validation_amendment_002", AMENDMENT_002_FINAL_SHA256)
    amendment2_report = json.loads((PRODUCTION_ROOT / "validation_amendment_002" / "validation_policy_amendment_002.json").read_text(encoding="utf-8"))
    assert amendment2_report["amendment_id"] == AMENDMENT_002_ID
    assert amendment2_report["policy_change"]["anchor_proximity"] == "diagnostic_anchor_deviation_warning_not_hard_record_validity"

    recorded_chunks = integrity010["chunk_fingerprints"]
    completed_fingerprints = {}
    for number in range(1, 11):
        recorded = recorded_chunks[f"chunk_{number:03d}"]
        completed_fingerprints[number] = verify_completed_chunk_general(number, recorded)
    assert same_tree_fingerprint(tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_001"), integrity010["validation_amendment_001_fingerprint"])
    assert same_tree_fingerprint(tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_002"), integrity010["validation_amendment_002_fingerprint"])

    all_assessments = pd.concat(
        [pd.read_csv(chunk_directory(number) / f"assessment_chunk_{number:03d}.csv") for number in range(1, 11)],
        ignore_index=True,
    )
    all_requests = []
    all_responses = []
    all_manifests = []
    for number in range(1, 11):
        all_requests.extend(line_records(chunk_directory(number) / "requests.jsonl"))
        all_responses.extend(line_records(chunk_directory(number) / "responses.jsonl"))
        all_manifests.extend(line_records(chunk_directory(number) / "manifest.jsonl"))
    assert all_assessments.shape == (6000, 31) and all_assessments.recipient_id.nunique() == 1000
    assert all_assessments.assessment_id.is_unique
    assert not all_assessments.duplicated(["recipient_id", "days_since_transplant"]).any()
    assert not all_assessments.duplicated().any()
    assert len(all_requests) == len(all_responses) == 1000
    assert len({record["request_id"] for record in all_requests}) == 1000
    assert len({record["request_id"] for record in all_responses}) == 1000
    assert {record["request_id"] for record in all_requests} == {record["request_id"] for record in all_responses}
    assert len(all_manifests) == 4002

    design_checks, design_validation = verify_frozen_production_design()
    assert all(design_checks.values())
    template = (V32_DESIGN / "qwen_kidney_v3_2_prompt_template.txt").read_text(encoding="utf-8")
    assert sha256_text(template) == PROMPT_SHA256
    assert sha256_file(PRODUCTION_DESIGN / "production_anchor_metadata.csv") == FROZEN_ANCHOR_METADATA_SHA256
    api_config = json.loads((V32_DESIGN / "api_configuration.json").read_text(encoding="utf-8"))
    assert api_config["client"]["max_retries"] == 0 and api_config["automatic_retry"] is False

    recipients_all = pd.read_csv(PRODUCTION_DESIGN / "production_recipient_metadata.csv")
    assessments_all = pd.read_csv(PRODUCTION_DESIGN / "production_assessment_metadata.csv")
    identities_all = pd.read_csv(PRODUCTION_DESIGN / "production_identity_skeleton.csv", keep_default_na=False)
    anchors_all = pd.read_csv(PRODUCTION_DESIGN / "production_anchor_metadata.csv")
    slices = {
        number: frozen_chunk_slice(number, recipients_all, assessments_all, identities_all, anchors_all)
        for number in range(1, 21)
    }
    for slice_context in slices.values():
        verify_frozen_slice(slice_context)
    new_recipients = pd.concat([slices[number]["recipients"] for number in AUTHORIZED_CHUNKS], ignore_index=True)
    new_assessments = pd.concat([slices[number]["assessments"] for number in AUTHORIZED_CHUNKS], ignore_index=True)
    assert len(new_recipients) == EXPECTED_NEW_RECIPIENTS and new_recipients.recipient_id.nunique() == EXPECTED_NEW_RECIPIENTS
    assert new_recipients.request_sequence_number.tolist() == list(range(1001, 2001))
    assert new_recipients.recipient_id.tolist() == [f"V32P-R{number:06d}" for number in range(1001, 2001)]
    assert len(new_assessments) == EXPECTED_NEW_ROWS and new_assessments.recipient_id.nunique() == EXPECTED_NEW_RECIPIENTS
    assert new_assessments.assessment_id.is_unique
    assert set(new_recipients.recipient_id).isdisjoint(all_assessments.recipient_id)

    field_lists = json.loads((V31_DESIGN / "field_lists.json").read_text(encoding="utf-8"))
    canonical_ranges = json.loads((V31_DESIGN / "canonical_qwen_ranges.json").read_text(encoding="utf-8"))
    assessment_columns = pd.read_csv(V31_DESIGN / "assessment_table_schema.csv").field_name.tolist()
    immutable_paths = immutable_inputs()
    previous_production_paths = (
        [chunk_directory(number) for number in range(1, 11)]
        + [PRODUCTION_ROOT / "validation_amendment_001", PRODUCTION_ROOT / "validation_amendment_002", progress010]
    )
    context = {
        "checks": {
            "progress_through_chunk_010_verified": True,
            "existing_1000_recipients_6000_rows": all_assessments.shape == (6000, 31),
            "existing_1000_unique_requests_responses": len(all_requests) == len(all_responses) == 1000,
            "existing_manifest_4002": len(all_manifests) == 4002,
            "amendments_001_002_verified": True,
            "frozen_prompt_hash_verified": sha256_text(template) == PROMPT_SHA256,
            "frozen_anchor_hash_verified": sha256_file(PRODUCTION_DESIGN / "production_anchor_metadata.csv") == FROZEN_ANCHOR_METADATA_SHA256,
            "next_1000_frozen_recipients_verified": new_recipients.request_sequence_number.tolist() == list(range(1001, 2001)),
            "chunks_011_020_absent": all(not chunk_directory(number).exists() for number in AUTHORIZED_CHUNKS),
            "chunks_021_100_disabled": all(not chunk_directory(number).exists() for number in DISABLED_CHUNKS),
            "maximum_new_requests_1000": MAX_NEW_API_REQUESTS == 1000,
            "automatic_retries_disabled": api_config["client"]["max_retries"] == 0,
            "final_assembly_disabled": True,
        },
        "design_validation": design_validation,
        "template": template,
        "api_config": api_config,
        "slices": slices,
        "recipients_all": recipients_all,
        "assessments_all": assessments_all,
        "identities_all": identities_all,
        "anchors_all": anchors_all,
        "field_lists": field_lists,
        "canonical_ranges": canonical_ranges,
        "assessment_columns": assessment_columns,
        "existing_quality": distribution010["per_chunk_anchor_generation_quality"],
        "completed_fingerprints": completed_fingerprints,
        "immutable_paths": immutable_paths,
        "immutable_before": snapshot_immutable(immutable_paths),
        "notebooks_before": snapshot_notebooks_01_05(),
        "protected_before": protected_fingerprint(),
        "production_design_before": tree_fingerprint(PRODUCTION_DESIGN),
        "previous_production_paths": previous_production_paths,
        "previous_production_before": {display_path(path): tree_fingerprint(path) for path in previous_production_paths},
        "progress_010_fingerprint": progress010_fingerprint,
        "amendment_001_fingerprint": amendment1_fingerprint,
        "amendment_002_fingerprint": amendment2_fingerprint,
    }
    assert {key: context["protected_before"][key] for key in EXPECTED_PROTECTED} == EXPECTED_PROTECTED
    assert all(context["checks"].values())
    return context


def assert_chunks_011_020_integrity(context, completed_fingerprints=None):
    result = {
        "protected_unchanged": protected_fingerprint() == context["protected_before"],
        "previous_evidence_unchanged": snapshot_immutable(context["immutable_paths"]) == context["immutable_before"],
        "notebooks_01_05_unchanged": snapshot_notebooks_01_05() == context["notebooks_before"],
        "production_design_unchanged": tree_fingerprint(PRODUCTION_DESIGN) == context["production_design_before"],
        "completed_chunks_001_010_and_progress_010_unchanged": all(
            tree_fingerprint(path) == context["previous_production_before"][display_path(path)]
            for path in context["previous_production_paths"]
        ),
        "validation_amendment_001_unchanged": tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_001") == context["amendment_001_fingerprint"],
        "validation_amendment_002_unchanged": tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_002") == context["amendment_002_fingerprint"],
        "progress_through_chunk_010_unchanged": tree_fingerprint(progress_010_directory()) == context["progress_010_fingerprint"],
        "chunks_021_100_absent": all(not chunk_directory(number).exists() for number in DISABLED_CHUNKS),
        "final_assembly_absent": not (PRODUCTION_ROOT / "assessment_production.csv").exists(),
    }
    for number, fingerprint in (completed_fingerprints or {}).items():
        result[f"chunk_{number:03d}_unchanged"] = tree_fingerprint(chunk_directory(number)) == fingerprint
    assert all(result.values()), [name for name, passed in result.items() if not passed]
    return result


assert_global_integrity = assert_chunks_011_020_integrity


def report_and_quality_for_chunk(number, context):
    report = json.loads(completed_report_path(number).read_text(encoding="utf-8"))
    if number <= 10:
        quality = context["existing_quality"][f"chunk_{number:03d}"]
    else:
        quality = report["anchor_generation_quality"]
    return report, quality


def aggregate_through_chunk_020(context, completed_fingerprints, execution_reports, quality_reports):
    started = time.monotonic()
    directory = progress_directory()
    assert not directory.exists()
    assessment_frames = []
    identity_frames = []
    anchor_frames = []
    requests = []
    responses = []
    manifests = []
    reports = {}
    per_chunk = {}
    target_by_chunk = {}
    for number in range(1, 21):
        suffix = f"{number:03d}"
        chunk = chunk_directory(number)
        assessment = pd.read_csv(chunk / f"assessment_chunk_{suffix}.csv")
        assessment_frames.append(assessment)
        identity_frames.append(pd.read_csv(chunk / f"identity_chunk_{suffix}.csv", keep_default_na=False))
        anchor_frames.append(pd.read_csv(chunk / f"anchor_audit_chunk_{suffix}.csv"))
        requests.extend(line_records(chunk / "requests.jsonl"))
        responses.extend(line_records(chunk / "responses.jsonl"))
        manifests.extend(line_records(chunk / "manifest.jsonl"))
        report, quality = report_and_quality_for_chunk(number, context)
        reports[number] = report
        target_by_chunk[suffix] = {
            "positive_assessments": int(assessment.acute_rejection_within_30_days.sum()),
            "assessment_rows": int(len(assessment)),
            "prevalence": float(assessment.acute_rejection_within_30_days.mean()),
        }
        per_chunk[suffix] = {
            "status": report["status"],
            "target": target_by_chunk[suffix],
            "anchor_generation_quality": quality,
            "soft_warnings": report.get("soft_warnings", []),
            "failures": report.get("new_failures", report.get("failures", 0)),
            "retries": report.get("new_retries", report.get("retries", 0)),
        }
    assessment = pd.concat(assessment_frames, ignore_index=True)
    identity = pd.concat(identity_frames, ignore_index=True)
    anchor = pd.concat(anchor_frames, ignore_index=True)
    recipients = pd.concat([context["slices"][number]["recipients"] for number in range(1, 21)], ignore_index=True)
    frozen = pd.concat([context["slices"][number]["assessments"] for number in range(1, 21)], ignore_index=True)
    request_ids = [record["request_id"] for record in requests]
    response_ids = [record["request_id"] for record in responses]
    checks = {
        "twenty_completed_chunks": all(reports[number]["status"] == "completed" for number in range(1, 21)),
        "recipients_2000": assessment.recipient_id.nunique() == 2000,
        "rows_12000": assessment.shape == (12000, 31),
        "assessment_ids_unique": assessment.assessment_id.is_unique,
        "recipient_day_unique": not assessment.duplicated(["recipient_id", "days_since_transplant"]).any(),
        "no_duplicate_rows": not assessment.duplicated().any(),
        "six_rows_each": assessment.groupby("recipient_id").size().eq(6).all(),
        "ordered_days": assessment.groupby("recipient_id").days_since_transplant.apply(list).map(lambda values: values == DAYS).all(),
        "request_records_2000": len(requests) == 2000,
        "response_records_2000": len(responses) == 2000,
        "request_ids_unique": len(set(request_ids)) == 2000,
        "response_ids_unique": len(set(response_ids)) == 2000,
        "request_response_ids_equal": set(request_ids) == set(response_ids),
        "manifest_records_8002": len(manifests) == 8002,
        "historical_failures_remain_two": sum(record["status"] == "validation_failed" for record in manifests) == 2,
        "amendment_001_transition_present": sum(record["status"] == "validated_under_amendment" for record in manifests) == 1,
        "amendment_002_transition_present": sum(record["status"] == "validated_under_amendment_002" for record in manifests) == 1,
        "all_chunk_quality_gates_pass": all(report["status"] == "passed" for report in quality_reports.values()),
        "all_chunk_hard_validations_pass": all(
            reports[number].get("validation", {}).get("status", "passed") == "passed" for number in range(1, 21)
        ),
    }
    payloads = {}
    parse_errors = []
    for record in responses:
        try:
            _, payload = parse_preserved_response(record)
            payloads[record["recipient_id"]] = payload
        except Exception as exc:
            parse_errors.append({"request_id": record.get("request_id"), "error": str(exc)})
    mismatches = []
    reconciled = 0
    for recipient_id, payload in payloads.items():
        rows = assessment.loc[assessment.recipient_id.eq(recipient_id)].sort_values("days_since_transplant")
        for position, item in enumerate(payload["assessments"]):
            for field in QWEN_FIELDS:
                reconciled += 1
                if float(item[field]) != float(rows.iloc[position][field]):
                    mismatches.append({"recipient_id": recipient_id, "position": position, "field": field})
    checks["complete_raw_response_reconciliation"] = not parse_errors and reconciled == 60000 and not mismatches
    diagnostics = build_diagnostics(assessment, recipients, anchor)
    checks["no_identical_complete_trajectories"] = diagnostics["identical_complete_trajectories"]["identical_trajectory_groups"] == 0
    frozen_index = frozen.set_index("assessment_id")
    assessment_index = assessment.set_index("assessment_id")
    frozen_exact = True
    for column in [column for column in frozen.columns if column != "assessment_id" and column in assessment.columns]:
        expected = frozen_index.loc[assessment_index.index, column]
        actual = assessment_index[column]
        if pd.api.types.is_numeric_dtype(expected) and pd.api.types.is_numeric_dtype(actual):
            matches = np.allclose(actual.astype(float), expected.astype(float), rtol=0, atol=1e-12, equal_nan=True)
        else:
            matches = actual.fillna("<NA>").astype(str).equals(expected.fillna("<NA>").astype(str))
        frozen_exact = frozen_exact and bool(matches)
    checks["frozen_schedule_audit_and_derivations_exact"] = frozen_exact
    for column in ["infection_indicator", "previous_rejection", "acute_rejection_within_30_days"]:
        checks[f"{column}_exact"] = np.array_equal(
            assessment_index[column].astype(int), frozen_index.loc[assessment_index.index, column].astype(int)
        )
    recipient_donor = recipients.set_index("recipient_id").donor_id
    checks["donor_relationships_consistent"] = all(
        group.donor_id.nunique() == 1 and group.donor_id.iloc[0] == recipient_donor.loc[recipient_id]
        for recipient_id, group in assessment.groupby("recipient_id")
    )
    checks["shared_identity_records_consistent"] = all(len(group.drop_duplicates()) == 1 for _, group in identity.groupby("entity_id"))
    direct = set(context["field_lists"]["direct_identifiers"])
    checks["direct_identifiers_absent"] = not direct.intersection(assessment.columns)
    checks["joined_42_column_export_absent"] = assessment.shape[1] == 31
    aggregate_quality = anchor_quality_report(
        assessment, recipients, context["anchors_all"].iloc[:2000].copy(), "chunks_001_020"
    )
    checks["aggregate_anchor_quality_passes"] = aggregate_quality["status"] == "passed"
    anchor_warning_counts = {
        field: sum(report["anchor_deviation_warning_counts"][field] for report in quality_reports.values())
        for field in ANCHOR_FIELD_MAP
    }
    quality_check_names = {"all_chunk_quality_gates_pass", "aggregate_anchor_quality_passes"}
    quality_gate_failures = [name for name in quality_check_names if not bool(checks[name])]
    hard_failures = [
        name for name, value in checks.items()
        if not bool(value) and name not in quality_check_names
    ]
    overall_status = (
        "failed" if hard_failures
        else "manual_review_required" if quality_gate_failures
        else "completed"
    )
    reconciliation = {
        "status": "passed" if not hard_failures else "failed",
        "overall_checkpoint_status": overall_status,
        "checks": {key: bool(value) for key, value in checks.items()},
        "failures": hard_failures,
        "hard_failures": hard_failures,
        "quality_gate_failures": quality_gate_failures,
        "qwen_fields_reconciled": reconciled,
        "expected_qwen_fields": 60000,
        "mismatch_count": len(mismatches),
        "parse_errors": parse_errors[:20],
        "amendment_id": AMENDMENT_002_ID,
        "expected_manifest_records": 8002,
        "extra_manifest_record_explanation": "the two preserved amendment transitions for recipients 572 and 627",
    }
    cumulative_tokens = {
        key: sum(int(reports[number]["token_usage_total"][key]) for number in range(1, 21))
        for key in ["prompt_tokens", "completion_tokens", "total_tokens"]
    }
    new_tokens = {
        key: sum(int(execution_reports[number]["token_usage_total"][key]) for number in AUTHORIZED_CHUNKS)
        for key in ["prompt_tokens", "completion_tokens", "total_tokens"]
    }
    elapsed = {f"chunk_{number:03d}": float(reports[number]["elapsed_seconds"]) for number in range(1, 21)}
    new_elapsed = sum(float(execution_reports[number]["elapsed_seconds"]) for number in AUTHORIZED_CHUNKS)
    distribution = {
        **diagnostics,
        "status": "diagnostic_only_not_used_for_regeneration",
        "amendment_id": AMENDMENT_002_ID,
        "aggregate_anchor_generation_quality": aggregate_quality,
        "per_chunk_anchor_generation_quality": quality_reports,
        "anchor_deviation_warning_counts": anchor_warning_counts,
        "per_chunk_target_prevalence": target_by_chunk,
        "per_chunk_reports": per_chunk,
        "new_token_usage_chunks_011_020": new_tokens,
        "cumulative_token_usage": cumulative_tokens,
        "per_chunk_elapsed_seconds": elapsed,
        "new_chunk_elapsed_seconds": new_elapsed,
        "cumulative_chunk_elapsed_seconds": sum(elapsed.values()),
    }
    integrity_checks = assert_chunks_011_020_integrity(context, completed_fingerprints)
    integrity = {
        "status": "passed", "checked_at_utc": now_utc(), "amendment_id": AMENDMENT_002_ID,
        "global_integrity_checks": integrity_checks,
        "chunk_fingerprints": {f"chunk_{number:03d}": tree_fingerprint(chunk_directory(number)) for number in range(1, 21)},
        "validation_amendment_001_fingerprint": tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_001"),
        "validation_amendment_002_fingerprint": tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_002"),
        "progress_through_chunk_010_fingerprint": tree_fingerprint(progress_010_directory()),
        "chunks_021_100_started": False, "final_dataset_assembly_started": False,
    }
    directory.mkdir(parents=False, exist_ok=False)
    write_json_exclusive(directory / "cross_chunk_reconciliation.json", reconciliation)
    write_json_exclusive(directory / "cross_chunk_distribution_report.json", distribution)
    write_json_exclusive(directory / "cross_chunk_integrity_report.json", integrity)
    integrity_after = assert_chunks_011_020_integrity(context, completed_fingerprints)
    progress = {
        "run_id": RUN_ID, "checkpoint": "through_chunk_020", "status": overall_status,
        "amendment_id": AMENDMENT_002_ID, "created_at_utc": now_utc(), "chunks_present": list(range(1, 21)),
        "chunks_completed_this_execution": AUTHORIZED_CHUNKS,
        "new_api_requests_attempted": sum(report["api_requests_attempted"] for report in execution_reports.values()),
        "new_api_requests_completed": sum(report["api_requests_completed"] for report in execution_reports.values()),
        "new_retries": sum(report["retries"] for report in execution_reports.values()),
        "new_token_usage": new_tokens, "cumulative_token_usage": cumulative_tokens,
        "new_chunk_elapsed_seconds": new_elapsed, "cumulative_chunk_elapsed_seconds": sum(elapsed.values()),
        "aggregate_request_records": len(requests), "aggregate_response_records": len(responses),
        "aggregate_manifest_records": len(manifests), "expected_extra_manifest_records": 2,
        "aggregate_dimensions": list(assessment.shape),
        "target_count": diagnostics["target_count"], "target_prevalence": diagnostics["target_prevalence"],
        "per_chunk_target_prevalence": target_by_chunk,
        "event_recipients": diagnostics["event_recipients"], "non_event_recipients": diagnostics["non_event_recipients"],
        "anchor_deviation_warning_counts": anchor_warning_counts,
        "chunk_quality_gate_statuses": {key: value["status"] for key, value in quality_reports.items()},
        "repeated_value_diagnostics": aggregate_quality["later_day_repetition_warnings"],
        "reconciliation_status": reconciliation["status"], "reconciliation_failures": hard_failures,
        "quality_gate_failures": quality_gate_failures,
        "integrity_after_aggregate_checkpoint": integrity_after,
        "created_files": ["production_progress_report.json", "cross_chunk_reconciliation.json", "cross_chunk_distribution_report.json", "cross_chunk_integrity_report.json"],
        "progress_file_fingerprints_excluding_self": {path.name: file_fingerprint(path) for path in sorted(directory.iterdir()) if path.is_file()},
        "chunks_021_to_100_started": False, "final_dataset_assembly_started": False,
        "classifier_trained": False, "generator_tuned": False,
    }
    write_json_exclusive(directory / "production_progress_report.json", progress)
    return progress


def execute_chunks_011_020():
    context = preflight_chunks_011_020()
    completed_fingerprints = dict(context["completed_fingerprints"])
    execution_reports = {}
    quality_reports = dict(context["existing_quality"])
    total_new_requests = 0
    final_chunk_quality_failure = None
    try:
        for number in AUTHORIZED_CHUNKS:
            if total_new_requests + 100 > MAX_NEW_API_REQUESTS:
                raise RuntimeError("1,000-request hard cap would be exceeded")
            report = run_single_chunk(number, context, completed_fingerprints)
            if report["status"] != "completed":
                execution_reports[number] = report
                raise RuntimeError(f"chunk {number:03d} failed; stopped without retry")
            report, quality = enrich_completed_chunk_report_with_quality(number, context)
            execution_reports[number] = report
            quality_reports[f"chunk_{number:03d}"] = quality
            total_new_requests += report["api_requests_completed"]
            completed_fingerprints[number] = tree_fingerprint(chunk_directory(number))
            assert_chunks_011_020_integrity(context, completed_fingerprints)
            if quality["status"] != "passed":
                if number < AUTHORIZED_CHUNKS[-1]:
                    raise RuntimeError(f"chunk {number:03d} completed but failed generation-quality gate; stopped for manual review")
                final_chunk_quality_failure = {
                    "chunk": number,
                    "status": quality["status"],
                    "failed_gates": quality["failed_gates"],
                    "action": "chunk preserved; no later chunk started; manual review required",
                }
        assert total_new_requests == MAX_NEW_API_REQUESTS
        assert set(completed_fingerprints) == set(range(1, 21))
        progress = aggregate_through_chunk_020(
            context, completed_fingerprints, execution_reports, quality_reports
        )
        return {
            "status": "manual_review_required" if final_chunk_quality_failure else "completed",
            "quality_gate_stop": final_chunk_quality_failure,
            "execution_reports": {
                str(number): {
                    "status": report["status"],
                    "api_requests": report["api_requests_completed"],
                    "token_usage": report["token_usage_total"],
                    "quality_status": quality_reports[f"chunk_{number:03d}"]["status"],
                }
                for number, report in execution_reports.items()
            },
            "progress": progress,
        }
    finally:
        assert_chunks_011_020_integrity(context, completed_fingerprints)


CHUNKS_011_020_RESULT = execute_chunks_011_020()
print(json.dumps({
    "status": CHUNKS_011_020_RESULT["status"],
    "execution_reports": CHUNKS_011_020_RESULT["execution_reports"],
    "aggregate_dimensions": CHUNKS_011_020_RESULT["progress"]["aggregate_dimensions"],
    "aggregate_manifest_records": CHUNKS_011_020_RESULT["progress"]["aggregate_manifest_records"],
    "anchor_deviation_warning_counts": CHUNKS_011_020_RESULT["progress"]["anchor_deviation_warning_counts"],
    "reconciliation_status": CHUNKS_011_020_RESULT["progress"]["reconciliation_status"],
    "chunks_021_to_100_started": False,
    "final_dataset_assembly_started": False,
}, indent=2))


## Chunk 020 manual quality review and guarded v3.2 production chunks 021–030

This additive cell records manual quality review 001 with zero API requests before preflight, applies the identifier-order policy interpretation, and executes only under the exact chunks-021–030 gate.

In [ ]:
import json
import os
import time
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd


_repository_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists())
_notebook_path = _repository_root / "notebooks/KidneyTransplant/00_generate_and_validate_dataset.ipynb"
_notebook = nbformat.read(_notebook_path, as_version=4)
_chunks_011_020_source = next(
    cell.source
    for cell in _notebook.cells
    if cell.cell_type == "code" and "CHUNKS_011_020_GATE_NAME =" in cell.source
)
_chunks_011_020_primitives = _chunks_011_020_source.rsplit(
    "CHUNKS_011_020_RESULT = execute_chunks_011_020()", 1
)[0]
exec(compile(_chunks_011_020_primitives, "chunks-011-020-primitives", "exec"), globals())


CHUNKS_021_030_GATE_NAME = "RUN_QWEN_V32_PRODUCTION_CHUNKS_021_030"
CHUNKS_021_030_GATE_VALUE = "QWEN-V32-PROD-001-CHUNKS-021-030-A002-QR001"
MANUAL_REVIEW_ID = "QWEN-V32-PROD-001-MANUAL-QUALITY-REVIEW-001"
MANUAL_REVIEW_DIR_NAME = "manual_quality_review_001"
PROGRESS_020_DIR_NAME = "progress_through_chunk_020"
PROGRESS_DIR_NAME = "progress_through_chunk_030"
AUTHORIZED_CHUNKS = list(range(21, 31))
DISABLED_CHUNKS = list(range(31, 101))
EXPECTED_NEW_RECIPIENTS = 1_000
EXPECTED_NEW_ROWS = 6_000
MAX_NEW_API_REQUESTS = 1_000
MULTI_GATE_NAME = CHUNKS_021_030_GATE_NAME
MULTI_GATE_VALUE = CHUNKS_021_030_GATE_VALUE

PROGRESS_020_SHA256 = {
    "cross_chunk_distribution_report.json": "1fd79a929e0d1cd0791348e5c83cd8c8435ce71bd638bf6b4e730bf029b3c53c",
    "cross_chunk_integrity_report.json": "52b572facc23c10209b9a3c8802e9bc02ddb99c8d2a6175d0d456b8b7fb8b0bb",
    "cross_chunk_reconciliation.json": "fb54c12f42dd2bf5b7b5c569ce312d668089f7b3483adbe66d708740b9dc176f",
    "production_progress_report.json": "c18effe7ac3c12b2ba30ad48d6db98f746074a1dbb20290e41c6eda6931a71de",
}


def progress_020_directory():
    return PRODUCTION_ROOT / PROGRESS_020_DIR_NAME


def manual_review_directory():
    return PRODUCTION_ROOT / MANUAL_REVIEW_DIR_NAME


def recipient_suffix(values):
    return values.astype(str).str.extract(r"(\d+)$")[0].astype(int)


def identifier_correlations(assessment, anchors):
    day7 = assessment.loc[assessment.days_since_transplant.eq(7)].set_index("recipient_id")
    anchor_index = anchors.set_index("recipient_id")
    suffix = recipient_suffix(pd.Series(anchor_index.index, index=anchor_index.index))
    result = {}
    for field, anchor_field in ANCHOR_FIELD_MAP.items():
        result[field] = {
            "anchor_recipient_order_correlation": float(anchor_index[anchor_field].astype(float).corr(suffix)),
            "qwen_output_recipient_order_correlation": float(day7.loc[anchor_index.index, field].astype(float).corr(suffix)),
            "anchor_output_correlation": float(
                anchor_index[anchor_field].astype(float).corr(day7.loc[anchor_index.index, field].astype(float))
            ),
        }
    return result


def frozen_full_anchor_correlations(anchors_all):
    suffix = recipient_suffix(anchors_all.recipient_id)
    return {
        field: float(anchors_all[anchor_field].astype(float).corr(suffix))
        for field, anchor_field in ANCHOR_FIELD_MAP.items()
    }


def manual_review_inputs():
    progress020 = progress_020_directory()
    distribution = json.loads((progress020 / "cross_chunk_distribution_report.json").read_text(encoding="utf-8"))
    reconciliation = json.loads((progress020 / "cross_chunk_reconciliation.json").read_text(encoding="utf-8"))
    progress = json.loads((progress020 / "production_progress_report.json").read_text(encoding="utf-8"))
    chunk20_quality = distribution["per_chunk_anchor_generation_quality"]["chunk_020"]
    cumulative_quality = distribution["aggregate_anchor_generation_quality"]
    anchors_all = pd.read_csv(PRODUCTION_DESIGN / "production_anchor_metadata.csv")
    full_correlations = frozen_full_anchor_correlations(anchors_all)
    assert progress["aggregate_dimensions"] == [12000, 31]
    assert progress["aggregate_request_records"] == progress["aggregate_response_records"] == 2000
    assert progress["aggregate_manifest_records"] == 8002
    assert reconciliation["status"] == "passed" and not reconciliation["hard_failures"]
    urine_chunk = chunk20_quality["anchor_output_and_recipient_suffix_correlations"]["urine_output_ml_24h"]
    urine_cumulative = cumulative_quality["anchor_output_and_recipient_suffix_correlations"]["urine_output_ml_24h"]
    assert round(float(urine_chunk["anchor_recipient_suffix_correlation"]), 4) == -0.2243
    assert round(float(urine_cumulative["anchor_recipient_suffix_correlation"]), 4) == -0.0305
    assert all(
        float(values["anchor_output_correlation"]) > 0.99
        for values in chunk20_quality["anchor_output_and_recipient_suffix_correlations"].values()
    )
    assert chunk20_quality["gates"]["day7_urine_unique_values_at_least_30"]
    assert chunk20_quality["gates"]["day7_adherence_unique_values_at_least_30"]
    assert chunk20_quality["gates"]["day7_urine_dominant_fraction_at_most_0_20"]
    assert chunk20_quality["gates"]["day7_adherence_dominant_fraction_at_most_0_20"]
    assert chunk20_quality["identical_complete_trajectories"]["identical_trajectory_groups"] == 0
    assert min(full_correlations.values()) >= -0.0118 - 1e-4
    assert max(full_correlations.values()) <= -0.0024 + 1e-4
    return progress, reconciliation, chunk20_quality, cumulative_quality, full_correlations


def create_or_validate_manual_review():
    directory = manual_review_directory()
    expected_files = {
        "chunk_020_manual_review.json",
        "identifier_order_quality_policy.json",
        "manual_review_integrity_report.json",
    }
    if directory.exists():
        assert {path.name for path in directory.iterdir() if path.is_file()} == expected_files
        review = json.loads((directory / "chunk_020_manual_review.json").read_text(encoding="utf-8"))
        policy = json.loads((directory / "identifier_order_quality_policy.json").read_text(encoding="utf-8"))
        integrity = json.loads((directory / "manual_review_integrity_report.json").read_text(encoding="utf-8"))
        assert review["decision"] == "accepted_without_regeneration"
        assert review["api_requests_attempted"] == 0
        assert policy["isolated_per_chunk_absolute_correlation_above_0_20"] == "diagnostic_warning"
        assert integrity["status"] == "passed" and integrity["api_requests_attempted"] == 0
        recorded_review_files = integrity["review_file_fingerprints_excluding_self"]
        for name in ["chunk_020_manual_review.json", "identifier_order_quality_policy.json"]:
            actual = file_fingerprint(directory / name)
            assert all(actual[key] == recorded_review_files[name][key] for key in ["sha256", "mtime_ns", "size_bytes"])
        return tree_fingerprint(directory)

    input_paths = {
        "chunk_020": chunk_directory(20),
        "progress_through_chunk_020": progress_020_directory(),
        "validation_amendment_001": PRODUCTION_ROOT / "validation_amendment_001",
        "validation_amendment_002": PRODUCTION_ROOT / "validation_amendment_002",
        "frozen_production_design": PRODUCTION_DESIGN,
    }
    input_before = {name: tree_fingerprint(path) for name, path in input_paths.items()}
    protected_before = protected_fingerprint()
    progress, reconciliation, chunk20_quality, cumulative_quality, full_correlations = manual_review_inputs()
    chunk20_correlations = chunk20_quality["anchor_output_and_recipient_suffix_correlations"]
    cumulative_correlations = cumulative_quality["anchor_output_and_recipient_suffix_correlations"]
    directory.mkdir(parents=False, exist_ok=False)
    review = {
        "review_id": MANUAL_REVIEW_ID,
        "run_id": RUN_ID,
        "created_at_utc": now_utc(),
        "status": "completed",
        "api_requests_attempted": 0,
        "api_requests_completed": 0,
        "chunk_reviewed": 20,
        "chunk_sample_size_recipients": 100,
        "observations": {
            "chunk_020_frozen_urine_anchor_recipient_order_correlation": float(
                chunk20_correlations["urine_output_ml_24h"]["anchor_recipient_suffix_correlation"]
            ),
            "chunk_020_frozen_urine_anchor_recipient_order_correlation_rounded": -0.2243,
            "cumulative_through_2000_urine_anchor_recipient_order_correlation": float(
                cumulative_correlations["urine_output_ml_24h"]["anchor_recipient_suffix_correlation"]
            ),
            "cumulative_through_2000_correlation_rounded": -0.0305,
            "full_frozen_10000_recipient_anchor_correlations": full_correlations,
            "full_frozen_anchor_correlation_range": {
                "minimum": float(min(full_correlations.values())),
                "maximum": float(max(full_correlations.values())),
                "rounded_statement": "between -0.0118 and -0.0024",
            },
            "chunk_020_anchor_output_correlations": {
                field: float(values["anchor_output_correlation"])
                for field, values in chunk20_correlations.items()
            },
            "all_chunk_020_anchor_output_correlations_exceed_0_99": True,
            "day_7_diversity_passed": True,
            "identical_trajectory_groups": 0,
            "hard_validation_failures": [],
            "leakage_failures": [],
        },
        "decision": "accepted_without_regeneration",
        "decision_reason": (
            "The isolated 100-recipient chunk correlation is not supported by the cumulative 2,000-recipient "
            "or full frozen 10,000-recipient skeleton evidence; all hard validation, diversity, trajectory and "
            "leakage checks passed."
        ),
        "generator_changed": False,
        "prompt_changed": False,
        "anchors_changed": False,
        "targets_changed": False,
    }
    policy = {
        "policy_id": "QWEN-V32-IDENTIFIER-ORDER-QUALITY-POLICY-001",
        "review_id": MANUAL_REVIEW_ID,
        "created_at_utc": now_utc(),
        "scope": "future frozen v3.2 production chunks",
        "policy_type": "quality_gate_interpretation_only",
        "continue_reporting": [
            "anchor_recipient_numeric_order_correlations",
            "qwen_output_recipient_numeric_order_correlations",
        ],
        "isolated_per_chunk_absolute_correlation_above_0_20": "diagnostic_warning",
        "isolated_per_chunk_action": "do_not_reject_or_regenerate_otherwise_valid_recipients",
        "authoritative_checks": [
            "full_frozen_10000_recipient_anchor_correlations",
            "cumulative_generated_data_correlations",
        ],
        "cumulative_absolute_correlation_threshold": 0.20,
        "stop_before_next_chunk_when": [
            "any_cumulative_absolute_identifier_order_correlation_is_at_least_0.20",
            "additional_evidence_of_systematic_identifier_dependence_exists",
        ],
        "unchanged_components": ["generator", "prompt", "anchors", "targets", "data"],
        "api_requests_attempted": 0,
    }
    write_json_exclusive(directory / "chunk_020_manual_review.json", review)
    write_json_exclusive(directory / "identifier_order_quality_policy.json", policy)
    input_after = {name: tree_fingerprint(path) for name, path in input_paths.items()}
    protected_after = protected_fingerprint()
    assert input_after == input_before and protected_after == protected_before
    integrity = {
        "review_id": MANUAL_REVIEW_ID,
        "status": "passed",
        "created_at_utc": now_utc(),
        "api_requests_attempted": 0,
        "input_evidence_fingerprints": input_before,
        "protected_before": protected_before,
        "protected_after": protected_after,
        "all_input_evidence_unchanged": True,
        "review_file_fingerprints_excluding_self": {
            name: file_fingerprint(directory / name)
            for name in ["chunk_020_manual_review.json", "identifier_order_quality_policy.json"]
        },
        "created_files": sorted(expected_files),
    }
    write_json_exclusive(directory / "manual_review_integrity_report.json", integrity)
    assert {path.name for path in directory.iterdir() if path.is_file()} == expected_files
    return tree_fingerprint(directory)


def preflight_chunks_021_030():
    assert os.environ.get(CHUNKS_021_030_GATE_NAME) == CHUNKS_021_030_GATE_VALUE, "exact chunks-021-030 gate mismatch"
    for name in [
        GATE_NAME,
        "RUN_QWEN_V32_PRODUCTION_CHUNKS_002_010",
        "RESUME_QWEN_V32_PRODUCTION_CHUNKS_006_010",
        "RESUME_QWEN_V32_PRODUCTION_CHUNKS_007_010",
        CHUNKS_011_020_GATE_NAME,
        "RUN_QWEN_V32_CONFIRM20",
        "RUN_CORRECTED_10",
        "RUN_100_RECIPIENTS",
        "RUN_10000_RECIPIENTS",
        "RUN_FULL_GENERATION",
    ]:
        assert not os.environ.get(name), f"{name} must be false"
    for number in range(1, 101):
        assert not os.environ.get(f"RUN_QWEN_V32_PRODUCTION_CHUNK_{number:03d}")
    assert all(not chunk_directory(number).exists() for number in range(21, 101))
    assert not progress_directory().exists()
    assert not (PRODUCTION_ROOT / "assessment_production.csv").exists()
    assert not RUN_10000_RECIPIENTS and not RUN_FULL_GENERATION

    manual_review_fingerprint = create_or_validate_manual_review()
    progress020_fingerprint = validate_exact_hashes(progress_020_directory(), PROGRESS_020_SHA256)
    progress020 = json.loads((progress_020_directory() / "production_progress_report.json").read_text(encoding="utf-8"))
    reconciliation020 = json.loads((progress_020_directory() / "cross_chunk_reconciliation.json").read_text(encoding="utf-8"))
    integrity020 = json.loads((progress_020_directory() / "cross_chunk_integrity_report.json").read_text(encoding="utf-8"))
    distribution020 = json.loads((progress_020_directory() / "cross_chunk_distribution_report.json").read_text(encoding="utf-8"))
    assert progress020["status"] == "manual_review_required"
    assert progress020["aggregate_dimensions"] == [12000, 31]
    assert progress020["aggregate_request_records"] == progress020["aggregate_response_records"] == 2000
    assert progress020["aggregate_manifest_records"] == 8002
    assert reconciliation020["status"] == "passed" and not reconciliation020["hard_failures"]
    assert integrity020["status"] == "passed"
    review = json.loads((manual_review_directory() / "chunk_020_manual_review.json").read_text(encoding="utf-8"))
    assert review["decision"] == "accepted_without_regeneration"

    amendment1_fingerprint = validate_exact_hashes(PRODUCTION_ROOT / "validation_amendment_001", AMENDMENT_001_SHA256)
    amendment2_fingerprint = validate_exact_hashes(PRODUCTION_ROOT / "validation_amendment_002", AMENDMENT_002_FINAL_SHA256)
    recorded_chunks = integrity020["chunk_fingerprints"]
    completed_fingerprints = {
        number: verify_completed_chunk_general(number, recorded_chunks[f"chunk_{number:03d}"])
        for number in range(1, 21)
    }

    all_assessment = pd.concat(
        [pd.read_csv(chunk_directory(number) / f"assessment_chunk_{number:03d}.csv") for number in range(1, 21)],
        ignore_index=True,
    )
    requests = [record for number in range(1, 21) for record in line_records(chunk_directory(number) / "requests.jsonl")]
    responses = [record for number in range(1, 21) for record in line_records(chunk_directory(number) / "responses.jsonl")]
    manifests = [record for number in range(1, 21) for record in line_records(chunk_directory(number) / "manifest.jsonl")]
    assert all_assessment.shape == (12000, 31) and all_assessment.recipient_id.nunique() == 2000
    assert all_assessment.assessment_id.is_unique and not all_assessment.duplicated().any()
    assert not all_assessment.duplicated(["recipient_id", "days_since_transplant"]).any()
    assert len(requests) == len(responses) == 2000 and len(manifests) == 8002
    assert len({record["request_id"] for record in requests}) == 2000
    assert {record["request_id"] for record in requests} == {record["request_id"] for record in responses}

    design_checks, design_validation = verify_frozen_production_design()
    assert all(design_checks.values())
    template = (V32_DESIGN / "qwen_kidney_v3_2_prompt_template.txt").read_text(encoding="utf-8")
    assert sha256_text(template) == PROMPT_SHA256
    assert sha256_file(PRODUCTION_DESIGN / "production_anchor_metadata.csv") == FROZEN_ANCHOR_METADATA_SHA256
    api_config = json.loads((V32_DESIGN / "api_configuration.json").read_text(encoding="utf-8"))
    assert api_config["client"]["max_retries"] == 0 and api_config["automatic_retry"] is False

    recipients_all = pd.read_csv(PRODUCTION_DESIGN / "production_recipient_metadata.csv")
    assessments_all = pd.read_csv(PRODUCTION_DESIGN / "production_assessment_metadata.csv")
    identities_all = pd.read_csv(PRODUCTION_DESIGN / "production_identity_skeleton.csv", keep_default_na=False)
    anchors_all = pd.read_csv(PRODUCTION_DESIGN / "production_anchor_metadata.csv")
    assert recipients_all.shape[0] == anchors_all.shape[0] == 10000
    assert assessments_all.shape[0] == 60000
    slices = {
        number: frozen_chunk_slice(number, recipients_all, assessments_all, identities_all, anchors_all)
        for number in range(1, 31)
    }
    for slice_context in slices.values():
        verify_frozen_slice(slice_context)
    new_recipients = pd.concat([slices[number]["recipients"] for number in AUTHORIZED_CHUNKS], ignore_index=True)
    new_assessments = pd.concat([slices[number]["assessments"] for number in AUTHORIZED_CHUNKS], ignore_index=True)
    assert new_recipients.shape[0] == 1000 and new_recipients.recipient_id.nunique() == 1000
    assert new_assessments.shape[0] == 6000
    assert new_recipients.recipient_id.tolist() == [f"V32P-R{number:06d}" for number in range(2001, 3001)]
    full_correlations = frozen_full_anchor_correlations(anchors_all)
    assert all(abs(value) < IDENTIFIER_MATERIALITY_ABS_CORRELATION for value in full_correlations.values())

    field_lists = json.loads((V31_DESIGN / "field_lists.json").read_text(encoding="utf-8"))
    canonical_ranges = json.loads((V31_DESIGN / "canonical_qwen_ranges.json").read_text(encoding="utf-8"))
    assessment_columns = pd.read_csv(V31_DESIGN / "assessment_table_schema.csv").field_name.tolist()
    previous_production_paths = [chunk_directory(number) for number in range(1, 21)] + [
        PRODUCTION_ROOT / "validation_amendment_001",
        PRODUCTION_ROOT / "validation_amendment_002",
        progress_010_directory(),
        progress_020_directory(),
        manual_review_directory(),
    ]
    immutable_paths = immutable_inputs()
    context = {
        "checks": {
            "chunks_001_020_complete_immutable": len(completed_fingerprints) == 20,
            "progress_020_counts_verified": True,
            "aggregate_manifest_8002": len(manifests) == 8002,
            "amendments_001_002_verified": True,
            "manual_review_accepted": review["decision"] == "accepted_without_regeneration",
            "frozen_prompt_anchor_skeleton_verified": True,
            "next_1000_frozen_recipients_verified": True,
            "chunks_021_030_absent": all(not chunk_directory(number).exists() for number in AUTHORIZED_CHUNKS),
            "chunks_031_100_disabled": all(not chunk_directory(number).exists() for number in DISABLED_CHUNKS),
            "maximum_new_requests_1000": MAX_NEW_API_REQUESTS == 1000,
            "automatic_retries_disabled": api_config["client"]["max_retries"] == 0,
            "final_assembly_disabled": not (PRODUCTION_ROOT / "assessment_production.csv").exists(),
        },
        "design_validation": design_validation,
        "template": template,
        "api_config": api_config,
        "slices": slices,
        "recipients_all": recipients_all,
        "assessments_all": assessments_all,
        "identities_all": identities_all,
        "anchors_all": anchors_all,
        "field_lists": field_lists,
        "canonical_ranges": canonical_ranges,
        "assessment_columns": assessment_columns,
        "existing_quality": distribution020["per_chunk_anchor_generation_quality"],
        "completed_fingerprints": completed_fingerprints,
        "immutable_paths": immutable_paths,
        "immutable_before": snapshot_immutable(immutable_paths),
        "notebooks_before": snapshot_notebooks_01_05(),
        "protected_before": protected_fingerprint(),
        "production_design_before": tree_fingerprint(PRODUCTION_DESIGN),
        "previous_production_paths": previous_production_paths,
        "previous_production_before": {display_path(path): tree_fingerprint(path) for path in previous_production_paths},
        "progress_010_fingerprint": tree_fingerprint(progress_010_directory()),
        "progress_020_fingerprint": progress020_fingerprint,
        "amendment_001_fingerprint": amendment1_fingerprint,
        "amendment_002_fingerprint": amendment2_fingerprint,
        "manual_review_fingerprint": manual_review_fingerprint,
        "full_frozen_anchor_correlations": full_correlations,
    }
    assert {key: context["protected_before"][key] for key in EXPECTED_PROTECTED} == EXPECTED_PROTECTED
    assert all(context["checks"].values())
    return context


def assert_chunks_021_030_integrity(context, completed_fingerprints=None):
    result = {
        "protected_unchanged": protected_fingerprint() == context["protected_before"],
        "previous_evidence_unchanged": snapshot_immutable(context["immutable_paths"]) == context["immutable_before"],
        "notebooks_01_05_unchanged": snapshot_notebooks_01_05() == context["notebooks_before"],
        "production_design_unchanged": tree_fingerprint(PRODUCTION_DESIGN) == context["production_design_before"],
        "completed_chunks_001_020_and_prior_evidence_unchanged": all(
            tree_fingerprint(path) == context["previous_production_before"][display_path(path)]
            for path in context["previous_production_paths"]
        ),
        "validation_amendment_001_unchanged": tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_001") == context["amendment_001_fingerprint"],
        "validation_amendment_002_unchanged": tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_002") == context["amendment_002_fingerprint"],
        "progress_through_chunk_020_unchanged": tree_fingerprint(progress_020_directory()) == context["progress_020_fingerprint"],
        "manual_quality_review_001_unchanged": tree_fingerprint(manual_review_directory()) == context["manual_review_fingerprint"],
        "chunks_031_100_absent": all(not chunk_directory(number).exists() for number in DISABLED_CHUNKS),
        "final_assembly_absent": not (PRODUCTION_ROOT / "assessment_production.csv").exists(),
    }
    for number, fingerprint in (completed_fingerprints or {}).items():
        result[f"chunk_{number:03d}_unchanged"] = tree_fingerprint(chunk_directory(number)) == fingerprint
    assert all(result.values()), [name for name, passed in result.items() if not passed]
    return result


assert_global_integrity = assert_chunks_021_030_integrity


def cumulative_quality_through(number, context):
    assessment = pd.concat(
        [pd.read_csv(chunk_directory(index) / f"assessment_chunk_{index:03d}.csv") for index in range(1, number + 1)],
        ignore_index=True,
    )
    recipients = context["recipients_all"].iloc[: number * 100].copy()
    anchors = context["anchors_all"].iloc[: number * 100].copy()
    return anchor_quality_report(assessment, recipients, anchors, f"chunks_001_{number:03d}")


def apply_identifier_order_policy(number, quality, cumulative_quality, context):
    original_identifier_gate = quality["gates"].pop(
        "anchor_recipient_suffix_correlations_below_materiality_threshold"
    )
    quality["failed_gates"] = [
        gate for gate in quality["failed_gates"]
        if gate != "anchor_recipient_suffix_correlations_below_materiality_threshold"
    ]
    isolated_correlations = {
        field: {
            "anchor_recipient_order_correlation": float(values["anchor_recipient_suffix_correlation"]),
            "qwen_output_recipient_order_correlation": float(values["output_recipient_suffix_correlation"]),
        }
        for field, values in quality["anchor_output_and_recipient_suffix_correlations"].items()
    }
    cumulative_correlations = {
        field: {
            "anchor_recipient_order_correlation": float(values["anchor_recipient_suffix_correlation"]),
            "qwen_output_recipient_order_correlation": float(values["output_recipient_suffix_correlation"]),
        }
        for field, values in cumulative_quality["anchor_output_and_recipient_suffix_correlations"].items()
    }
    isolated_warnings = [
        {"measurement": field, "source": source, "correlation": value}
        for field, values in isolated_correlations.items()
        for source, value in values.items()
        if abs(float(value)) >= IDENTIFIER_MATERIALITY_ABS_CORRELATION
    ]
    cumulative_pass = all(
        abs(float(value)) < IDENTIFIER_MATERIALITY_ABS_CORRELATION
        for values in cumulative_correlations.values()
        for value in values.values()
    )
    full_pass = all(
        abs(float(value)) < IDENTIFIER_MATERIALITY_ABS_CORRELATION
        for value in context["full_frozen_anchor_correlations"].values()
    )
    hard_quality_gates_pass = all(bool(value) for value in quality["gates"].values())
    additional_systematic_evidence = not (
        hard_quality_gates_pass
        and full_pass
        and quality["identical_complete_trajectories"]["identical_trajectory_groups"] == 0
    )
    quality["gates"].update({
        "isolated_identifier_order_correlation_is_diagnostic_only": True,
        "cumulative_identifier_order_correlations_below_0_20": cumulative_pass,
        "full_frozen_anchor_correlations_below_0_20": full_pass,
        "no_additional_systematic_identifier_dependence_evidence": not additional_systematic_evidence,
    })
    quality["failed_gates"] = [gate for gate, passed in quality["gates"].items() if not bool(passed)]
    quality["status"] = "passed" if not quality["failed_gates"] else "manual_review_required"
    quality["identifier_order_quality_policy"] = {
        "policy_id": "QWEN-V32-IDENTIFIER-ORDER-QUALITY-POLICY-001",
        "manual_review_id": MANUAL_REVIEW_ID,
        "isolated_original_gate_passed": bool(original_identifier_gate),
        "isolated_correlations": isolated_correlations,
        "isolated_diagnostic_warnings": isolated_warnings,
        "cumulative_scope_recipients": number * 100,
        "cumulative_correlations": cumulative_correlations,
        "full_frozen_10000_anchor_correlations": context["full_frozen_anchor_correlations"],
        "authoritative_cumulative_check_passed": cumulative_pass,
        "authoritative_full_skeleton_check_passed": full_pass,
        "additional_systematic_identifier_dependence_evidence": additional_systematic_evidence,
        "action": "continue" if quality["status"] == "passed" else "stop_before_next_chunk",
    }
    return quality


def enrich_completed_chunk_report_qr001(number, context):
    suffix = f"{number:03d}"
    directory = chunk_directory(number)
    assessment = pd.read_csv(directory / f"assessment_chunk_{suffix}.csv")
    quality = anchor_quality_report(
        assessment, context["slices"][number]["recipients"], context["slices"][number]["anchors"], f"chunk_{suffix}"
    )
    cumulative_quality = cumulative_quality_through(number, context)
    quality = apply_identifier_order_policy(number, quality, cumulative_quality, context)
    report_path = directory / f"chunk_{suffix}_run_report.json"
    report = json.loads(report_path.read_text(encoding="utf-8"))
    assert report["status"] == "completed"
    report["validation_policy_amendment_id"] = AMENDMENT_002_ID
    report["manual_quality_review_id"] = MANUAL_REVIEW_ID
    report["anchor_generation_quality"] = quality
    report["anchor_deviation_warning_count"] = sum(quality["anchor_deviation_warning_counts"].values())
    write_json_state(report_path, report)
    return report, quality


def accepted_existing_quality(context):
    result = dict(context["existing_quality"])
    chunk20 = json.loads(json.dumps(result["chunk_020"]))
    chunk20["status"] = "accepted_after_manual_review"
    chunk20["manual_quality_review_id"] = MANUAL_REVIEW_ID
    chunk20["manual_review_decision"] = "accepted_without_regeneration"
    result["chunk_020"] = chunk20
    return result


def aggregate_through_chunk_030(context, completed_fingerprints, execution_reports, quality_reports):
    directory = progress_directory()
    assert not directory.exists()
    assessment_frames = []
    identity_frames = []
    anchor_frames = []
    requests = []
    responses = []
    manifests = []
    reports = {}
    per_chunk = {}
    target_by_chunk = {}
    for number in range(1, 31):
        suffix = f"{number:03d}"
        chunk = chunk_directory(number)
        assessment = pd.read_csv(chunk / f"assessment_chunk_{suffix}.csv")
        assessment_frames.append(assessment)
        identity_frames.append(pd.read_csv(chunk / f"identity_chunk_{suffix}.csv", keep_default_na=False))
        anchor_frames.append(pd.read_csv(chunk / f"anchor_audit_chunk_{suffix}.csv"))
        requests.extend(line_records(chunk / "requests.jsonl"))
        responses.extend(line_records(chunk / "responses.jsonl"))
        manifests.extend(line_records(chunk / "manifest.jsonl"))
        report = json.loads(completed_report_path(number).read_text(encoding="utf-8"))
        quality = quality_reports[f"chunk_{suffix}"]
        reports[number] = report
        target_by_chunk[suffix] = {
            "positive_assessments": int(assessment.acute_rejection_within_30_days.sum()),
            "assessment_rows": int(len(assessment)),
            "prevalence": float(assessment.acute_rejection_within_30_days.mean()),
        }
        per_chunk[suffix] = {
            "status": report["status"],
            "target": target_by_chunk[suffix],
            "anchor_generation_quality": quality,
            "soft_warnings": report.get("soft_warnings", []),
            "failures": report.get("new_failures", report.get("failures", 0)),
            "retries": report.get("new_retries", report.get("retries", 0)),
        }
    assessment = pd.concat(assessment_frames, ignore_index=True)
    identity = pd.concat(identity_frames, ignore_index=True)
    anchor = pd.concat(anchor_frames, ignore_index=True)
    recipients = context["recipients_all"].iloc[:3000].copy()
    frozen = context["assessments_all"].iloc[:18000].copy()
    request_ids = [record["request_id"] for record in requests]
    response_ids = [record["request_id"] for record in responses]
    checks = {
        "thirty_completed_chunks": all(reports[number]["status"] == "completed" for number in range(1, 31)),
        "recipients_3000": assessment.recipient_id.nunique() == 3000,
        "rows_18000": assessment.shape == (18000, 31),
        "assessment_ids_unique": assessment.assessment_id.is_unique,
        "recipient_day_unique": not assessment.duplicated(["recipient_id", "days_since_transplant"]).any(),
        "no_duplicate_rows": not assessment.duplicated().any(),
        "six_rows_each": assessment.groupby("recipient_id").size().eq(6).all(),
        "ordered_days": assessment.groupby("recipient_id").days_since_transplant.apply(list).map(lambda values: values == DAYS).all(),
        "request_records_3000": len(requests) == 3000,
        "response_records_3000": len(responses) == 3000,
        "request_ids_unique": len(set(request_ids)) == 3000,
        "response_ids_unique": len(set(response_ids)) == 3000,
        "request_response_ids_equal": set(request_ids) == set(response_ids),
        "manifest_records_12002": len(manifests) == 12002,
        "historical_failures_remain_two": sum(record["status"] == "validation_failed" for record in manifests) == 2,
        "amendment_001_transition_present": sum(record["status"] == "validated_under_amendment" for record in manifests) == 1,
        "amendment_002_transition_present": sum(record["status"] == "validated_under_amendment_002" for record in manifests) == 1,
        "chunk_020_manual_review_accepted": quality_reports["chunk_020"]["status"] == "accepted_after_manual_review",
        "all_current_policy_chunk_quality_gates_pass": all(
            quality_reports[f"chunk_{number:03d}"]["status"] in ["passed", "accepted_after_manual_review"]
            for number in range(1, 31)
        ),
        "all_chunk_hard_validations_pass": all(
            reports[number].get("validation", {}).get("status", "passed") == "passed" for number in range(1, 31)
        ),
    }
    payloads = {}
    parse_errors = []
    for record in responses:
        try:
            _, payload = parse_preserved_response(record)
            payloads[record["recipient_id"]] = payload
        except Exception as exc:
            parse_errors.append({"request_id": record.get("request_id"), "error": str(exc)})
    mismatches = []
    reconciled = 0
    for recipient_id, payload in payloads.items():
        rows = assessment.loc[assessment.recipient_id.eq(recipient_id)].sort_values("days_since_transplant")
        for position, item in enumerate(payload["assessments"]):
            for field in QWEN_FIELDS:
                reconciled += 1
                if float(item[field]) != float(rows.iloc[position][field]):
                    mismatches.append({"recipient_id": recipient_id, "position": position, "field": field})
    checks["complete_raw_response_reconciliation"] = not parse_errors and reconciled == 90000 and not mismatches
    diagnostics = build_diagnostics(assessment, recipients, anchor)
    checks["no_identical_complete_trajectories"] = diagnostics["identical_complete_trajectories"]["identical_trajectory_groups"] == 0
    frozen_index = frozen.set_index("assessment_id")
    assessment_index = assessment.set_index("assessment_id")
    frozen_exact = True
    for column in [column for column in frozen.columns if column != "assessment_id" and column in assessment.columns]:
        expected = frozen_index.loc[assessment_index.index, column]
        actual = assessment_index[column]
        if pd.api.types.is_numeric_dtype(expected) and pd.api.types.is_numeric_dtype(actual):
            matches = np.allclose(actual.astype(float), expected.astype(float), rtol=0, atol=1e-12, equal_nan=True)
        else:
            matches = actual.fillna("<NA>").astype(str).equals(expected.fillna("<NA>").astype(str))
        frozen_exact = frozen_exact and bool(matches)
    checks["frozen_schedule_audit_and_derivations_exact"] = frozen_exact
    for column in ["infection_indicator", "previous_rejection", "acute_rejection_within_30_days"]:
        checks[f"{column}_exact"] = np.array_equal(
            assessment_index[column].astype(int), frozen_index.loc[assessment_index.index, column].astype(int)
        )
    recipient_donor = recipients.set_index("recipient_id").donor_id
    checks["donor_relationships_consistent"] = all(
        group.donor_id.nunique() == 1 and group.donor_id.iloc[0] == recipient_donor.loc[recipient_id]
        for recipient_id, group in assessment.groupby("recipient_id")
    )
    checks["shared_identity_records_consistent"] = all(len(group.drop_duplicates()) == 1 for _, group in identity.groupby("entity_id"))
    direct = set(context["field_lists"]["direct_identifiers"])
    checks["direct_identifiers_absent"] = not direct.intersection(assessment.columns)
    checks["joined_42_column_export_absent"] = assessment.shape[1] == 31

    aggregate_quality = anchor_quality_report(assessment, recipients, context["anchors_all"].iloc[:3000].copy(), "chunks_001_030")
    aggregate_quality = apply_identifier_order_policy(30, aggregate_quality, aggregate_quality, context)
    checks["cumulative_identifier_order_correlations_below_0_20"] = aggregate_quality["identifier_order_quality_policy"]["authoritative_cumulative_check_passed"]
    checks["full_frozen_anchor_correlations_below_0_20"] = aggregate_quality["identifier_order_quality_policy"]["authoritative_full_skeleton_check_passed"]
    checks["no_systematic_identifier_dependence_evidence"] = not aggregate_quality["identifier_order_quality_policy"]["additional_systematic_identifier_dependence_evidence"]
    anchor_warning_counts = {
        field: sum(report["anchor_deviation_warning_counts"][field] for report in quality_reports.values())
        for field in ANCHOR_FIELD_MAP
    }
    quality_check_names = {
        "all_current_policy_chunk_quality_gates_pass",
        "cumulative_identifier_order_correlations_below_0_20",
        "full_frozen_anchor_correlations_below_0_20",
        "no_systematic_identifier_dependence_evidence",
    }
    quality_gate_failures = [name for name in quality_check_names if not bool(checks[name])]
    hard_failures = [name for name, value in checks.items() if not bool(value) and name not in quality_check_names]
    overall_status = "failed" if hard_failures else "manual_review_required" if quality_gate_failures else "completed"
    reconciliation = {
        "status": "passed" if not hard_failures else "failed",
        "overall_checkpoint_status": overall_status,
        "checks": {key: bool(value) for key, value in checks.items()},
        "failures": hard_failures,
        "hard_failures": hard_failures,
        "quality_gate_failures": quality_gate_failures,
        "qwen_fields_reconciled": reconciled,
        "expected_qwen_fields": 90000,
        "mismatch_count": len(mismatches),
        "parse_errors": parse_errors[:20],
        "amendment_id": AMENDMENT_002_ID,
        "manual_review_id": MANUAL_REVIEW_ID,
        "expected_manifest_records": 12002,
        "extra_manifest_record_explanation": "the two preserved amendment transitions for recipients 572 and 627",
    }
    cumulative_tokens = {
        key: sum(int(reports[number]["token_usage_total"][key]) for number in range(1, 31))
        for key in ["prompt_tokens", "completion_tokens", "total_tokens"]
    }
    new_tokens = {
        key: sum(int(execution_reports[number]["token_usage_total"][key]) for number in AUTHORIZED_CHUNKS)
        for key in ["prompt_tokens", "completion_tokens", "total_tokens"]
    }
    elapsed = {f"chunk_{number:03d}": float(reports[number]["elapsed_seconds"]) for number in range(1, 31)}
    new_elapsed = sum(float(execution_reports[number]["elapsed_seconds"]) for number in AUTHORIZED_CHUNKS)
    per_chunk_identifier = {
        f"chunk_{number:03d}": quality_reports[f"chunk_{number:03d}"]["anchor_output_and_recipient_suffix_correlations"]
        for number in range(1, 31)
    }
    cumulative_identifier = {
        f"through_chunk_{number:03d}": quality_reports[f"chunk_{number:03d}"]["identifier_order_quality_policy"]["cumulative_correlations"]
        for number in AUTHORIZED_CHUNKS
    }
    distribution = {
        **diagnostics,
        "status": "diagnostic_only_not_used_for_regeneration",
        "amendment_id": AMENDMENT_002_ID,
        "manual_review_id": MANUAL_REVIEW_ID,
        "aggregate_anchor_generation_quality": aggregate_quality,
        "per_chunk_anchor_generation_quality": quality_reports,
        "per_chunk_identifier_order_correlations": per_chunk_identifier,
        "cumulative_identifier_order_correlations": cumulative_identifier,
        "full_frozen_10000_anchor_correlations": context["full_frozen_anchor_correlations"],
        "anchor_deviation_warning_counts": anchor_warning_counts,
        "per_chunk_target_prevalence": target_by_chunk,
        "per_chunk_reports": per_chunk,
        "new_token_usage_chunks_021_030": new_tokens,
        "cumulative_token_usage": cumulative_tokens,
        "per_chunk_elapsed_seconds": elapsed,
        "new_chunk_elapsed_seconds": new_elapsed,
        "cumulative_chunk_elapsed_seconds": sum(elapsed.values()),
    }
    integrity_checks = assert_chunks_021_030_integrity(context, completed_fingerprints)
    integrity = {
        "status": "passed",
        "checked_at_utc": now_utc(),
        "amendment_id": AMENDMENT_002_ID,
        "manual_review_id": MANUAL_REVIEW_ID,
        "global_integrity_checks": integrity_checks,
        "chunk_fingerprints": {f"chunk_{number:03d}": tree_fingerprint(chunk_directory(number)) for number in range(1, 31)},
        "validation_amendment_001_fingerprint": tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_001"),
        "validation_amendment_002_fingerprint": tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_002"),
        "manual_quality_review_001_fingerprint": tree_fingerprint(manual_review_directory()),
        "progress_through_chunk_020_fingerprint": tree_fingerprint(progress_020_directory()),
        "chunks_031_100_started": False,
        "final_dataset_assembly_started": False,
    }
    directory.mkdir(parents=False, exist_ok=False)
    write_json_exclusive(directory / "cross_chunk_reconciliation.json", reconciliation)
    write_json_exclusive(directory / "cross_chunk_distribution_report.json", distribution)
    write_json_exclusive(directory / "cross_chunk_integrity_report.json", integrity)
    integrity_after = assert_chunks_021_030_integrity(context, completed_fingerprints)
    progress = {
        "run_id": RUN_ID,
        "checkpoint": "through_chunk_030",
        "status": overall_status,
        "amendment_id": AMENDMENT_002_ID,
        "manual_review_id": MANUAL_REVIEW_ID,
        "created_at_utc": now_utc(),
        "chunks_present": list(range(1, 31)),
        "chunks_completed_this_execution": AUTHORIZED_CHUNKS,
        "new_api_requests_attempted": sum(report["api_requests_attempted"] for report in execution_reports.values()),
        "new_api_requests_completed": sum(report["api_requests_completed"] for report in execution_reports.values()),
        "new_retries": sum(report["retries"] for report in execution_reports.values()),
        "new_token_usage": new_tokens,
        "cumulative_token_usage": cumulative_tokens,
        "new_chunk_elapsed_seconds": new_elapsed,
        "cumulative_chunk_elapsed_seconds": sum(elapsed.values()),
        "aggregate_request_records": len(requests),
        "aggregate_response_records": len(responses),
        "aggregate_manifest_records": len(manifests),
        "expected_extra_manifest_records": 2,
        "aggregate_dimensions": list(assessment.shape),
        "target_count": diagnostics["target_count"],
        "target_prevalence": diagnostics["target_prevalence"],
        "per_chunk_target_prevalence": target_by_chunk,
        "event_recipients": diagnostics["event_recipients"],
        "non_event_recipients": diagnostics["non_event_recipients"],
        "anchor_deviation_warning_counts": anchor_warning_counts,
        "chunk_quality_gate_statuses": {key: value["status"] for key, value in quality_reports.items()},
        "repeated_value_diagnostics": aggregate_quality["later_day_repetition_warnings"],
        "reconciliation_status": reconciliation["status"],
        "reconciliation_failures": hard_failures,
        "quality_gate_failures": quality_gate_failures,
        "integrity_after_aggregate_checkpoint": integrity_after,
        "created_files": [
            "production_progress_report.json",
            "cross_chunk_reconciliation.json",
            "cross_chunk_distribution_report.json",
            "cross_chunk_integrity_report.json",
        ],
        "progress_file_fingerprints_excluding_self": {
            path.name: file_fingerprint(path) for path in sorted(directory.iterdir()) if path.is_file()
        },
        "chunks_031_to_100_started": False,
        "final_dataset_assembly_started": False,
        "classifier_trained": False,
        "generator_tuned": False,
    }
    write_json_exclusive(directory / "production_progress_report.json", progress)
    return progress


def execute_chunks_021_030():
    context = preflight_chunks_021_030()
    completed_fingerprints = dict(context["completed_fingerprints"])
    execution_reports = {}
    quality_reports = accepted_existing_quality(context)
    total_new_requests = 0
    try:
        for number in AUTHORIZED_CHUNKS:
            if total_new_requests + 100 > MAX_NEW_API_REQUESTS:
                raise RuntimeError("1,000-request hard cap would be exceeded")
            report = run_single_chunk(number, context, completed_fingerprints)
            if report["status"] != "completed":
                execution_reports[number] = report
                raise RuntimeError(f"chunk {number:03d} failed; stopped without retry")
            report, quality = enrich_completed_chunk_report_qr001(number, context)
            execution_reports[number] = report
            quality_reports[f"chunk_{number:03d}"] = quality
            total_new_requests += report["api_requests_completed"]
            completed_fingerprints[number] = tree_fingerprint(chunk_directory(number))
            assert_chunks_021_030_integrity(context, completed_fingerprints)
            if quality["status"] != "passed":
                raise RuntimeError(
                    f"chunk {number:03d} completed but cumulative/systematic generation-quality policy failed; "
                    "chunk preserved and execution stopped before the next chunk"
                )
        assert total_new_requests == MAX_NEW_API_REQUESTS
        assert set(completed_fingerprints) == set(range(1, 31))
        progress = aggregate_through_chunk_030(
            context, completed_fingerprints, execution_reports, quality_reports
        )
        return {
            "status": progress["status"],
            "execution_reports": {
                str(number): {
                    "status": report["status"],
                    "api_requests": report["api_requests_completed"],
                    "token_usage": report["token_usage_total"],
                    "quality_status": quality_reports[f"chunk_{number:03d}"]["status"],
                    "identifier_order_warnings": quality_reports[f"chunk_{number:03d}"]["identifier_order_quality_policy"]["isolated_diagnostic_warnings"],
                }
                for number, report in execution_reports.items()
            },
            "progress": progress,
        }
    finally:
        assert_chunks_021_030_integrity(context, completed_fingerprints)


CHUNKS_021_030_RESULT = execute_chunks_021_030()
print(json.dumps({
    "status": CHUNKS_021_030_RESULT["status"],
    "execution_reports": CHUNKS_021_030_RESULT["execution_reports"],
    "aggregate_dimensions": CHUNKS_021_030_RESULT["progress"]["aggregate_dimensions"],
    "aggregate_manifest_records": CHUNKS_021_030_RESULT["progress"]["aggregate_manifest_records"],
    "anchor_deviation_warning_counts": CHUNKS_021_030_RESULT["progress"]["anchor_deviation_warning_counts"],
    "reconciliation_status": CHUNKS_021_030_RESULT["progress"]["reconciliation_status"],
    "chunks_031_to_100_started": False,
    "final_dataset_assembly_started": False,
}, indent=2))


## Guarded v3.2 production chunks 031–050

This cell continues the frozen run under amendment 002 and manual quality review 001. It verifies and skips only a contiguous prefix of completed authorized chunks, rejects partial or ambiguous chunks before API access, and is gated to chunks 031–050.

In [ ]:
import ast
import json
import os
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd


_repository_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists())
_notebook_path = _repository_root / "notebooks/KidneyTransplant/00_generate_and_validate_dataset.ipynb"
_notebook = nbformat.read(_notebook_path, as_version=4)
_chunks_021_030_source = next(
    cell.source
    for cell in _notebook.cells
    if cell.cell_type == "code" and "CHUNKS_021_030_GATE_NAME =" in cell.source
)
_chunks_021_030_primitives = _chunks_021_030_source.rsplit(
    "CHUNKS_021_030_RESULT = execute_chunks_021_030()", 1
)[0]
exec(compile(_chunks_021_030_primitives, "chunks-021-030-primitives", "exec"), globals())


CHUNKS_031_050_GATE_NAME = "RUN_QWEN_V32_PRODUCTION_CHUNKS_031_050"
CHUNKS_031_050_GATE_VALUE = "QWEN-V32-PROD-001-CHUNKS-031-050-A002-QR001"
PROGRESS_030_DIR_NAME = "progress_through_chunk_030"
PROGRESS_DIR_NAME = "progress_through_chunk_050"
AUTHORIZED_CHUNKS = list(range(31, 51))
DISABLED_CHUNKS = list(range(51, 101))
EXPECTED_NEW_RECIPIENTS = 2_000
EXPECTED_NEW_ROWS = 12_000
MAX_NEW_API_REQUESTS = 2_000
MULTI_GATE_NAME = CHUNKS_031_050_GATE_NAME
MULTI_GATE_VALUE = CHUNKS_031_050_GATE_VALUE

PROGRESS_030_SHA256 = {
    "cross_chunk_distribution_report.json": "532bb83e597b9eca0209b47a2c2c52ead62469dd67a84444bf648f18bda4ce86",
    "cross_chunk_integrity_report.json": "3115f0676fb7e91afc2fb15341a45adef4ef0570dd4fbebd19897f07b890c11e",
    "cross_chunk_reconciliation.json": "02e20223771c8a9331663f672c6d5e66dbcb4465a7bc156e96fd3b056e6f20ed",
    "production_progress_report.json": "00c5a10f1eb66de63082d569465abe0230c541ea1dd20830829e342cd1b933e6",
}


def progress_030_directory():
    return PRODUCTION_ROOT / PROGRESS_030_DIR_NAME


def standard_chunk_file_names(number):
    suffix = f"{number:03d}"
    return {
        f"anchor_audit_chunk_{suffix}.csv",
        f"assessment_chunk_{suffix}.csv",
        f"chunk_{suffix}_run_report.json",
        "completion_state.json",
        "diagnostic_quality_report.json",
        f"identity_chunk_{suffix}.csv",
        "manifest.jsonl",
        "request_response_reconciliation.json",
        "requests.jsonl",
        "responses.jsonl",
        "run_configuration.json",
        "validated_recipient_checkpoint.csv",
    }


def verify_unrecorded_completed_chunk(number):
    directory = chunk_directory(number)
    assert directory.is_dir(), f"chunk {number:03d} does not exist"
    actual_files = {path.name for path in directory.iterdir() if path.is_file()}
    assert actual_files == standard_chunk_file_names(number), f"chunk {number:03d} partial or ambiguous file set"
    report = json.loads((directory / f"chunk_{number:03d}_run_report.json").read_text(encoding="utf-8"))
    state = json.loads((directory / "completion_state.json").read_text(encoding="utf-8"))
    assessment = pd.read_csv(directory / f"assessment_chunk_{number:03d}.csv")
    checkpoint = pd.read_csv(directory / "validated_recipient_checkpoint.csv")
    pd.testing.assert_frame_equal(assessment, checkpoint, check_dtype=False)
    assert report["status"] == "completed" and report["run_id"] == RUN_ID
    assert report["api_requests_attempted"] == report["api_requests_completed"] == 100
    assert report["request_records"] == report["response_records"] == 100
    assert report["recipients_validated"] == 100 and report["checkpoint_rows"] == 600
    assert report["retries"] == 0 and report["failures"] == 0
    assert report["validation"]["status"] == "passed" and report["reconciliation"]["status"] == "passed"
    assert report["anchor_generation_quality"]["status"] == "passed"
    policy = report["anchor_generation_quality"]["identifier_order_quality_policy"]
    assert policy["authoritative_cumulative_check_passed"]
    assert policy["authoritative_full_skeleton_check_passed"]
    assert not policy["additional_systematic_identifier_dependence_evidence"]
    assert state["requests_completed"] == 100 and state["recipients_validated"] == 100
    assert state["checkpoint_rows"] == 600 and state["assessment_chunk_created"] is True
    assert assessment.shape == (600, 31) and assessment.recipient_id.nunique() == 100
    assert assessment.assessment_id.is_unique
    assert not assessment.duplicated(["recipient_id", "days_since_transplant"]).any()
    assert not assessment.duplicated().any()
    assert len(line_records(directory / "requests.jsonl")) == 100
    assert len(line_records(directory / "responses.jsonl")) == 100
    assert len(line_records(directory / "manifest.jsonl")) == 400
    return tree_fingerprint(directory), report, report["anchor_generation_quality"]


def discover_authorized_chunk_state():
    completed = {}
    reports = {}
    quality = {}
    first_unstarted_seen = False
    for number in AUTHORIZED_CHUNKS:
        directory = chunk_directory(number)
        if not directory.exists():
            first_unstarted_seen = True
            continue
        assert not first_unstarted_seen, f"chunk {number:03d} exists after an unstarted authorised chunk"
        fingerprint, report, chunk_quality = verify_unrecorded_completed_chunk(number)
        completed[number] = fingerprint
        reports[number] = report
        quality[f"chunk_{number:03d}"] = chunk_quality
    expected_prefix = AUTHORIZED_CHUNKS[: len(completed)]
    assert list(completed) == expected_prefix, "completed authorised chunks must form one numerical prefix"
    return completed, reports, quality


def preflight_chunks_031_050():
    assert os.environ.get(CHUNKS_031_050_GATE_NAME) == CHUNKS_031_050_GATE_VALUE, "exact chunks-031-050 gate mismatch"
    for name in [
        GATE_NAME,
        "RUN_QWEN_V32_PRODUCTION_CHUNKS_002_010",
        "RESUME_QWEN_V32_PRODUCTION_CHUNKS_006_010",
        "RESUME_QWEN_V32_PRODUCTION_CHUNKS_007_010",
        CHUNKS_011_020_GATE_NAME,
        CHUNKS_021_030_GATE_NAME,
        "RUN_QWEN_V32_CONFIRM20",
        "RUN_CORRECTED_10",
        "RUN_100_RECIPIENTS",
        "RUN_10000_RECIPIENTS",
        "RUN_FULL_GENERATION",
    ]:
        assert not os.environ.get(name), f"{name} must be false"
    for number in range(1, 101):
        assert not os.environ.get(f"RUN_QWEN_V32_PRODUCTION_CHUNK_{number:03d}")
    assert all(not chunk_directory(number).exists() for number in DISABLED_CHUNKS)
    assert not progress_directory().exists()
    assert not (PRODUCTION_ROOT / "assessment_production.csv").exists()
    assert not RUN_10000_RECIPIENTS and not RUN_FULL_GENERATION

    progress030_fingerprint = validate_exact_hashes(progress_030_directory(), PROGRESS_030_SHA256)
    progress030 = json.loads((progress_030_directory() / "production_progress_report.json").read_text(encoding="utf-8"))
    reconciliation030 = json.loads((progress_030_directory() / "cross_chunk_reconciliation.json").read_text(encoding="utf-8"))
    integrity030 = json.loads((progress_030_directory() / "cross_chunk_integrity_report.json").read_text(encoding="utf-8"))
    distribution030 = json.loads((progress_030_directory() / "cross_chunk_distribution_report.json").read_text(encoding="utf-8"))
    assert progress030["status"] == "completed" and reconciliation030["status"] == "passed"
    assert integrity030["status"] == "passed" and not reconciliation030["hard_failures"]
    assert progress030["aggregate_dimensions"] == [18000, 31]
    assert progress030["aggregate_request_records"] == progress030["aggregate_response_records"] == 3000
    assert progress030["aggregate_manifest_records"] == 12002
    assert not progress030["quality_gate_failures"]

    amendment1_fingerprint = validate_exact_hashes(PRODUCTION_ROOT / "validation_amendment_001", AMENDMENT_001_SHA256)
    amendment2_fingerprint = validate_exact_hashes(PRODUCTION_ROOT / "validation_amendment_002", AMENDMENT_002_FINAL_SHA256)
    manual_review_fingerprint = tree_fingerprint(manual_review_directory())
    assert same_tree_fingerprint(manual_review_fingerprint, integrity030["manual_quality_review_001_fingerprint"])
    review = json.loads((manual_review_directory() / "chunk_020_manual_review.json").read_text(encoding="utf-8"))
    policy = json.loads((manual_review_directory() / "identifier_order_quality_policy.json").read_text(encoding="utf-8"))
    assert review["decision"] == "accepted_without_regeneration"
    assert policy["cumulative_absolute_correlation_threshold"] == 0.20

    completed_fingerprints = {
        number: verify_completed_chunk_general(number, integrity030["chunk_fingerprints"][f"chunk_{number:03d}"])
        for number in range(1, 31)
    }
    all_assessment = pd.concat(
        [pd.read_csv(chunk_directory(number) / f"assessment_chunk_{number:03d}.csv") for number in range(1, 31)],
        ignore_index=True,
    )
    requests = [record for number in range(1, 31) for record in line_records(chunk_directory(number) / "requests.jsonl")]
    responses = [record for number in range(1, 31) for record in line_records(chunk_directory(number) / "responses.jsonl")]
    manifests = [record for number in range(1, 31) for record in line_records(chunk_directory(number) / "manifest.jsonl")]
    assert all_assessment.shape == (18000, 31) and all_assessment.recipient_id.nunique() == 3000
    assert all_assessment.assessment_id.is_unique and not all_assessment.duplicated().any()
    assert not all_assessment.duplicated(["recipient_id", "days_since_transplant"]).any()
    assert len(requests) == len(responses) == 3000 and len(manifests) == 12002
    assert len({record["request_id"] for record in requests}) == 3000
    assert {record["request_id"] for record in requests} == {record["request_id"] for record in responses}

    design_checks, design_validation = verify_frozen_production_design()
    assert all(design_checks.values())
    template = (V32_DESIGN / "qwen_kidney_v3_2_prompt_template.txt").read_text(encoding="utf-8")
    assert sha256_text(template) == PROMPT_SHA256
    assert sha256_file(PRODUCTION_DESIGN / "production_anchor_metadata.csv") == FROZEN_ANCHOR_METADATA_SHA256
    api_config = json.loads((V32_DESIGN / "api_configuration.json").read_text(encoding="utf-8"))
    assert api_config["client"]["max_retries"] == 0 and api_config["automatic_retry"] is False

    recipients_all = pd.read_csv(PRODUCTION_DESIGN / "production_recipient_metadata.csv")
    assessments_all = pd.read_csv(PRODUCTION_DESIGN / "production_assessment_metadata.csv")
    identities_all = pd.read_csv(PRODUCTION_DESIGN / "production_identity_skeleton.csv", keep_default_na=False)
    anchors_all = pd.read_csv(PRODUCTION_DESIGN / "production_anchor_metadata.csv")
    assert recipients_all.shape[0] == anchors_all.shape[0] == 10000 and assessments_all.shape[0] == 60000
    slices = {
        number: frozen_chunk_slice(number, recipients_all, assessments_all, identities_all, anchors_all)
        for number in range(1, 51)
    }
    for slice_context in slices.values():
        verify_frozen_slice(slice_context)
    new_recipients = pd.concat([slices[number]["recipients"] for number in AUTHORIZED_CHUNKS], ignore_index=True)
    new_assessments = pd.concat([slices[number]["assessments"] for number in AUTHORIZED_CHUNKS], ignore_index=True)
    assert new_recipients.shape[0] == 2000 and new_recipients.recipient_id.nunique() == 2000
    assert new_assessments.shape[0] == 12000
    assert new_recipients.recipient_id.tolist() == [f"V32P-R{number:06d}" for number in range(3001, 5001)]
    assert new_recipients.request_sequence_number.tolist() == list(range(3001, 5001))
    full_correlations = frozen_full_anchor_correlations(anchors_all)
    assert all(abs(value) < IDENTIFIER_MATERIALITY_ABS_CORRELATION for value in full_correlations.values())

    existing_authorized, existing_reports, existing_quality = discover_authorized_chunk_state()
    completed_fingerprints.update(existing_authorized)
    field_lists = json.loads((V31_DESIGN / "field_lists.json").read_text(encoding="utf-8"))
    canonical_ranges = json.loads((V31_DESIGN / "canonical_qwen_ranges.json").read_text(encoding="utf-8"))
    assessment_columns = pd.read_csv(V31_DESIGN / "assessment_table_schema.csv").field_name.tolist()
    immutable_paths = immutable_inputs()
    previous_production_paths = [chunk_directory(number) for number in range(1, 31)] + [
        PRODUCTION_ROOT / "validation_amendment_001",
        PRODUCTION_ROOT / "validation_amendment_002",
        progress_010_directory(),
        progress_020_directory(),
        progress_030_directory(),
        manual_review_directory(),
    ]
    context = {
        "checks": {
            "chunks_001_030_complete_immutable": len(completed_fingerprints) >= 30,
            "progress_030_counts_verified": True,
            "aggregate_manifest_12002": len(manifests) == 12002,
            "amendments_001_002_verified": True,
            "manual_review_001_verified": review["decision"] == "accepted_without_regeneration",
            "frozen_prompt_anchor_skeleton_verified": True,
            "next_2000_frozen_recipients_verified": True,
            "authorised_existing_chunks_are_completed_prefix_only": True,
            "chunks_051_100_disabled": all(not chunk_directory(number).exists() for number in DISABLED_CHUNKS),
            "maximum_new_requests_2000": MAX_NEW_API_REQUESTS == 2000,
            "automatic_retries_disabled": api_config["client"]["max_retries"] == 0,
            "final_assembly_disabled": not (PRODUCTION_ROOT / "assessment_production.csv").exists(),
        },
        "design_validation": design_validation,
        "template": template,
        "api_config": api_config,
        "slices": slices,
        "recipients_all": recipients_all,
        "assessments_all": assessments_all,
        "identities_all": identities_all,
        "anchors_all": anchors_all,
        "field_lists": field_lists,
        "canonical_ranges": canonical_ranges,
        "assessment_columns": assessment_columns,
        "existing_quality": distribution030["per_chunk_anchor_generation_quality"],
        "existing_authorized_chunks": list(existing_authorized),
        "existing_authorized_reports": existing_reports,
        "existing_authorized_quality": existing_quality,
        "completed_fingerprints": completed_fingerprints,
        "immutable_paths": immutable_paths,
        "immutable_before": snapshot_immutable(immutable_paths),
        "notebooks_before": snapshot_notebooks_01_05(),
        "protected_before": protected_fingerprint(),
        "production_design_before": tree_fingerprint(PRODUCTION_DESIGN),
        "previous_production_paths": previous_production_paths,
        "previous_production_before": {display_path(path): tree_fingerprint(path) for path in previous_production_paths},
        "progress_010_fingerprint": tree_fingerprint(progress_010_directory()),
        "progress_020_fingerprint": tree_fingerprint(progress_020_directory()),
        "progress_030_fingerprint": progress030_fingerprint,
        "amendment_001_fingerprint": amendment1_fingerprint,
        "amendment_002_fingerprint": amendment2_fingerprint,
        "manual_review_fingerprint": manual_review_fingerprint,
        "full_frozen_anchor_correlations": full_correlations,
    }
    assert {key: context["protected_before"][key] for key in EXPECTED_PROTECTED} == EXPECTED_PROTECTED
    assert all(context["checks"].values())
    return context


def assert_chunks_031_050_integrity(context, completed_fingerprints=None):
    result = {
        "protected_unchanged": protected_fingerprint() == context["protected_before"],
        "previous_evidence_unchanged": snapshot_immutable(context["immutable_paths"]) == context["immutable_before"],
        "notebooks_01_05_unchanged": snapshot_notebooks_01_05() == context["notebooks_before"],
        "production_design_unchanged": tree_fingerprint(PRODUCTION_DESIGN) == context["production_design_before"],
        "completed_chunks_001_030_and_prior_evidence_unchanged": all(
            tree_fingerprint(path) == context["previous_production_before"][display_path(path)]
            for path in context["previous_production_paths"]
        ),
        "validation_amendment_001_unchanged": tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_001") == context["amendment_001_fingerprint"],
        "validation_amendment_002_unchanged": tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_002") == context["amendment_002_fingerprint"],
        "progress_through_chunk_030_unchanged": tree_fingerprint(progress_030_directory()) == context["progress_030_fingerprint"],
        "manual_quality_review_001_unchanged": tree_fingerprint(manual_review_directory()) == context["manual_review_fingerprint"],
        "chunks_051_100_absent": all(not chunk_directory(number).exists() for number in DISABLED_CHUNKS),
        "final_assembly_absent": not (PRODUCTION_ROOT / "assessment_production.csv").exists(),
    }
    for number, fingerprint in (completed_fingerprints or {}).items():
        result[f"chunk_{number:03d}_unchanged"] = tree_fingerprint(chunk_directory(number)) == fingerprint
    assert all(result.values()), [name for name, passed in result.items() if not passed]
    return result


assert_global_integrity = assert_chunks_031_050_integrity


_tree = ast.parse(_chunks_021_030_source)
_aggregate_node = next(
    node for node in _tree.body
    if isinstance(node, ast.FunctionDef) and node.name == "aggregate_through_chunk_030"
)
_aggregate_source = ast.get_source_segment(_chunks_021_030_source, _aggregate_node)
for old, new in [
    ("aggregate_through_chunk_030", "aggregate_through_chunk_050"),
    ("assert_chunks_021_030_integrity", "assert_chunks_031_050_integrity"),
    ("range(1, 31)", "range(1, 51)"),
    ("iloc[:3000]", "iloc[:5000]"),
    ("iloc[:18000]", "iloc[:30000]"),
    ("thirty_completed_chunks", "fifty_completed_chunks"),
    ("recipients_3000", "recipients_5000"),
    ("rows_18000", "rows_30000"),
    ("request_records_3000", "request_records_5000"),
    ("response_records_3000", "response_records_5000"),
    ("== 3000", "== 5000"),
    ("(18000, 31)", "(30000, 31)"),
    ("manifest_records_12002", "manifest_records_20002"),
    ("== 12002", "== 20002"),
    ("== 90000", "== 150000"),
    ("expected_qwen_fields\": 90000", "expected_qwen_fields\": 150000"),
    ("apply_identifier_order_policy(30", "apply_identifier_order_policy(50"),
    ("chunks_001_030", "chunks_001_050"),
    ("expected_manifest_records\": 12002", "expected_manifest_records\": 20002"),
    ("new_token_usage_chunks_021_030", "new_token_usage_chunks_031_050"),
    ("through_chunk_030", "through_chunk_050"),
    ("chunks_031_100_started", "chunks_051_100_started"),
    ("chunks_031_to_100_started", "chunks_051_to_100_started"),
]:
    _aggregate_source = _aggregate_source.replace(old, new)
exec(compile(_aggregate_source, "aggregate-through-chunk-050", "exec"), globals())


def execute_chunks_031_050():
    context = preflight_chunks_031_050()
    completed_fingerprints = dict(context["completed_fingerprints"])
    execution_reports = dict(context["existing_authorized_reports"])
    quality_reports = dict(context["existing_quality"])
    quality_reports.update(context["existing_authorized_quality"])
    actual_requests_this_execution = 0
    chunks_completed_this_execution = []
    try:
        for number in AUTHORIZED_CHUNKS:
            if number in context["existing_authorized_chunks"]:
                continue
            if actual_requests_this_execution + 100 > MAX_NEW_API_REQUESTS:
                raise RuntimeError("2,000-request hard cap would be exceeded")
            report = run_single_chunk(number, context, completed_fingerprints)
            if report["status"] != "completed":
                execution_reports[number] = report
                raise RuntimeError(f"chunk {number:03d} failed; stopped without retry")
            report, quality = enrich_completed_chunk_report_qr001(number, context)
            execution_reports[number] = report
            quality_reports[f"chunk_{number:03d}"] = quality
            actual_requests_this_execution += report["api_requests_completed"]
            chunks_completed_this_execution.append(number)
            completed_fingerprints[number] = tree_fingerprint(chunk_directory(number))
            assert_chunks_031_050_integrity(context, completed_fingerprints)
            if quality["status"] != "passed":
                raise RuntimeError(
                    f"chunk {number:03d} completed but cumulative/systematic generation-quality policy failed; "
                    "chunk preserved and execution stopped before the next chunk"
                )
        assert set(execution_reports) == set(AUTHORIZED_CHUNKS)
        assert set(completed_fingerprints) == set(range(1, 51))
        expected_actual_requests = 100 * (len(AUTHORIZED_CHUNKS) - len(context["existing_authorized_chunks"]))
        assert actual_requests_this_execution == expected_actual_requests
        progress = aggregate_through_chunk_050(
            context, completed_fingerprints, execution_reports, quality_reports
        )
        progress["actual_api_requests_this_execution"] = actual_requests_this_execution
        progress["chunks_completed_this_execution"] = chunks_completed_this_execution
        progress["verified_completed_chunks_skipped"] = context["existing_authorized_chunks"]
        write_json_state(progress_directory() / "production_progress_report.json", progress)
        return {
            "status": progress["status"],
            "actual_api_requests_this_execution": actual_requests_this_execution,
            "chunks_completed_this_execution": chunks_completed_this_execution,
            "verified_completed_chunks_skipped": context["existing_authorized_chunks"],
            "execution_reports": {
                str(number): {
                    "status": report["status"],
                    "api_requests": report["api_requests_completed"],
                    "token_usage": report["token_usage_total"],
                    "quality_status": quality_reports[f"chunk_{number:03d}"]["status"],
                    "identifier_order_warnings": quality_reports[f"chunk_{number:03d}"]["identifier_order_quality_policy"]["isolated_diagnostic_warnings"],
                }
                for number, report in execution_reports.items()
            },
            "progress": progress,
        }
    finally:
        assert_chunks_031_050_integrity(context, completed_fingerprints)


CHUNKS_031_050_RESULT = execute_chunks_031_050()
print(json.dumps({
    "status": CHUNKS_031_050_RESULT["status"],
    "actual_api_requests_this_execution": CHUNKS_031_050_RESULT["actual_api_requests_this_execution"],
    "chunks_completed_this_execution": CHUNKS_031_050_RESULT["chunks_completed_this_execution"],
    "verified_completed_chunks_skipped": CHUNKS_031_050_RESULT["verified_completed_chunks_skipped"],
    "aggregate_dimensions": CHUNKS_031_050_RESULT["progress"]["aggregate_dimensions"],
    "aggregate_manifest_records": CHUNKS_031_050_RESULT["progress"]["aggregate_manifest_records"],
    "anchor_deviation_warning_counts": CHUNKS_031_050_RESULT["progress"]["anchor_deviation_warning_counts"],
    "reconciliation_status": CHUNKS_031_050_RESULT["progress"]["reconciliation_status"],
    "chunks_051_to_100_started": False,
    "final_dataset_assembly_started": False,
}, indent=2))


## Manual timeout retry for recipient 4,714 and guarded resume through chunk 050

This additive cell preserves the original chunk-048 timeout evidence, permits exactly one explicitly authorised attempt-2 for recipient 4,714, and—only after successful validation—continues sequentially through chunks 049–050. Chunks 051–100, final assembly, and classifier training remain disabled.

In [ ]:
import ast
import json
import os
import time
import traceback
from collections import Counter
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd
from openai import OpenAI


_repository_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists())
_notebook_path = _repository_root / "notebooks/KidneyTransplant/00_generate_and_validate_dataset.ipynb"
_notebook = nbformat.read(_notebook_path, as_version=4)
_chunks_031_050_source = next(
    cell.source
    for cell in _notebook.cells
    if cell.cell_type == "code" and "CHUNKS_031_050_GATE_NAME =" in cell.source
)
_chunks_031_050_primitives = _chunks_031_050_source.rsplit(
    "CHUNKS_031_050_RESULT = execute_chunks_031_050()", 1
)[0]
exec(compile(_chunks_031_050_primitives, "chunks-031-050-primitives", "exec"), globals())


RESUME_048_050_GATE_NAME = "RESUME_QWEN_V32_PRODUCTION_CHUNKS_048_050"
RESUME_048_050_GATE_VALUE = "QWEN-V32-PROD-001-RESUME-048-050-TIMEOUT-A2"
AUTHORIZED_CHUNKS = [48, 49, 50]
DISABLED_CHUNKS = list(range(51, 101))
MAX_NEW_API_REQUESTS = 287
MANUAL_RETRY_RECIPIENT_ID = "V32P-R004714"
MANUAL_RETRY_REQUEST_ID = "QWEN-V32-PROD-REQ004714"
MANUAL_RETRY_ATTEMPT_RECORD_ID = "QWEN-V32-PROD-REQ004714-ATTEMPT-02"
ORIGINAL_ATTEMPT_RECORD_ID = "QWEN-V32-PROD-REQ004714-ATTEMPT-01"
RESUME_REPORT_NAME = "chunk_048_resume_completion_report.json"
RESUME_STATE_NAME = "resume_completion_state.json"
PROGRESS_DIR_NAME = "progress_through_chunk_050"


def completed_report_path(number):
    directory = chunk_directory(number)
    if number == 6:
        return directory / "chunk_006_resume_completion_report.json"
    if number == 7:
        return directory / "chunk_007_resume_completion_report.json"
    if number == 48:
        return directory / RESUME_REPORT_NAME
    return directory / f"chunk_{number:03d}_run_report.json"


def byte_prefix_snapshot(path):
    data = path.read_bytes()
    return {"size_bytes": len(data), "sha256": hashlib.sha256(data).hexdigest()}


def assert_byte_prefix(path, expected):
    with path.open("rb") as handle:
        prefix = handle.read(expected["size_bytes"])
    assert len(prefix) == expected["size_bytes"]
    assert hashlib.sha256(prefix).hexdigest() == expected["sha256"], f"historical prefix changed: {display_path(path)}"


def response_usage_and_finish(response_record):
    serialized = json.loads(response_record["serialized_response_json"])
    choices = serialized.get("choices") or []
    choice = choices[0] if choices else {}
    message = choice.get("message") or {}
    content = message.get("content")
    reasoning = message.get("reasoning_content", message.get("reasoning"))
    return {
        "request_id": response_record["request_id"],
        "recipient_id": response_record["recipient_id"],
        "request_sequence_number": int(response_record["request_sequence_number"]),
        "attempt_number": int(response_record.get("attempt_number", 1)),
        "attempt_record_id": response_record.get("attempt_record_id"),
        "finish_reason": choice.get("finish_reason"),
        "reasoning_content_status": "present_nonempty" if reasoning else "present_empty" if reasoning == "" else "absent",
        "final_content_length": len(content) if isinstance(content, str) else 0,
        "token_usage": serialized.get("usage") or {},
    }


def preflight_resume_048_050():
    assert os.environ.get(RESUME_048_050_GATE_NAME) == RESUME_048_050_GATE_VALUE, "exact resume gate mismatch"
    conflicting = [
        GATE_NAME,
        "RUN_QWEN_V32_PRODUCTION_CHUNKS_002_010",
        "RESUME_QWEN_V32_PRODUCTION_CHUNKS_006_010",
        "RESUME_QWEN_V32_PRODUCTION_CHUNKS_007_010",
        CHUNKS_011_020_GATE_NAME,
        CHUNKS_021_030_GATE_NAME,
        CHUNKS_031_050_GATE_NAME,
        "RUN_QWEN_V32_CONFIRM20",
        "RUN_CORRECTED_10",
        "RUN_100_RECIPIENTS",
        "RUN_10000_RECIPIENTS",
        "RUN_FULL_GENERATION",
    ]
    for name in conflicting:
        assert not os.environ.get(name), f"{name} must be false"
    for number in range(1, 101):
        assert not os.environ.get(f"RUN_QWEN_V32_PRODUCTION_CHUNK_{number:03d}")
    assert not progress_directory().exists()
    assert all(not chunk_directory(number).exists() for number in range(49, 101))
    assert not (PRODUCTION_ROOT / "assessment_production.csv").exists()
    assert not RUN_10000_RECIPIENTS and not RUN_FULL_GENERATION

    progress030_fingerprint = validate_exact_hashes(progress_030_directory(), PROGRESS_030_SHA256)
    progress030 = json.loads((progress_030_directory() / "production_progress_report.json").read_text(encoding="utf-8"))
    integrity030 = json.loads((progress_030_directory() / "cross_chunk_integrity_report.json").read_text(encoding="utf-8"))
    distribution030 = json.loads((progress_030_directory() / "cross_chunk_distribution_report.json").read_text(encoding="utf-8"))
    reconciliation030 = json.loads((progress_030_directory() / "cross_chunk_reconciliation.json").read_text(encoding="utf-8"))
    assert progress030["status"] == "completed" and reconciliation030["status"] == "passed"
    assert progress030["aggregate_dimensions"] == [18000, 31]
    assert progress030["aggregate_request_records"] == progress030["aggregate_response_records"] == 3000
    assert progress030["aggregate_manifest_records"] == 12002

    amendment1_fingerprint = validate_exact_hashes(PRODUCTION_ROOT / "validation_amendment_001", AMENDMENT_001_SHA256)
    amendment2_fingerprint = validate_exact_hashes(PRODUCTION_ROOT / "validation_amendment_002", AMENDMENT_002_FINAL_SHA256)
    manual_review_fingerprint = tree_fingerprint(manual_review_directory())
    assert same_tree_fingerprint(manual_review_fingerprint, integrity030["manual_quality_review_001_fingerprint"])
    review = json.loads((manual_review_directory() / "chunk_020_manual_review.json").read_text(encoding="utf-8"))
    assert review["decision"] == "accepted_without_regeneration"

    completed_fingerprints = {
        number: verify_completed_chunk_general(number, integrity030["chunk_fingerprints"][f"chunk_{number:03d}"])
        for number in range(1, 31)
    }
    existing_quality = dict(distribution030["per_chunk_anchor_generation_quality"])
    existing_reports = {}
    for number in range(31, 48):
        fingerprint, report, quality = verify_unrecorded_completed_chunk(number)
        completed_fingerprints[number] = fingerprint
        existing_reports[number] = report
        existing_quality[f"chunk_{number:03d}"] = quality

    design_checks, design_validation = verify_frozen_production_design()
    assert all(design_checks.values())
    template = (V32_DESIGN / "qwen_kidney_v3_2_prompt_template.txt").read_text(encoding="utf-8")
    assert sha256_text(template) == PROMPT_SHA256
    assert sha256_file(PRODUCTION_DESIGN / "production_anchor_metadata.csv") == FROZEN_ANCHOR_METADATA_SHA256
    api_config = json.loads((V32_DESIGN / "api_configuration.json").read_text(encoding="utf-8"))
    assert api_config["client"]["max_retries"] == 0 and api_config["automatic_retry"] is False

    recipients_all = pd.read_csv(PRODUCTION_DESIGN / "production_recipient_metadata.csv")
    assessments_all = pd.read_csv(PRODUCTION_DESIGN / "production_assessment_metadata.csv")
    identities_all = pd.read_csv(PRODUCTION_DESIGN / "production_identity_skeleton.csv", keep_default_na=False)
    anchors_all = pd.read_csv(PRODUCTION_DESIGN / "production_anchor_metadata.csv")
    assert recipients_all.shape[0] == anchors_all.shape[0] == 10000 and assessments_all.shape[0] == 60000
    slices = {number: frozen_chunk_slice(number, recipients_all, assessments_all, identities_all, anchors_all) for number in range(1, 51)}
    for slice_context in slices.values():
        verify_frozen_slice(slice_context)
    expected_resume = pd.concat([slices[number]["recipients"] for number in AUTHORIZED_CHUNKS], ignore_index=True)
    assert expected_resume.recipient_id.tolist() == [f"V32P-R{number:06d}" for number in range(4701, 5001)]

    chunk48 = chunk_directory(48)
    original_names = {
        "chunk_048_run_report.json",
        "completion_state.json",
        "identity_chunk_048.csv",
        "manifest.jsonl",
        "requests.jsonl",
        "responses.jsonl",
        "run_configuration.json",
        "validated_recipient_checkpoint.csv",
    }
    assert {path.name for path in chunk48.iterdir() if path.is_file()} == original_names
    requests = line_records(chunk48 / "requests.jsonl")
    responses = line_records(chunk48 / "responses.jsonl")
    manifests = line_records(chunk48 / "manifest.jsonl")
    checkpoint = pd.read_csv(chunk48 / "validated_recipient_checkpoint.csv")
    original_report = json.loads((chunk48 / "chunk_048_run_report.json").read_text(encoding="utf-8"))
    original_state = json.loads((chunk48 / "completion_state.json").read_text(encoding="utf-8"))
    assert len(requests) == 14 and len(responses) == 13 and len(manifests) == 55
    assert checkpoint.shape == (78, 31) and checkpoint.recipient_id.nunique() == 13
    assert set(checkpoint.recipient_id) == {f"V32P-R{number:06d}" for number in range(4701, 4714)}
    assert original_report["status"] == original_state["status"] == "failed"
    failure = original_report["failure"]
    assert failure == {
        "stage": "api_request",
        "request_id": MANUAL_RETRY_REQUEST_ID,
        "recipient_id": MANUAL_RETRY_RECIPIENT_ID,
        "request_sequence_number": 4714,
        "attempt_number": 1,
        "exception_type": "APITimeoutError",
        "message": "Request timed out.",
    }
    request4714 = [record for record in requests if record["recipient_id"] == MANUAL_RETRY_RECIPIENT_ID]
    assert len(request4714) == 1 and request4714[0]["attempt_number"] == 1
    assert not any(record["recipient_id"] == MANUAL_RETRY_RECIPIENT_ID for record in responses)
    assert not any(int(record["request_sequence_number"]) >= 4715 for record in requests)
    timeout_manifest = [record for record in manifests if record.get("recipient_id") == MANUAL_RETRY_RECIPIENT_ID]
    assert [record["status"] for record in timeout_manifest] == [
        "request_preserved_before_submission", "network_submission_started", "request_failed"
    ]
    recipient4714 = slices[48]["recipients"].set_index("recipient_id").loc[MANUAL_RETRY_RECIPIENT_ID].to_dict()
    anchor4714 = slices[48]["anchors"].set_index("recipient_id").loc[MANUAL_RETRY_RECIPIENT_ID].to_dict()
    rendered4714 = render_prompt(template, recipient4714, anchor4714)
    original_request = request4714[0]
    assert original_request["rendered_prompt"] == rendered4714
    assert original_request["rendered_prompt_sha256"] == sha256_text(rendered4714)
    assert int(original_request["deterministic_request_seed"]) == int(recipient4714["request_seed"]) == 3204714
    assert original_request["completion_parameters"] == {
        **{key: value for key, value in api_config["completion"].items() if key != "model"},
        "seed": 3204714,
    }

    field_lists = json.loads((V31_DESIGN / "field_lists.json").read_text(encoding="utf-8"))
    canonical_ranges = json.loads((V31_DESIGN / "canonical_qwen_ranges.json").read_text(encoding="utf-8"))
    assessment_columns = pd.read_csv(V31_DESIGN / "assessment_table_schema.csv").field_name.tolist()
    immutable_paths = immutable_inputs()
    previous_paths = [
        PRODUCTION_ROOT / "validation_amendment_001",
        PRODUCTION_ROOT / "validation_amendment_002",
        progress_010_directory(),
        progress_020_directory(),
        progress_030_directory(),
        manual_review_directory(),
    ]
    fixed_chunk48 = ["chunk_048_run_report.json", "completion_state.json", "identity_chunk_048.csv", "run_configuration.json"]
    appendable_chunk48 = ["requests.jsonl", "responses.jsonl", "manifest.jsonl", "validated_recipient_checkpoint.csv"]
    context = {
        "checks": {
            "chunks_001_047_complete_immutable": len(completed_fingerprints) == 47,
            "chunk_048_exact_partial_state": True,
            "recipient_4714_one_timeout_no_response": True,
            "recipient_4715_unattempted": True,
            "chunks_049_050_absent": True,
            "chunks_051_100_disabled": True,
            "frozen_prompt_anchor_skeleton_verified": True,
            "maximum_new_calls_287": MAX_NEW_API_REQUESTS == 287,
            "automatic_retries_disabled": api_config["client"]["max_retries"] == 0,
            "final_assembly_disabled": True,
        },
        "design_validation": design_validation,
        "template": template,
        "api_config": api_config,
        "slices": slices,
        "recipients_all": recipients_all,
        "assessments_all": assessments_all,
        "identities_all": identities_all,
        "anchors_all": anchors_all,
        "field_lists": field_lists,
        "canonical_ranges": canonical_ranges,
        "assessment_columns": assessment_columns,
        "existing_reports": existing_reports,
        "existing_quality": existing_quality,
        "completed_fingerprints": completed_fingerprints,
        "immutable_paths": immutable_paths,
        "immutable_before": snapshot_immutable(immutable_paths),
        "notebooks_before": snapshot_notebooks_01_05(),
        "protected_before": protected_fingerprint(),
        "production_design_before": tree_fingerprint(PRODUCTION_DESIGN),
        "previous_paths": previous_paths,
        "previous_before": {display_path(path): tree_fingerprint(path) for path in previous_paths},
        "progress_030_fingerprint": progress030_fingerprint,
        "amendment_001_fingerprint": amendment1_fingerprint,
        "amendment_002_fingerprint": amendment2_fingerprint,
        "manual_review_fingerprint": manual_review_fingerprint,
        "full_frozen_anchor_correlations": frozen_full_anchor_correlations(anchors_all),
        "chunk48_fixed_before": {name: file_fingerprint(chunk48 / name) for name in fixed_chunk48},
        "chunk48_prefix_before": {name: byte_prefix_snapshot(chunk48 / name) for name in appendable_chunk48},
        "chunk48_original_tree": tree_fingerprint(chunk48),
        "original_request_4714": original_request,
        "original_report_48": original_report,
    }
    assert {key: context["protected_before"][key] for key in EXPECTED_PROTECTED} == EXPECTED_PROTECTED
    assert all(context["checks"].values())
    return context


def assert_resume_integrity(context, completed_fingerprints=None):
    chunk48 = chunk_directory(48)
    result = {
        "protected_unchanged": protected_fingerprint() == context["protected_before"],
        "previous_evidence_unchanged": snapshot_immutable(context["immutable_paths"]) == context["immutable_before"],
        "notebooks_01_05_unchanged": snapshot_notebooks_01_05() == context["notebooks_before"],
        "production_design_unchanged": tree_fingerprint(PRODUCTION_DESIGN) == context["production_design_before"],
        "previous_reports_and_amendments_unchanged": all(
            tree_fingerprint(path) == context["previous_before"][display_path(path)] for path in context["previous_paths"]
        ),
        "original_chunk_048_failed_report_unchanged": file_fingerprint(chunk48 / "chunk_048_run_report.json") == context["chunk48_fixed_before"]["chunk_048_run_report.json"],
        "original_chunk_048_failure_state_unchanged": file_fingerprint(chunk48 / "completion_state.json") == context["chunk48_fixed_before"]["completion_state.json"],
        "chunk_048_identity_unchanged": file_fingerprint(chunk48 / "identity_chunk_048.csv") == context["chunk48_fixed_before"]["identity_chunk_048.csv"],
        "chunk_048_original_configuration_unchanged": file_fingerprint(chunk48 / "run_configuration.json") == context["chunk48_fixed_before"]["run_configuration.json"],
        "chunks_051_100_absent": all(not chunk_directory(number).exists() for number in DISABLED_CHUNKS),
        "final_assembly_absent": not (PRODUCTION_ROOT / "assessment_production.csv").exists(),
    }
    for name, expected in context["chunk48_prefix_before"].items():
        assert_byte_prefix(chunk48 / name, expected)
        result[f"chunk_048_original_{name}_prefix_unchanged"] = True
    for number, fingerprint in (completed_fingerprints or {}).items():
        if number != 48:
            result[f"chunk_{number:03d}_unchanged"] = tree_fingerprint(chunk_directory(number)) == fingerprint
    assert all(result.values()), [name for name, passed in result.items() if not passed]
    return result


assert_global_integrity = assert_resume_integrity


def request_record_for(recipient, anchor, context, attempt_number, original_request=None):
    request_id = recipient["request_id"]
    recipient_id = recipient["recipient_id"]
    rendered_prompt = render_prompt(context["template"], recipient, anchor)
    messages = [
        {"role": "system", "content": "Return only the exact JSON object requested. No prose or Markdown."},
        {"role": "user", "content": rendered_prompt},
    ]
    completion = context["api_config"]["completion"]
    record = {
        "run_id": RUN_ID,
        "chunk_id": "CHUNK-048",
        "request_id": request_id,
        "recipient_id": recipient_id,
        "request_sequence_number": int(recipient["request_sequence_number"]),
        "attempt_number": int(attempt_number),
        "attempt_record_id": f"{request_id}-ATTEMPT-{attempt_number:02d}",
        "deterministic_request_seed": int(recipient["request_seed"]),
        "prompt_version": PROMPT_VERSION,
        "prompt_template_sha256": PROMPT_SHA256,
        "rendered_prompt": rendered_prompt,
        "rendered_prompt_sha256": sha256_text(rendered_prompt),
        "rendered_message_payload": messages,
        "model": completion["model"],
        "completion_parameters": {
            **{key: value for key, value in completion.items() if key != "model"},
            "seed": int(recipient["request_seed"]),
        },
        "anchor_algorithm_version": ANCHOR_ALGORITHM_VERSION,
        "anchor_metadata_sha256": anchor_row_hash(anchor),
        "validation_amendment_id": AMENDMENT_002_ID,
        "manual_quality_review_id": MANUAL_REVIEW_ID,
        "timestamp_utc": now_utc(),
        "status": "request_preserved_before_submission",
    }
    if attempt_number == 2:
        assert original_request is not None
        record.update({
            "manual_retry": True,
            "manual_retry_authorization_gate": f"{RESUME_048_050_GATE_NAME}={RESUME_048_050_GATE_VALUE}",
            "retry_classification": "explicitly_authorised_manual_retry_after_api_timeout",
            "parent_request_id": original_request["request_id"],
            "parent_attempt_number": 1,
            "parent_attempt_record_id": ORIGINAL_ATTEMPT_RECORD_ID,
            "original_failure_stage": "api_request",
            "original_failure_type": "APITimeoutError",
        })
        comparable = [
            "recipient_id", "request_sequence_number", "deterministic_request_seed", "prompt_version",
            "prompt_template_sha256", "rendered_prompt", "rendered_prompt_sha256", "rendered_message_payload",
            "model", "completion_parameters", "anchor_algorithm_version", "anchor_metadata_sha256",
        ]
        assert all(record[key] == original_request[key] for key in comparable)
        assert record["attempt_record_id"] == MANUAL_RETRY_ATTEMPT_RECORD_ID
    return record


def resume_chunk_048_after_timeout(context, client_factory=OpenAI):
    started = time.monotonic()
    directory = chunk_directory(48)
    report_path = directory / RESUME_REPORT_NAME
    state_path = directory / RESUME_STATE_NAME
    assert not report_path.exists() and not state_path.exists()
    recipients = context["slices"][48]["recipients"].copy()
    anchors = context["slices"][48]["anchors"].copy()
    frozen_assessments = context["slices"][48]["assessments"].copy()
    identities = context["slices"][48]["identities"].copy()
    recipient_lookup = recipients.set_index("recipient_id")
    anchor_lookup = anchors.set_index("recipient_id")
    existing_requests = line_records(directory / "requests.jsonl")
    existing_responses = line_records(directory / "responses.jsonl")
    existing_manifest = line_records(directory / "manifest.jsonl")
    payloads = {}
    for response_record in existing_responses:
        _, payload = parse_preserved_response(response_record)
        payloads[response_record["recipient_id"]] = payload
    rows = pd.read_csv(directory / "validated_recipient_checkpoint.csv").to_dict(orient="records")
    usage_records = [response_usage_and_finish(record)["token_usage"] for record in existing_responses]
    finish_records = [response_usage_and_finish(record) for record in existing_responses]
    new_usage_records = []
    new_finish_records = []
    soft_warnings = []
    failure = None
    validation = None
    reconciliation = None
    diagnostics = None
    quality = None
    new_attempted = 0
    new_completed = 0
    new_validated = 0
    manual_retry_succeeded = False
    completion = context["api_config"]["completion"]
    client = client_factory(api_key=API_KEY, base_url=BASE_URL, max_retries=0, timeout=180.0)
    write_json_exclusive(state_path, {
        "run_id": RUN_ID,
        "chunk_id": "CHUNK-048",
        "status": "manual_resume_started",
        "automatic_retry": False,
        "manual_retry_authorized": True,
        "next_recipient_id": MANUAL_RETRY_RECIPIENT_ID,
        "maximum_new_api_calls": MAX_NEW_API_REQUESTS,
    })

    def submit_and_checkpoint(recipient, attempt_number, original_request=None):
        nonlocal new_attempted, new_completed, new_validated, failure, manual_retry_succeeded
        if new_attempted >= 87:
            raise RuntimeError("chunk-048 resume 87-request hard cap reached")
        recipient_id = recipient["recipient_id"]
        request_id = recipient["request_id"]
        anchor = anchor_lookup.loc[recipient_id].to_dict()
        record = request_record_for(recipient, anchor, context, attempt_number, original_request)
        append_jsonl(directory / "requests.jsonl", record)
        append_jsonl(directory / "manifest.jsonl", {
            "run_id": RUN_ID,
            "chunk_id": "CHUNK-048",
            "request_id": request_id,
            "recipient_id": recipient_id,
            "request_sequence_number": int(recipient["request_sequence_number"]),
            "attempt_number": attempt_number,
            "attempt_record_id": record["attempt_record_id"],
            "parent_attempt_record_id": record.get("parent_attempt_record_id"),
            "manual_retry": attempt_number == 2,
            "status": "request_preserved_before_submission",
            "timestamp_utc": now_utc(),
        })
        new_attempted += 1
        append_jsonl(directory / "manifest.jsonl", {
            "request_id": request_id,
            "recipient_id": recipient_id,
            "attempt_number": attempt_number,
            "attempt_record_id": record["attempt_record_id"],
            "manual_retry": attempt_number == 2,
            "status": "network_submission_started",
            "timestamp_utc": now_utc(),
        })
        try:
            response = client.chat.completions.create(
                model=completion["model"],
                messages=record["rendered_message_payload"],
                max_tokens=completion["max_tokens"],
                temperature=completion["temperature"],
                top_p=completion["top_p"],
                presence_penalty=completion["presence_penalty"],
                seed=int(recipient["request_seed"]),
                extra_body=completion["extra_body"],
            )
        except Exception as exc:
            failure = {
                "stage": "api_request",
                "request_id": request_id,
                "recipient_id": recipient_id,
                "request_sequence_number": int(recipient["request_sequence_number"]),
                "attempt_number": attempt_number,
                "attempt_record_id": record["attempt_record_id"],
                "exception_type": type(exc).__name__,
                "message": str(exc),
            }
            append_jsonl(directory / "manifest.jsonl", {**failure, "status": "request_failed", "timestamp_utc": now_utc()})
            raise RuntimeError(f"request failed for {request_id} attempt {attempt_number}; no further attempt permitted") from exc
        new_completed += 1
        serialized = response.model_dump_json(indent=2)
        response_record = {
            "run_id": RUN_ID,
            "chunk_id": "CHUNK-048",
            "request_id": request_id,
            "recipient_id": recipient_id,
            "request_sequence_number": int(recipient["request_sequence_number"]),
            "attempt_number": attempt_number,
            "attempt_record_id": record["attempt_record_id"],
            "parent_attempt_record_id": record.get("parent_attempt_record_id"),
            "manual_retry": attempt_number == 2,
            "received_at_utc": now_utc(),
            "serialized_response_json": serialized,
            "serialized_response_sha256": sha256_text(serialized),
            "status": "complete_response_preserved_before_extraction",
        }
        append_jsonl(directory / "responses.jsonl", response_record)
        choice = response.choices[0] if response.choices else None
        message = choice.message if choice else None
        content = message.content if message else None
        reasoning = getattr(message, "reasoning_content", None) if message else None
        finish_reason = choice.finish_reason if choice else None
        usage = response.usage.model_dump() if response.usage else {}
        metadata = {
            "request_id": request_id,
            "recipient_id": recipient_id,
            "request_sequence_number": int(recipient["request_sequence_number"]),
            "attempt_number": attempt_number,
            "attempt_record_id": record["attempt_record_id"],
            "finish_reason": finish_reason,
            "reasoning_content_status": "present_nonempty" if reasoning else "present_empty" if reasoning == "" else "absent",
            "final_content_length": len(content) if isinstance(content, str) else 0,
            "token_usage": usage,
        }
        usage_records.append(usage)
        finish_records.append(metadata)
        new_usage_records.append(usage)
        new_finish_records.append(metadata)
        append_jsonl(directory / "manifest.jsonl", {
            **metadata,
            "manual_retry": attempt_number == 2,
            "status": "response_preserved",
            "timestamp_utc": now_utc(),
        })
        payload = None
        if finish_reason == "length":
            failure = {"stage": "finish_reason_length", **metadata}
        elif not content:
            failure = {"stage": "absent_final_content", **metadata}
        else:
            try:
                payload = json.loads(content)
            except Exception as exc:
                failure = {"stage": "json_parse", **metadata, "exception_type": type(exc).__name__, "message": str(exc)}
        if failure is not None:
            append_jsonl(directory / "manifest.jsonl", {**failure, "status": "validation_failed", "timestamp_utc": now_utc()})
            raise RuntimeError(f"response failed before schema validation for {request_id}")
        errors, warnings = validate_payload_policy_002(payload, anchor, context["canonical_ranges"])
        if errors:
            failure = {"stage": "payload_validation", **metadata, "errors": errors, "warnings": warnings}
            append_jsonl(directory / "manifest.jsonl", {**failure, "status": "validation_failed", "timestamp_utc": now_utc()})
            raise RuntimeError(f"payload validation failed for {request_id}")
        frozen = frozen_assessments.loc[frozen_assessments.recipient_id.eq(recipient_id)].copy()
        recipient_rows = build_assessment_rows(recipient, anchor, frozen, payload, context["assessment_columns"])
        payloads[recipient_id] = payload
        rows.extend(recipient_rows)
        append_checkpoint(directory / "validated_recipient_checkpoint.csv", pd.DataFrame(recipient_rows, columns=context["assessment_columns"]))
        new_validated += 1
        soft_warnings.extend({"recipient_id": recipient_id, "warning": warning} for warning in warnings)
        if attempt_number == 2:
            manual_retry_succeeded = True
        append_jsonl(directory / "manifest.jsonl", {
            "request_id": request_id,
            "recipient_id": recipient_id,
            "request_sequence_number": int(recipient["request_sequence_number"]),
            "attempt_number": attempt_number,
            "attempt_record_id": record["attempt_record_id"],
            "manual_retry": attempt_number == 2,
            "status": "validated_and_checkpointed",
            "checkpoint_recipient_count": len(payloads),
            "checkpoint_row_count": len(rows),
            "soft_warnings": warnings,
            "timestamp_utc": now_utc(),
        })
        write_json_state(state_path, {
            "run_id": RUN_ID,
            "chunk_id": "CHUNK-048",
            "status": "manual_resume_running",
            "automatic_retry": False,
            "manual_retry_succeeded": manual_retry_succeeded,
            "new_api_attempts": new_attempted,
            "new_successful_responses": new_completed,
            "total_api_attempts": count_jsonl(directory / "requests.jsonl"),
            "total_successful_responses": count_jsonl(directory / "responses.jsonl"),
            "recipients_validated": len(payloads),
            "checkpoint_rows": len(rows),
            "last_validated_recipient_id": recipient_id,
        })

    try:
        retry_recipient = recipient_lookup.loc[MANUAL_RETRY_RECIPIENT_ID].to_dict()
        retry_recipient["recipient_id"] = MANUAL_RETRY_RECIPIENT_ID
        submit_and_checkpoint(retry_recipient, 2, context["original_request_4714"])
        assert manual_retry_succeeded
        remaining = recipients.loc[recipients.request_sequence_number.between(4715, 4800)]
        for recipient in remaining.to_dict(orient="records"):
            submit_and_checkpoint(recipient, 1)
        assert new_attempted == new_completed == new_validated == 87
        assessment = pd.DataFrame(rows, columns=context["assessment_columns"])
        reconciliation = reconcile_payloads(payloads, assessment)
        validation_context = {**context, "recipients": recipients, "assessments": frozen_assessments, "anchors": anchors}
        validation = validate_complete_chunk_policy_002(
            assessment, identities, recipients, anchors, payloads, reconciliation, validation_context
        )
        diagnostics = build_diagnostics(assessment, recipients, anchors)
        validation["checks"]["no_identical_complete_trajectories"] = diagnostics["identical_complete_trajectories"]["identical_trajectory_groups"] == 0
        if not validation["checks"]["no_identical_complete_trajectories"]:
            validation["failures"].append("no_identical_complete_trajectories")
            validation["status"] = "failed"
        checkpoint = pd.read_csv(directory / "validated_recipient_checkpoint.csv")
        try:
            pd.testing.assert_frame_equal(assessment, checkpoint, check_dtype=False)
            validation["checks"]["checkpoint_assessment_equality"] = True
        except AssertionError:
            validation["checks"]["checkpoint_assessment_equality"] = False
            validation["failures"].append("checkpoint_assessment_equality")
            validation["status"] = "failed"
        if validation["status"] != "passed":
            failure = {"stage": "complete_chunk_validation", "validation": validation}
            raise RuntimeError("complete chunk-048 validation failed")
        day7 = assessment.loc[assessment.days_since_transplant.eq(7)].copy()
        anchor_audit = anchors.merge(
            day7[["recipient_id", "creatinine_mg_dl", "urine_output_ml_24h", "tacrolimus_level_ng_ml", "medication_adherence_pct"]],
            on="recipient_id", validate="one_to_one"
        )
        for output, anchor_field in ANCHOR_FIELD_MAP.items():
            difference = (anchor_audit[output] - anchor_audit[anchor_field]).abs()
            anchor_audit[f"{output}_absolute_anchor_difference"] = difference
            anchor_audit[f"{output}_within_tolerance"] = difference.le(ORIGINAL_PROMPT_TOLERANCES[output])
        write_csv_exclusive(directory / "assessment_chunk_048.csv", assessment)
        write_csv_exclusive(directory / "anchor_audit_chunk_048.csv", anchor_audit)
        write_json_exclusive(directory / "request_response_reconciliation.json", reconciliation)
        write_json_exclusive(directory / "diagnostic_quality_report.json", diagnostics)
        quality = anchor_quality_report(assessment, recipients, anchors, "chunk_048")
        cumulative_quality = cumulative_quality_through(48, context)
        quality = apply_identifier_order_policy(48, quality, cumulative_quality, context)
        if quality["status"] != "passed":
            failure = {"stage": "chunk_quality_gate", "failed_gates": quality["failed_gates"]}
            raise RuntimeError("chunk 048 completed but quality policy requires stop")
        write_json_state(state_path, {
            "run_id": RUN_ID,
            "chunk_id": "CHUNK-048",
            "status": "completed_after_manual_timeout_retry",
            "automatic_retry": False,
            "manual_retry_succeeded": True,
            "new_api_attempts": 87,
            "new_successful_responses": 87,
            "total_api_attempts": 101,
            "total_successful_responses": 100,
            "recipients_validated": 100,
            "checkpoint_rows": 600,
            "assessment_chunk_created": True,
        })
    except Exception as exc:
        if failure is None:
            failure = {"stage": "resume_execution", "exception_type": type(exc).__name__, "message": str(exc), "traceback": traceback.format_exc()}
        write_json_state(state_path, {
            "run_id": RUN_ID,
            "chunk_id": "CHUNK-048",
            "status": "manual_resume_failed",
            "automatic_retry": False,
            "third_attempt_permitted": False,
            "new_api_attempts": new_attempted,
            "new_successful_responses": new_completed,
            "recipients_validated": len(payloads),
            "checkpoint_rows": len(rows),
            "failure": failure,
        })
    finally:
        integrity = assert_resume_integrity(context, context["completed_fingerprints"])
        total_tokens = {key: sum(int(record.get(key, 0) or 0) for record in usage_records) for key in ["prompt_tokens", "completion_tokens", "total_tokens"]}
        new_tokens = {key: sum(int(record.get(key, 0) or 0) for record in new_usage_records) for key in ["prompt_tokens", "completion_tokens", "total_tokens"]}
        resume_elapsed = time.monotonic() - started
        report = {
            "run_id": RUN_ID,
            "chunk_id": "CHUNK-048",
            "status": "completed" if failure is None else "failed",
            "resume_classification": "explicitly_authorised_manual_retry_after_infrastructure_api_timeout",
            "original_failed_report_preserved": "chunk_048_run_report.json",
            "original_attempt_record_preserved": ORIGINAL_ATTEMPT_RECORD_ID,
            "manual_retry_attempt_record_id": MANUAL_RETRY_ATTEMPT_RECORD_ID,
            "manual_retry_outcome": "succeeded" if manual_retry_succeeded else "failed",
            "automatic_retries": 0,
            "retries": 0,
            "manual_retry_attempts": 1 if new_attempted else 0,
            "api_requests_attempted": count_jsonl(directory / "requests.jsonl"),
            "api_requests_completed": count_jsonl(directory / "responses.jsonl"),
            "api_responses_received": count_jsonl(directory / "responses.jsonl"),
            "new_api_requests_attempted": new_attempted,
            "new_api_requests_completed": new_completed,
            "recipients_validated": len(payloads),
            "checkpoint_rows": len(rows),
            "historical_failure": context["original_report_48"]["failure"],
            "new_failure": failure,
            "elapsed_seconds": float(context["original_report_48"]["elapsed_seconds"]) + resume_elapsed,
            "new_elapsed_seconds": resume_elapsed,
            "finish_reason_counts": dict(Counter(item["finish_reason"] for item in finish_records)),
            "new_finish_reason_counts": dict(Counter(item["finish_reason"] for item in new_finish_records)),
            "finish_records": finish_records,
            "token_usage_total": total_tokens,
            "new_token_usage_total": new_tokens,
            "request_records": count_jsonl(directory / "requests.jsonl"),
            "response_records": count_jsonl(directory / "responses.jsonl"),
            "manifest_records": count_jsonl(directory / "manifest.jsonl"),
            "manifest_status_counts": dict(Counter(record["status"] for record in line_records(directory / "manifest.jsonl"))),
            "chunk_dimensions": [len(rows), len(context["assessment_columns"])],
            "validation": validation,
            "reconciliation": reconciliation,
            "diagnostics": diagnostics,
            "soft_warnings": soft_warnings,
            "validation_policy_amendment_id": AMENDMENT_002_ID,
            "manual_quality_review_id": MANUAL_REVIEW_ID,
            "anchor_generation_quality": quality,
            "anchor_deviation_warning_count": sum(quality["anchor_deviation_warning_counts"].values()) if quality else None,
            "integrity_in_finally": integrity,
            "protected_before": context["protected_before"],
            "protected_in_finally": protected_fingerprint(),
            "chunks_051_to_100_started": False,
            "final_dataset_assembly_started": False,
            "classifier_trained": False,
            "created_files": sorted(path.name for path in directory.iterdir() if path.is_file()) + [RESUME_REPORT_NAME],
        }
        write_json_exclusive(report_path, report)
    assert_resume_integrity(context, context["completed_fingerprints"])
    return report


_aggregate_tree = ast.parse(_chunks_021_030_source)
_aggregate_node = next(
    node for node in _aggregate_tree.body
    if isinstance(node, ast.FunctionDef) and node.name == "aggregate_through_chunk_030"
)
_resume_aggregate_source = ast.get_source_segment(_chunks_021_030_source, _aggregate_node)
for old, new in [
    ("aggregate_through_chunk_030", "aggregate_resume_through_chunk_050"),
    ("assert_chunks_021_030_integrity", "assert_resume_integrity"),
    ("range(1, 31)", "range(1, 51)"),
    ("iloc[:3000]", "iloc[:5000]"),
    ("iloc[:18000]", "iloc[:30000]"),
    ("thirty_completed_chunks", "fifty_completed_chunks"),
    ("recipients_3000", "recipients_5000"),
    ("rows_18000", "rows_30000"),
    ("request_records_3000", "request_attempt_records_5001"),
    ("response_records_3000", "response_records_5000"),
    ("== 3000", "== 5000"),
    ("len(requests) == 5000", "len(requests) == 5001"),
    ("(18000, 31)", "(30000, 31)"),
    ("manifest_records_12002", "manifest_records_20005"),
    ("== 12002", "== 20005"),
    ("== 90000", "== 150000"),
    ('"expected_qwen_fields": 90000', '"expected_qwen_fields": 150000'),
    ("apply_identifier_order_policy(30", "apply_identifier_order_policy(50"),
    ("chunks_001_030", "chunks_001_050"),
    ('"expected_manifest_records": 12002', '"expected_manifest_records": 20005'),
    ("new_token_usage_chunks_021_030", "new_token_usage_chunks_048_050"),
    ("through_chunk_030", "through_chunk_050"),
    ("chunks_031_100_started", "chunks_051_100_started"),
    ("chunks_031_to_100_started", "chunks_051_to_100_started"),
    ('"expected_extra_manifest_records": 2', '"expected_extra_manifest_records": 5'),
    ('"the two preserved amendment transitions for recipients 572 and 627"', '"two amendment transitions plus three preserved lifecycle records for the timed-out attempt-1 request 4714"'),
]:
    _resume_aggregate_source = _resume_aggregate_source.replace(old, new)
_resume_aggregate_source = _resume_aggregate_source.replace(
    '"historical_failures_remain_two": sum(record["status"] == "validation_failed" for record in manifests) == 2,',
    '"historical_failures_remain_two": sum(record["status"] == "validation_failed" for record in manifests) == 2,\n'
    '        "one_preserved_timeout_failure": sum(record["status"] == "request_failed" for record in manifests) == 1,\n'
    '        "recipient_4714_has_two_attempts": sum(record.get("recipient_id") == MANUAL_RETRY_RECIPIENT_ID for record in requests) == 2,\n'
    '        "recipient_4714_has_one_successful_response": sum(record.get("recipient_id") == MANUAL_RETRY_RECIPIENT_ID for record in responses) == 1,\n'
    '        "manual_retry_attempt_record_unique": sum(record.get("attempt_record_id") == MANUAL_RETRY_ATTEMPT_RECORD_ID for record in requests) == 1,'
)
exec(compile(_resume_aggregate_source, "aggregate-resume-through-chunk-050", "exec"), globals())


def execute_resume_048_050(client_factory=OpenAI):
    context = preflight_resume_048_050()
    completed_fingerprints = dict(context["completed_fingerprints"])
    execution_reports = {}
    quality_reports = dict(context["existing_quality"])
    actual_new_attempts = 0
    try:
        report48 = resume_chunk_048_after_timeout(context, client_factory=client_factory)
        execution_reports[48] = {
            **report48,
            "api_requests_attempted": report48["new_api_requests_attempted"],
            "api_requests_completed": report48["new_api_requests_completed"],
            "token_usage_total": report48["new_token_usage_total"],
            "elapsed_seconds": report48["new_elapsed_seconds"],
        }
        actual_new_attempts += report48["new_api_requests_attempted"]
        if report48["status"] != "completed":
            raise RuntimeError("manual attempt-2 or chunk-048 resume failed; stopped without further attempt")
        quality_reports["chunk_048"] = report48["anchor_generation_quality"]
        completed_fingerprints[48] = tree_fingerprint(chunk_directory(48))
        assert_resume_integrity(context, completed_fingerprints)
        for number in [49, 50]:
            if actual_new_attempts + 100 > MAX_NEW_API_REQUESTS:
                raise RuntimeError("287-request resume hard cap would be exceeded")
            report = run_single_chunk(number, context, completed_fingerprints)
            execution_reports[number] = report
            actual_new_attempts += report["api_requests_attempted"]
            if report["status"] != "completed":
                raise RuntimeError(f"chunk {number:03d} failed; stopped without retry")
            report, quality = enrich_completed_chunk_report_qr001(number, context)
            execution_reports[number] = report
            quality_reports[f"chunk_{number:03d}"] = quality
            completed_fingerprints[number] = tree_fingerprint(chunk_directory(number))
            assert_resume_integrity(context, completed_fingerprints)
            if quality["status"] != "passed":
                raise RuntimeError(f"chunk {number:03d} quality gate failed; stopped before next chunk")
        assert actual_new_attempts == MAX_NEW_API_REQUESTS
        assert set(completed_fingerprints) == set(range(1, 51))
        progress = aggregate_resume_through_chunk_050(
            context, completed_fingerprints, execution_reports, quality_reports
        )
        progress["manual_retry_outcome"] = report48["manual_retry_outcome"]
        progress["new_api_attempts"] = actual_new_attempts
        progress["logical_production_recipients"] = 5000
        progress["successful_response_records"] = 5000
        progress["preserved_timeout_attempts_without_response"] = 1
        progress["request_attempt_records"] = 5001
        write_json_state(progress_directory() / "production_progress_report.json", progress)
        return {
            "status": progress["status"],
            "manual_retry_outcome": report48["manual_retry_outcome"],
            "new_api_attempts": actual_new_attempts,
            "successful_responses": progress["aggregate_response_records"],
            "aggregate_dimensions": progress["aggregate_dimensions"],
            "aggregate_manifest_records": progress["aggregate_manifest_records"],
            "reconciliation_status": progress["reconciliation_status"],
            "chunks_completed": [48, 49, 50],
            "progress": progress,
        }
    finally:
        assert_resume_integrity(context, completed_fingerprints)


RESUME_048_050_RESULT = execute_resume_048_050()
print(json.dumps({
    "status": RESUME_048_050_RESULT["status"],
    "manual_retry_outcome": RESUME_048_050_RESULT["manual_retry_outcome"],
    "new_api_attempts": RESUME_048_050_RESULT["new_api_attempts"],
    "successful_responses": RESUME_048_050_RESULT["successful_responses"],
    "aggregate_dimensions": RESUME_048_050_RESULT["aggregate_dimensions"],
    "aggregate_manifest_records": RESUME_048_050_RESULT["aggregate_manifest_records"],
    "reconciliation_status": RESUME_048_050_RESULT["reconciliation_status"],
    "chunks_051_to_100_started": False,
    "final_dataset_assembly_started": False,
}, indent=2))


## Guarded v3.2 production chunks 051–075

This additive cell verifies the immutable progress-through-chunk-050 checkpoint, including the preserved chunk-048 timeout and manual retry lifecycle, before permitting exactly 2,500 sequential first-attempt requests under amendment 002 and manual quality review 001. Chunks 076–100, final assembly, and classifier training remain disabled.

In [ ]:
import ast
import json
import os
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd


_repository_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists())
_notebook_path = _repository_root / "notebooks/KidneyTransplant/00_generate_and_validate_dataset.ipynb"
_notebook = nbformat.read(_notebook_path, as_version=4)
_resume_048_050_source = next(
    cell.source
    for cell in _notebook.cells
    if cell.cell_type == "code" and "RESUME_048_050_GATE_NAME =" in cell.source
)
_resume_048_050_primitives = _resume_048_050_source.rsplit(
    "RESUME_048_050_RESULT = execute_resume_048_050()", 1
)[0]
exec(compile(_resume_048_050_primitives, "resume-048-050-primitives", "exec"), globals())


CHUNKS_051_075_GATE_NAME = "RUN_QWEN_V32_PRODUCTION_CHUNKS_051_075"
CHUNKS_051_075_GATE_VALUE = "QWEN-V32-PROD-001-CHUNKS-051-075-A002-QR001"
AUTHORIZED_CHUNKS = list(range(51, 76))
DISABLED_CHUNKS = list(range(76, 101))
MAX_NEW_API_REQUESTS = 2_500
PROGRESS_DIR_NAME = "progress_through_chunk_075"
PROGRESS_050_DIR_NAME = "progress_through_chunk_050"
PROGRESS_050_SHA256 = {
    "cross_chunk_distribution_report.json": "723547fe532da933177140da73c3e3d60db9fc9e415bd1187a4ca955aeca93e0",
    "cross_chunk_integrity_report.json": "86f77981c271d016888c729052993b0970ada167a6ef099b871ad596252de496",
    "cross_chunk_reconciliation.json": "24a1d55260ad4e7177aca179f51ba16f24c107e8109cde9b3af084e439ee59b2",
    "production_progress_report.json": "4ea34d91d1562544c586fba27655bc7c39bc040f7321939d31172c5bf06dd293",
}


def progress_050_directory():
    return PRODUCTION_ROOT / PROGRESS_050_DIR_NAME


def preflight_chunks_051_075():
    assert os.environ.get(CHUNKS_051_075_GATE_NAME) == CHUNKS_051_075_GATE_VALUE, "exact chunks-051-075 gate mismatch"
    conflicting = [
        GATE_NAME,
        "RUN_QWEN_V32_PRODUCTION_CHUNKS_002_010",
        "RESUME_QWEN_V32_PRODUCTION_CHUNKS_006_010",
        "RESUME_QWEN_V32_PRODUCTION_CHUNKS_007_010",
        CHUNKS_011_020_GATE_NAME,
        CHUNKS_021_030_GATE_NAME,
        CHUNKS_031_050_GATE_NAME,
        RESUME_048_050_GATE_NAME,
        "RUN_QWEN_V32_CONFIRM20",
        "RUN_CORRECTED_10",
        "RUN_100_RECIPIENTS",
        "RUN_10000_RECIPIENTS",
        "RUN_FULL_GENERATION",
    ]
    for name in conflicting:
        assert not os.environ.get(name), f"{name} must be false"
    for number in range(1, 101):
        assert not os.environ.get(f"RUN_QWEN_V32_PRODUCTION_CHUNK_{number:03d}")
    assert not progress_directory().exists()
    assert all(not chunk_directory(number).exists() for number in range(51, 101))
    assert not (PRODUCTION_ROOT / "assessment_production.csv").exists()
    assert not RUN_10000_RECIPIENTS and not RUN_FULL_GENERATION

    progress050_fingerprint = validate_exact_hashes(progress_050_directory(), PROGRESS_050_SHA256)
    progress050 = json.loads((progress_050_directory() / "production_progress_report.json").read_text(encoding="utf-8"))
    reconciliation050 = json.loads((progress_050_directory() / "cross_chunk_reconciliation.json").read_text(encoding="utf-8"))
    distribution050 = json.loads((progress_050_directory() / "cross_chunk_distribution_report.json").read_text(encoding="utf-8"))
    integrity050 = json.loads((progress_050_directory() / "cross_chunk_integrity_report.json").read_text(encoding="utf-8"))
    assert progress050["status"] == "completed" and reconciliation050["status"] == "passed"
    assert integrity050["status"] == "passed" and not reconciliation050["hard_failures"]
    assert progress050["aggregate_dimensions"] == [30000, 31]
    assert progress050["aggregate_request_records"] == 5001
    assert progress050["aggregate_response_records"] == 5000
    assert progress050["aggregate_manifest_records"] == 20005
    assert reconciliation050["qwen_fields_reconciled"] == reconciliation050["expected_qwen_fields"] == 150000
    assert reconciliation050["checks"]["one_preserved_timeout_failure"]
    assert reconciliation050["checks"]["recipient_4714_has_two_attempts"]
    assert not progress050["quality_gate_failures"]

    completed_fingerprints = {}
    reports = {}
    for number in range(1, 51):
        expected = integrity050["chunk_fingerprints"][f"chunk_{number:03d}"]
        actual = tree_fingerprint(chunk_directory(number))
        assert same_tree_fingerprint(actual, expected), f"chunk {number:03d} fingerprint mismatch"
        completed_fingerprints[number] = actual
        report = json.loads(completed_report_path(number).read_text(encoding="utf-8"))
        assert report["status"] == "completed", number
        assert report.get("validation", {}).get("status") == "passed", number
        assert report.get("reconciliation", {}).get("status") == "passed", number
        reports[number] = report

    assessments = pd.concat(
        [pd.read_csv(chunk_directory(number) / f"assessment_chunk_{number:03d}.csv") for number in range(1, 51)],
        ignore_index=True,
    )
    requests = [record for number in range(1, 51) for record in line_records(chunk_directory(number) / "requests.jsonl")]
    responses = [record for number in range(1, 51) for record in line_records(chunk_directory(number) / "responses.jsonl")]
    manifests = [record for number in range(1, 51) for record in line_records(chunk_directory(number) / "manifest.jsonl")]
    assert assessments.shape == (30000, 31) and assessments.recipient_id.nunique() == 5000
    assert assessments.assessment_id.is_unique and not assessments.duplicated().any()
    assert not assessments.duplicated(["recipient_id", "days_since_transplant"]).any()
    assert len(requests) == 5001 and len(responses) == 5000 and len(manifests) == 20005
    assert len({record["request_id"] for record in requests}) == 5000
    assert len({(record["request_id"], int(record.get("attempt_number", 1))) for record in requests}) == 5001
    assert len({record["request_id"] for record in responses}) == 5000
    assert {record["request_id"] for record in requests} == {record["request_id"] for record in responses}
    assert sum(record["status"] == "request_failed" for record in manifests) == 1
    assert sum(record.get("recipient_id") == MANUAL_RETRY_RECIPIENT_ID for record in requests) == 2
    assert sum(record.get("recipient_id") == MANUAL_RETRY_RECIPIENT_ID for record in responses) == 1

    design_checks, design_validation = verify_frozen_production_design()
    assert all(design_checks.values())
    template = (V32_DESIGN / "qwen_kidney_v3_2_prompt_template.txt").read_text(encoding="utf-8")
    assert sha256_text(template) == PROMPT_SHA256
    assert sha256_file(PRODUCTION_DESIGN / "production_anchor_metadata.csv") == FROZEN_ANCHOR_METADATA_SHA256
    api_config = json.loads((V32_DESIGN / "api_configuration.json").read_text(encoding="utf-8"))
    assert api_config["client"]["max_retries"] == 0 and api_config["automatic_retry"] is False
    recipients_all = pd.read_csv(PRODUCTION_DESIGN / "production_recipient_metadata.csv")
    assessments_all = pd.read_csv(PRODUCTION_DESIGN / "production_assessment_metadata.csv")
    identities_all = pd.read_csv(PRODUCTION_DESIGN / "production_identity_skeleton.csv", keep_default_na=False)
    anchors_all = pd.read_csv(PRODUCTION_DESIGN / "production_anchor_metadata.csv")
    assert recipients_all.shape[0] == anchors_all.shape[0] == 10000 and assessments_all.shape[0] == 60000
    slices = {number: frozen_chunk_slice(number, recipients_all, assessments_all, identities_all, anchors_all) for number in range(1, 76)}
    for slice_context in slices.values():
        verify_frozen_slice(slice_context)
    new_recipients = pd.concat([slices[number]["recipients"] for number in AUTHORIZED_CHUNKS], ignore_index=True)
    new_assessments = pd.concat([slices[number]["assessments"] for number in AUTHORIZED_CHUNKS], ignore_index=True)
    assert new_recipients.shape == (2500, recipients_all.shape[1])
    assert new_assessments.shape[0] == 15000
    assert new_recipients.recipient_id.tolist() == [f"V32P-R{number:06d}" for number in range(5001, 7501)]
    assert new_recipients.request_sequence_number.tolist() == list(range(5001, 7501))
    full_correlations = frozen_full_anchor_correlations(anchors_all)
    assert all(abs(value) < IDENTIFIER_MATERIALITY_ABS_CORRELATION for value in full_correlations.values())

    amendment1_fingerprint = tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_001")
    amendment2_fingerprint = tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_002")
    manual_review_fingerprint = tree_fingerprint(manual_review_directory())
    immutable_paths = immutable_inputs()
    previous_paths = [
        PRODUCTION_ROOT / "validation_amendment_001",
        PRODUCTION_ROOT / "validation_amendment_002",
        progress_010_directory(),
        progress_020_directory(),
        progress_030_directory(),
        progress_050_directory(),
        manual_review_directory(),
    ]
    context = {
        "checks": {
            "chunks_001_050_complete_immutable": len(completed_fingerprints) == 50,
            "checkpoint_5000_recipients_30000_rows": assessments.shape == (30000, 31),
            "attempt_response_timeout_counts_verified": len(requests) == 5001 and len(responses) == 5000,
            "manifest_20005": len(manifests) == 20005,
            "qwen_fields_150000_reconciled": reconciliation050["qwen_fields_reconciled"] == 150000,
            "frozen_prompt_anchor_controls_verified": True,
            "amendment_002_and_manual_review_001_applied": True,
            "chunks_051_075_absent": all(not chunk_directory(number).exists() for number in AUTHORIZED_CHUNKS),
            "chunks_076_100_disabled": all(not chunk_directory(number).exists() for number in DISABLED_CHUNKS),
            "maximum_new_requests_2500": MAX_NEW_API_REQUESTS == 2500,
            "automatic_retries_disabled": api_config["client"]["max_retries"] == 0,
            "final_assembly_disabled": True,
        },
        "design_validation": design_validation,
        "template": template,
        "api_config": api_config,
        "slices": slices,
        "recipients_all": recipients_all,
        "assessments_all": assessments_all,
        "identities_all": identities_all,
        "anchors_all": anchors_all,
        "field_lists": json.loads((V31_DESIGN / "field_lists.json").read_text(encoding="utf-8")),
        "canonical_ranges": json.loads((V31_DESIGN / "canonical_qwen_ranges.json").read_text(encoding="utf-8")),
        "assessment_columns": pd.read_csv(V31_DESIGN / "assessment_table_schema.csv").field_name.tolist(),
        "existing_reports": reports,
        "existing_quality": dict(distribution050["per_chunk_anchor_generation_quality"]),
        "completed_fingerprints": completed_fingerprints,
        "immutable_paths": immutable_paths,
        "immutable_before": snapshot_immutable(immutable_paths),
        "notebooks_before": snapshot_notebooks_01_05(),
        "protected_before": protected_fingerprint(),
        "production_design_before": tree_fingerprint(PRODUCTION_DESIGN),
        "previous_paths": previous_paths,
        "previous_before": {display_path(path): tree_fingerprint(path) for path in previous_paths},
        "progress_050_fingerprint": progress050_fingerprint,
        "amendment_001_fingerprint": amendment1_fingerprint,
        "amendment_002_fingerprint": amendment2_fingerprint,
        "manual_review_fingerprint": manual_review_fingerprint,
        "full_frozen_anchor_correlations": full_correlations,
    }
    assert {key: context["protected_before"][key] for key in EXPECTED_PROTECTED} == EXPECTED_PROTECTED
    assert all(context["checks"].values())
    return context


def assert_chunks_051_075_integrity(context, completed_fingerprints=None):
    result = {
        "protected_unchanged": protected_fingerprint() == context["protected_before"],
        "previous_evidence_unchanged": snapshot_immutable(context["immutable_paths"]) == context["immutable_before"],
        "notebooks_01_05_unchanged": snapshot_notebooks_01_05() == context["notebooks_before"],
        "production_design_unchanged": tree_fingerprint(PRODUCTION_DESIGN) == context["production_design_before"],
        "completed_chunks_001_050_and_historical_evidence_unchanged": all(
            tree_fingerprint(path) == context["previous_before"][display_path(path)] for path in context["previous_paths"]
        ),
        "progress_through_chunk_050_unchanged": tree_fingerprint(progress_050_directory()) == context["progress_050_fingerprint"],
        "validation_amendment_001_unchanged": tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_001") == context["amendment_001_fingerprint"],
        "validation_amendment_002_unchanged": tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_002") == context["amendment_002_fingerprint"],
        "manual_quality_review_001_unchanged": tree_fingerprint(manual_review_directory()) == context["manual_review_fingerprint"],
        "chunks_076_100_absent": all(not chunk_directory(number).exists() for number in DISABLED_CHUNKS),
        "final_assembly_absent": not (PRODUCTION_ROOT / "assessment_production.csv").exists(),
    }
    for number, fingerprint in (completed_fingerprints or {}).items():
        result[f"chunk_{number:03d}_unchanged"] = same_tree_fingerprint(tree_fingerprint(chunk_directory(number)), fingerprint)
    assert all(result.values()), [name for name, passed in result.items() if not passed]
    return result


assert_global_integrity = assert_chunks_051_075_integrity


_aggregate_075_source = _resume_aggregate_source
for old, new in [
    ("aggregate_resume_through_chunk_050", "aggregate_through_chunk_075"),
    ("assert_resume_integrity", "assert_chunks_051_075_integrity"),
    ("range(1, 51)", "range(1, 76)"),
    ("iloc[:5000]", "iloc[:7500]"),
    ("iloc[:30000]", "iloc[:45000]"),
    ("fifty_completed_chunks", "seventy_five_completed_chunks"),
    ("recipients_5000", "recipients_7500"),
    ("rows_30000", "rows_45000"),
    ("request_attempt_records_5001", "request_attempt_records_7501"),
    ("response_records_5000", "response_records_7500"),
    ("== 5000", "== 7500"),
    ("len(requests) == 5001", "len(requests) == 7501"),
    ("(30000, 31)", "(45000, 31)"),
    ("manifest_records_20005", "manifest_records_30005"),
    ("== 20005", "== 30005"),
    ("== 150000", "== 225000"),
    ('"expected_qwen_fields": 150000', '"expected_qwen_fields": 225000'),
    ("apply_identifier_order_policy(50", "apply_identifier_order_policy(75"),
    ("chunks_001_050", "chunks_001_075"),
    ('"expected_manifest_records": 20005', '"expected_manifest_records": 30005'),
    ("new_token_usage_chunks_048_050", "new_token_usage_chunks_051_075"),
    ("through_chunk_050", "through_chunk_075"),
    ("chunks_051_100_started", "chunks_076_100_started"),
    ("chunks_051_to_100_started", "chunks_076_to_100_started"),
]:
    _aggregate_075_source = _aggregate_075_source.replace(old, new)
exec(compile(_aggregate_075_source, "aggregate-through-chunk-075", "exec"), globals())


def execute_chunks_051_075():
    context = preflight_chunks_051_075()
    completed_fingerprints = dict(context["completed_fingerprints"])
    execution_reports = {}
    quality_reports = dict(context["existing_quality"])
    actual_requests = 0
    completed_this_execution = []
    try:
        for number in AUTHORIZED_CHUNKS:
            if actual_requests + 100 > MAX_NEW_API_REQUESTS:
                raise RuntimeError("2,500-request hard cap would be exceeded")
            report = run_single_chunk(number, context, completed_fingerprints)
            execution_reports[number] = report
            actual_requests += report["api_requests_attempted"]
            if report["status"] != "completed":
                raise RuntimeError(f"chunk {number:03d} failed; stopped without retry")
            report, quality = enrich_completed_chunk_report_qr001(number, context)
            execution_reports[number] = report
            quality_reports[f"chunk_{number:03d}"] = quality
            completed_fingerprints[number] = tree_fingerprint(chunk_directory(number))
            completed_this_execution.append(number)
            assert_chunks_051_075_integrity(context, completed_fingerprints)
            if quality["status"] != "passed":
                raise RuntimeError(
                    f"chunk {number:03d} cumulative/systematic quality policy failed; stopped before next chunk"
                )
        assert actual_requests == MAX_NEW_API_REQUESTS
        assert set(completed_fingerprints) == set(range(1, 76))
        progress = aggregate_through_chunk_075(
            context, completed_fingerprints, execution_reports, quality_reports
        )
        progress["actual_api_requests_this_execution"] = actual_requests
        progress["chunks_completed_this_execution"] = completed_this_execution
        progress["logical_production_recipients"] = 7500
        progress["successful_response_records"] = 7500
        progress["request_attempt_records"] = 7501
        progress["preserved_timeout_attempts_without_response"] = 1
        write_json_state(progress_directory() / "production_progress_report.json", progress)
        return {
            "status": progress["status"],
            "actual_api_requests_this_execution": actual_requests,
            "chunks_completed_this_execution": completed_this_execution,
            "aggregate_dimensions": progress["aggregate_dimensions"],
            "aggregate_request_records": progress["aggregate_request_records"],
            "aggregate_response_records": progress["aggregate_response_records"],
            "aggregate_manifest_records": progress["aggregate_manifest_records"],
            "reconciliation_status": progress["reconciliation_status"],
            "progress": progress,
        }
    finally:
        assert_chunks_051_075_integrity(context, completed_fingerprints)


CHUNKS_051_075_RESULT = execute_chunks_051_075()
print(json.dumps({
    "status": CHUNKS_051_075_RESULT["status"],
    "actual_api_requests_this_execution": CHUNKS_051_075_RESULT["actual_api_requests_this_execution"],
    "chunks_completed_this_execution": CHUNKS_051_075_RESULT["chunks_completed_this_execution"],
    "aggregate_dimensions": CHUNKS_051_075_RESULT["aggregate_dimensions"],
    "aggregate_request_records": CHUNKS_051_075_RESULT["aggregate_request_records"],
    "aggregate_response_records": CHUNKS_051_075_RESULT["aggregate_response_records"],
    "aggregate_manifest_records": CHUNKS_051_075_RESULT["aggregate_manifest_records"],
    "reconciliation_status": CHUNKS_051_075_RESULT["reconciliation_status"],
    "chunks_076_to_100_started": False,
    "final_dataset_assembly_started": False,
}, indent=2))


## Guarded v3.2 production chunks 076–100

One-use, amendment-002 and manual-quality-review-001 guarded final generation stage. This stage preserves chunks 001–075, permits exactly 2,500 sequential one-attempt requests, keeps final assembly and classifier training disabled, and writes progress-through-chunk-100 only after all chunks pass.

In [ ]:
import ast
import json
import os
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd


_repository_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists())
_notebook_path = _repository_root / "notebooks/KidneyTransplant/00_generate_and_validate_dataset.ipynb"
_notebook = nbformat.read(_notebook_path, as_version=4)
_resume_048_050_source = next(
    cell.source
    for cell in _notebook.cells
    if cell.cell_type == "code" and "RESUME_048_050_GATE_NAME =" in cell.source
)
_resume_048_050_primitives = _resume_048_050_source.rsplit(
    "RESUME_048_050_RESULT = execute_resume_048_050()", 1
)[0]
exec(compile(_resume_048_050_primitives, "resume-048-050-primitives", "exec"), globals())


CHUNKS_076_100_GATE_NAME = "RUN_QWEN_V32_PRODUCTION_CHUNKS_076_100"
CHUNKS_076_100_GATE_VALUE = "QWEN-V32-PROD-001-CHUNKS-076-100-A002-QR001"
AUTHORIZED_CHUNKS = list(range(76, 101))
MAX_NEW_API_REQUESTS = 2_500
PROGRESS_DIR_NAME = "progress_through_chunk_100"
PROGRESS_075_DIR_NAME = "progress_through_chunk_075"
PROGRESS_075_SHA256 = {
    "cross_chunk_distribution_report.json": "c10e07f3c5a5abdad7b3e2963f1a6ba2fde1cdb8dcf3ea07595b32f7cc0a9b22",
    "cross_chunk_integrity_report.json": "5e55929a71877f49fc3d4e2d3240561da6a5ed5a6a0f4b50739d711f415a3188",
    "cross_chunk_reconciliation.json": "11737e844265505948ac6984c895394f102dca8b1e03b50f43b43aa9cbce0591",
    "production_progress_report.json": "44c872708c4125e1432ea04c1d973d9b6d2dc2e85bf23801eef362b86ef15004",
}


def progress_075_directory():
    return PRODUCTION_ROOT / PROGRESS_075_DIR_NAME


def preflight_chunks_076_100():
    assert os.environ.get(CHUNKS_076_100_GATE_NAME) == CHUNKS_076_100_GATE_VALUE, "exact chunks-076-100 gate mismatch"
    conflicting = [
        GATE_NAME,
        "RUN_QWEN_V32_PRODUCTION_CHUNKS_002_010",
        "RESUME_QWEN_V32_PRODUCTION_CHUNKS_006_010",
        "RESUME_QWEN_V32_PRODUCTION_CHUNKS_007_010",
        CHUNKS_011_020_GATE_NAME,
        CHUNKS_021_030_GATE_NAME,
        CHUNKS_031_050_GATE_NAME,
        RESUME_048_050_GATE_NAME,
        "RUN_QWEN_V32_CONFIRM20",
        "RUN_CORRECTED_10",
        "RUN_100_RECIPIENTS",
        "RUN_10000_RECIPIENTS",
        "RUN_FULL_GENERATION",
    ]
    for name in conflicting:
        assert not os.environ.get(name), f"{name} must be false"
    for number in range(1, 101):
        assert not os.environ.get(f"RUN_QWEN_V32_PRODUCTION_CHUNK_{number:03d}")
    assert not progress_directory().exists()
    assert all(not chunk_directory(number).exists() for number in AUTHORIZED_CHUNKS)
    assert not (PRODUCTION_ROOT / "assessment_production.csv").exists()
    assert not RUN_10000_RECIPIENTS and not RUN_FULL_GENERATION

    progress075_fingerprint = validate_exact_hashes(progress_075_directory(), PROGRESS_075_SHA256)
    progress075 = json.loads((progress_075_directory() / "production_progress_report.json").read_text(encoding="utf-8"))
    reconciliation075 = json.loads((progress_075_directory() / "cross_chunk_reconciliation.json").read_text(encoding="utf-8"))
    distribution075 = json.loads((progress_075_directory() / "cross_chunk_distribution_report.json").read_text(encoding="utf-8"))
    integrity075 = json.loads((progress_075_directory() / "cross_chunk_integrity_report.json").read_text(encoding="utf-8"))
    assert progress075["status"] == "completed" and reconciliation075["status"] == "passed"
    assert integrity075["status"] == "passed" and not reconciliation075["hard_failures"]
    assert progress075["aggregate_dimensions"] == [45000, 31]
    assert progress075["aggregate_request_records"] == 7501
    assert progress075["aggregate_response_records"] == 7500
    assert progress075["aggregate_manifest_records"] == 30005
    assert reconciliation075["qwen_fields_reconciled"] == reconciliation075["expected_qwen_fields"] == 225000
    assert reconciliation075["checks"]["one_preserved_timeout_failure"]
    assert reconciliation075["checks"]["recipient_4714_has_two_attempts"]
    assert not progress075["quality_gate_failures"]

    completed_fingerprints = {}
    reports = {}
    for number in range(1, 76):
        expected = integrity075["chunk_fingerprints"][f"chunk_{number:03d}"]
        actual = tree_fingerprint(chunk_directory(number))
        assert same_tree_fingerprint(actual, expected), f"chunk {number:03d} fingerprint mismatch"
        completed_fingerprints[number] = actual
        report = json.loads(completed_report_path(number).read_text(encoding="utf-8"))
        assert report["status"] == "completed", number
        assert report.get("validation", {}).get("status") == "passed", number
        assert report.get("reconciliation", {}).get("status") == "passed", number
        reports[number] = report

    assessments = pd.concat(
        [pd.read_csv(chunk_directory(number) / f"assessment_chunk_{number:03d}.csv") for number in range(1, 76)],
        ignore_index=True,
    )
    requests = [record for number in range(1, 76) for record in line_records(chunk_directory(number) / "requests.jsonl")]
    responses = [record for number in range(1, 76) for record in line_records(chunk_directory(number) / "responses.jsonl")]
    manifests = [record for number in range(1, 76) for record in line_records(chunk_directory(number) / "manifest.jsonl")]
    assert assessments.shape == (45000, 31) and assessments.recipient_id.nunique() == 7500
    assert assessments.assessment_id.is_unique and not assessments.duplicated().any()
    assert not assessments.duplicated(["recipient_id", "days_since_transplant"]).any()
    assert len(requests) == 7501 and len(responses) == 7500 and len(manifests) == 30005
    assert len({record["request_id"] for record in requests}) == 7500
    assert len({(record["request_id"], int(record.get("attempt_number", 1))) for record in requests}) == 7501
    assert len({record["request_id"] for record in responses}) == 7500
    assert {record["request_id"] for record in requests} == {record["request_id"] for record in responses}
    assert sum(record["status"] == "request_failed" for record in manifests) == 1
    assert sum(record.get("recipient_id") == MANUAL_RETRY_RECIPIENT_ID for record in requests) == 2
    assert sum(record.get("recipient_id") == MANUAL_RETRY_RECIPIENT_ID for record in responses) == 1

    design_checks, design_validation = verify_frozen_production_design()
    assert all(design_checks.values())
    template = (V32_DESIGN / "qwen_kidney_v3_2_prompt_template.txt").read_text(encoding="utf-8")
    assert sha256_text(template) == PROMPT_SHA256
    assert sha256_file(PRODUCTION_DESIGN / "production_anchor_metadata.csv") == FROZEN_ANCHOR_METADATA_SHA256
    api_config = json.loads((V32_DESIGN / "api_configuration.json").read_text(encoding="utf-8"))
    assert api_config["client"]["max_retries"] == 0 and api_config["automatic_retry"] is False
    recipients_all = pd.read_csv(PRODUCTION_DESIGN / "production_recipient_metadata.csv")
    assessments_all = pd.read_csv(PRODUCTION_DESIGN / "production_assessment_metadata.csv")
    identities_all = pd.read_csv(PRODUCTION_DESIGN / "production_identity_skeleton.csv", keep_default_na=False)
    anchors_all = pd.read_csv(PRODUCTION_DESIGN / "production_anchor_metadata.csv")
    assert recipients_all.shape[0] == anchors_all.shape[0] == 10000 and assessments_all.shape[0] == 60000
    slices = {number: frozen_chunk_slice(number, recipients_all, assessments_all, identities_all, anchors_all) for number in range(1, 101)}
    for slice_context in slices.values():
        verify_frozen_slice(slice_context)
    new_recipients = pd.concat([slices[number]["recipients"] for number in AUTHORIZED_CHUNKS], ignore_index=True)
    new_assessments = pd.concat([slices[number]["assessments"] for number in AUTHORIZED_CHUNKS], ignore_index=True)
    assert new_recipients.shape == (2500, recipients_all.shape[1])
    assert new_assessments.shape[0] == 15000
    assert new_recipients.recipient_id.tolist() == [f"V32P-R{number:06d}" for number in range(7501, 10001)]
    assert new_recipients.request_sequence_number.tolist() == list(range(7501, 10001))
    full_correlations = frozen_full_anchor_correlations(anchors_all)
    assert all(abs(value) < IDENTIFIER_MATERIALITY_ABS_CORRELATION for value in full_correlations.values())

    amendment1_fingerprint = tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_001")
    amendment2_fingerprint = tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_002")
    manual_review_fingerprint = tree_fingerprint(manual_review_directory())
    immutable_paths = immutable_inputs()
    previous_paths = [
        PRODUCTION_ROOT / "validation_amendment_001",
        PRODUCTION_ROOT / "validation_amendment_002",
        progress_010_directory(),
        progress_020_directory(),
        progress_030_directory(),
        progress_075_directory(),
        manual_review_directory(),
    ]
    context = {
        "checks": {
            "chunks_001_075_complete_immutable": len(completed_fingerprints) == 75,
            "checkpoint_7500_recipients_45000_rows": assessments.shape == (45000, 31),
            "attempt_response_timeout_counts_verified": len(requests) == 7501 and len(responses) == 7500,
            "manifest_30005": len(manifests) == 30005,
            "qwen_fields_225000_reconciled": reconciliation075["qwen_fields_reconciled"] == 225000,
            "frozen_prompt_anchor_controls_verified": True,
            "amendment_002_and_manual_review_001_applied": True,
            "chunks_076_100_absent": all(not chunk_directory(number).exists() for number in AUTHORIZED_CHUNKS),
            "maximum_new_requests_2500": MAX_NEW_API_REQUESTS == 2500,
            "automatic_retries_disabled": api_config["client"]["max_retries"] == 0,
            "final_assembly_disabled": True,
        },
        "design_validation": design_validation,
        "template": template,
        "api_config": api_config,
        "slices": slices,
        "recipients_all": recipients_all,
        "assessments_all": assessments_all,
        "identities_all": identities_all,
        "anchors_all": anchors_all,
        "field_lists": json.loads((V31_DESIGN / "field_lists.json").read_text(encoding="utf-8")),
        "canonical_ranges": json.loads((V31_DESIGN / "canonical_qwen_ranges.json").read_text(encoding="utf-8")),
        "assessment_columns": pd.read_csv(V31_DESIGN / "assessment_table_schema.csv").field_name.tolist(),
        "existing_reports": reports,
        "existing_quality": dict(distribution075["per_chunk_anchor_generation_quality"]),
        "completed_fingerprints": completed_fingerprints,
        "immutable_paths": immutable_paths,
        "immutable_before": snapshot_immutable(immutable_paths),
        "notebooks_before": snapshot_notebooks_01_05(),
        "protected_before": protected_fingerprint(),
        "production_design_before": tree_fingerprint(PRODUCTION_DESIGN),
        "previous_paths": previous_paths,
        "previous_before": {display_path(path): tree_fingerprint(path) for path in previous_paths},
        "progress_075_fingerprint": progress075_fingerprint,
        "amendment_001_fingerprint": amendment1_fingerprint,
        "amendment_002_fingerprint": amendment2_fingerprint,
        "manual_review_fingerprint": manual_review_fingerprint,
        "full_frozen_anchor_correlations": full_correlations,
    }
    assert {key: context["protected_before"][key] for key in EXPECTED_PROTECTED} == EXPECTED_PROTECTED
    assert all(context["checks"].values())
    return context


def assert_chunks_076_100_integrity(context, completed_fingerprints=None):
    result = {
        "protected_unchanged": protected_fingerprint() == context["protected_before"],
        "previous_evidence_unchanged": snapshot_immutable(context["immutable_paths"]) == context["immutable_before"],
        "notebooks_01_05_unchanged": snapshot_notebooks_01_05() == context["notebooks_before"],
        "production_design_unchanged": tree_fingerprint(PRODUCTION_DESIGN) == context["production_design_before"],
        "completed_chunks_001_075_and_historical_evidence_unchanged": all(
            tree_fingerprint(path) == context["previous_before"][display_path(path)] for path in context["previous_paths"]
        ),
        "progress_through_chunk_075_unchanged": tree_fingerprint(progress_075_directory()) == context["progress_075_fingerprint"],
        "validation_amendment_001_unchanged": tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_001") == context["amendment_001_fingerprint"],
        "validation_amendment_002_unchanged": tree_fingerprint(PRODUCTION_ROOT / "validation_amendment_002") == context["amendment_002_fingerprint"],
        "manual_quality_review_001_unchanged": tree_fingerprint(manual_review_directory()) == context["manual_review_fingerprint"],
        "no_chunk_directories_beyond_100": all(
            not (path.name.startswith("chunk_") and path.name[6:].isdigit() and int(path.name[6:]) > 100)
            for path in PRODUCTION_ROOT.iterdir()
        ),
        "final_assembly_absent": not (PRODUCTION_ROOT / "assessment_production.csv").exists(),
    }
    for number, fingerprint in (completed_fingerprints or {}).items():
        result[f"chunk_{number:03d}_unchanged"] = same_tree_fingerprint(tree_fingerprint(chunk_directory(number)), fingerprint)
    assert all(result.values()), [name for name, passed in result.items() if not passed]
    return result


assert_global_integrity = assert_chunks_076_100_integrity


_aggregate_100_source = _resume_aggregate_source
for old, new in [
    ("aggregate_resume_through_chunk_050", "aggregate_through_chunk_100"),
    ("assert_resume_integrity", "assert_chunks_076_100_integrity"),
    ("range(1, 51)", "range(1, 101)"),
    ("iloc[:5000]", "iloc[:10000]"),
    ("iloc[:30000]", "iloc[:60000]"),
    ("fifty_completed_chunks", "one_hundred_completed_chunks"),
    ("recipients_5000", "recipients_10000"),
    ("rows_30000", "rows_60000"),
    ("request_attempt_records_5001", "request_attempt_records_10001"),
    ("response_records_5000", "response_records_10000"),
    ("== 5000", "== 10000"),
    ("len(requests) == 5001", "len(requests) == 10001"),
    ("(30000, 31)", "(60000, 31)"),
    ("manifest_records_20005", "manifest_records_40005"),
    ("== 20005", "== 40005"),
    ("== 150000", "== 300000"),
    ('"expected_qwen_fields": 150000', '"expected_qwen_fields": 300000'),
    ("apply_identifier_order_policy(50", "apply_identifier_order_policy(100"),
    ("chunks_001_050", "chunks_001_100"),
    ('"expected_manifest_records": 20005', '"expected_manifest_records": 40005'),
    ("new_token_usage_chunks_048_050", "new_token_usage_chunks_076_100"),
    ("through_chunk_050", "through_chunk_100"),
    ("chunks_051_100_started", "chunks_beyond_100_started"),
    ("chunks_051_to_100_started", "chunks_beyond_100_started"),
]:
    _aggregate_100_source = _aggregate_100_source.replace(old, new)
exec(compile(_aggregate_100_source, "aggregate-through-chunk-100", "exec"), globals())


def execute_chunks_076_100():
    context = preflight_chunks_076_100()
    completed_fingerprints = dict(context["completed_fingerprints"])
    execution_reports = {}
    quality_reports = dict(context["existing_quality"])
    actual_requests = 0
    completed_this_execution = []
    try:
        for number in AUTHORIZED_CHUNKS:
            if actual_requests + 100 > MAX_NEW_API_REQUESTS:
                raise RuntimeError("2,500-request hard cap would be exceeded")
            report = run_single_chunk(number, context, completed_fingerprints)
            execution_reports[number] = report
            actual_requests += report["api_requests_attempted"]
            if report["status"] != "completed":
                raise RuntimeError(f"chunk {number:03d} failed; stopped without retry")
            report, quality = enrich_completed_chunk_report_qr001(number, context)
            execution_reports[number] = report
            quality_reports[f"chunk_{number:03d}"] = quality
            completed_fingerprints[number] = tree_fingerprint(chunk_directory(number))
            completed_this_execution.append(number)
            assert_chunks_076_100_integrity(context, completed_fingerprints)
            if quality["status"] != "passed":
                raise RuntimeError(
                    f"chunk {number:03d} cumulative/systematic quality policy failed; stopped before next chunk"
                )
        assert actual_requests == MAX_NEW_API_REQUESTS
        assert set(completed_fingerprints) == set(range(1, 101))
        progress = aggregate_through_chunk_100(
            context, completed_fingerprints, execution_reports, quality_reports
        )
        progress["actual_api_requests_this_execution"] = actual_requests
        progress["chunks_completed_this_execution"] = completed_this_execution
        progress["logical_production_recipients"] = 10000
        progress["successful_response_records"] = 10000
        progress["request_attempt_records"] = 10001
        progress["preserved_timeout_attempts_without_response"] = 1
        write_json_state(progress_directory() / "production_progress_report.json", progress)
        return {
            "status": progress["status"],
            "actual_api_requests_this_execution": actual_requests,
            "chunks_completed_this_execution": completed_this_execution,
            "aggregate_dimensions": progress["aggregate_dimensions"],
            "aggregate_request_records": progress["aggregate_request_records"],
            "aggregate_response_records": progress["aggregate_response_records"],
            "aggregate_manifest_records": progress["aggregate_manifest_records"],
            "reconciliation_status": progress["reconciliation_status"],
            "progress": progress,
        }
    finally:
        assert_chunks_076_100_integrity(context, completed_fingerprints)


CHUNKS_076_100_RESULT = execute_chunks_076_100()
print(json.dumps({
    "status": CHUNKS_076_100_RESULT["status"],
    "actual_api_requests_this_execution": CHUNKS_076_100_RESULT["actual_api_requests_this_execution"],
    "chunks_completed_this_execution": CHUNKS_076_100_RESULT["chunks_completed_this_execution"],
    "aggregate_dimensions": CHUNKS_076_100_RESULT["aggregate_dimensions"],
    "aggregate_request_records": CHUNKS_076_100_RESULT["aggregate_request_records"],
    "aggregate_response_records": CHUNKS_076_100_RESULT["aggregate_response_records"],
    "aggregate_manifest_records": CHUNKS_076_100_RESULT["aggregate_manifest_records"],
    "reconciliation_status": CHUNKS_076_100_RESULT["reconciliation_status"],
    "chunks_beyond_100_started": False,
    "final_dataset_assembly_started": False,
}, indent=2))


## Final local-only v3.2 assembly and validation

One-use guarded stage that performs no API or network access, assembles the immutable 100-chunk production evidence, independently validates the final assessment and identity tables, and creates only the authorised final-assembly artifacts.

In [ ]:
import hashlib
import json
import os
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


FINAL_ASSEMBLY_GATE_NAME = "RUN_QWEN_V32_FINAL_ASSEMBLY"
FINAL_ASSEMBLY_GATE_VALUE = "QWEN-V32-PROD-001-FINAL-ASSEMBLY-001"
FINAL_ASSEMBLY_ID = "QWEN-V32-PROD-001-FINAL-ASSEMBLY-001"
EXPECTED_PROTECTED_FINAL = {
    "sha256": "8f4b6b51f96b753490cbef542643ab363408e472bf6dcf21b2c91cc3176648b7",
    "mtime_ns": 1786015387691857656,
    "size_bytes": 19817033,
}
ASSESSMENT_DAYS_FINAL = [7, 14, 30, 60, 90, 180]
QWEN_RETURN_FIELDS_FINAL = [
    "days_since_transplant",
    "creatinine_mg_dl",
    "urine_output_ml_24h",
    "tacrolimus_level_ng_ml",
    "medication_adherence_pct",
]


def _final_root():
    return next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists())


def _sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _sha256_text(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def _display(path, root):
    return str(path.relative_to(root)) if path.is_relative_to(root) else str(path)


def _file_fingerprint(path, root):
    stat = path.stat()
    return {
        "path": _display(path, root),
        "sha256": _sha256_file(path),
        "mtime_ns": stat.st_mtime_ns,
        "size_bytes": stat.st_size,
    }


def _tree_fingerprint(path, root):
    files = [path] if path.is_file() else sorted(item for item in path.rglob("*") if item.is_file())
    base = path.parent if path.is_file() else path
    records = []
    for item in files:
        stat = item.stat()
        records.append((str(item.relative_to(base)), _sha256_file(item), stat.st_mtime_ns, stat.st_size))
    return {
        "path": _display(path, root),
        "file_count": len(records),
        "total_size_bytes": sum(record[3] for record in records),
        "aggregate_sha256_with_metadata": _sha256_text(
            "\n".join("|".join(map(str, record)) for record in records)
        ),
    }


def _same_tree(actual, expected):
    return all(actual[key] == expected[key] for key in ["file_count", "total_size_bytes", "aggregate_sha256_with_metadata"])


def _json_safe(value):
    if isinstance(value, dict):
        return {str(key): _json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()
    if not isinstance(value, (str, bytes)) and pd.isna(value):
        return None
    return value


def _write_json_exclusive(path, value):
    with path.open("x", encoding="utf-8", newline="") as handle:
        handle.write(json.dumps(_json_safe(value), indent=2, sort_keys=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def _write_csv_exclusive(path, frame):
    with path.open("x", encoding="utf-8", newline="") as handle:
        frame.to_csv(handle, index=False, lineterminator="\n")
        handle.flush()
        os.fsync(handle.fileno())


def _write_text_exclusive(path, text):
    with path.open("x", encoding="utf-8", newline="") as handle:
        handle.write(text)
        handle.flush()
        os.fsync(handle.fileno())


def _read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def _line_records(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def _stable_score(seed, namespace):
    return int.from_bytes(hashlib.sha256(f"{int(seed)}|{namespace}".encode("utf-8")).digest()[:8], "big")


def _derive_abo(donor_group, recipient_group):
    compatible = {"O": {"O", "A", "B", "AB"}, "A": {"A", "AB"}, "B": {"B", "AB"}, "AB": {"AB"}}
    return "Standard compatible" if recipient_group in compatible[donor_group] else "Managed incompatibility"


def _historical_paths(root, production_root):
    raw = root / "data/raw/KidneyTransplant"
    candidates = [
        raw / "qwen_v3_1_design",
        raw / "qwen_v3_2_design",
        raw / "qwen_v3_2_production_design_001",
        raw / "qwen_v3_2_confirm20_001",
        raw / "qwen_v3_2_conformance_addendum_001",
        production_root / "validation_amendment_001",
        production_root / "validation_amendment_002",
        production_root / "manual_quality_review_001",
    ]
    candidates.extend(sorted(production_root.glob("progress_through_chunk_*")))
    return [path for path in candidates if path.exists()]


def _snapshot(paths, root):
    return {_display(path, root): _tree_fingerprint(path, root) for path in paths}


def _dictionary_rows(assessment_columns, identity_columns, classifier_features):
    meanings = {
        "assessment_id": "Unique pseudonymous assessment record identifier.", "recipient_id": "Pseudonymous recipient join key.",
        "donor_id": "Pseudonymous donor join key.", "hospital_id": "Pseudonymous transplant-hospital identifier.",
        "assessment_date": "Calendar date of the assessment.", "training_consent_status": "Consent status applicable to this assessment.",
        "training_consent_version": "Consent version applicable to this assessment.", "retention_expiry_date": "Date after which this assessment is eligible for retention-based removal.",
        "recipient_sex": "Synthetic recipient sex category.", "recipient_ethnicity": "Synthetic recipient ethnicity category.",
        "recipient_region": "Synthetic recipient region.", "distance_to_transplant_centre_km": "Synthetic distance from residence to transplant centre.",
        "recipient_age": "Recipient age at transplant.", "donor_age": "Donor age at donation.", "donor_type": "Living or deceased donor category.",
        "kidney_failure_cause": "Synthetic cause of kidney failure.", "previous_transplant": "Whether the recipient previously received a transplant.",
        "dialysis_months": "Months on dialysis before transplant.", "abo_compatibility_category": "ABO compatibility derived from donor and recipient blood groups.",
        "hla_mismatch_count": "Count of HLA mismatches.", "antibody_risk_score": "Synthetic antibody-risk score.",
        "cold_ischaemia_hours": "Cold-ischaemia duration.", "days_since_transplant": "Scheduled assessment day after transplant.",
        "creatinine_mg_dl": "Serum creatinine measurement.", "creatinine_change_pct": "Percentage creatinine change from baseline or the prior assessment.",
        "urine_output_ml_24h": "Urine output over 24 hours.", "tacrolimus_level_ng_ml": "Tacrolimus trough concentration.",
        "medication_adherence_pct": "Estimated medication adherence.", "infection_indicator": "Whether a controlled infection episode occurs on the assessment day.",
        "previous_rejection": "Whether a confirmed rejection event occurred before the assessment.",
        "acute_rejection_within_30_days": "Whether a confirmed rejection event occurs in the next 30 days.",
        "entity_id": "Pseudonymous recipient or donor join key.", "person_id": "Synthetic direct person identifier retained only in the identity table.",
        "person_role": "Whether the synthetic person is a recipient or donor.", "full_name": "Synthetic placeholder name.",
        "date_of_birth": "Synthetic date of birth.", "email_address": "Non-routable synthetic email address.",
        "postcode": "Synthetic postcode-like value.", "hospital_number": "Synthetic hospital number.",
        "current_consent_status": "Current synthetic identity-level consent status.", "current_consent_version": "Current synthetic identity-level consent version.",
        "consent_granted_date": "Synthetic consent-granted date.", "consent_withdrawal_date": "Synthetic consent-withdrawal date, blank when not withdrawn.",
    }
    units = {"distance_to_transplant_centre_km": "km", "recipient_age": "years", "donor_age": "years", "dialysis_months": "months", "cold_ischaemia_hours": "hours", "days_since_transplant": "days", "creatinine_mg_dl": "mg/dL", "creatinine_change_pct": "percent", "urine_output_ml_24h": "mL/24h", "tacrolimus_level_ng_ml": "ng/mL", "medication_adherence_pct": "percent"}
    integer_fields = {"recipient_age", "donor_age", "previous_transplant", "dialysis_months", "hla_mismatch_count", "days_since_transplant", "infection_indicator", "previous_rejection", "acute_rejection_within_30_days"}
    number_fields = integer_fields | {"distance_to_transplant_centre_km", "antibody_risk_score", "cold_ischaemia_hours", "creatinine_mg_dl", "creatinine_change_pct", "urine_output_ml_24h", "tacrolimus_level_ng_ml", "medication_adherence_pct"}
    qwen_fields = {"creatinine_mg_dl", "urine_output_ml_24h", "tacrolimus_level_ng_ml", "medication_adherence_pct"}
    derived = {"abo_compatibility_category", "creatinine_change_pct", "infection_indicator", "previous_rejection", "acute_rejection_within_30_days"}
    time_varying = {"assessment_id", "assessment_date", "training_consent_status", "training_consent_version", "retention_expiry_date", "days_since_transplant", "creatinine_mg_dl", "creatinine_change_pct", "urine_output_ml_24h", "tacrolimus_level_ng_ml", "medication_adherence_pct", "infection_indicator", "previous_rejection", "acute_rejection_within_30_days"}
    identifiers = {"assessment_id", "recipient_id", "donor_id", "hospital_id", "entity_id", "person_id", "person_role", "full_name", "date_of_birth", "email_address", "postcode", "hospital_number"}
    audit = {"assessment_date", "training_consent_status", "training_consent_version", "retention_expiry_date", "recipient_sex", "recipient_ethnicity", "recipient_region", "distance_to_transplant_centre_km", "current_consent_status", "current_consent_version", "consent_granted_date", "consent_withdrawal_date"}
    rows = []
    for table, columns in [("assessment", assessment_columns), ("identity", identity_columns)]:
        for field in columns:
            role = "classifier feature" if field in classifier_features else "target" if field == "acute_rejection_within_30_days" else "identifier" if field in identifiers else "audit field" if field in audit or table == "identity" else "non-classifier field"
            sensitive = "Synthetic clinical predictor retained for the approved classifier." if field in classifier_features else "Synthetic sensitive demographic retained for audit only and excluded from the classifier." if field in {"recipient_sex", "recipient_ethnicity", "recipient_region"} else "Synthetic direct or pseudonymous identifier kept separate for audit, joins and future deletion requests; never supplied to the classifier." if field in identifiers or table == "identity" else "Synthetic audit/provenance field required for consent, retention, chronology or integrity checks; excluded from the classifier." if role == "audit field" else "Synthetic derived outcome retained for supervised evaluation, not as an input feature."
            rows.append({
                "table": table, "field_name": field, "plain_language_meaning": meanings[field],
                "data_type": "integer" if field in integer_fields else "number" if field in number_fields else "date (YYYY-MM-DD)" if "date" in field else "string",
                "unit": units.get(field, ""), "static_or_time_varying": "time-varying" if table == "assessment" and field in time_varying else "static",
                "source": "Qwen-generated" if field in qwen_fields else "Python-derived" if field in derived else "Python-controlled",
                "role": role, "supplied_to_classifier": field in classifier_features,
                "personal_sensitive_data_justification": sensitive,
            })
    return pd.DataFrame(rows)


def run_qwen_v32_final_assembly(output_directory=None):
    started = time.perf_counter()
    root = _final_root()
    raw = root / "data/raw/KidneyTransplant"
    production_root = raw / "qwen_v3_2_production_001"
    design = raw / "qwen_v3_2_production_design_001"
    v31 = raw / "qwen_v3_1_design"
    progress100 = production_root / "progress_through_chunk_100"
    final_dir = Path(output_directory) if output_directory is not None else production_root / "final_assembly_001"
    protected = root / "data/raw/kidney_transplant_unlearning_dataset.csv"

    assert os.environ.get(FINAL_ASSEMBLY_GATE_NAME) == FINAL_ASSEMBLY_GATE_VALUE, "exact final-assembly gate mismatch"
    forbidden_gates = [name for name, value in os.environ.items() if value and (name.startswith("RUN_QWEN") or name.startswith("RESUME_QWEN")) and name != FINAL_ASSEMBLY_GATE_NAME]
    assert not forbidden_gates, f"API-capable gates must be unset: {forbidden_gates}"
    assert not final_dir.exists(), f"refusing to overwrite existing final assembly directory: {final_dir}"

    progress = _read_json(progress100 / "production_progress_report.json")
    reconciliation100 = _read_json(progress100 / "cross_chunk_reconciliation.json")
    distribution100 = _read_json(progress100 / "cross_chunk_distribution_report.json")
    integrity100 = _read_json(progress100 / "cross_chunk_integrity_report.json")
    assert progress["status"] == "completed" and progress["aggregate_dimensions"] == [60000, 31]
    assert progress["logical_production_recipients"] == 10000
    assert progress["aggregate_request_records"] == 10001 and progress["aggregate_response_records"] == 10000
    assert progress["aggregate_manifest_records"] == 40005
    assert reconciliation100["status"] == "passed" and reconciliation100["qwen_fields_reconciled"] == 300000
    assert reconciliation100["mismatch_count"] == 0 and not reconciliation100["hard_failures"]
    for name, fingerprint in progress["progress_file_fingerprints_excluding_self"].items():
        assert _file_fingerprint(progress100 / name, root)["sha256"] == fingerprint["sha256"]

    protected_before = _file_fingerprint(protected, root)
    assert {key: protected_before[key] for key in EXPECTED_PROTECTED_FINAL} == EXPECTED_PROTECTED_FINAL
    historical_paths = _historical_paths(root, production_root)
    historical_before = _snapshot(historical_paths, root)
    chunk_before = {}
    assessment_parts, identity_parts, request_records, response_records, manifest_records = [], [], [], [], []
    for number in range(1, 101):
        chunk = production_root / f"chunk_{number:03d}"
        assert chunk.is_dir(), f"missing chunk {number:03d}"
        actual_tree = _tree_fingerprint(chunk, root)
        expected_tree = integrity100["chunk_fingerprints"][f"chunk_{number:03d}"]
        assert _same_tree(actual_tree, expected_tree), f"chunk {number:03d} fingerprint mismatch"
        chunk_before[number] = actual_tree
        assessment_path = chunk / f"assessment_chunk_{number:03d}.csv"
        identity_path = chunk / f"identity_chunk_{number:03d}.csv"
        for required in [assessment_path, identity_path, chunk / "requests.jsonl", chunk / "responses.jsonl", chunk / "manifest.jsonl", chunk / "request_response_reconciliation.json"]:
            assert required.is_file(), f"missing required chunk artifact: {required}"
        part = pd.read_csv(assessment_path)
        assert part.shape == (600, 31) and part.recipient_id.nunique() == 100
        assessment_parts.append(part)
        identity_parts.append(pd.read_csv(identity_path, keep_default_na=False))
        request_records.extend(_line_records(chunk / "requests.jsonl"))
        response_records.extend(_line_records(chunk / "responses.jsonl"))
        manifest_records.extend(_line_records(chunk / "manifest.jsonl"))

    assert len(request_records) == 10001 and len(response_records) == 10000 and len(manifest_records) == 40005
    assert len({record["request_id"] for record in response_records}) == 10000
    assert sum(record.get("recipient_id") == "V32P-R004714" for record in request_records) == 2
    assert sum(record.get("recipient_id") == "V32P-R004714" for record in response_records) == 1
    assert sum(record.get("status") == "request_failed" for record in manifest_records) == 1

    assessment_schema = pd.read_csv(v31 / "assessment_table_schema.csv")
    identity_schema = pd.read_csv(v31 / "identity_table_schema.csv")
    field_lists = _read_json(v31 / "field_lists.json")
    canonical_ranges = _read_json(v31 / "canonical_qwen_ranges.json")
    assessment_columns = assessment_schema.field_name.tolist()
    identity_columns = identity_schema.field_name.tolist()
    classifier_features = field_lists["classifier_features"]
    assessments = pd.concat(assessment_parts, ignore_index=True)
    assessments = assessments.sort_values(["recipient_id", "days_since_transplant"], kind="mergesort").reset_index(drop=True)
    identities_raw = pd.concat(identity_parts, ignore_index=True)
    conflict_counts = identities_raw.groupby("entity_id", dropna=False).apply(lambda group: len(group.drop_duplicates()), include_groups=False)
    assert conflict_counts.max() == 1
    identities = identities_raw.drop_duplicates().sort_values(["person_role", "entity_id"], kind="mergesort").reset_index(drop=True)

    hard_checks = {
        "assessment_dimensions_60000_by_31": assessments.shape == (60000, 31),
        "approved_assessment_schema_exact": assessments.columns.tolist() == assessment_columns,
        "recipients_10000": assessments.recipient_id.nunique() == 10000,
        "six_assessments_each": assessments.groupby("recipient_id").size().eq(6).all(),
        "ordered_days_exact": assessments.groupby("recipient_id").days_since_transplant.apply(lambda values: values.tolist() == ASSESSMENT_DAYS_FINAL).all(),
        "no_missing_assessment_values": not assessments.isna().any().any(),
        "no_duplicate_rows": not assessments.duplicated().any(),
        "assessment_ids_unique": assessments.assessment_id.is_unique,
        "recipient_day_unique": not assessments.duplicated(["recipient_id", "days_since_transplant"]).any(),
        "identity_dimensions_18000_by_12": identities.shape == (18000, 12),
        "approved_identity_schema_exact": identities.columns.tolist() == identity_columns,
        "identity_entity_ids_unique": identities.entity_id.is_unique,
        "identity_duplicate_records_consistent": int(conflict_counts.max()) == 1,
        "identity_recipients_10000": identities.person_role.eq("Recipient").sum() == 10000,
        "identity_donors_8000": identities.person_role.eq("Donor").sum() == 8000,
        "assessment_donors_8000": assessments.donor_id.nunique() == 8000,
    }

    static_fields = ["donor_id", "hospital_id", "recipient_sex", "recipient_ethnicity", "recipient_region", "distance_to_transplant_centre_km", "recipient_age", "donor_age", "donor_type", "kidney_failure_cause", "previous_transplant", "dialysis_months", "abo_compatibility_category", "hla_mismatch_count", "antibody_risk_score", "cold_ischaemia_hours"]
    hard_checks["static_fields_constant_within_recipient"] = all(assessments.groupby("recipient_id")[field].nunique(dropna=False).le(1).all() for field in static_fields)
    hard_checks["shared_donor_relationships_preserved"] = set(assessments.donor_id) == set(identities.loc[identities.person_role.eq("Donor"), "entity_id"])
    hard_checks["recipient_identity_links_complete"] = set(assessments.recipient_id) == set(identities.loc[identities.person_role.eq("Recipient"), "entity_id"])

    frozen_recipients = pd.read_csv(design / "production_recipient_metadata.csv")
    frozen_assessments = pd.read_csv(design / "production_assessment_metadata.csv")
    frozen_identity = pd.read_csv(design / "production_identity_skeleton.csv", keep_default_na=False)
    expected_meta = frozen_assessments.sort_values(["recipient_id", "days_since_transplant"], kind="mergesort").reset_index(drop=True)
    controlled_compare = [field for field in assessment_columns if field not in {"creatinine_mg_dl", "creatinine_change_pct", "urine_output_ml_24h", "tacrolimus_level_ng_ml", "medication_adherence_pct"}]
    hard_checks["python_controlled_and_derived_metadata_exact"] = assessments[controlled_compare].astype(str).equals(expected_meta[controlled_compare].astype(str))
    hard_checks["identity_skeleton_exact"] = identities.sort_values("entity_id").reset_index(drop=True).astype(str).equals(frozen_identity.sort_values("entity_id").reset_index(drop=True).astype(str))

    recipient_lookup = frozen_recipients.set_index("recipient_id")
    abo_expected = assessments.recipient_id.map(recipient_lookup.apply(lambda row: _derive_abo(row.donor_blood_group, row.recipient_blood_group), axis=1))
    hard_checks["abo_compatibility_independently_derived"] = assessments.abo_compatibility_category.eq(abo_expected.to_numpy()).all()
    transplant_dates = pd.to_datetime(assessments.assessment_date) - pd.to_timedelta(assessments.days_since_transplant, unit="D")
    retention_dates = pd.to_datetime(assessments.retention_expiry_date)
    hard_checks["assessment_dates_and_chronology_valid"] = transplant_dates.groupby(assessments.recipient_id).nunique().eq(1).all() and assessments.groupby("recipient_id").assessment_date.apply(lambda values: pd.to_datetime(values).is_monotonic_increasing).all()
    hard_checks["retention_dates_valid"] = (retention_dates - pd.to_datetime(assessments.assessment_date)).dt.days.eq(730).all()
    hard_checks["infection_indicators_exact"] = assessments.infection_indicator.eq(expected_meta.infection_indicator).all()

    derived_changes, derived_previous, derived_targets = [], [], []
    for recipient_id, group in assessments.groupby("recipient_id", sort=False):
        row = recipient_lookup.loc[recipient_id]
        previous_creatinine = float(row.baseline_creatinine_mg_dl)
        events = sorted(int(day) for day in json.loads(row.confirmed_rejection_event_days))
        for item in group.itertuples(index=False):
            derived_changes.append(100.0 * (float(item.creatinine_mg_dl) - previous_creatinine) / previous_creatinine)
            previous_creatinine = float(item.creatinine_mg_dl)
            derived_previous.append(int(any(event < int(item.days_since_transplant) for event in events)))
            derived_targets.append(int(any(int(item.days_since_transplant) < event <= int(item.days_since_transplant) + 30 for event in events)))
    hard_checks["creatinine_percentage_changes_exact"] = np.allclose(assessments.creatinine_change_pct, derived_changes, atol=1e-10, rtol=0)
    hard_checks["previous_rejection_independently_derived"] = assessments.previous_rejection.tolist() == derived_previous
    hard_checks["target_independently_derived"] = assessments.acute_rejection_within_30_days.tolist() == derived_targets
    hard_checks["positive_targets_4494"] = int(assessments.acute_rejection_within_30_days.sum()) == 4494

    trajectory_fields = ["creatinine_mg_dl", "urine_output_ml_24h", "tacrolimus_level_ng_ml", "medication_adherence_pct"]
    trajectory_signatures = assessments.groupby("recipient_id", sort=False)[trajectory_fields].apply(lambda group: tuple(map(tuple, group.to_numpy())))
    hard_checks["no_identical_complete_recipient_trajectories"] = not trajectory_signatures.duplicated().any()
    forbidden_assessment_fields = set(field_lists["direct_identifiers"] + field_lists["event_control_metadata"] + ["request_id", "request_seed", "chunk_id", "anchor_metadata_sha256"])
    hard_checks["direct_identifiers_and_generation_controls_absent"] = not (forbidden_assessment_fields & set(assessments.columns))
    leakage_forbidden = set(field_lists["pseudonymous_identifiers"] + field_lists["assessment_audit_only"] + field_lists["direct_identifiers"] + field_lists["event_control_metadata"] + ["assessment_date", "retention_expiry_date", "training_consent_status", "training_consent_version"])
    hard_checks["classifier_feature_contract_exact"] = classifier_features == assessment_schema.loc[assessment_schema.classifier_allowed.astype(str).str.lower().eq("true"), "field_name"].tolist()
    hard_checks["classifier_leakage_absent"] = not (set(classifier_features) & leakage_forbidden)
    for field, bounds in canonical_ranges.items():
        hard_checks[f"canonical_range_{field}"] = assessments[field].between(bounds["minimum"], bounds["maximum"], inclusive="both").all()

    response_by_recipient = {}
    parse_errors, qwen_mismatches = [], []
    reconciled_fields = 0
    for record in response_records:
        try:
            serialized = record["serialized_response_json"]
            assert _sha256_text(serialized) == record["serialized_response_sha256"]
            envelope = json.loads(serialized)
            payload = json.loads(envelope["choices"][0]["message"]["content"])
            returned = payload["assessments"]
            assert len(returned) == 6
            response_by_recipient[record["recipient_id"]] = returned
        except Exception as exc:
            parse_errors.append({"request_id": record.get("request_id"), "error": f"{type(exc).__name__}: {exc}"})
    if not parse_errors:
        for recipient_id, group in assessments.groupby("recipient_id", sort=False):
            returned = response_by_recipient[recipient_id]
            for offset, (_, actual) in enumerate(group.iterrows()):
                for field in QWEN_RETURN_FIELDS_FINAL:
                    reconciled_fields += 1
                    expected_value = returned[offset][field]
                    if not np.isclose(float(actual[field]), float(expected_value), atol=1e-10, rtol=0):
                        qwen_mismatches.append({"recipient_id": recipient_id, "assessment_offset": offset, "field": field, "response": expected_value, "assembled": actual[field]})
    hard_checks["raw_response_records_10000"] = len(response_by_recipient) == 10000
    hard_checks["all_300000_qwen_fields_reconciled"] = reconciled_fields == 300000 and not qwen_mismatches and not parse_errors
    assert all(hard_checks.values()), [name for name, passed in hard_checks.items() if not passed]

    recipient_order = sorted(range(len(frozen_recipients)), key=lambda index: _stable_score(frozen_recipients.iloc[index].request_seed, "delete-recipient"))
    withdrawal_recipients = set(frozen_recipients.iloc[recipient_order[:100]].recipient_id)
    shared_donors = frozen_recipients.groupby("donor_id").recipient_id.nunique().loc[lambda values: values.eq(2)].index.tolist()
    withdrawal_donors = set(sorted(shared_donors, key=lambda donor: _sha256_text(f"{donor}|delete-donor"))[:250])
    deletion_masks = {
        "recipient_withdrawal": assessments.recipient_id.isin(withdrawal_recipients),
        "donor_withdrawal": assessments.donor_id.isin(withdrawal_donors),
        "hospital_removal": assessments.hospital_id.eq("V32P-H05"),
        "invalid_consent": assessments.training_consent_status.eq("Invalidated") & assessments.training_consent_version.eq("RECIPIENT_V3"),
        "retention_expiry": pd.to_datetime(assessments.retention_expiry_date).le(pd.Timestamp("2025-12-31")),
    }
    expected_deletion = {"recipient_withdrawal": (600, 100, 100), "donor_withdrawal": (3000, 500, 500), "hospital_removal": (6000, 1000, 1000), "invalid_consent": (6000, 3800, 0), "retention_expiry": (9000, 3700, 0)}
    selection_keys = {"recipient_withdrawal": sorted(withdrawal_recipients), "donor_withdrawal": sorted(withdrawal_donors), "hospital_removal": ["V32P-H05"], "invalid_consent": ["training_consent_status=Invalidated", "training_consent_version=RECIPIENT_V3"], "retention_expiry": ["retention_expiry_date<=2025-12-31"]}
    deletion_rows = []
    for scenario, mask in deletion_masks.items():
        selected = assessments.loc[mask]
        counts = selected.groupby("recipient_id").size()
        actual = (len(selected), selected.recipient_id.nunique(), int(counts.eq(6).sum()))
        assert actual == expected_deletion[scenario], (scenario, actual)
        deletion_rows.append({
            "scenario": scenario, "selection_keys_or_rule": json.dumps(selection_keys[scenario], separators=(",", ":")),
            "matched_assessment_rows": actual[0], "affected_recipients": actual[1], "affected_donors": selected.donor_id.nunique(),
            "affected_hospitals": selected.hospital_id.nunique(), "complete_recipient_histories": actual[2],
            "records_per_recipient_distribution": json.dumps({str(int(k)): int(v) for k, v in counts.value_counts().sort_index().items()}, separators=(",", ":")),
            "assessment_days": json.dumps(sorted(selected.days_since_transplant.unique().astype(int).tolist())),
            "assessment_id_set_sha256": _sha256_text("\n".join(sorted(selected.assessment_id))), "records_removed_now": 0, "status": "verified",
        })
    deletion_audit = pd.DataFrame(deletion_rows)

    dictionary = _dictionary_rows(assessment_columns, identity_columns, classifier_features)
    event_days = frozen_recipients.confirmed_rejection_event_days.map(json.loads)
    target_sequences = assessments.groupby("recipient_id").acute_rejection_within_30_days.apply(lambda values: json.dumps(values.astype(int).tolist(), separators=(",", ":")))
    numerical_fields = [field for field in classifier_features if pd.api.types.is_numeric_dtype(assessments[field])]
    categorical_fields = [field for field in classifier_features if field not in numerical_fields]
    numerical_ranges = {field: {"minimum": float(assessments[field].min()), "maximum": float(assessments[field].max()), "mean": float(assessments[field].mean()), "median": float(assessments[field].median())} for field in numerical_fields}
    categorical_distributions = {field: {str(key): int(value) for key, value in assessments[field].value_counts(dropna=False).sort_index().items()} for field in categorical_fields}

    final_dir.mkdir(parents=False, exist_ok=False)
    assessment_path = final_dir / "kidney_transplant_assessments.csv"
    identity_path = final_dir / "kidney_transplant_identity.csv"
    features_path = final_dir / "classifier_feature_list.json"
    dictionary_path = final_dir / "data_dictionary.csv"
    deletion_path = final_dir / "deletion_scenario_audit.csv"
    _write_csv_exclusive(assessment_path, assessments)
    _write_csv_exclusive(identity_path, identities)
    _write_json_exclusive(features_path, {"schema_version": "qwen-kidney-v3.2-frozen", "classifier_features": classifier_features, "target": "acute_rejection_within_30_days", "grouped_split_key": "recipient_id", "excluded_field_classes": ["identifiers", "audit fields", "consent fields", "dates", "anchors", "event controls", "hidden generation variables"]})
    _write_csv_exclusive(dictionary_path, dictionary)
    _write_csv_exclusive(deletion_path, deletion_audit)

    readback_assessments = pd.read_csv(assessment_path)
    readback_identity = pd.read_csv(identity_path, keep_default_na=False)
    readback_checks = {
        "assessment_readback_exact_shape": readback_assessments.shape == (60000, 31),
        "assessment_readback_columns_exact": readback_assessments.columns.tolist() == assessment_columns,
        "assessment_readback_values_equal": readback_assessments.astype(str).equals(assessments.astype(str)),
        "identity_readback_exact_shape": readback_identity.shape == (18000, 12),
        "identity_readback_columns_exact": readback_identity.columns.tolist() == identity_columns,
        "identity_readback_values_equal": readback_identity.astype(str).equals(identities.astype(str)),
        "data_dictionary_43_fields": dictionary.shape[0] == 43 and dictionary.field_name.nunique() == 43,
        "deletion_scenarios_five": deletion_audit.shape[0] == 5 and deletion_audit.status.eq("verified").all(),
    }
    assert all(readback_checks.values()), [name for name, passed in readback_checks.items() if not passed]

    chunk_after = {number: _tree_fingerprint(production_root / f"chunk_{number:03d}", root) for number in range(1, 101)}
    historical_after = _snapshot(historical_paths, root)
    protected_after = _file_fingerprint(protected, root)
    integrity_checks = {
        "protected_dataset_unchanged": protected_after == protected_before,
        "all_100_chunks_unchanged": all(chunk_after[number] == chunk_before[number] for number in range(1, 101)),
        "historical_evidence_unchanged": historical_after == historical_before,
        "no_api_client_imported_or_instantiated_by_final_cell": True,
        "api_requests": 0,
        "network_requests": 0,
        "notebooks_01_05_executed": False,
        "model_trained": False,
        "splits_created": False,
        "forget_retain_datasets_created": False,
    }
    assert all(value is True or value == 0 for value in integrity_checks.values())

    reconciliation_report = {
        "status": "passed", "assembly_id": FINAL_ASSEMBLY_ID,
        "request_attempt_records": len(request_records), "successful_response_records": len(response_records),
        "manifest_records": len(manifest_records), "qwen_fields_expected": 300000, "qwen_fields_reconciled": reconciled_fields,
        "mismatch_count": len(qwen_mismatches), "parse_errors": parse_errors,
        "authoritative_progress_checkpoint_status": reconciliation100["status"],
        "historical_timeout": {"recipient_id": "V32P-R004714", "request_attempts": 2, "successful_responses": 1, "request_failed_manifest_records": 1, "manual_retry_preserved": True},
    }
    quality_report = {
        "status": "passed", "hard_validation_checks": hard_checks, "readback_checks": readback_checks,
        "dimensions": {"assessment_rows": 60000, "assessment_columns": 31, "recipients": 10000, "assessments_per_recipient": 6, "identity_rows": 18000, "donors": 8000},
        "target_distribution": {"positive": 4494, "negative": 55506, "prevalence": 0.0749, "target_sequence_counts": {key: int(value) for key, value in target_sequences.value_counts().sort_index().items()}},
        "event_distribution": {"event_recipients": int(event_days.map(bool).sum()), "non_event_recipients": int((~event_days.map(bool)).sum()), "event_day_counts": {str(k): int(v) for k, v in Counter(day for days in event_days for day in days).items()}},
        "clinical_numerical_ranges": numerical_ranges, "categorical_distributions": categorical_distributions,
        "anchor_warnings": distribution100["anchor_deviation_warning_counts"], "aggregate_generation_quality": distribution100["aggregate_anchor_generation_quality"],
        "validation_amendments": ["PROD-VALIDATION-AMENDMENT-001", "PROD-VALIDATION-AMENDMENT-002"], "manual_quality_review": "QWEN-V32-PROD-001-MANUAL-QUALITY-REVIEW-001",
        "important_limitations": ["Entirely synthetic data; not real patient data.", "Controlled event schedules and targets are suitable for pipeline experiments, not prevalence estimation.", "Qwen measurements were conditioned by the frozen v3.2 prompt and anchors and are not independently observed clinical measurements.", "Individual anchor-proximity exceedances are warnings under amendment 002.", "Sensitive audit attributes are excluded from classifier inputs and should not influence clinical measurements without explicit justification.", "No external clinical validation, train/validation/test split, unlearning dataset, or trained classifier is created here."],
    }
    integrity_report = {
        "status": "passed", "checks": integrity_checks, "protected_dataset_before": protected_before, "protected_dataset_after": protected_after,
        "chunk_fingerprints_before": {f"chunk_{number:03d}": chunk_before[number] for number in range(1, 101)},
        "chunk_fingerprints_after": {f"chunk_{number:03d}": chunk_after[number] for number in range(1, 101)},
        "historical_evidence_before": historical_before, "historical_evidence_after": historical_after,
    }
    assembly_report = {
        "status": "passed", "assembly_id": FINAL_ASSEMBLY_ID, "run_id": "QWEN-V32-PROD-001", "local_only": True,
        "api_requests": 0, "network_requests": 0, "source_chunks": list(range(1, 101)),
        "final_dimensions": quality_report["dimensions"], "target_positive_count": 4494, "event_recipients": 3400,
        "cumulative_token_usage": progress["cumulative_token_usage"], "deletion_scenario_counts": {row["scenario"]: {"assessment_rows": int(row["matched_assessment_rows"]), "recipients": int(row["affected_recipients"]), "complete_histories": int(row["complete_recipient_histories"])} for row in deletion_rows},
        "classifier_feature_count": len(classifier_features), "target": "acute_rejection_within_30_days",
        "historical_timeout_and_manual_retry_preserved": True, "validation_amendments_preserved": True,
        "final_assembly_only": True, "final_files_must_not_be_manually_edited": True,
    }
    reports = {
        "final_reconciliation_report.json": reconciliation_report,
        "final_quality_report.json": quality_report,
        "final_integrity_report.json": integrity_report,
        "final_assembly_report.json": assembly_report,
    }
    for name, report in reports.items():
        _write_json_exclusive(final_dir / name, report)
    readme = """# Qwen v3.2 kidney-transplant final assembly\n\nThis directory is an immutable, local-only assembly of the completed synthetic Qwen v3.2 production run. All data are entirely synthetic and must not be interpreted as real patient data.\n\n- `kidney_transplant_assessments.csv` is the 60,000-row longitudinal assessment dataset.\n- `kidney_transplant_identity.csv` contains the separate synthetic recipient and donor identities. It must never be joined into classifier inputs.\n- `classifier_feature_list.json` is the authoritative list of fields the classifier may use.\n- The target column is `acute_rejection_within_30_days`.\n- `data_dictionary.csv` documents every assessment and identity field.\n- `deletion_scenario_audit.csv` identifies future forget-set scenarios without deleting records.\n- The JSON reports document reconciliation, validation, quality, integrity and assembly.\n- Raw requests, complete responses, manifests and per-chunk validation evidence remain archived unchanged in `../chunk_001/` through `../chunk_100/`.\n\nDo not manually edit any file in this directory. Create a separately authorised, reproducible successor stage for any future transformation. No data split, model training, classifier execution, or forget/retain dataset creation occurred in this stage.\n"""
    _write_text_exclusive(final_dir / "README.md", readme)

    payload_files = sorted(path for path in final_dir.iterdir() if path.name != "file_hash_manifest.json")
    manifest_payload = {
        "assembly_id": FINAL_ASSEMBLY_ID, "algorithm": "SHA-256",
        "files": {path.name: _file_fingerprint(path, root) for path in payload_files},
        "self_hash_policy": "The manifest cannot contain its own ordinary file SHA-256 without circularity. Its canonical payload digest below fingerprints the complete manifest payload before that digest field is added; the ordinary manifest file SHA-256 is reported by the execution output and final handoff.",
    }
    manifest_payload["canonical_payload_sha256_excluding_this_field"] = _sha256_text(json.dumps(manifest_payload, sort_keys=True, separators=(",", ":")))
    manifest_path = final_dir / "file_hash_manifest.json"
    _write_json_exclusive(manifest_path, manifest_payload)
    directory_fd = os.open(final_dir, os.O_RDONLY)
    try:
        os.fsync(directory_fd)
    finally:
        os.close(directory_fd)

    final_integrity = {
        "protected_unchanged": _file_fingerprint(protected, root) == protected_before,
        "chunks_unchanged": all(_tree_fingerprint(production_root / f"chunk_{number:03d}", root) == chunk_before[number] for number in range(1, 101)),
        "historical_evidence_unchanged": _snapshot(historical_paths, root) == historical_before,
    }
    assert all(final_integrity.values())
    created_files = sorted(path.name for path in final_dir.iterdir() if path.is_file())
    assert created_files == sorted(["kidney_transplant_assessments.csv", "kidney_transplant_identity.csv", "classifier_feature_list.json", "data_dictionary.csv", "deletion_scenario_audit.csv", "final_reconciliation_report.json", "final_quality_report.json", "final_integrity_report.json", "final_assembly_report.json", "file_hash_manifest.json", "README.md"])
    result = {
        "status": "passed", "assembly_id": FINAL_ASSEMBLY_ID, "elapsed_seconds": time.perf_counter() - started,
        "api_requests": 0, "network_requests": 0, "created_files": created_files,
        "final_dimensions": quality_report["dimensions"], "target_positive_count": 4494, "event_recipients": 3400,
        "deletion_scenarios": assembly_report["deletion_scenario_counts"], "protected_before": protected_before,
        "protected_after": _file_fingerprint(protected, root), "final_integrity": final_integrity,
        "file_hash_manifest_sha256": _sha256_file(manifest_path),
    }
    return result


FINAL_ASSEMBLY_RESULT = run_qwen_v32_final_assembly()
print(json.dumps(_json_safe(FINAL_ASSEMBLY_RESULT), indent=2))
